<a href="https://colab.research.google.com/github/jejenfel/MACA/blob/main/source_code/Preprocessing_MACA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create Output Directories

In [ ]:
import os

# Define base output directory
OUTPUT_BASE_DIR = '/content/processed_data'

# Define specific output folders
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')
OUTPUT_FOLDER_TRACER_STUDY = os.path.join(OUTPUT_BASE_DIR, 'tracer_study_data')
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')

# Create these folders if they don't exist
os.makedirs(OUTPUT_FOLDER_INDUSTRY, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_CURRICULUM, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_TRACER_STUDY, exist_ok=True)
os.makedirs(OUTPUT_FOLDER_DICTIONARY, exist_ok=True)

print(f"Output directories created:")
print(f"- Industry Data: {OUTPUT_FOLDER_INDUSTRY}")
print(f"- Curriculum Data: {OUTPUT_FOLDER_CURRICULUM}")
print(f"- Tracer Study Data: {OUTPUT_FOLDER_TRACER_STUDY}")
print(f"- Dictionary Data: {OUTPUT_FOLDER_DICTIONARY}")

Output directories created:
- Industry Data: /content/processed_data/industry_data
- Curriculum Data: /content/processed_data/curriculum_data
- Tracer Study Data: /content/processed_data/tracer_study_data
- Dictionary Data: /content/processed_data/dictionary_data


## Mount G-Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive berhasil di-mount.")

Mounted at /content/drive
Google Drive berhasil di-mount.


# JOB PORTAL

In [ ]:
import pandas as pd
import os

# --- MODIFIED: Load CSV files from Google Drive folder ---
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/"

linkedin_files = []
jobstreet_files = []

# List all files in the directory
try:
    all_files = os.listdir(DRIVE_FOLDER_PATH)
except FileNotFoundError:
    print(f"Error: Google Drive folder '{DRIVE_FOLDER_PATH}' not found. Please check the path and ensure Drive is mounted.")
    exit()

for f in all_files:
    if f.startswith('linkedin_data_') and f.endswith('.csv'):
        linkedin_files.append(os.path.join(DRIVE_FOLDER_PATH, f))
    elif f.startswith('jobstreet_data_') and f.endswith('.csv'):
        jobstreet_files.append(os.path.join(DRIVE_FOLDER_PATH, f))

print(f"Found {len(linkedin_files)} LinkedIn data files.")
print(f"Found {len(jobstreet_files)} JobStreet data files.")

list_df_linkedin = []
list_df_jobstreet = []

# Load LinkedIn files
for fpath in linkedin_files:
    try:
        df = pd.read_csv(fpath, encoding='utf8', on_bad_lines='skip')
        list_df_linkedin.append(df)
        print(f"Successfully loaded {os.path.basename(fpath)} with {len(df)} records.")
    except Exception as e:
        print(f"Error loading {fpath}: {e}")

# Load JobStreet files
for fpath in jobstreet_files:
    try:
        df = pd.read_csv(fpath, encoding='utf8', on_bad_lines='skip')
        list_df_jobstreet.append(df)
        print(f"Successfully loaded {os.path.basename(fpath)} with {len(df)} records.")
    except Exception as e:
        print(f"Error loading {fpath}: {e}")

if not list_df_linkedin and not list_df_jobstreet:
    print("No valid LinkedIn or JobStreet CSV files found. Exiting.")
    exit()

df_linkedin = pd.DataFrame()
if list_df_linkedin:
    df_linkedin = pd.concat(list_df_linkedin, ignore_index=True)
    print(f"Total LinkedIn records after concatenation: {len(df_linkedin)}")
else:
    print("No LinkedIn data loaded. df_linkedin will be empty.")

df_jobstreet = pd.DataFrame()
if list_df_jobstreet:
    df_jobstreet = pd.concat(list_df_jobstreet, ignore_index=True)
    print(f"Total JobStreet records after concatenation: {len(df_jobstreet)}")
else:
    print("No JobStreet data loaded. df_jobstreet will be empty.")

print(f"Total records from all loaded files: {len(df_linkedin) + len(df_jobstreet)}")
# --- END MODIFIED SECTION ---


# Print original column names for validation
print(f"LinkedIn Original Columns (after concatenation): {df_linkedin.columns.tolist()}")
print(f"JobStreet Original Columns (after concatenation): {df_jobstreet.columns.tolist()}")

# Define the unified schema columns
unified_schema_columns = [
    'job_title',
    'job_description_html',
    'company_name',
    'location',
    'date_posted',
    'applicant_count',
    'source' # 'source' is added by the code, so it should be part of the final unified schema for concatenation
]

# --- Process LinkedIn Data --- #
df_linkedin_processed = df_linkedin.copy() # Start with a copy of the concatenated LinkedIn data

# Define LinkedIn column mapping
linkedin_column_map = {
    'title': 'job_title',
    'descriptionHtml': 'job_description_html',
    'companyName': 'company_name',
    'postedAt': 'date_posted',
    'applicantsCount': 'applicant_count'
    # 'location' is typically already 'location', no explicit rename needed here
}

# Rename columns ONLY if the original columns exist
linkedin_rename_map = {old_col: new_col for old_col, new_col in linkedin_column_map.items() if old_col in df_linkedin_processed.columns}
df_linkedin_processed = df_linkedin_processed.rename(columns=linkedin_rename_map)

# Add column source with values "LinkedIn"
df_linkedin_processed['source'] = 'LinkedIn'

# --- Process JobStreet Data --- #
df_jobstreet_processed = df_jobstreet.copy() # Start with a copy of the concatenated JobStreet data

# Remove these JobStreet columns and do not use them (as per original code, but do not fail if not exist)
cols_to_drop_jobstreet = [
    'company/id',
    'company/logo',
    'company/description',
    'description' # As per rule 3, only jobDescriptionHtml is to be used
]
existing_cols_to_drop_jobstreet = [col for col in cols_to_drop_jobstreet if col in df_jobstreet_processed.columns]
df_jobstreet_processed = df_jobstreet_processed.drop(columns=existing_cols_to_drop_jobstreet, errors='ignore')

# Define JobStreet column mapping
jobstreet_column_map = {
    'title': 'job_title',
    'jobDescriptionHtml': 'job_description_html',
    'company/name': 'company_name',
    'postDate': 'date_posted'
    # 'location' is typically already 'location', no explicit rename needed here
}

# Rename columns ONLY if the original columns exist
jobstreet_rename_map = {old_col: new_col for old_col, new_col in jobstreet_column_map.items() if old_col in df_jobstreet_processed.columns}
df_jobstreet_processed = df_jobstreet_processed.rename(columns=jobstreet_rename_map)

# Create applicant_count for JobStreet and fill it with 0
# Only add if 'applicant_count' is in the target schema and not already present
if 'applicant_count' in unified_schema_columns and 'applicant_count' not in df_jobstreet_processed.columns:
    df_jobstreet_processed['applicant_count'] = 0

# Add column source with values "JobStreet"
df_jobstreet_processed['source'] = 'JobStreet'

# --- Ensure Compatible Schema for Concatenation ---
# This step ensures both final dataframes have all columns specified in unified_schema_columns, in order.
# Missing columns will be added and filled with pd.NA

df_linkedin_final = pd.DataFrame(columns=unified_schema_columns) # Initialize with target schema
for col in unified_schema_columns:
    if col in df_linkedin_processed.columns:
        df_linkedin_final[col] = df_linkedin_processed[col]
    else:
        df_linkedin_final[col] = pd.NA

df_jobstreet_final = pd.DataFrame(columns=unified_schema_columns) # Initialize with target schema
for col in unified_schema_columns:
    if col in df_jobstreet_processed.columns:
        df_jobstreet_final[col] = df_jobstreet_processed[col]
    else:
        df_jobstreet_final[col] = pd.NA

# Print column names after standardization (Task 8)
print("\n--- Columns after Standardization ---")
print(f"LinkedIn Final Columns: {df_linkedin_final.columns.tolist()}")
print(f"JobStreet Final Columns: {df_jobstreet_final.columns.tolist()}")

# --- Combine Data --- #
# 7. Concatenate LinkedIn and JobStreet into ONE DataFrame: industry_demand_df
industry_demand_df = pd.concat([df_linkedin_final, df_jobstreet_final], ignore_index=True)

# 8. Print shapes
print("\n--- Dataset Shapes ---")
print(f"Shape of LinkedIn dataset after processing: {df_linkedin_final.shape}")
print(f"Shape of JobStreet dataset after processing: {df_jobstreet_final.shape}")
print(f"Shape of final merged dataset (industry_demand_df): {industry_demand_df.shape}")

# 10. Print dataframe columns and head() (as done in previous task, useful for verification)
print("\n--- Final Merged Dataset Summary ---")
print("Column names of the final merged dataset:")
print(industry_demand_df.columns.tolist())

print("\nFirst 5 rows of the final merged dataset:")
display(industry_demand_df.head())

# --- NEW: Define output path --- #
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 11. Save the final dataset as industry_demand_combined.csv
output_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_combined.csv')
industry_demand_df.to_csv(output_filepath, index=False)
print(f"\nUnified dataset successfully saved to '{output_filepath}'.")

Found 2 LinkedIn data files.
Found 2 JobStreet data files.
Successfully loaded linkedin_data_2.csv with 952 records.
Successfully loaded linkedin_data_1.csv with 959 records.
Successfully loaded jobstreet_data_2.csv with 3234 records.
Successfully loaded jobstreet_data_1.csv with 2048 records.
Total LinkedIn records after concatenation: 1911
Total JobStreet records after concatenation: 5282
Total records from all loaded files: 7193
LinkedIn Original Columns (after concatenation): ['applicantsCount', 'companyName', 'descriptionHtml', 'descriptionText', 'location', 'postedAt', 'title']
JobStreet Original Columns (after concatenation): ['company/description', 'company/id', 'company/logo', 'company/name', 'description', 'jobDescriptionHtml', 'location', 'postDate', 'title']

--- Columns after Standardization ---
LinkedIn Final Columns: ['job_title', 'job_description_html', 'company_name', 'location', 'date_posted', 'applicant_count', 'source']
JobStreet Final Columns: ['job_title', 'job_de

,job_title,job_description_html,company_name,location,date_posted,applicant_count,source
0,Sr Officer-Account Executive,<strong>Job Summary<br><br></strong>Account Ma...,PT. Indosat Tbk,"Jakarta, Indonesia",2026-01-26,200,LinkedIn
1,IT Officer Pontianak,<strong>Responsibilities<br><br></strong><ul><...,Wings Group Indonesia (Sayap Mas Utama),"Pontianak, West Kalimantan, Indonesia",2025-09-15,200,LinkedIn
2,Sales Control Tower Officer,<strong>Responsibilities<br><br></strong><ul><...,Wings Group Indonesia (Sayap Mas Utama),"Jakarta, Jakarta, Indonesia",2025-05-03,200,LinkedIn
3,Brand/Packaging Designer,<strong>Responsibilities<br><br></strong><ul><...,Wings Group Indonesia (Sayap Mas Utama),"Jakarta, Jakarta, Indonesia",2025-06-30,200,LinkedIn
4,Staff / Officer Race,<strong>Jobdesk<br><br></strong>Indonesia Muda...,PT Indonesia Muda Kreasi (IM Road Runner),"Tangerang, Banten, Indonesia",2025-12-11,127,LinkedIn



Unified dataset successfully saved to '/content/processed_data/industry_data/industry_demand_combined.csv'.


## Cleaning

In [ ]:
import pandas as pd
import re
import html # Standard Python library for HTML entity handling
from bs4 import BeautifulSoup # Added for HTML cleaning
import numpy as np # For np.nan
import os

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 1. Load industry_demand_combined.csv
try:
    # Load from the new output folder
    input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_combined.csv')
    df = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully from '{OUTPUT_FOLDER_INDUSTRY}'.")
except FileNotFoundError as e:
    print(f"Error: Make sure '{os.path.basename(input_filepath)}' is in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    exit()

# Ensure job_description_html is string type and handle NaNs for safe processing
df['job_description_html'] = df['job_description_html'].astype(str).fillna('')

# --- Start of MODIFIED Duplicate Handling Logic for Transparency --- #

# Convert date_posted to datetime format for date comparison, using 'mixed' format for robustness
# and explicitly make them UTC to ensure comparability
df['date_posted'] = pd.to_datetime(df['date_posted'], format='mixed', utc=True)

# Sort the DataFrame by identifying columns and then by date_posted
# This is crucial for the flagging logic to correctly identify chronological duplicates
df_sorted = df.sort_values(by=['job_title', 'company_name', 'job_description_html', 'date_posted']).copy()

# Initialize new flagging columns
df_sorted['is_duplicate'] = False
df_sorted['duplicate_group_id'] = np.nan # Use np.nan for numerical IDs that might be empty
df_sorted['date_diff_days'] = np.nan

# Define a function to flag duplicates within each grouped set of identical jobs
def flag_duplicates_in_group(group):
    # Ensure operating on a copy to avoid SettingWithCopyWarning
    group = group.copy()

    if len(group) <= 1:
        return group # No duplicates in a group of 1 or less

    last_retained_date = group.iloc[0]['date_posted']
    has_actual_duplicates = False

    for i in range(1, len(group)):
        current_date = group.iloc[i]['date_posted']

        if (current_date - last_retained_date).days <= 7:
            # This row is considered a duplicate within the window
            group.loc[group.index[i], 'is_duplicate'] = True
            group.loc[group.index[i], 'date_diff_days'] = (current_date - last_retained_date).days
            has_actual_duplicates = True
        else:
            # This job posting is sufficiently distant, treat as a new 'unique' reference
            last_retained_date = current_date

    # This function will return the group with 'is_duplicate' and 'date_diff_days' populated.
    # 'duplicate_group_id' will be populated globally after this apply.
    return group

# Assign a temporary group key first, so we can use it as a base for duplicate_group_id later
df_sorted['temp_group_key'] = df_sorted.groupby(['job_title', 'company_name', 'job_description_html']).ngroup()
df_flagged_all = df_sorted.groupby('temp_group_key', group_keys=False).apply(flag_duplicates_in_group)

# Now, assign sequential duplicate_group_id only to groups that actually contain duplicates
duplicate_group_ids_map = {}
current_unique_duplicate_id = 0
for temp_id in df_flagged_all['temp_group_key'].unique():
    if df_flagged_all[(df_flagged_all['temp_group_key'] == temp_id) & (df_flagged_all['is_duplicate'] == True)].shape[0] > 0:
        duplicate_group_ids_map[temp_id] = current_unique_duplicate_id
        current_unique_duplicate_id += 1

# Apply the mapped duplicate group IDs. Groups without actual duplicates will remain np.nan.
df_flagged_all['duplicate_group_id'] = df_flagged_all['temp_group_key'].map(duplicate_group_ids_map).fillna(np.nan)

# Drop the temporary group id column, as it's no longer needed
df_flagged_all = df_flagged_all.drop(columns=['temp_group_key'])


# --- Display Inspection Information --- #
df_duplicates_for_inspection = df_flagged_all[df_flagged_all['is_duplicate'] == True].copy()

total_detected_duplicate_groups = df_duplicates_for_inspection['duplicate_group_id'].nunique()
total_rows_flagged_as_duplicates = df_duplicates_for_inspection.shape[0]

print(f"Total number of detected duplicate groups: {total_detected_duplicate_groups}")
print(f"Total number of rows flagged as duplicates: {total_rows_flagged_as_duplicates}")
print("\nSample rows from the duplicate dataset (df_duplicates_for_inspection):")
display(df_duplicates_for_inspection[['job_title', 'company_name', 'date_posted', 'is_duplicate', 'duplicate_group_id', 'date_diff_days']].head(10))

# --- Final Removal of Duplicates from the main dataset --- #
# The final 'df' should only contain the non-duplicate entries.
initial_rows_before_flagging = df.shape[0] # Original number of rows before any flagging or removal
df = df_flagged_all[df_flagged_all['is_duplicate'] == False].reset_index(drop=True)

# Remove the flagging columns from the final 'df' to keep its structure clean,
# as these columns were primarily for inspection and not part of the final output schema.
df = df.drop(columns=['is_duplicate', 'duplicate_group_id', 'date_diff_days'])

# Calculate removed rows based on the flagging logic
duplicate_rows_removed = initial_rows_before_flagging - df.shape[0]
print(f"Removed {duplicate_rows_removed} duplicate rows using date window logic (after flagging and inspection).")
print(f"Final rows after deduplication: {df.shape[0]}")

# --- End of MODIFIED Duplicate Handling Logic --- #

# --- Start of NEW IT-job filtering logic --- #
print("\n--- Filtering for IT-related job positions ---")

# Define IT-related keywords based on corpus and examples from prompt.
# These are used to determine domain relevance and are not considered a skill dictionary.
# The terms are intentionally broad to capture the IT domain.
it_keywords = [
    'programming', 'software', 'data', 'system', 'network', 'cloud', 'application',
    'developer', 'engineer', 'it', 'information technology', 'cybersecurity',
    'devops', 'web', 'backend', 'frontend', 'ai', 'artificial intelligence',
    'machine learning', 'database', 'infrastructure', 'analyst'
]
it_keywords_pattern = '|'.join(it_keywords)

# Create is_it_related column by checking for keywords in job_title or job_description_clean
# Using .str.contains with regex=True for pattern matching
df['is_it_related'] = (
    df['job_title'].str.contains(it_keywords_pattern, case=False, na=False) |
    df['job_description_html'].str.contains(it_keywords_pattern, case=False, na=False)
)

# Count IT and non-IT related jobs before removal
total_it_jobs_before_removal = df['is_it_related'].sum()
total_non_it_jobs_before_removal = len(df) - total_it_jobs_before_removal

print(f"Total IT-related jobs identified: {total_it_jobs_before_removal}")
print(f"Total non-IT related jobs identified: {total_non_it_jobs_before_removal}")

# Create a separate DataFrame for non-IT job postings for inspection
df_non_it_for_inspection = df[~df['is_it_related']].copy()
print("\nSample rows from the non-IT job dataset (df_non_it_for_inspection):")
# Display only relevant columns for non-IT inspection
display(df_non_it_for_inspection[['job_title', 'company_name', 'source', 'is_it_related']].head(10))

# Remove non-IT job postings from the main dataset
initial_rows_before_it_filter = df.shape[0]
df = df[df['is_it_related']].reset_index(drop=True)

total_non_it_jobs_removed = initial_rows_before_it_filter - df.shape[0]

print(f"Total non-IT jobs removed from the main dataset: {total_non_it_jobs_removed}")
print(f"Final rows after IT filtering: {df.shape[0]}")

# Remove the 'is_it_related' column from the final DataFrame
df = df.drop(columns=['is_it_related'])

# 7. Keep all non-text columns unchanged - this is implicitly handled as we only modify job_description_html and add job_description_clean.

# 8. Display basic validation outputs for the FINAL cleaned dataset:
print("\n--- Cleaned Dataset Summary (After Final Deduplication and IT Filtering) ---")
print(f"Shape of the cleaned dataset: {df.shape}")
print("Column names of the cleaned dataset:")
print(df.columns.tolist())

print("\nFirst 5 rows of the cleaned dataset (showing relevant columns):")
display(df[['job_title', 'company_name', 'job_description_html', 'date_posted']].head()) # Added date_posted for verification

# 9. Save the cleaned dataset as industry_demand_cleaned.csv
output_filepath_cleaned = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_cleaned.csv')
df.to_csv(output_filepath_cleaned, index=False)
print(f"\nCleaned dataset successfully saved to '{output_filepath_cleaned}'.")

Dataset 'industry_demand_combined.csv' loaded successfully from '/content/processed_data/industry_data'.


/tmp/ipykernel_8876/132161433.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_flagged_all = df_sorted.groupby('temp_group_key', group_keys=False).apply(flag_duplicates_in_group)


Total number of detected duplicate groups: 366
Total number of rows flagged as duplicates: 5383

Sample rows from the duplicate dataset (df_duplicates_for_inspection):


,job_title,company_name,date_posted,is_duplicate,duplicate_group_id,date_diff_days
1944,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
1976,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2009,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2041,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2073,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2105,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2136,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2167,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2199,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0
2231,AMAG Graduated Program (Management Trainee),Asuransi Multi Artha Guna,2026-01-23 10:27:07+00:00,True,0.0,0.0


Removed 5485 duplicate rows using date window logic (after flagging and inspection).
Final rows after deduplication: 1708

--- Filtering for IT-related job positions ---
Total IT-related jobs identified: 1699
Total non-IT related jobs identified: 9

Sample rows from the non-IT job dataset (df_non_it_for_inspection):


,job_title,company_name,source,is_it_related
113,Art Director,Flock | Storikka,LinkedIn,False
741,Host Live Streaming,PT. Larocking Keleva Global,LinkedIn,False
742,Host Live Streaming,PT. Larocking Keleva Global,LinkedIn,False
1633,Tugboat Able Seaman,PT Habco Primatama,LinkedIn,False
1702,试油工程师（外派）,PT. Enecal Indonesia,LinkedIn,False
1703,试油工程师（外派）,PT. Enecal Indonesia,LinkedIn,False
1704,車載用コネクタのプロジェクトマネジメント業務,JAE,LinkedIn,False
1705,電気・電子回路設計＜浜松工場＞,MinebeaMitsumi Philippines,LinkedIn,False
1707,项目经理（外派）,PT. Enecal Indonesia,LinkedIn,False


Total non-IT jobs removed from the main dataset: 9
Final rows after IT filtering: 1699

--- Cleaned Dataset Summary (After Final Deduplication and IT Filtering) ---
Shape of the cleaned dataset: (1699, 7)
Column names of the cleaned dataset:
['job_title', 'job_description_html', 'company_name', 'location', 'date_posted', 'applicant_count', 'source']

First 5 rows of the cleaned dataset (showing relevant columns):


,job_title,company_name,job_description_html,date_posted
0,(Dev Core) Front-End Developer,cmlabs,DEVELOPER<br><br>JUNIOR<br><br>INTERNSHIP<br><...,2025-09-22 00:00:00+00:00
1,14 フルスタックエンジニア,AI inside Inc.,スケールも、難易度も、常識外。AIプラットフォームの核心へ。<br><br><strong>...,2025-07-10 00:00:00+00:00
2,16 バックエンドエンジニア,AI inside Inc.,未踏のAIプラットフォームを、構想から築く。<br><br><strong>Descript...,2025-07-10 00:00:00+00:00
3,3D Design & Animation Intern,PT SMART Tbk,<strong>Responsibilities<br><br></strong>As a ...,2025-12-19 00:00:00+00:00
4,3D Designer,Octarine,<strong>Responsibilities<br><br></strong><ul><...,2026-01-27 00:00:00+00:00



Cleaned dataset successfully saved to '/content/processed_data/industry_data/industry_demand_cleaned.csv'.


## New Competency Unit Extraction Pipeline

In [ ]:
import pandas as pd
import re
import html # Standard Python library for HTML entity handling
from bs4 import BeautifulSoup # Added for HTML cleaning
import nltk
from nltk.tokenize import sent_tokenize
import os

# Download necessary NLTK data
nltk.download('punkt', quiet=True)

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 1. Load Data: industry_demand_combined.csv
input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_cleaned.csv')
try:
    df_industry_raw = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: Make sure '{os.path.basename(input_filepath)}' is in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    df_industry_raw = pd.DataFrame() # Create empty DataFrame to avoid errors

# Exit if DataFrame is empty
if df_industry_raw.empty:
    print("Exiting further processing as the DataFrame is empty.")
    raise SystemExit("DataFrame is empty, cannot proceed.")

# 2. Use job_description_html as main source, rename or copy into 'combined_text'
df_industry_raw['combined_text'] = df_industry_raw['job_description_html'].astype(str).fillna('')

print(f"Initial DataFrame shape: {df_industry_raw.shape}")
print("First 5 rows of 'combined_text':")
display(df_industry_raw[['job_title', 'combined_text']].head())

Dataset 'industry_demand_cleaned.csv' loaded successfully.
Initial DataFrame shape: (1699, 8)
First 5 rows of 'combined_text':


,job_title,combined_text
0,(Dev Core) Front-End Developer,DEVELOPER<br><br>JUNIOR<br><br>INTERNSHIP<br><...
1,14 フルスタックエンジニア,スケールも、難易度も、常識外。AIプラットフォームの核心へ。<br><br><strong>...
2,16 バックエンドエンジニア,未踏のAIプラットフォームを、構想から築く。<br><br><strong>Descript...
3,3D Design & Animation Intern,<strong>Responsibilities<br><br></strong>As a ...
4,3D Designer,<strong>Responsibilities<br><br></strong><ul><...


### 3. Remove HTML Tags

This step cleans the `combined_text` column by removing HTML tags using BeautifulSoup and decoding HTML entities. This ensures that only the textual content remains, preserving the semantic structure.

In [ ]:
def clean_html(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    soup = BeautifulSoup(text, 'html.parser')

    # 3. Handle Line Breaks (br)
    for br_tag in soup.find_all('br'):
        br_tag.replace_with('\n')

    # 1. Handle Section Headings (strong, h1-h6)
    for tag_name in ['strong', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
        for tag in soup.find_all(tag_name):
            # Replace the tag with a marker, its text content (joining inner text with space), and then a newline
            tag.replace_with(
                f"\n[SECTION] {tag.get_text(separator='\n').strip()}\n"
            )
    # 2. Handle Bullet Items (li)
    for li_tag in soup.find_all('li'):
        # Replace <li> with [BULLET] text content (joining inner text with space) and a newline
        li_tag.replace_with(f"[BULLET] {li_tag.get_text(separator=' ')}\n")

    # 4. Handle Paragraphs (p)
    for p_tag in soup.find_all('p'):
        # Replace <p> with its text content (joining inner text with space) surrounded by double newlines
        p_tag.replace_with(f"\n\n{p_tag.get_text(separator=' ')}\n\n")

    # 5. Handle Table Cells (td)
    for td_tag in soup.find_all('td'):
        # Replace <td> with [TABLE_ITEM] text content (joining inner text with space)
        td_tag.replace_with(f"[TABLE_ITEM] {td_tag.get_text(separator=' ')}")

    # Extract text from the modified soup. We've already injected newlines and markers as text nodes.
    text = soup.get_text()

    # Decode HTML entities
    text = html.unescape(text)

    # Normalize newlines: Reduce any sequence of 3 or more newlines to exactly two newlines.
    # This preserves single newlines from list items and double newlines from paragraphs.
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Strip any leading/trailing newlines that might be artifacts from multiple replace_with calls.
    text = text.strip()

    # Normalize spaces: reduce multiple spaces to a single space
    text = re.sub(r'[ \t]+', ' ', text)
    text = text.strip()
    return text

df_industry_raw['combined_text'] = df_industry_raw['combined_text'].apply(clean_html)

print("HTML cleaning complete. First 5 rows of cleaned 'combined_text':")
display(df_industry_raw[['job_title', 'combined_text']].head())

HTML cleaning complete. First 5 rows of cleaned 'combined_text':


,job_title,combined_text
0,(Dev Core) Front-End Developer,DEVELOPER\n\nJUNIOR\n\nINTERNSHIP\n\nUpdated 2...
1,14 フルスタックエンジニア,スケールも、難易度も、常識外。AIプラットフォームの核心へ。\n\n[SECTION] De...
2,16 バックエンドエンジニア,未踏のAIプラットフォームを、構想から築く。\n\n[SECTION] Descriptio...
3,3D Design & Animation Intern,[SECTION] Responsibilities\nAs a 3D Design & A...
4,3D Designer,[SECTION] Responsibilities\n[BULLET] Product &...


### 4. Apply Light Preprocessing

This step applies light preprocessing to the cleaned text. It includes lowercasing, normalizing whitespace, removing excessive symbols (while preserving technical terminology), and normalizing bullet points. Aggressive stemming or stopword removal is avoided to retain semantic meaning.

In [ ]:
import re
import pandas as pd

# 4. Apply Light Preprocessing
def light_preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''

    text = text.lower()

    # # Normalize newlines
    # text = re.sub(r'\n+', '\n', text)

    # Normalize spaces
    text = re.sub(r'[ \t]+', ' ', text)

    text = text.strip()

    return text

df_industry_raw['combined_text'] = df_industry_raw['combined_text'].apply(light_preprocess_text)

print("Light preprocessing complete. First 5 rows of preprocessed 'combined_text':")
display(df_industry_raw[['job_title', 'combined_text']].head())

Light preprocessing complete. First 5 rows of preprocessed 'combined_text':


,job_title,combined_text
0,(Dev Core) Front-End Developer,developer\n\njunior\n\ninternship\n\nupdated 2...
1,14 フルスタックエンジニア,スケールも、難易度も、常識外。aiプラットフォームの核心へ。\n\n[section] de...
2,16 バックエンドエンジニア,未踏のaiプラットフォームを、構想から築く。\n\n[section] descriptio...
3,3D Design & Animation Intern,[section] responsibilities\nas a 3d design & a...
4,3D Designer,[section] responsibilities\n[bullet] product &...


### Save Light Preprocessing Results

This step saves the `df_industry_raw` DataFrame, which now contains the text after light preprocessing in the `combined_text` column, to a CSV file. This allows for persistent storage of this intermediate step.

In [ ]:
import os

# Define the output file path
output_filepath_light_preprocess = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_light_preprocessed.csv')

# Save the DataFrame to CSV
df_industry_raw.to_csv(output_filepath_light_preprocess, index=False)

print(f"Light preprocessed data saved to: {output_filepath_light_preprocess}")
print("First 5 rows of the saved DataFrame:")
display(df_industry_raw.head())

Light preprocessed data saved to: /content/processed_data/industry_data/industry_demand_light_preprocessed.csv
First 5 rows of the saved DataFrame:


,job_title,job_description_html,company_name,location,date_posted,applicant_count,source,combined_text
0,(Dev Core) Front-End Developer,DEVELOPER<br><br>JUNIOR<br><br>INTERNSHIP<br><...,cmlabs,"Malang, East Java, Indonesia",2025-09-22 00:00:00+00:00,124,LinkedIn,developer\n\njunior\n\ninternship\n\nupdated 2...
1,14 フルスタックエンジニア,スケールも、難易度も、常識外。AIプラットフォームの核心へ。<br><br><strong>...,AI inside Inc.,"Lembang, West Java, Indonesia",2025-07-10 00:00:00+00:00,26,LinkedIn,スケールも、難易度も、常識外。aiプラットフォームの核心へ。\n\n[section] de...
2,16 バックエンドエンジニア,未踏のAIプラットフォームを、構想から築く。<br><br><strong>Descript...,AI inside Inc.,"Lembang, West Java, Indonesia",2025-07-10 00:00:00+00:00,25,LinkedIn,未踏のaiプラットフォームを、構想から築く。\n\n[section] descriptio...
3,3D Design & Animation Intern,<strong>Responsibilities<br><br></strong>As a ...,PT SMART Tbk,"Jakarta, Jakarta, Indonesia",2025-12-19 00:00:00+00:00,101,LinkedIn,[section] responsibilities\nas a 3d design & a...
4,3D Designer,<strong>Responsibilities<br><br></strong><ul><...,Octarine,"Bogor, West Java, Indonesia",2026-01-27 00:00:00+00:00,33,LinkedIn,[section] responsibilities\n[bullet] product &...


### 5. Split Text into Sentences

This step divides the preprocessed text into semantic sentences. It uses NLTK's `sent_tokenize` for standard punctuation-based splitting and incorporates custom logic to handle bullet points and multiple newline characters as potential sentence boundaries.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
import re
import pandas as pd

# 5. Split Text into Sentences
def split_into_competency_units(text):

    if pd.isna(text) or not isinstance(text, str):
        return []

    chunks = []

    for line in text.split('\n'):

        line = line.strip()

        if not line:
            continue

        chunks.append(line)

    return chunks

df_industry_raw['sentences'] = df_industry_raw['combined_text'].apply(split_into_competency_units)

# Explode the DataFrame to have one row per sentence for easier processing of competency units
df_sentences = df_industry_raw.explode('sentences').reset_index(drop=True)
df_sentences = df_sentences[df_sentences['sentences'].str.strip() != ''].copy() # Remove empty sentences

print(f"Total sentences extracted: {len(df_sentences)}")
print("First 10 extracted sentences:")
display(df_sentences[['job_title', 'sentences']].head(10))

Total sentences extracted: 37942
First 10 extracted sentences:


,job_title,sentences
0,(Dev Core) Front-End Developer,developer
1,(Dev Core) Front-End Developer,junior
2,(Dev Core) Front-End Developer,internship
3,(Dev Core) Front-End Developer,updated 2023-05-22
4,(Dev Core) Front-End Developer,copy link to share
5,(Dev Core) Front-End Developer,[section] job link
6,(Dev Core) Front-End Developer,position (dev core) front-end developer
7,(Dev Core) Front-End Developer,team developer
8,(Dev Core) Front-End Developer,position level junior
9,(Dev Core) Front-End Developer,position type internship


### Extract Parent Section

In [ ]:
import pandas as pd
import re

print("Starting Section Tracking & Competency Unit Extraction...")

records = []

for idx, row in df_industry_raw.iterrows():

    job_title = row.get('job_title', '')
    text = row.get('combined_text', '')

    if pd.isna(text):
        continue

    current_section = "unknown"

    lines = text.split('\n')

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # --------------------------------------------------
        # SECTION
        # --------------------------------------------------
        if line.lower().startswith('[section]'):

            current_section = (
                line.replace('[SECTION]', '')
                    .replace('[section]', '')
                    .strip()
            )

            continue

        # --------------------------------------------------
        # BULLET
        # --------------------------------------------------
        if line.lower().startswith('[bullet]'):

            competency_unit = (
                line.replace('[BULLET]', '')
                    .replace('[bullet]', '')
                    .strip()
            )

            if competency_unit:

                records.append({
                    'job_title': job_title,
                    'parent_section': current_section,
                    'competency_unit': competency_unit
                })

            continue

        # --------------------------------------------------
        # NON-MARKER TEXT
        # --------------------------------------------------
        # simpan juga jika ternyata ada isi penting
        if len(line.split()) >= 2:

            records.append({
                'job_title': job_title,
                'parent_section': current_section,
                'competency_unit': line
            })

df_competency_units = pd.DataFrame(records)

print(f"Total competency units extracted: {len(df_competency_units)}")

display(df_competency_units.head(20))

Starting Section Tracking & Competency Unit Extraction...
Total competency units extracted: 30333


,job_title,parent_section,competency_unit
0,(Dev Core) Front-End Developer,unknown,updated 2023-05-22
1,(Dev Core) Front-End Developer,unknown,copy link to share
2,(Dev Core) Front-End Developer,job link,position (dev core) front-end developer
3,(Dev Core) Front-End Developer,job link,team developer
4,(Dev Core) Front-End Developer,job link,position level junior
5,(Dev Core) Front-End Developer,job link,position type internship
6,(Dev Core) Front-End Developer,job link,"position location malang, indonesia"
7,(Dev Core) Front-End Developer,job link,company description
8,(Dev Core) Front-End Developer,job link,qualification
9,(Dev Core) Front-End Developer,job link,job description


In [ ]:
section_candidates = []

for idx, row in df_competency_units.iterrows():

    text = str(row['competency_unit']).strip()

    # kandidat heading
    if (
        len(text.split()) <= 5
        and len(text) < 60
        and not text.endswith('.')
    ):

        section_candidates.append(text)

df_section_candidates = pd.DataFrame({
    'candidate_heading': list(set(section_candidates))
})

display(df_section_candidates.head(50))

section_distribution = (
    df_competency_units['parent_section']
    .value_counts()
)

display(section_distribution.head(50))

df_competency_units[
    df_competency_units["parent_section"]=="unknown"
].sample(50)
display(df_competency_units.head(50))

,candidate_heading
0,kemampuan komunikasi yang baik
1,excellent career development opportunities
2,marine mammals keeper
3,provide l2 support
4,to join our
5,expertise in checkpoint
6,materi training
7,lokasi: gft hong kong
8,monitor project implementation
9,the university


,count
parent_section,
unknown,3978
requirements,3420
responsibilities,2153
job description,1041
persyaratan,943
skills,874
minimum qualifications,747
qualifications,708
deskripsi pekerjaan,635


,job_title,parent_section,competency_unit
0,(Dev Core) Front-End Developer,unknown,updated 2023-05-22
1,(Dev Core) Front-End Developer,unknown,copy link to share
2,(Dev Core) Front-End Developer,job link,position (dev core) front-end developer
3,(Dev Core) Front-End Developer,job link,team developer
4,(Dev Core) Front-End Developer,job link,position level junior
5,(Dev Core) Front-End Developer,job link,position type internship
6,(Dev Core) Front-End Developer,job link,"position location malang, indonesia"
7,(Dev Core) Front-End Developer,job link,company description
8,(Dev Core) Front-End Developer,job link,qualification
9,(Dev Core) Front-End Developer,job link,job description


In [ ]:
print(df_competency_units.columns.tolist())

['job_title', 'parent_section', 'competency_unit']


### SECTION NORMALIZATION & FILTERING

In [ ]:
import pandas as pd
import numpy as np
import re

print("=== SECTION NORMALIZATION & FILTERING ===")

# =====================================================
# 1. NORMALIZE HEADING
# =====================================================

def normalize_heading(text):

    if pd.isna(text):
        return 'unknown'

    text = str(text).lower().strip()

    # remove trailing punctuation
    text = re.sub(r'[:\-–—.]+$', '', text)

    # normalize spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# =====================================================
# 2. SECTION MAPPING
# =====================================================

def map_section(section):

    s = normalize_heading(section)

    # -------------------------
    # COMPETENCY
    # -------------------------
    competency_keywords = [

        # =====================
        # Requirement
        # =====================
        'requirement',
        'requirements',
        'required',
        'persyaratan',
        'syarat',

        # =====================
        # Qualification
        # =====================
        'qualification',
        'qualifications',
        'kualifikasi',

        # =====================
        # Skills
        # =====================
        'skill',
        'skills',
        'tools',
        'competenc',      # competency, competencies
        'technical skill',
        'technical competenc',
        'soft skill',
        'behavioral competenc',
        'leadership competenc',

        # =====================
        # Experience
        # =====================
        'experience',
        'professional experience',
        'required experience',

        # =====================
        # Candidate Profile
        # =====================
        'candidate profile',
        'ideal profile',
        'about you',
        'who we need',
        'what you bring',
        'looking for',
        'person we are looking for',
        'fit in this job',
        'need to have',
        'nice to have',
        'nice-to-have',
        'good to have',
        'bonus point if',
        'must-have',

        # =====================
        # Knowledge / Attributes
        # =====================
        'knowledge',
        'attributes',
        'knowledge, skills',
        'skills and qualifications',
        'skills and attributes',
        'qualifications and experience',

        # =====================
        # Education
        # =====================
        'academic qualification',
        'academic qualifications',
        'education requirement',

        # =====================
        # Certification
        # =====================
        'certification',
        'certifications',

        # =====================
        # Physical / Mental
        # =====================
        'physical and mental'

        'what you will need',
        'what you will bring',
        'need to be successful',
        'qualified candidates',
        'preferred',
        'preferred character',
        'what do you have',
        'education',
        'profile',
        'essential',
        'requisiti',
        'competenze',
        'minimum required',
        'mandatory',

        "what you will need",
        "what you'll need",
        'who are you',
        'who you are',
        'character',
        'is this you',
        'specialized tools',
        'curriculum vitae'
    ]

    if any(k in s for k in competency_keywords):
        return 'competency'

    # -------------------------
    # ACTIVITY
    # -------------------------
    activity_keywords = [

        # =====================
        # Responsibilities
        # =====================
        'responsibil',
        'responsibilit',
        'responsibilites',
        'responbilities',

        # =====================
        # Duties
        # =====================
        'duties',
        'duty',

        # =====================
        # Accountabilities
        # =====================
        'accountabil',

        # =====================
        # Role
        # =====================
        'about the job',
        'about this role',
        'about the position',
        'role overview',
        'role description',
        'job overview',
        'role',

        # =====================
        # Job Description
        # =====================
        'job description',
        'job descriptions',
        'job desc',
        'jobdesc',
        'jobdesk',
        'job desk',
        'deskripsi pekerjaan',
        'description',

        # =====================
        # Job Summary / Brief
        # =====================
        'job summary',
        'job brief',
        'job scope',

        # =====================
        # Scope of Work
        # =====================
        'scope of work',
        'lingkup pekerjaan',

        # =====================
        # Work To Be Done
        # =====================
        'what you will do',
        'what you’ll do',
        "what you'll do",

        'what you will be doing',
        "what you'll be doing",
        'be doing',
        'you will',
        'what will you do',
        'the work you’ll do',

        'your day-to-day',
        "what you'll do on a typical day",
        "‘day to day’",
        "'day-to-day'",
        'your day',

        'what we’ll work on together',
        'look',

        # =====================
        # Indonesian
        # =====================
        'tanggung jawab',
        'tugas utama',
        'tugas dan tanggung jawab',
        'spesifikasi pekerjaan',

        # =====================
        # Misc
        # =====================
        'essential functions',
        'core responsibilities',
        'area of responsibilities',
        'key responsibility',
        'primary responsibilities',
        'primary duties'
        'job purpose',
        'position overview',
        'focus areas',
        'activities',
        'project execution',
        'attività',
        'responsabilità',
        'job purpose',
        'responsible',
        'you will:',
        'position summary',
        'capable to',
    ]

    if any(k in s for k in activity_keywords):
        return 'activity'

    # -------------------------
    # COMPANY INFO
    # -------------------------
    company_keywords = [

        # =====================
        # Company Introduction
        # =====================
        'about us',
        'about the team',
        'about company',
        'company description',
        'who are we',
        'about neon',
        'who we',
        'company',
        'company description',

        # =====================
        # Why Join
        # =====================
        'why join',
        'why michelin',

        # =====================
        # Benefits
        # =====================
        'benefit',
        'benefits',
        'what we offer',
        'what you will get',
        "what's in it for you",
        'what cybertrend can offer you',

        # =====================
        # Company Culture
        # =====================
        'core beliefs',
        'mission',
        'vision',
        'philosophy',

        # =====================
        # Career / Application
        # =====================
        'career path',
        'how to apply',
        'job link',
        'current vacancies',

        # =====================
        # Misc Company Info
        # =====================
        'startup culture',
        'more about the bank',
        'how did it all start',
        'your perks',
        'apply online',
        'group',
        'our product',
        'the opportunity',
        'about mtu',
        'about moladin',
        'how did we manage',
        'what we do to make you successful',
        'inspire motion',
        'location',
        'why work with us'
    ]

    if any(k in s for k in company_keywords):
        return 'company_info'

    return 'unknown'


# =====================================================
# 3. APPLY MAPPING
# =====================================================

df_competency_units['parent_section_normalized'] = (

    df_competency_units['parent_section']
    .fillna('unknown')
    .apply(map_section)
)

# =====================================================
# 4. FILTER RELEVANT SECTIONS
# =====================================================

KEEP_SECTIONS = [

    'competency',
    'activity',
    'company_info',
    'unknown'
]

df_competency_units_filtered = (

    df_competency_units[
        df_competency_units[
            'parent_section_normalized'
        ].isin(KEEP_SECTIONS)
    ]
    .copy()
)

# =====================================================
# 5. BASIC CLEANING
# =====================================================

df_competency_units_filtered['competency_unit'] = (

    df_competency_units_filtered['competency_unit']
    .fillna('')
    .astype(str)
    .str.strip()
)

df_competency_units_filtered = (

    df_competency_units_filtered[
        df_competency_units_filtered[
            'competency_unit'
        ].str.len() > 0
    ]
)

# =====================================================
# 6. REMOVE DUPLICATES
# =====================================================

df_competency_units_filtered = (

    df_competency_units_filtered
    .drop_duplicates(
        subset=[
            'job_title',
            'parent_section_normalized',
            'competency_unit'
        ]
    )
    .reset_index(drop=True)
)

# =====================================================
# 7. UNKNOWN ANALYSIS
# =====================================================

df_unknown = (

    df_competency_units_filtered[
        df_competency_units_filtered[
            'parent_section_normalized'
        ] == 'unknown'
    ]
)

print("\n===== UNKNOWN HEADINGS =====")

display(

    df_unknown['parent_section']
    .value_counts()
    .head(60)
)

# =====================================================
# 8. SUMMARY
# =====================================================

print("\n===== SECTION DISTRIBUTION =====")

display(

    df_competency_units_filtered[
        'parent_section_normalized'
    ].value_counts()
)

print("\n===== DATASET SIZE =====")

print(
    f"Original rows : {len(df_competency_units):,}"
)

print(
    f"Filtered rows : {len(df_competency_units_filtered):,}"
)

print(
    f"Unknown rows : {len(df_unknown):,}"
)

# =====================================================
# 9. SAVE
# =====================================================

output_dir = "/content/processed_data/industry_data"

df_competency_units_filtered.to_csv(
    f"{output_dir}/industry_competency_units_filtered.csv",
    index=False
)

=== SECTION NORMALIZATION & FILTERING ===

===== UNKNOWN HEADINGS =====


,count
parent_section,
unknown,3651
en,45
id,45
category,29
consulting,27
hris / hcms platforms,25
manual and automation testing,25
about product designer,24
it security operations officer,22



===== SECTION DISTRIBUTION =====


,count
parent_section_normalized,
competency,10735
activity,7995
unknown,7073
company_info,1211



===== DATASET SIZE =====
Original rows : 30,333
Filtered rows : 27,014
Unknown rows : 7,073


In [ ]:
import pandas as pd
import numpy as np

print("=== UNKNOWN SECTION RECOVERY ===")

# =====================================================
# 1. FORWARD FILL UNKNOWN
# =====================================================

df_fixed = df_competency_units.copy()

# Simpan hasil awal
df_fixed['parent_section_original'] = (
    df_fixed['parent_section_normalized']
)

# Unknown -> NaN
df_fixed['section_temp'] = (
    df_fixed['parent_section_normalized']
    .replace('unknown', np.nan)
)

# Forward fill per lowongan
df_fixed['parent_section_fixed'] = (

    df_fixed
    .groupby('job_title')['section_temp']
    .ffill()
)

# Isi NaN yang masih tersisa
df_fixed['parent_section_fixed'] = (
    df_fixed['parent_section_fixed']
    .fillna('unknown')
)

# =====================================================
# 2. RULE-BASED CLASSIFICATION
# =====================================================

competency_keywords = [

    'experience',
    'experiences',
    'skill',
    'skills',
    'knowledge',
    'ability',
    'abilities',
    'qualification',
    'qualifications',
    'education',
    'degree',
    'certification',
    'certifications',
    'familiar',
    'proficient',
    'proficiency',
    'communication',
    'teamwork',
    'leadership',
    'understand',
    'understanding',
    'knowledge of',
    'knowledge in',

    'mampu',
    'menguasai',
    'memahami',
    'mengerti',
    'berpengalaman',
    'pengalaman',
    'qualification',
    'qualifications',
    'knowledge',
    'experience',
    'skill',
    'skills',
    'ability',
    'abilities',
    'certification',
    'sertifikasi',
    'familiar',
    'proficient',

    'vmware',
    'sql',
    'python',
    'excel',

    'creative',
    'analytical',
    'positive vibes',
    'jujur',
    'ceria',

    'what we expect from you',
    'what we’re looking for',
    'what you will need',
    'requirement',
    'able to',
    'cepat belajar',
    'memiliki',
    'mahir',
    'keep',
    'good',
    'seeking',
    'terbiasa',
    'bekerja sama'
]

activity_keywords = [

    'responsible',
    'responsibilities',
    'develop',
    'design',
    'implement',
    'maintain',
    'manage',
    'monitor',
    'support',
    'coordinate',
    'execute',
    'analyze',
    'analyse',
    'build',
    'create',
    'conduct',
    'perform',
    'review',
    'deliver',
    'lead',
    'melakukan',
    'mengelola',
    'mengembangkan',
    'menganalisis',
    'menangani',
    'bertanggung jawab',
    'berkoordinasi',
    'memonitor',

    'gather',
    'gathers',
    'document',
    'documents',

    'assist',
    'assisting',

    'prepare',
    'preparing',

    'support',
    'supporting',

    'collaborate',
    'collaborating',

    'troubleshoot',
    'troubleshooting',

    'maintain',
    'maintaining',

    'develop',
    'developing',

    'design',
    'designing',

    'implement',
    'implementing',

    'resolve',
    'resolving',

    'menyusun',
    'menulis',

    'memberikan bantuan',

    'berkomunikasi',

    'membuat laporan',

    'project planning',

    'documentation',

    'compliance',
    'memastikan',
    'ensure',
    'membantu',
    'membuat',
    'mengumpulkan',
    'provide',
    'menyusun',
    'deploment',
    'memberikan',
    'menyediakan',
    'memasukkan',
    'mencatat',
    'melindungi',
    'act',
    'evaluate',
    'identify',
    'integrate',
    'mengatur'
]

company_keywords = [

    'benefit',
    'benefits',
    'career',
    'join us',
    'company',
    'organization',
    'our team',
    'holiday',
    'insurance',
    'salary',
    'allowance',
    '福利',
    '休日',
    '休暇'
]

def classify_unknown(text):

    text = str(text).lower()

    if any(k in text for k in competency_keywords):
        return 'competency'

    if any(k in text for k in activity_keywords):
        return 'activity'

    if any(k in text for k in company_keywords):
        return 'company_info'

    return 'unknown'

# =====================================================
# 3. CLASSIFY REMAINING UNKNOWN
# =====================================================

mask_unknown = (
    df_fixed['parent_section_fixed']
    == 'unknown'
)

df_fixed.loc[
    mask_unknown,
    'parent_section_fixed'
] = (

    df_fixed.loc[
        mask_unknown,
        'competency_unit'
    ]
    .apply(classify_unknown)
)

# =====================================================
# 4. UNKNOWN ANALYSIS
# =====================================================

df_unknown_final = (

    df_fixed[
        df_fixed['parent_section_fixed']
        == 'unknown'
    ]
)

print("\n===== ORIGINAL SECTION =====")

display(
    df_fixed['parent_section_original']
    .value_counts()
)

print("\n===== FIXED SECTION =====")

display(
    df_fixed['parent_section_fixed']
    .value_counts()
)

print("\n===== FINAL UNKNOWN SAMPLE =====")

if len(df_unknown_final) > 0:

    display(

        df_unknown_final[
            [
                'job_title',
                'parent_section',
                'competency_unit'
            ]
        ]
        .sample(
            min(60, len(df_unknown_final)),
            random_state=42
        )
    )

print("\n===== SUMMARY =====")

print(
    f"Total rows : {len(df_fixed):,}"
)

print(
    f"Remaining unknown : {len(df_unknown_final):,}"
)

print(
    f"Recovered rows : "
    f"{(df_fixed['parent_section_original'] == 'unknown').sum() - len(df_unknown_final):,}"
)

# =====================================================
# 5. SAVE
# =====================================================

output_dir = "/content/processed_data/industry_data"

df_fixed.to_csv(
    f"{output_dir}/industry_sections_recovered.csv",
    index=False
)

df_unknown_final.to_csv(
    f"{output_dir}/industry_remaining_unknown.csv",
    index=False
)

print("\nFiles saved successfully.")

=== UNKNOWN SECTION RECOVERY ===

===== ORIGINAL SECTION =====


,count
parent_section_original,
competency,12343
activity,8842
unknown,7757
company_info,1391



===== FIXED SECTION =====


,count
parent_section_fixed,
competency,15183
activity,11697
company_info,1998
unknown,1455



===== FINAL UNKNOWN SAMPLE =====


,job_title,parent_section,competency_unit
14968,IT Monitoring,unknown,shift ii : 16.00 – 00.00
29761,Web Developer,unknown,δωρεά
12692,Graphic Designer - ID Team,graphic designer (gd),"start date: flexible, but the sooner the better"
27519,System Development,unknown,"sampai: 31 desember, 2025"
27215,Supervisor IT,unknown,pendidikan minimal s1 teknik komputer
17381,Ingenieur Elektrotechnik (w/m/d),was wir dir bieten,"ein umfeld ohne unnötige bürokratie, dafür mit..."
5072,Content Planner,unknown,we have now based in few of the fastest growin...
29510,Veiligheidskundige,unknown,een uitdagende functie binnen een snelgroeiend...
29520,Veiligheidskundige,unknown,bij ploegam werken jouw collega’s iedere dag v...
23755,Quality Assurance & Improvement Manager (2W Au...,preventive maintenance & reliability,"define maintenance standards, inspection routi..."



===== SUMMARY =====
Total rows : 30,333
Remaining unknown : 1,455
Recovered rows : 6,302

Files saved successfully.


In [ ]:
display(df_competency_units.head(60))

,job_title,parent_section,competency_unit,parent_section_normalized
0,(Dev Core) Front-End Developer,unknown,updated 2023-05-22,unknown
1,(Dev Core) Front-End Developer,unknown,copy link to share,unknown
2,(Dev Core) Front-End Developer,job link,position (dev core) front-end developer,company_info
3,(Dev Core) Front-End Developer,job link,team developer,company_info
4,(Dev Core) Front-End Developer,job link,position level junior,company_info
5,(Dev Core) Front-End Developer,job link,position type internship,company_info
6,(Dev Core) Front-End Developer,job link,"position location malang, indonesia",company_info
7,(Dev Core) Front-End Developer,job link,company description,company_info
8,(Dev Core) Front-End Developer,job link,qualification,company_info
9,(Dev Core) Front-End Developer,job link,job description,company_info


### Translation

In [ ]:
import pandas as pd
import numpy as np
import re

print("=== INDUSTRY TEXT CLEANING ===")

df_industry_final = df_fixed.copy()

# =====================================================
# NOISE PATTERNS
# =====================================================

NOISE_PATTERNS = [

    r'updated\s*\d{4}-\d{2}-\d{2}',
    r'last updated.*',
    r'copy link to share',
    r'share this job',
    r'apply now',
    r'apply online',
    r'click here',
    r'read more',
    r'job link',

    r'please note that only short-listed candidates.*',

    r'tanggal penutupan.*',
    r'closing date.*',
    r'application deadline.*',

    r'kpis?\s*/\s*dimensions?:?',

    r'^soft skills:?$',
    r'^hard skills:?$',
    r'^skills:?$',
    r'^competencies:?$',
    r'^requirements:?$',
    r'^qualifications:?$',

    r'^1\s*-\s*3\s*years$',
    r'^2\s*-\s*4\s*years$',
    r'^3\s*-\s*5\s*tahun$',

    r'^\d+\s*years?$',
    r'^\d+\+\s*years?$',

    r'^as a\s*[\'"“”‘’]?$',

    r'^e[\-\s]?mail:?$',

    r'^drop portfolio here.*$',
    r'^drop portofolio here.*$',
    r'^upload cv.*$',
    r'^submit application.*$',

    r'^powered by.*$',

    r'^-\s*to\s+\w+.*$',
    r'^—\s*to\s+\w+.*$',

    r'^:$',

    r'^\s*$'
]

# =====================================================
# COMPANY INFO KEYWORDS
# =====================================================

COMPANY_INFO_KEYWORDS = [

    # benefits
    'company perks',
    'employee benefits',
    'benefits',
    'competitive compensation',
    'salary',
    'bonus',
    'equity',
    'health insurance',
    'employment insurance',
    'paid leave',
    'unpaid leave',

    # work environment
    'hybrid working',
    'hybrid work',
    'flexible working hours',
    'work-life balance',
    'comfort office',
    'breakfast and lunch',

    # learning & culture
    'peer learning',
    'sharing session',
    'webinar',
    'company values',
    'our mission',
    'our vision',
    'our culture',
    'company culture',

    # recruitment
    'recruitment pipeline',
    'recruitment process',
    'application review',
    'pre-assessment test',
    'hr interview',
    'user interview',
    'final interview',
    'job offer',

    # company profile
    'who are we',
    'about us',
    'why join us',
    'what we offer',
    'join our team',
    'our company',

    # perks
    'career development opportunities',
    'team building',
    'coffee time',
    'happy hours',
    'networking opportunities',

    # values
    'our purpose',
    'our values',
    'we exist to',
    'we take pride in',

    # common phrases
    'people × profit × planet',
    'employee wellbeing',
    'employee engagement',
    "what's on offer?",
    'powered by'
]

# =====================================================
# PREPARE TEXT
# =====================================================

df_industry_final['competency_unit_clean'] = (

    df_industry_final['competency_unit']
    .fillna('')
    .astype(str)
    .str.strip()

)

# =====================================================
# REMOVE URL ROWS
# =====================================================

url_mask = (

    df_industry_final['competency_unit_clean']
    .str.contains(
        r'https?://|www\.',
        case=False,
        na=False
    )

)

df_removed_url = df_industry_final[url_mask].copy()

df_industry_final = df_industry_final[~url_mask].copy()

print(f"Removed URL rows: {len(df_removed_url):,}")

# =====================================================
# REMOVE EMAIL ROWS
# =====================================================

email_mask = (

    df_industry_final['competency_unit_clean']
    .str.contains(
        r'\S+@\S+',
        case=False,
        na=False
    )

)

df_removed_email = df_industry_final[email_mask].copy()

df_industry_final = df_industry_final[~email_mask].copy()

print(f"Removed email rows: {len(df_removed_email):,}")

# =====================================================
# REMOVE NOISE PATTERN ROWS
# =====================================================

noise_pattern = "|".join(NOISE_PATTERNS)

noise_mask = (

    df_industry_final['competency_unit_clean']
    .str.lower()
    .str.contains(
        noise_pattern,
        regex=True,
        na=False
    )

)

df_removed_noise = (

    df_industry_final[
        noise_mask
    ].copy()

)

df_industry_final = (

    df_industry_final[
        ~noise_mask
    ].copy()

)

print(
    f"Removed noise rows: {len(df_removed_noise):,}"
)

# =====================================================
# REMOVE COMPANY INFO ROWS
# =====================================================

company_pattern = "|".join(

    re.escape(keyword.lower())

    for keyword in COMPANY_INFO_KEYWORDS

)

company_mask = (

    df_industry_final[
        'competency_unit_clean'
    ]
    .str.lower()
    .str.contains(
        company_pattern,
        regex=True,
        na=False
    )

)

df_removed_company_info = (

    df_industry_final[
        company_mask
    ].copy()

)

df_industry_final = (

    df_industry_final[
        ~company_mask
    ].copy()

)

print(
    f"Removed company info rows: {len(df_removed_company_info):,}"
)

# =====================================================
# FINAL CLEAN
# =====================================================

df_industry_final[
    'competency_unit_clean'
] = (

    df_industry_final[
        'competency_unit_clean'
    ]
    .str.replace(
        r'\s+',
        ' ',
        regex=True
    )
    .str.strip()

)

df_industry_final = (

    df_industry_final[
        df_industry_final[
            'competency_unit_clean'
        ].str.len() > 0
    ]
    .copy()

)

# =====================================================
# KEEP ONLY COMPETENCY + ACTIVITY
# =====================================================

valid_sections = [

    'competency',
    'activity'

]

section_mask = (

    df_industry_final[
        'parent_section_fixed'
    ]
    .isin(valid_sections)

)

df_removed_section = (

    df_industry_final[
        ~section_mask
    ]
    .copy()

)

df_industry_final = (

    df_industry_final[
        section_mask
    ]
    .copy()

)

print(
    f"Removed invalid sections: "
    f"{len(df_removed_section):,}"
)

print("\nRemoved section distribution:")

print(

    df_removed_section[
        'parent_section_fixed'
    ]
    .value_counts(
        dropna=False
    )

)

# =====================================================
# SUMMARY
# =====================================================

print("\n===== SUMMARY =====")

print(
    f"Final rows: {len(df_industry_final):,}"
)

print(
    f"Removed URL: {len(df_removed_url):,}"
)

print(
    f"Removed Email: {len(df_removed_email):,}"
)

print(
    f"Removed Noise: {len(df_removed_noise):,}"
)

print(
    f"Removed Company Info: {len(df_removed_company_info):,}"
)

print(
    f"Unique competency text: "
    f"{df_industry_final['competency_unit_clean'].nunique():,}"
)

print("\n===== FINAL SECTION DISTRIBUTION =====")

print(

    df_industry_final[
        'parent_section_fixed'
    ]
    .value_counts()

)

display(

    df_industry_final[
        [
            'job_title',
            'parent_section_fixed',
            'competency_unit',
            'competency_unit_clean'
        ]
    ]
    .sample(
        30,
        random_state=42
    )

)

=== INDUSTRY TEXT CLEANING ===
Removed URL rows: 45
Removed email rows: 41
Removed noise rows: 169
Removed company info rows: 1,213
Removed invalid sections: 3,178

Removed section distribution:
parent_section_fixed
company_info    1770
unknown         1408
Name: count, dtype: int64

===== SUMMARY =====
Final rows: 25,687
Removed URL: 45
Removed Email: 41
Removed Noise: 169
Removed Company Info: 1,213
Unique competency text: 20,793

===== FINAL SECTION DISTRIBUTION =====
parent_section_fixed
competency    14180
activity      11507
Name: count, dtype: int64


,job_title,parent_section_fixed,competency_unit,competency_unit_clean
24884,SAP MM CONSULTANT,activity,conduct post implementation review to support ...,conduct post implementation review to support ...
5853,Data Analyst,competency,minimum gpa 3.00.,minimum gpa 3.00.
23062,Project Manager,activity,contribute to lessons learned and internal kno...,contribute to lessons learned and internal kno...
10412,Front-End Engineer,activity,testing and debugging the software: to ensure ...,testing and debugging the software: to ensure ...
20472,Network Administrator,activity,identify opportunities for system improvements...,identify opportunities for system improvements...
10061,Front End Developer,activity,identify and mitigate frontend-related risks i...,identify and mitigate frontend-related risks i...
1738,Application Development,activity,actively participate in team meetings and prov...,actively participate in team meetings and prov...
26063,Software Developer,competency,experience collaborating with the purchasing d...,experience collaborating with the purchasing d...
23792,Quality Assurance (Contract),activity,•communicate issues and provide feedback to de...,•communicate issues and provide feedback to de...
21741,Process Excellence Analyst,activity,collaborate with stakeholders to define clear ...,collaborate with stakeholders to define clear ...


In [ ]:
# =====================================================
# INSTALL LIBRARY
# =====================================================

!pip install deep-translator tqdm -q

# =====================================================
# IMPORT
# =====================================================

import pandas as pd
import numpy as np
import re
import time

from tqdm import tqdm
from deep_translator import GoogleTranslator

print("=== START TRANSLATION PIPELINE ===")

# =====================================================
# 1. BUILD UNIQUE TRANSLATION CACHE
# =====================================================

translation_cache = pd.DataFrame({

    'original_text':

        df_industry_final[
            'competency_unit_clean'
        ]
        .fillna('')
        .astype(str)
        .str.strip()
        .unique()

})

translation_cache = (

    translation_cache[
        translation_cache[
            'original_text'
        ].str.len() > 0
    ]
    .copy()
    .reset_index(drop=True)

)

print(
    f"Unique texts to process: {len(translation_cache):,}"
)

# =====================================================
# 2. TRANSLATOR
# =====================================================

translator_id = GoogleTranslator(
    source='auto',
    target='id'
)

translator_en = GoogleTranslator(
    source='auto',
    target='en'
)

def translate_text(text, target_lang):

    try:

        text = str(text).strip()

        if text == "":
            return ""

        translated = GoogleTranslator(
            source='auto',
            target=target_lang
        ).translate(text)

        return translated

    except Exception as e:

        print(
            f"Translation error: {str(e)[:100]}"
        )

        return text

# =====================================================
# 3. RESUME SUPPORT
# =====================================================

PROGRESS_FILE = "/content/translation_progress.csv"

try:

    existing_progress = pd.read_csv(
        PROGRESS_FILE
    )

    translated_dict = {}

    for _, row in existing_progress.iterrows():

        translated_dict[
            row['original_text']
        ] = {

            'translated_id':
                row['translated_id'],

            'translated_en':
                row['translated_en']

        }

    print(
        f"Loaded previous progress: {len(translated_dict):,}"
    )

except:

    translated_dict = {}

    print(
        "No previous progress found."
    )

# =====================================================
# 4. TRANSLATION LOOP
# =====================================================

remaining_texts = (

    translation_cache[
        ~translation_cache[
            'original_text'
        ].isin(
            translated_dict.keys()
        )
    ]

)

print(
    f"Remaining texts: {len(remaining_texts):,}"
)

counter = 0

for text in tqdm(

    remaining_texts[
        'original_text'
    ]

):

    translated_id = translate_text(
        text,
        'id'
    )

    translated_en = translate_text(
        text,
        'en'
    )

    translated_dict[text] = {

        'translated_id':
            translated_id,

        'translated_en':
            translated_en

    }

    counter += 1

    # -------------------------
    # SAVE EVERY 500 ROWS
    # -------------------------

    if counter % 500 == 0:

        temp_df = pd.DataFrame([

            {

                'original_text': k,

                'translated_id':
                    v['translated_id'],

                'translated_en':
                    v['translated_en']

            }

            for k, v in translated_dict.items()

        ])

        temp_df.to_csv(

            PROGRESS_FILE,

            index=False

        )

        print(
            f"Checkpoint saved: {len(temp_df):,}"
        )

    # -------------------------
    # SMALL DELAY
    # -------------------------

    time.sleep(0.05)

# =====================================================
# 5. FINAL CACHE
# =====================================================

translation_cache_final = pd.DataFrame([

    {

        'original_text': k,

        'translated_id':
            v['translated_id'],

        'translated_en':
            v['translated_en']

    }

    for k, v in translated_dict.items()

])

translation_cache_final.to_csv(

    "/content/translation_cache_industry.csv",

    index=False

)

print(
    f"Translation cache saved: {len(translation_cache_final):,}"
)

# =====================================================
# 6. MERGE BACK
# =====================================================

translation_lookup_id = dict(
    zip(
        translation_cache_final[
            'original_text'
        ],
        translation_cache_final[
            'translated_id'
        ]
    )
)

translation_lookup_en = dict(
    zip(
        translation_cache_final[
            'original_text'
        ],
        translation_cache_final[
            'translated_en'
        ]
    )
)

df_industry_final[
    'competency_unit_id'
] = (

    df_industry_final[
        'competency_unit_clean'
    ]
    .map(
        translation_lookup_id
    )

)

df_industry_final[
    'competency_unit_en'
] = (

    df_industry_final[
        'competency_unit_clean'
    ]
    .map(
        translation_lookup_en
    )

)

# =====================================================
# 7. QUALITY CHECK
# =====================================================

print("\n===== SAMPLE RESULT =====")

display(

    df_industry_final[
        [
            'competency_unit',
            'competency_unit_clean',
            'competency_unit_id',
            'competency_unit_en'
        ]
    ]
    .sample(
        50,
        random_state=42
    )

)

# =====================================================
# 8. SAVE FINAL DATASET
# =====================================================

OUTPUT_PATH = (

    "/content/processed_data/"
    "industry_data/"
    "industry_final_translated.csv"
)

df_industry_final.to_csv(

    OUTPUT_PATH,

    index=False

)

print(
    f"\nDataset saved to:\n{OUTPUT_PATH}"
)

# =====================================================
# 9. SUMMARY
# =====================================================

print("\n===== SUMMARY =====")

print(
    f"Rows: {len(df_industry_final):,}"
)

print(
    f"Unique competency text: "
    f"{df_industry_final['competency_unit_clean'].nunique():,}"
)

print(
    f"Translated text: "
    f"{df_industry_final['competency_unit_id'].notna().sum():,}"
)

print(
    f"Translated text: "
    f"{df_industry_final['competency_unit_en'].notna().sum():,}"
)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.2 MB/s eta 0:00:00
=== START TRANSLATION PIPELINE ===
Unique texts to process: 20,793
No previous progress found.
Remaining texts: 20,793


  2%|▏         | 500/20793 [06:12<3:49:47,  1.47it/s]

Checkpoint saved: 500


  5%|▍         | 1000/20793 [12:17<4:14:59,  1.29it/s]

Checkpoint saved: 1,000


  6%|▌         | 1295/20793 [16:04<4:02:05,  1.34it/s]


KeyboardInterrupt: 

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_INDUSTRY = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_INDUSTRY}")

# Define the file to copy
filename_to_copy = 'industry_final_translated.csv'
source_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, filename_to_copy)
destination_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, filename_to_copy)

try:
    shutil.copy2(source_filepath, destination_filepath)
    print(f"Successfully copied '{filename_to_copy}' to '{destination_filepath}'.")
except FileNotFoundError:
    print(f"Error: Source file '{filename_to_copy}' not found at '{source_filepath}'.")
except Exception as e:
    print(f"An error occurred while copying '{filename_to_copy}': {e}")

### Final Cleaning

In [ ]:
import pandas as pd
import re

# =====================================
# LOAD DATA
# =====================================

from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Path file CSV di Google Drive
file_path = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/industry_final_translated.csv"

# Load data
df = pd.read_csv(file_path)

print(df.shape)
df.head()

# =====================================
# NOISE PATTERNS
# =====================================

noise_patterns = [

    # ==================================================
    # COMPANY PROFILE
    # ==================================================
    r"about us",
    r"our company",
    r"company profile",
    r"who we are",
    r"established in",
    r"founded in",
    r"located in",
    r"headquartered",
    r"our mission",
    r"our vision",
    r"specializes in",
    r"expand to a bigger market",
    r"global scale",
    r"serve users",
    r"our products?",
    r"our services?",
    r"kita",
    r"we are",
    r"anda",
    r"you are",
    r"visi kami",
    r"misi kami",
    r"mengapa memilih kami?"
    r"kami ingin sekali bertemu anda!",


    # ==================================================
    # EMPLOYEE BENEFITS
    # ==================================================
    r"financial well-being",
    r"employees'? financial",
    r"meet(s)? their expectations",
    r"cozy office",
    r"comfortable facilities",
    r"pleasant atmosphere",
    r"food will be perfect",
    r"cheer up everyone",
    r"garden",
    r"employee benefits?",
    r"benefits package",
    r"health insurance",
    r"annual leave",
    r"competitive salary",
    r"career growth",
    r"work life balance",

    # ==================================================
    # COMPANY CULTURE
    # ==================================================
    r"everyone can grow",
    r"knowledgeable employees",
    r"biggest assets",
    r"we care about",
    r"we want to make sure",
    r"driven by the idea",
    r"working in a pleasant atmosphere",

    # ==================================================
    # HIRING PROCESS
    # ==================================================
    r"hiring stages?",
    r"recruitment process",
    r"selection process",
    r"interview process",
    r"casual interview",
    r"document screening",
    r"reference check",
    r"offer interview",
    r"aptitude test",
    r"final interview",

    # ==================================================
    # INTERVIEW STAGES
    # ==================================================
    r"in this interview",
    r"in this stage",
    r"you will meet the user",
    r"our hr will examine",
    r"determine whether you are",
    r"candidate we have been looking for",
    r"role you're applying for",
    r"position you are applying for",

    # ==================================================
    # CANDIDATE ASSESSMENT
    # ==================================================
    r"we will assess",
    r"assess specific skills",
    r"assess how well",
    r"technical proficiency",
    r"suitability for the role",
    r"background, personality",
    r"strengths and weaknesses",
    r"technical skills and personality align",

    # ==================================================
    # WEBSITE / NAVIGATION
    # ==================================================
    r"click here",
    r"read more",
    r"learn more",
    r"visit our website",
    r"follow us",

    # ==================================================
    # BLOG / NEWS
    # ==================================================
    r"press release",
    r"latest information",
    r"latest news",
    r"conference",
    r"blog article",

    r"about ai inside products",
    r"about the worldview",
    r"work with buddy",
    r"next-generation office",
    r"common sense of the future",
    r"future creator",
    r"work style reform",
    r"beyond ai ocr",
    r"autonomous ai development",
    r"large-scale language model",
    r"learning center",
    r"heylix",
    r"azabudai hills",
    r"tokyo",
    r"minato-ku",
    r"azabudai",
    r"mori jp tower",
    r"\d+-\d+-\d+",
    r"\b\d{1,4}-\d{1,4}-\d{1,4}\b",

    r"salary",
    r"allowance",
    r"telework allowance",
    r"business allowance",
    r"paid vacation",
    r"vacation",
    r"leave",
    r"holidays",
    r"days off",
    r"childcare leave",
    r"sick leave",
    r"refreshment leave",
    r"condolence leave",
    r"commuting transportation",

    r"work from anywhere",
    r"working conditions",
    r"job type",
    r"work hours",
    r"core time",
    r"flexible time",
    r"probation period",
    r"telework",
    r"remote work",
    r"hybrid system",
    r"come to the office",
    r"office or telework",
    r"willing to be placed",

    r"\byen\b",
    r"\$",
    r"salary",
    r"allowance",
    r"\d{1,2}:\d{2}",
    r"days off",
    r"vacation",
    r"leave",

    r"win business with ai",
    r"taken away by ai",
    r"future creator",
    r"common sense of the future",
    r"next norm",
    r"next-generation office",
    r"work with buddy",

    r"^we are looking for$",
    r"^job type$",
    r"^work hours$",
    r"^working conditions$",
    r"^probation period$",
    r"^benefits$",
    r"^requirements$",
    r"^preferred experiences$",

    r"join the company",
    r"work day will be postponed",
    r"come to the office",
    r"office or telework",
    r"telework possible",
    r"work from anywhere",
    r"hybrid system",

    r"salary",
    r"allowance",
    r"transportation expenses",
    r"paid vacation",
    r"yen/month",
    r"yen",
    r"basic salary",
    r"business allowance",
    r"telework allowance",
    r"internship"
    r""
]

# =====================================
# COMBINE REGEX
# =====================================

combined_pattern = "|".join(noise_patterns)

# =====================================
# FILTER COLUMN
# =====================================

df["competency_unit_en"] = (
    df["competency_unit_en"]
    .fillna("")
    .astype(str)
)

# =====================================
# REMOVE ROWS CONTAINING NOISE
# =====================================

before = len(df)

df_clean = df[
    ~df["competency_unit_en"]
        .str.lower()
        .str.contains(
            combined_pattern,
            regex=True,
            na=False
        )
].copy()

after = len(df_clean)

print(f"Before : {before}")
print(f"After  : {after}")
print(f"Removed: {before-after}")

# =====================================
# SPLIT MULTI-SENTENCE ROWS
# =====================================

before = len(df_clean)
split_columns = [
    "competency_unit",
    "competency_unit_clean",
    "competency_unit_id",
    "competency_unit_en"
]

def split_text(text):

    if pd.isna(text):
        return []

    text = str(text)

    text = re.sub(r'\|', '. ', text)
    text = re.sub(r'━━', '. ', text)
    text = re.sub(r'──', '. ', text)
    text = re.sub(r'—', '. ', text)

    sentences = re.split(
        r'(?<=[.!?;。！？])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if s.strip()
    ]


new_rows = []

for _, row in df_clean.iterrows():

    split_results = {}

    max_len = 1

    for col in split_columns:

        parts = split_text(row[col])

        if len(parts) == 0:
            parts = [row[col]]

        split_results[col] = parts
        max_len = max(max_len, len(parts))

    for i in range(max_len):

        new_row = row.copy()

        for col in split_columns:

            parts = split_results[col]

            if i < len(parts):
                new_row[col] = parts[i]
            else:
                new_row[col] = parts[-1]

        new_rows.append(new_row)

df_split = pd.DataFrame(new_rows)

print(df_split.shape)

after = len(df_split)

print(f"Before : {before}")
print(f"After  : {after}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(25687, 10)


/tmp/ipykernel_8876/2909180463.py:266: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Before : 25687
After  : 24055
Removed: 1632
(26033, 10)
Before : 24055
After  : 26033


In [ ]:
# =====================================================
# BUILD BILINGUAL TEXT
# =====================================================

def build_embedding_text(row):

    id_text = str(
        row['competency_unit_id']
    ).strip()

    en_text = str(
        row['competency_unit_en']
    ).strip()

    # handle nan
    if id_text.lower() == 'nan':
        id_text = ''

    if en_text.lower() == 'nan':
        en_text = ''

    # jika keduanya kosong
    if id_text == '' and en_text == '':
        return ''

    # jika sama persis
    if id_text.lower() == en_text.lower():
        return id_text

    # hanya ada indonesia
    if en_text == '':
        return id_text

    # hanya ada english
    if id_text == '':
        return en_text

    # gabungkan bilingual
    return f"{id_text} {en_text}"


df_split[
    'competency_embedding_text'
] = df_split.apply(
    build_embedding_text,
    axis=1
)

print(
    "Embedding text created:",
    len(df_split)
)

display(

    df_split[
        [
            'competency_unit_id',
            'competency_unit_en',
            'competency_embedding_text'
        ]
    ]
    .sample(
        20,
        random_state=42
    )

)

print(
    f"Unique competency text: "
    f"{df_split['competency_unit_clean'].nunique():,}"
)


Embedding text created: 26033


,competency_unit_id,competency_unit_en,competency_embedding_text
18320,memberikan rekomendasi mitigasi yang realistis...,provide mitigation recommendations that are re...,memberikan rekomendasi mitigasi yang realistis...
15500,mengembangkan dan memelihara laporan analitik ...,develop and maintain client analytics reports.,mengembangkan dan memelihara laporan analitik ...
13800,kirimkan lamaran Anda,submit your application,kirimkan lamaran Anda submit your application
25094,sikap berorientasi solusi;,solution – oriented attitude;,sikap berorientasi solusi; solution – oriented...
13561,kemampuan untuk bekerja secara efektif dengan ...,ability to work effectively with cross-functio...,kemampuan untuk bekerja secara efektif dengan ...
16011,pemecahan masalah:,problem-solving:,pemecahan masalah: problem-solving:
18679,pemahaman yang kuat tentang manajemen proyek b...,solid understanding of project management base...,pemahaman yang kuat tentang manajemen proyek b...
11184,membuat laporan harian dan bulanan terkait akt...,make daily and monthly reports related to help...,membuat laporan harian dan bulanan terkait akt...
9767,"memiliki kemampuan komunikasi yang baik, wawas...","have good communication skills, broad insight ...","memiliki kemampuan komunikasi yang baik, wawas..."
18178,perhatian yang sangat baik terhadap detail (ak...,"excellent attention to detail (spec accuracy, ...",perhatian yang sangat baik terhadap detail (ak...


Unique competency text: 20,982


In [ ]:
import pandas as pd
import re
import os

# =====================================
# NOISE PATTERNS
# =====================================

noise_patterns = [

    r'\bminimal ipk\b',
    r'\bipk minimal\b',
    r'\bgpa minimum\b',
    r'\bgpa minimal\b',
    r'\bminimum gpa\b',
    r'\bgpa\s*>\s*\d',
    r'\bipk\s*>\s*\d',

    r'\bfresh graduate\b',
    r'\bfresh graduates\b',
    r'\blulusan baru\b',

    r'\bpendidikan minimal\b',
    r'\bminimum pendidikan\b',
    r'\bminimal pendidikan\b',

    r'\busia maksimal\b',
    r'\busia minimum\b',
    r'\bumur maksimal\b',
    r'\bumur minimum\b',

    r'\bberdomisili\b',
    r'\bdomisili\b',

    r'\bsim a\b',
    r'\bsim b\b',
    r'\bsim c\b',

    r'\bmemiliki kendaraan pribadi\b',
    r'\bsepeda motor pribadi\b',

    r'\bvaksin\b',
    r'\bvaccinated\b',
    r'\bfully vaccinated\b',

    r'\bnot allergic\b',
    r'\btidak alergi\b',

    r'\bmale only\b',
    r'\bfemale only\b',

    r'\bminimum \d+ years\b',
    r'\bminimum \d+ year\b',

    r'^\d+\s*years?$',
    r'^\d+\+\s*years?$',

    r'\babout us\b',
    r'\babout the company\b',
    r'\bwho we are\b',

    r'\bour company\b',
    r'\bour culture\b',
    r'\bcompany culture\b',

    r'\bour mission\b',
    r'\bour vision\b',
    r'\bour values\b',

    r'\bwe take pride\b',
    r'\bwe believe\b',
    r'\bwe are committed\b',

    r'\bjoin our team\b',
    r'\bjoin us\b',

    r'\bcareer development\b',
    r'\bcareer growth\b',

    r'\bworking environment\b',
    r'\bwork environment\b',

    r'\bemployee wellbeing\b',
    r'\bemployee engagement\b',

    r'\bcompany perks\b',
    r'\bemployee benefits\b',
    r'\bcompetitive compensation\b',

    r'\bhybrid working\b',
    r'\bhybrid work\b',

    r'\bflexible working hours\b',

    r'\bwork-life balance\b',

    r'\bpaid leave\b',
    r'\bunpaid leave\b',

    r'\bhealth insurance\b',
    r'\bemployment insurance\b',

    r'\bbreakfast and lunch\b',
    r'\bcoffee time\b',
    r'\bhappy hours\b',

    r'\bsharing session\b',
    r'\bpeer learning\b',
    r'\bwebinar\b',

    r'\brecruitment process\b',
    r'\brecruitment pipeline\b',

    r'\bapplication review\b',

    r'\bpre-assessment test\b',
    r'\bassessment test\b',

    r'\bhr interview\b',
    r'\buser interview\b',
    r'\bfinal interview\b',

    r'\bjob offer\b',
    r'\boffering\b',

    r'\bselection process\b',

    r'\bshort-listed candidates\b',
    r'\bshortlisted candidates\b',

    r'\bapply now\b',
    r'\bapply online\b',

    r'\bconfidential\b',
    r'\bconfidentiality\b',

    r'\bdisclaimer\b',

    r'\bprivacy policy\b',

    r'\bintended recipient\b',

    r'\bmust not use\b',
    r'\bmust not disseminate\b',

    r'\bcopyright\b',

    r'\breserves the right\b',

    r'\blegal action\b',

    r'\bimmigration sponsorship\b',
    r'\bsponsorship\b',

    r'\bnot provide sponsorship\b',
    r'error\s*500',
    r'server error',

    r'that.?s an error',

    r'powered by',

    r'login logout',

    r'cookie policy',
    r'this website uses cookies',

    r'click here',

    r'read more',

    r'copy link to share',

    r'updated\s*\d{4}-\d{2}-\d{2}',

    # =====================================================
    # COMPANY PROFILE / PROMOTIONAL TEXT
    # =====================================================

    r'.*private island.*',
    r'.*resort.*',
    r'.*hospitality.*',
    r'.*south china sea.*',
    r'.*find serenity.*',
    r'.*another private island.*',
    r'.*part of something special and iconic.*',
    r'.*favourite beverages and food brands.*',
    r'.*all with full ownership and visibility.*',
    r'.*clear niche.*restaurant.*retail.*only.*',

    # ==================================================
    # COMPANY PROFILE
    # ==================================================
    r"\bis a global\b",
    r"\bis part of\b",
    r"\bsubsidiary of\b",
    r"\boperating program\b",
    r"\bscience-led\b",
    r"\bcompany\b",
    r"\borganization\b",
    r"\bnonprofit\b",
    r"\bglobal luxury group\b",
    r"\bheadquartered\b",
    r"\bfounded\b",
    r"\bestablished\b",

    # =====================================================
    # EEO / LEGAL DISCLAIMER
    # =====================================================

    r'.*equal opportunity.*',
    r'.*does not discriminate.*',
    r'.*all qualified applicants.*',
    r'.*protected veteran.*',
    r'.*gender identity.*',
    r'.*sexual orientation.*',
    r'.*criminal background.*',
    r'.*criminal record.*',
    r'.*citizenship status.*',
    r'.*marital status.*',
    r'.*veteran status.*',
    r'.*national origin.*',
    r'.*race.*religion.*sex.*',
    r'.*inclusive environment.*',

    # =====================================================
    # IMMIGRATION / RELOCATION
    # =====================================================

    r'.*visa.*immigration.*',
    r'.*immigration sponsorship.*',
    r'.*relocation support.*',

    # =====================================================
    # APPLICATION FORM / WEBSITE ARTIFACTS
    # =====================================================

    r'.*copy the link.*',
    r'.*or copy the link.*',
    r'.*visit linkedin.*',
    r'.*social network.*web links.*',
    r'.*role summary.*',
    r'.*no file chosen.*',
    r'.*pdf.*\.rar.*\.zip.*',
    r'.*2 mb max.*',
    r'.*resume.*',

    # =====================================================
    # PHYSICAL REQUIREMENT
    # =====================================================

    r'.*minimum height.*',
    r'.*proportional weight.*',
    r'.*ideal body weight.*',
    r'.*lift.*50 pounds.*',
    r'.*lifting and carrying.*',
    r'.*attractive.*',
    r'.*well groomed.*',
    r'.*berpenampilan menarik.*',
    r'.*penampilan yang terawat.*',

    # ==================================================
    # AGE / GENDER
    # ==================================================
    r"years? old",
    r"age\s*:?",
    r"age under",
    r"age max",
    r"male",
    r"female",
    r"men are preferred",
    r"women",
    r"gender",
    r"single",
    r"never been married",
    r"regardless of gender",
    r"\b\d{1,2}\s*years?\s*old\b",
    r"\bage\s*:?\s*\d+",
    r"\bage\s*\d+\s*-\s*\d+",

    # =====================================================
    # HEALTH RESTRICTION
    # =====================================================

    r'.*not color blind.*',
    r'.*not tattooed.*',
    r'.*not pierced.*',
    r'.*dust allergy.*',
    r'.*vision problems.*',
    r'.*covid-19 vaccinated.*',
    r'.*color differentiation.*',
    r'.*differentiat.*color.*',
    r'.*ketajaman penglihatan.*',
    r'.*vision acuity.*',
    r'.*membedakan warna.*',
    r'.*dust allergy.*',
    r'.*riwayat penyakit pernafasan.*',
    r'.*bath temperature.*',
    r'.*suhu mandi.*',

    # =====================================================
    # RECRUITMENT / ELIGIBILITY
    # =====================================================

    r'.*family currently works.*bank mandiri.*',
    r'.*no issues in working for financial services.*',
    r'.*no-work-no-pay.*',

    # =====================================================
    # MACHINE TRANSLATION ARTIFACTS
    # =====================================================

    r'^only human\.?$',
    r'^pathways into\.?$',
    r'^\s*and\s*$',
    r'^\s*dan\s*$',
    r'.*\d{4,}.*',
    r'.*welsh language.*',
    r'.*speak.*welsh.*',
    r'.*interview.*welsh.*',
    r'.*application.*welsh.*',
    r'.*submitted in english.*',
    r'.*submitted in welsh.*',
    r'.*bilingual institution.*',
    r'.*language strategy.*',
    r'.*welsh speakers.*',
    r'.*role holder.*welsh.*',

    # Recruitment / website artifact

    r'.*submitting file to portal.*',
    r'.*portal website.*',

    r'.*open the link.*',
    r'.*link cannot be opened.*',

    r'.*karir.*',
    r'.*career page.*',

    r'.*please.*delete.*',

    r'.*\(if the link cannot be opened.*',

    r'^additional information$',

    r'.*clear reproduction steps.*',

    # =====================================================
    # EMPLOYER BRANDING / COMPANY STORY
    # =====================================================

    r'.*exceptional ey experience.*',
    r'.*whenever you join.*',
    r'.*however long you stay.*',
    r'.*lasts a lifetime.*',
    r'.*designed to last a lifetime.*',

    r'.*our story began.*',
    r'.*our founders.*',
    r'.*road trip.*',
    r'.*motel was born.*',

    # =====================================================
    # TOURISM / RESORT PROMOTIONAL TEXT
    # =====================================================

    r'.*find serenity.*',
    r'.*hidden coastal enclaves.*',
    r'.*nature.?s allure.*',
    r'.*therapeutic retreat.*',

    # =====================================================
    # BAR / LIFESTYLE MARKETING
    # =====================================================

    r'.*world-class experimental cocktails.*',
    r'.*mobster syndicate.*',
    r'.*untold stories.*',
    r'.*transport you back in time.*',
    r'.*glamour.*decadence.*underworld.*',

    # =====================================================
    # LOCATION / PLACEMENT
    # =====================================================

    r'.*location:.*',
    r'.*implementation location.*',
    r'.*placement location.*',

    r'.*head office.*',
    r'.*jobsite.*',

    r'.*penempatan.*',
    r'.*placement.*',

    r'.*working in.*',
    r'.*ready to work in.*',

    r'.*jakarta.*',
    r'.*cikarang.*',
    r'.*sunter.*',
    r'.*denpasar.*',
    r'.*tangerang.*',
    r'.*kalimantan.*',

    # ==================================================
    # LOCATION
    # ==================================================
    r"sumatera",
    r"jakarta",
    r"batam",
    r"china",
    r"japan",
    r"korea",
    r"southeast asia",
    r"tokyo",
    r"singapore",

    # ==================================================
    # PLACEMENT / TRAVEL
    # ==================================================
    r"willing to be based",
    r"willing to travel",
    r"travel overseas",
    r"based overseas",
    r"multiple sites",
    r"travel to",
    r"placed at",
    r"placement",
    r"relocation",

    # =====================================================
    # EMPLOYER BRANDING
    # =====================================================

    r'.*local at heart.*',
    r'.*local economies.*',
    r'.*communities we love.*',

    r'.*world-class sales.*',
    r'.*manufacturing capabilities.*',

    r'.*unrivalled relationships.*',
    r'.*favourite brands.*',

    # =====================================================
    # RECRUITMENT CTA
    # =====================================================

    r'.*if you meet the requirement.*',
    r'.*open positions.*',
    r'.*we are looking for.*',
    r'.*we.?re looking for.*',
    r'.*we are seeking.*',
    r'.*currently seeking.*',
    r'.*currently looking for.*',
    r'.*what we.?re looking for.*',
    r'.*who we.?re looking for.*',
    r'.*look for the following.*',
    r'.*acquisition news.*',
    r'.*looking for you.*',
    r'^looking for$',

    # =====================================================
    # CAREER PROMOTION
    # =====================================================

    r'.*brighter career future.*',
    r'.*not just offer.*job.*',

    # =====================================
    # MOBILITY / PLACEMENT
    # =====================================

    r'.*able to be placed anywhere.*',
    r'.*hybrid or onsite arrangement.*',

    # =====================================
    # VEHICLE REQUIREMENT
    # =====================================

    r'.*having motorcycle.*',
    r'.*driver license.*',
    r'.*drive a car.*preferred.*',

    # =====================================
    # BROKEN TRANSLATION FRAGMENTS
    # =====================================

    r'^driving innovation$',
    r'.*essential for driving$',
    r'.*also for driving$',

    r'.*global leader.*',
    r'.*premium labels.*',
    r'.*competitive markets.*',
    r'.*consumer experiences.*',
    r'.*over a century.*printing experience.*',
    r'.*consumer-driven innovations.*',
    r'.*sustainable packaging solutions.*',
    r'.*amea amea.*',
    r'.*is a premium.*company.*',
    r'.*serving local manufacturers.*',

    # =====================================
    # RECRUITMENT TEMPLATE
    # =====================================

    r'^personal attributes:?$',
    r'^atribut pribadi:?$',

    r'^ideal profile:?$',
    r'^profil ideal:?$',

    r'^grade point average:?$',
    r'^nilai rata-rata:?$',

    # =====================================
    # APPLICATION FORM ARTIFACT
    # =====================================

    r'.*tahu informasi loker dari.*',
    r'.*scan qris.*',
    r'.*psikotes.*',

    # =====================================
    # EMPLOYER BRANDING
    # =====================================

    r'.*public recognition.*',
    r'.*track your achievements.*',
    r'.*build a personal portfolio.*',

    # =====================================================
    # EMPLOYER BRANDING / CULTURE TEXT
    # =====================================================

    r'.*you will not be able to enjoy this work.*',
    r'.*doing it the way.*always done it.*',
    r'.*you don.?t just meet deadlines.*',
    r'.*not everyone can work like you.*',
    r'.*you won.?t be a cog in the machine.*',
    r'.*much more than just a job.*',
    r'.*how did we manage to do that.*',
    r'.*competitors cannot keep up.*',
    r'.*but that is not right.*',

    # ==================================================
    # COMPANY CULTURE
    # ==================================================
    r"we have graduates",
    r"time flies beautifully",
    r"we don.?t just",
    r"our people",
    r"our culture",
    r"our values",
    r"join our team",

    # =====================================================
    # INFORMAL COMPANY CULTURE
    # =====================================================

    r'.*among us.*',
    r'.*avalon.*',
    r'.*loyal servant of the arthur.*',
    r'.*script running.*campaign launching.*',

    # =====================================================
    # LEGAL DISCLAIMER
    # =====================================================

    r'.*shall not be liable.*',
    r'.*loss, damage or consequences.*',
    r'.*communication or offer of employment.*',

    # =====================================================
    # RECRUITMENT FILLER
    # =====================================================

    r'.*this role is not.*',
    r'.*peran ini tidak diperuntukkan.*',

    # =====================================================
    # PLUS VALUE / PREFERENCE FRAGMENTS
    # =====================================================

    r'^is a plus\.?$',
    r'^adalah nilai tambah\.?$',
    r'^\(nilai tambah\)$',
    r'^\(value added\)$',
    r'^be added value\.?$',

    # =====================================================
    # NON-PROFESSIONAL INTEREST
    # =====================================================

    r'.*kpopers.*',

    # Hobi
    r'.*enjoy reading books.*',

    # Benefit perusahaan
    r'.*coffee and cake.*',
    r'.*specialty coffee.*',
    r'.*freshly baked cookies.*',
    r'.*bpjs.*',
    r'.*health benefit.*',
    r'.*insurance benefit.*',
    r'.*基本給.*',
    r'.*円.*',
    r'.*\d{3,},\d+円.*',
    r'.*salary.*',
    r'.*basic salary.*',

    # Preferensi pribadi
    r'.*love dog.*',
    r'.*love cat.*',

    # Exclusion statement
    r'.*not illustration-focused.*',
    r'.*no 3d.*mold design.*',
    r'.*general construction-only profiles.*',
    r'.*will not be considered.*',

    # =====================================================
    # EDUCATION DOCUMENTS
    # =====================================================

    r'.*transcript of final grades.*',
    r'.*transkrip universitas.*',
    r'.*latest university transcript.*',
    r'.*proof of education.*',
    r'.*bukti pendidikan.*',
    r'.*verifiable documentation.*',
    r'.*supporting evidence.*',
    r'.*tangkapan layar.*',
    r'.*audio recordings.*',
    r'.*screenshots.*',
    r'.*videos.*',
    r'.*logs.*',
    r'.*photo 3x4.*',
    r'.*foto 3x4.*',
    r'.*pas photo.*',

    # =====================================================
    # APPLICATION FORM ARTIFACT
    # =====================================================

    r'.*fill in glints.*',
    r'.*nama group loker.*',
    r'.*position name.*',

    # =====================================================
    # RECRUITMENT PROGRAM
    # =====================================================

    r'.*how to apply.*',
    r'.*cara melamar.*',
    r'.*provided upon joining.*',
    r'.*upon joining.*',
    r'.*1 hour course.*',
    r'.*completing a.*course.*',

    r'.*service bond.*',
    r'.*ikatan dinas.*',
    r'.*intensive training.*',

    # ==================================================
    # RECRUITMENT CTA
    # ==================================================
    r"if the answer is yes",
    r"please read",
    r"simply send us",
    r"apply now",
    r"send your cv",
    r"submit your application",

    # =====================================================
    # JOB OPENING INTRO
    # =====================================================

    r'.*is seeking a.*',
    r'.*sedang mencari.*',
    r'.*looking for.*',
    r'.*motivated intern.*',
    r'.*several openings are available.*',
    r'.*multiple domains and levels.*',

    # =====================================================
    # APPLICATION DEADLINE
    # =====================================================

    r'.*deadline.*application.*',
    r'.*deadline for receiving applications.*',
    r'.*batas waktu penerimaan lamaran.*',
    r'.*applications must be submitted by.*',
    r'.*permohonan harus diserahkan.*',
    r'.*latest submission.*',
    r'.*pengiriman terakhir.*',
    r'.*paling lambat.*',
    r'.*selambat-lambatnya.*',

    # =====================================================
    # DATE / PERIOD
    # =====================================================

    r'.*period:.*',
    r'.*periode:.*',

    r'.*\b(january|february|march|april|may|june|july|august|september|october|november|december)\b.*',

    r'.*\b(januari|februari|maret|april|mei|juni|juli|agustus|september|oktober|november|desember)\b.*',

    r'.*\b20\d{2}\b.*',

    r'.*\d{1,2}/\d{1,2}/\d{4}.*',

    # =====================================================
    # AVAILABILITY / JOINING DATE
    # =====================================================

    r'.*availability asap.*',
    r'.*immediate availability.*',
    r'.*expected start date.*',
    r'.*start date.*',
    r'.*join immediately.*',
    r'.*immediate joiners preferred.*',
    r'.*available to start.*',
    r'.*able to join.*',
    r'.*dapat segera bergabung.*',
    r'.*bisa bergabung.*',
    r'.*ketersediaan segera.*',
    r'.*ketersediaan secepatnya.*',

    # =====================================================
    # POSITION OPENING PERIOD
    # =====================================================

    r'.*position will be open.*',
    r'.*applications accepted on an ongoing basis.*',
    r'.*until the position is filled.*',

    # =====================================================
    # COMPANY EXPANSION
    # =====================================================

    r'.*will open under.*brand.*',
    r'.*hotel properties.*will open.*',

    # =====================================================
    # NARRATIVE MARKETING
    # =====================================================

    r'.*this is just the beginning.*',
    r'.*ini baru permulaan.*',
    r'.*sound interesting.*',
    r'.*does this sound like you.*',
    r'.*sounds like you.*',
    r'.*have a question.*',
    r'.*do you have any questions.*',
    r'.*let us know if.*',

    # =====================================================
    # EXPERIENCE REQUIREMENT
    # =====================================================

    r'.*minimum \d+.*years of experience.*',
    r'.*pengalaman minimal.*tahun.*',
    r'.*minimal \d+.*tahun.*',
    r'.*more than \d+ years of experience.*',
    r'.*lebih dari \d+ tahun pengalaman.*',
    r"minimum of .* years",
    r"at least .* years",
    r"\d+\+?\s*years? experience",
    r"\d+\s*years? experience",
    r"minimum experience",
    r"work experience",
    r"experience in",
    r"strong experience",
    r"experience with",
    r"\b\d+\+?\s*years?\b",
    r"\bat least\b.*\byears?\b",

    # =====================================================
    # FOOTWEAR INDUSTRY FALSE POSITIVE
    # =====================================================

    r'.*shoe development.*',
    r'.*finished shoes.*',
    r'.*quality of sample.*',

    # =====================================================
    # BOOTCAMP PROMOTION
    # =====================================================

    r'.*opening.*bootcamp.*program.*',
    r'.*through this bootcamp.*',
    r'.*labs bootcamp.*',
    r'.*twenty-week program.*',
    r'.*program dua puluh minggu.*',
    r'.*young talents.*',
    r'.*working in a global organisation.*',
    r'.*bootcamp.*is a plus.*',
    r'.*participated in.*bootcamp.*',

    # =====================================================
    # GPA / IPK
    # =====================================================

    r'.*gpa.*',
    r'.*ipk.*',
    r'.*min gpa.*',
    r'.*minimum gpa.*',
    r'.*ipk min.*',
    r'.*gpa of.*',
    r'.*gpa.*',
    r'.*ipk.*',

    # =====================================================
    # SINGLE TOKEN ARTIFACTS
    # =====================================================

    r'^\d+\.\d+$',
    r'^minimum$',
    r'^minimum\.$',

    # EEO / DIVERSITY / INCLUSION

    r'.*equal opportunity.*',
    r'.*equal-opportunity employer.*',
    r'.*inclusive employer.*',
    r'.*diversity.*inclusion.*',
    r'.*equality.*diversity.*',
    r'.*does not discriminate.*',
    r'.*qualified applicants.*',
    r'.*without regard to.*',
    r'.*protected status.*',
    r'.*protected characteristic.*',

    r'.*gender identity.*',
    r'.*gender expression.*',
    r'.*sexual orientation.*',
    r'.*citizenship status.*',
    r'.*marital status.*',
    r'.*veteran status.*',
    r'.*national origin.*',
    r'.*ethnicity.*',
    r'.*social class.*',
    r'.*genetic information.*',

    r'.*male applicants.*',
    r'.*preference.*male.*',
    r'.*male\/female.*',
    r'.*female\/male.*',
    r'.*married women.*',
    r'.*mommies.*',

    r'©\s*\d{4}.*',

    # =====================================================
    # SOCIAL MEDIA FOOTER
    # =====================================================

    r'.*follow .* on facebook.*',
    r'.*follow .* on instagram.*',
    r'.*ikuti .* di facebook.*',
    r'.*ikuti .* di instagram.*',

    r'.*find us on.*instagram.*',
    r'.*website and instagram.*',

    # ==================================================
    # SOCIAL MEDIA / WEBSITE
    # ==================================================
    r"linkedin",
    r"facebook",
    r"instagram",
    r"whatsapp",
    r"copy link",
    r"share",
    r"print",
    r"read more",
    r"click here",

    # =====================================================
    # LICENSE / ADMINISTRATIVE REQUIREMENT
    # =====================================================

    r'.*str aktif.*',
    r'.*bersedia membuat sip.*',

    # =====================================================
    # RECRUITMENT SENTENCE
    # =====================================================

    r'.*is seeking a skilled.*',
    r'.*mencari tenaga terampil.*',

    # =====================================================
    # MACHINE TRANSLATION ARTIFACT
    # =====================================================

    r'.*high wycombe.*',

    # Remote / work arrangement
    r'\bfully remote\b',
    r'\b100%\s*remote\b',
    r'\bremote type\b',
    r'\bthis position is fully remote\b',
    r'\bwe are 100%\s*remote\b',
    r'^\s*remote\s*$',
    r'^\s*terpencil\s*$',
    r'.*willing to move to.*',
    r'.*bersedia pindah ke.*',
    r'.*relocate to.*',

    # Recruitment contact
    r'.*contact whatsapp recruitment.*',
    r'.*hubungi whatsapp recruitment.*',
    r'.*for further information.*contact.*',
    r'.*whatsapp recruitment.*',
    r'.*08\d+.*',

    # CV / Resume
    r'\bcv\b',
    r'\bcurriculum vitae\b',
    r'\bcover letter\b',
    r'\bsurat pengantar\b',
    r'\bcv screening\b',
    r'\bpenyaringan cv\b',
    r'\bupload your cv\b',
    r'\bsend your cv\b',

    # Application process
    r'\bsubmit your application\b',
    r'\bapplication form\b',
    r'\bformulir aplikasi\b',
    r'\bregistration page\b',
    r'\bhalaman pendaftaran\b',

    # Recruitment contact
    r'\bplease send your cv\b',
    r'\bsend your cv to\b',
    r'\bemail protected\b',

    # Administrative artifacts
    r'.*missing information.*',
    r'.*information.*corrections.*',

    r'.*workstation labels?.*',
    r'.*desktops?/laptops? labels?.*',
    r'.*office assets?.*',

    r'.*mobile numbers?.*',
    r'.*desk extension lists?.*',

    r'.*engagement letters?.*',
    r'.*updating all promotional materials.*',
    r'.*update booking systems?.*',

    r'.*transport canada.*license.*',
    r'.*ame license.*',
    r'.*legally work in canada.*',
    r'.*valid.*license required.*',

    r'.*joint venture partnership.*',
    r'.*life insurance company.*',

    r'.*120 years of experience.*',

    r'.*fitch.*',
    r'.*standard & poor.*',

    r'.*best ceo.*',

    r'.*international business networks.*',

    r'.*sehat jasmani dan rohani.*',
    r'.*health certificate.*',
    r'.*surat sehat.*',
    r'.*physically and mentally healthy.*',
    r'.*discover alam sutera.*',
    r'.*best town for cycling.*',
    r'.*expired.*',

    r'.*pocket money.*',
    r'.*uang saku.*',

    r'.*accommodation.*',
    r'.*penginapan.*',

    r'.*standby during idul fitri.*',
    r'.*libur idul fitri.*',

    # ==================================================
    # EDUCATION
    # ==================================================
    r"minimum bachelor's",
    r"minimum bachelor",
    r"bachelor'?s degree",
    r"diploma'?s degree",
    r"minimum diploma",
    r"graduates? from",
    r"graduated in",
    r"degree in any major",
    r"degree in",
    r"major in",
    r"education degree",
    r"\bbachelor\b",
    r"\bdiploma\b",
    r"\bs1\b",
    r"\bd3\b",
    r"\bd4\b",
    r"\bmaster'?s\b",
    r"\bphd\b",
    r"\buniversity\b",
    r"\bacademy\b",
    r"\bminimum\b.*\bdegree\b",
    r"\bgraduate[sd]?\b",

    # ==================================================
    # SALARY / BENEFIT
    # ==================================================
    r"salary",
    r"allowance",
    r"benefits?",
    r"insurance",
    r"bonus",
    r"transportation",
    r"vacation",
    r"leave",
    r"holiday",
    r"compensation",
    r"yen",
  ]

# =====================================
# COMBINE REGEX
# =====================================

combined_pattern = "|".join(noise_patterns)

# =====================================
# FILTER COLUMN
# =====================================

df_split["competency_embedding_text"] = (
    df_split["competency_embedding_text"]
    .fillna("")
    .astype(str)
)

# =====================================
# REMOVE ROWS CONTAINING NOISE
# =====================================

before = len(df_split)

df_clean_again = df_split[
    ~df_split["competency_embedding_text"]
        .str.lower()
        .str.contains(
            combined_pattern,
            regex=True,
            na=False
        )
].copy()

after = len(df_clean_again)

print(f"Before : {before}")
print(f"After  : {after}")
print(f"Removed: {before-after}")

# =====================================================
# REMOVE DUPLICATE COMPETENCIES
# =====================================================

print("Before deduplication:", len(df_clean_again))

df_competency_unique = (

    df_clean_again[
        [
            'competency_unit',
            'competency_unit_id',
            'competency_unit_en',
            'competency_embedding_text'
        ]
    ]
    .dropna(
        subset=['competency_embedding_text']
    )
    .drop_duplicates(
        subset=['competency_embedding_text']
    )
    .reset_index(drop=True)

)

print("After deduplication:", len(df_competency_unique))

display(
    df_competency_unique.sample(
        20,
        random_state=42
    )
)
print(df_competency_unique.shape)


# =====================================
# SAVE
# =====================================

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Path file output
output_file = os.path.join(
    OUTPUT_FOLDER_INDUSTRY,
    "industry_final.csv"
)

# Save
df_competency_unique.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {output_file}")


/tmp/ipykernel_8876/2523887278.py:1062: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Before : 26033
After  : 18240
Removed: 7793
Before deduplication: 18240
After deduplication: 14650


,competency_unit,competency_unit_id,competency_unit_en,competency_embedding_text
6489,mengoordinasikan program dan kegiatan antarbag...,mengoordinasikan program dan kegiatan antarbag...,coordinating programs and activities between H...,mengoordinasikan program dan kegiatan antarbag...
13874,technical engineer,insinyur teknik,technical engineer,insinyur teknik technical engineer
14407,menjaga keandalan dan kontinuitas layanan ti u...,menjaga keandalan dan kontinuitas layanan ti u...,maintain reliability and continuity of IT serv...,menjaga keandalan dan kontinuitas layanan ti u...
7855,mampu menjaga jaringan (lan & wan),mampu memelihara jaringan (lan & wan),capable of maintaining the network (lan & wan),mampu memelihara jaringan (lan & wan) capable ...
4696,able to work effectively and result-oriented w...,mampu bekerja secara efektif dan berorientasi ...,able to work effectively and result-oriented w...,mampu bekerja secara efektif dan berorientasi ...
11701,strong analytical skills,kemampuan analitis yang kuat,strong analytical skills,kemampuan analitis yang kuat strong analytical...
13597,mahir menggunakan tools desain perangkat lunak...,mahir menggunakan tools desain perangkat lunak...,Proficient in using software design and system...,mahir menggunakan tools desain perangkat lunak...
14609,menganalisis metrik performa video dan berkola...,menganalisis metrik performa video dan berkola...,analyze video performance metrics and collabor...,menganalisis metrik performa video dan berkola...
9647,familiar with data processing tools,akrab dengan alat pemrosesan data,familiar with data processing tools,akrab dengan alat pemrosesan data familiar wit...
2796,support and partner with teams across the ente...,mendukung dan bermitra dengan tim di seluruh p...,support and partner with teams across the ente...,mendukung dan bermitra dengan tim di seluruh p...


(14650, 4)
Saved: /content/processed_data/industry_data/industry_final.csv


In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_INDUSTRY = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_INDUSTRY}")

# Define the file to copy
filename_to_copy = 'industry_final.csv'
source_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, filename_to_copy)
destination_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, filename_to_copy)

try:
    shutil.copy2(source_filepath, destination_filepath)
    print(f"Successfully copied '{filename_to_copy}' to '{destination_filepath}'.")
except FileNotFoundError:
    print(f"Error: Source file '{filename_to_copy}' not found at '{source_filepath}'.")
except Exception as e:
    print(f"An error occurred while copying '{filename_to_copy}': {e}")

Mounted at /content/drive
Created Google Drive directory: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/
Successfully copied 'industry_final.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/industry_final.csv'.


## Linguistic Processing (Tokenization, Stopwords, Lemmetization)

In [ ]:
import sys

# Install wordninja library
!{sys.executable} -m pip install wordninja

print("wordninja installation complete.")

In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords, words # Import 'words' corpus
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import sys # Import sys for sys.exit()
import os # Import os for path manipulation
import re # Import re for regex operations
import wordninja # Import wordninja for word splitting

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Download necessary NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) # Explicitly download 'punkt_tab' as suggested by the traceback
nltk.download('averaged_perceptron_tagger', quiet=True) # Good practice to include for lemmatization context if needed
nltk.download('words', quiet=True) # Download 'words' corpus for English vocabulary check

# Load wordninja dictionary (this might take a moment)
print("Loading wordninja dictionary for word splitting...")
# wordninja automatically loads its default dictionary upon first use of split()
# We can call it once to ensure it's loaded and ready, or let it lazily load.
# For now, we'll assume the import is sufficient and the first split call will handle it.

# 1. Load industry_demand_cleaned.csv
try:
    input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_translated.csv')
    df = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully from '{OUTPUT_FOLDER_INDUSTRY}'.")
except FileNotFoundError as e:
    print(f"Error: Make sure '{os.path.basename(input_filepath)}' is in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    sys.exit(1) # Use sys.exit() to ensure termination

# 2. Ensure job_description_clean is treated as text and handle missing values safely.
df['translated_text'] = df['translated_text'].astype(str).fillna('')

# Initialize NLTK components

# Custom stopwords based on common job posting terms that are not skills
custom_stopwords_to_add = {
    # education/degree
    'bachelor', 'master', 'degree', 'phd', 'graduate', 'undergraduate', 'education', 'educational', 'fresh',
    # experience
    'year', 'years', 'minimum', 'school', 'student', 'final', 'major', 'diploma', 'university',
    'gpa',
    # job context
    'job', 'description', 'role', 'position', 'posting', 'vacancy', 'career', 'opportunity', 'recruitment', 'interview', 'payment', 'hired',
    'insurance', 'money', 'time', 'benefit', 'wfo', 'wfh', 'onsite', 'fulltime', 'parttime', 'hybrid', 'provide', 'employment', 'bonus', 'employee',
    'unpaid', 'annual', 'hiring', 'internship',
    # requirement words
    'requirement', 'required', 'qualification', 'responsibility', 'experience', 'portofolio',
    # general words
    'ability', 'able', 'work', 'candidate', 'field', 'related', 'level', 'skill', 'expertise', 'knowledge', 'placed', 'company', 'office',
    'salary', 'deadline',
    # additional noise multi-language
    'aan', 'een', 'voor', 'van', 'met', 'com', 'ask', 'ntt', 'sure', 'anger', 'berne',
    'insta',
    # other common words
    'etc', 'apply', 'detail', 'member', 'internal', 'external', 'player', 'need', 'dan', 'make',
    'party', 'form', 'feature', 'world', 'roll', 'bank', 'jakarta', 'southeast', 'lab', 'facility',
    'brand', 'city', 'java', 'today', 'equity', 'daily', 'high', 'breakfast', 'record', 'weekly',
    'webinar', 'everyone', 'leave', 'hour', 'want', 'anyone', 'live', 'worklife', 'range', 'gender',
    'sexual', 'month', 'cmlabs', 'pleasant', 'participate', 'biaya', 'reach', 'comfort',
    # indo noise words
    'anda', 'mohon', 'tidak', 'akan', 'jika', 'dalam', 'bentuk', 'pihak', 'kami', 'segera', 'apabila',
    'apapun'
}

# Combine NLTK English stopwords with custom stopwords
stop_words_english_base = set(stopwords.words('english'))
stop_words_english = stop_words_english_base.union(custom_stopwords_to_add)

# Define words to explicitly preserve, overriding stopword lists
words_to_preserve = {'must', 'should'}
stop_words_english = stop_words_english.difference(words_to_preserve)

lemmatizer = WordNetLemmatizer()

# Define common tech abbreviations to preserve (updated to include user's exceptions)
tech_abbreviations = {'ai', 'ui', 'it', 'go', 'js', 'ml', 'bi', 'qa', 'os', 'sql', 'python', 'api'}

# English vocabulary for strong filtering (C. OPTIONAL)
english_vocab = set(words.words())

# Indonesian stopwords and patterns for filtering (B. FILTER BAHASA INDONESIA)
indo_stopwords = [
    "yang", "dan", "atau", "pada", "saat", "untuk", "dengan",
    "dari", "ke", "di", "ini", "itu", "adalah"
]

indo_patterns = [
    r'^me', r'^di', r'^ke', r'^ber', r'^per',
    r'kan$', r'an$', r'nya$'
]

# Define the linguistic processing function (renamed to reflect English-only processing)
def process_text_english(text):
    # E. STRUKTUR FINAL PIPELINE
    # 1. Protect phrase ("bahasa indonesia")
    text_preprocessed = text.replace("bahasa indonesia", "bahasa_indonesia_placeholder")

    # Initialize list for tokens that pass early Indonesian filtering
    filtered_indo_tokens = []

    # 2. Lowercase (initial split for early Indo filtering)
    # 3. Indo filtering (stopword + pattern) 🔥
    for word in text_preprocessed.split():
        word_lower = word.lower()

        # If token is an Indonesian stopword, skip it
        if word_lower in indo_stopwords:
            continue

        # If token matches an Indonesian pattern, skip it
        if any(re.match(pattern, word_lower) for pattern in indo_patterns):
            continue

        # If word passes Indo filtering, add to list for further processing
        filtered_indo_tokens.append(word)

    # Rejoin words to process with wordninja
    text_after_indo_filter = ' '.join(filtered_indo_tokens)

    # 4. Word splitting (wordninja, sudah dibatasi)
    processed_words_from_wordninja = []
    for word in text_after_indo_filter.split():
        # B. PERBAIKI WORDNINJA - Tambahkan proteksi untuk kata Indonesia
        # Jika kata cocok dengan indo_patterns, jangan di-split oleh wordninja
        if any(re.match(pattern, word.lower()) for pattern in indo_patterns):
            processed_words_from_wordninja.append(word)
            continue

        # B. PERBAIKI WORDNINJA - Batasi penggunaan wordninja
        # Hanya gunakan wordninja jika panjang kata > 12 dan hanya terdiri dari alfabet
        if len(word) > 12 and word.isalpha():
            split_result = wordninja.split(word)
            # Jika hasil split menghasilkan lebih dari 1 kata dan semua hasil berupa alfabet
            if len(split_result) > 1 and all(s.isalpha() for s in split_result):
                processed_words_from_wordninja.extend(split_result)
            else:
                # Jika tidak, gunakan kata asli
                processed_words_from_wordninja.append(word)
        else:
            processed_words_from_wordninja.append(word)

    # Rejoin words after splitting for further processing (tokenization)
    text_after_splitting = ' '.join(processed_words_from_wordninja)

    # 5. Tokenization
    tokens = word_tokenize(text_after_splitting)

    processed_tokens = []
    for word in tokens:
        # Lowercase token
        word_lower = word.lower()

        # 6. Remove number & symbol
        # Remove numbers (e.g., 'python3' -> 'python').
        word_lower_no_digits = re.sub(r'\d+', '', word_lower)

        # If token is not purely alphabetic after number removal, skip it
        if not word_lower_no_digits.isalpha():
            continue

        # Use the cleaned word for further checks
        cleaned_word = word_lower_no_digits

        # 7. Filter token (length, alpha) & D. TAMBAHKAN FILTER FRAGMENT (ANTI-NOISE)
        # Remove empty tokens or tokens <= 3 chars unless they are tech abbreviations
        if not cleaned_word.strip() or (len(cleaned_word) <= 3 and cleaned_word not in tech_abbreviations):
            continue

        # 8. English vocab filtering (soft)
        # C. PERBAIKI ENGLISH VOCAB FILTERING (SOFT FILTER)
        # Jika kata tidak ada di english_vocab, tetapi terlihat seperti kata Indonesia (sesuai pola), baru skip.
        # Ini memungkinkan technical terms yang tidak ada di english_vocab untuk tetap dipertahankan.
        if cleaned_word not in english_vocab:
            if any(re.match(pattern, cleaned_word) for pattern in indo_patterns):
                continue # Skip if it's not English and looks Indonesian
            # Else: If not in English vocab but doesn't look Indonesian, keep it (might be a valid tech term)

        # 9. Lemmatization
        lemmatized_word = lemmatizer.lemmatize(cleaned_word)

        # 10. Stopword removal
        if lemmatized_word in stop_words_english:
            continue

        processed_tokens.append(lemmatized_word)

    # 11. Join text
    processed_text_final_unrestored = ' '.join(processed_tokens)

    # 12. Restore phrase ("bahasa indonesia")
    final_result = processed_text_final_unrestored.replace("bahasa_indonesia_placeholder", "bahasa indonesia")

    return final_result

print("\nApplying English linguistic processing...")
df['job_description_processed'] = df['translated_text'].apply(process_text_english)

# F. OUTPUT VALIDASI
print("\n--- Contoh Pembersihan Bahasa (F. OUTPUT VALIDASI) ---")
sample_texts = [
    "Ini adalah contoh teks yang akan diproses dengan bahasa indonesia dan Python3.",
    "Saya mengerjakan proyek dengan machine learning dan mengembangkan fitur baru.",
    "Perusahaan ini mencari kandidat yang berpengalaman di bidang data dan juga memiliki kemampuan komunikasi.",
    "Melakukan analisis data untuk mendapatkan insight dari laporan keuangan. Beberapa orang bilang ini susah.",
    "mengerjakan proyek dengan mengelola data", # Test for 'mengerjakan' fragment
    "mengembangkan sistem aplikasi", # Test for 'mengembangkan' fragment
    "sebuah dap gan pro" # Test for short noise tokens
]

# Helper function to demonstrate processing and report token counts
def _process_and_report_example(original_text, processing_func):
    # Tokenize original text for initial count (after protecting phrase, similar to func internal flow)
    temp_original_text_protected = original_text.replace("bahasa indonesia", "bahasa_indonesia_placeholder")
    original_tokens = word_tokenize(temp_original_text_protected.lower()) # Use word_tokenize for more accurate initial count

    processed_text = processing_func(original_text)
    processed_tokens = word_tokenize(processed_text)

    print(f"  Original Text: '{original_text}'")
    print(f"  Processed Text: '{processed_text}'")
    print(f"  Initial Token Count (approx): {len(original_tokens)}")
    print(f"  Processed Token Count: {len(processed_tokens)}")
    print(f"  Tokens Removed (approx): {len(original_tokens) - len(processed_tokens)}\n")

print("Contoh proses pada beberapa kalimat:")
for i, text in enumerate(sample_texts):
    print(f"Contoh {i+1}:")
    _process_and_report_example(text, process_text_english)

# Display example before and after splitting (manual check for some words)
print("\n--- Word Splitting Examples ---")
example_words = ["crossfunctional", "teammanagement", "bigdataengineer", "cloudnative", "fullstackdeveloperperformance"] # Added longer word for new wordninja limit
for word in example_words:
    # Simulate the wordninja logic from process_text_english
    processed_word = []
    if any(re.match(pattern, word.lower()) for pattern in indo_patterns): # Check if protected by Indo patterns
        processed_word.append(word)
    elif len(word) > 12 and word.isalpha(): # New wordninja length limit
        split_result = wordninja.split(word)
        if len(split_result) > 1 and all(s.isalpha() for s in split_result):
            processed_word.extend(split_result)
        else:
            processed_word.append(word)
    else:
        processed_word.append(word)

    print(f"'{word}' before splitting: '{word}', after splitting: '{' '.join(processed_word)}'")

# 7. Keep all original columns unchanged (implicit, as we're adding a new column)

# 8. Display basic validation outputs:
print("\n--- Processed Dataset Summary ---")
print(f"Shape of the processed dataset: {df.shape}")
print("Column names of the processed dataset:")
print(df.columns.tolist())

print("\nFirst 5 rows of job_description_processed:")
display(df[['job_title', 'translated_text', 'job_description_processed']].head())

# 9. Save the processed dataset as industry_demand_processed.csv
output_filepath_processed = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_processed.csv')
df.to_csv(output_filepath_processed, index=False)
print(f"\nProcessed dataset successfully saved to '{output_filepath_processed}'.")

## Bi-Gram Analysis

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import scipy.sparse # To save sparse matrix
import os

# Definisikan path folder output (sesuai dengan setup notebook Anda)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 1. dan 2. Load dataset dari file "industry_demand_processed.csv"
# Pastikan file ini ada di folder yang benar, yaitu OUTPUT_FOLDER_INDUSTRY
input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_processed.csv')
try:
    df_industry = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' berhasil dimuat.")
except FileNotFoundError:
    print(f"Error: File '{os.path.basename(input_filepath)}' tidak ditemukan di '{OUTPUT_FOLDER_INDUSTRY}'.")
    print("Mohon pastikan file hasil proses sebelumnya sudah tersimpan di lokasi tersebut.")
    exit()

# 3. Gunakan kolom teks "job_description_processed"
# Pastikan kolom ini sudah bersih dan siap untuk ekstraksi n-gram.
text_data = df_industry['job_description_processed'].astype(str).fillna('')

print(f"\nTotal dokumen untuk diolah: {len(text_data)}")

# 4. Gunakan CountVectorizer dengan ngram_range=(2,2) untuk bi-gram
# Ini akan mengekstrak hanya pasangan kata berurutan (bi-gram).
vectorizer = CountVectorizer(ngram_range=(2, 2))

# Melakukan proses vectorization (fit dan transform)
print("Melakukan vectorization (ekstraksi bi-gram)...\n")
X_bigrams = vectorizer.fit_transform(text_data)

# 5. Menampilkan hasil:

#    * Daftar bi-gram (feature names)
print("--- Daftar Bi-gram (Feature Names) ---")
feature_names = vectorizer.get_feature_names_out()
print(f"Total bi-gram unik yang terdeteksi: {len(feature_names)}")
print("20 contoh bi-gram pertama:")
print(feature_names[:20]) # Menampilkan 20 bi-gram pertama sebagai contoh

#    * Matriks hasil vectorization
print("\n--- Matriks Hasil Vectorization (Sparse Matrix) ---")
print(f"Shape matriks (dokumen x bi-gram): {X_bigrams.shape}")
print("Matriks ini berbentuk sparse, menunjukkan frekuensi bi-gram di setiap dokumen.")
# HINDARI: display(pd.DataFrame(X_bigrams.toarray(), columns=feature_names).head()) karena memori
# Sebaliknya, tampilkan properti sparse matrix atau slice kecil:
print(f"Jumlah elemen non-nol (total frekuensi bi-gram): {X_bigrams.nnz}")
print("Untuk melihat sebagian matriks (misalnya, 5 baris pertama dan 10 kolom pertama):")
# Contoh melihat bagian kecil dari matriks dense (pilih kolom secara spesifik)
# Ini masih bisa berat jika feature_names terlalu banyak, jadi kita ambil 10 kolom pertama saja.
if len(feature_names) > 10:
    sample_cols_indices = np.random.choice(len(feature_names), 10, replace=False)
    sample_cols = [feature_names[i] for i in sorted(sample_cols_indices)]
    display(pd.DataFrame(X_bigrams[:5, sample_cols_indices].toarray(), columns=sample_cols))
else:
    display(pd.DataFrame(X_bigrams[:5, :].toarray(), columns=feature_names))


#    * 10 bi-gram paling sering muncul
print("\n--- 10 Bi-gram Paling Sering Muncul --- (Sebelum Filtering Panjang Kata)")
# Menjumlahkan frekuensi setiap bi-gram di seluruh dokumen
bigram_counts = X_bigrams.sum(axis=0)

# Membuat list tuple (bi-gram, frekuensi)
bigram_freq_list = [(feature_names[i], bigram_counts[0, i]) for i in range(len(feature_names))]

# Mengurutkan berdasarkan frekuensi secara menurun
bigram_freq_list.sort(key=lambda x: x[1], reverse=True)

# Menampilkan 10 bi-gram teratas sebelum filtering
for bigram, count in bigram_freq_list[:10]:
    print(f"'{bigram}': {int(count)}")

# NEW: Filtering bi-grams berdasarkan panjang kata
filtered_bigram_freq_list = []
for bigram, count in bigram_freq_list:
    words_in_bigram = bigram.split()
    # Pastikan itu adalah bi-gram (terdiri dari 2 kata)
    if len(words_in_bigram) == 2:
        word1, word2 = words_in_bigram
        # Filter bi-gram jika kedua katanya memiliki panjang lebih dari 2 karakter
        if len(word1) > 2 and len(word2) > 2:
            filtered_bigram_freq_list.append((bigram, count))

print("\n--- 20 Bi-gram Paling Sering Muncul (Setelah Filtering Panjang Kata > 2) ---")
# Menampilkan 20 bi-gram teratas setelah filtering
for bigram, count in filtered_bigram_freq_list[:20]:
    print(f"'{bigram}': {int(count)}")

print("\nProses ekstraksi bi-gram selesai. Matriks `X_bigrams` sekarang berisi data Anda dalam bentuk vektor bi-gram.")

# --- NEW: Save bi-gram results --- #
# Save the sparse matrix (document-term matrix)
npy_filename_bigram = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_document_term_matrix.npz')
scipy.sparse.save_npz(npy_filename_bigram, X_bigrams)
print(f"\nBi-gram document-term matrix saved as '{npy_filename_bigram}'.")

# Save the vocabulary (feature names) as a CSV or text file for easy loading
vocabulary_df_bigram = pd.DataFrame(feature_names, columns=['bigram'])
vocabulary_filepath_bigram = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_vocabulary.csv')
vocabulary_df_bigram.to_csv(vocabulary_filepath_bigram, index=False)
print(f"Bi-gram vocabulary saved as '{vocabulary_filepath_bigram}'.")


## N-Gram Analysis

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import scipy.sparse # To save sparse matrix
import os # Import os for path manipulation
import sys # Import sys for sys.exit()

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 1. Load the file industry_demand_processed.csv
try:
    input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'industry_demand_processed.csv')
    df = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully from '{OUTPUT_FOLDER_INDUSTRY}'.")
except FileNotFoundError as e:
    print(f"Error: Make sure '{os.path.basename(input_filepath)}' is in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    sys.exit(1)

# Ensure job_description_processed is string type and handle NaNs safely
df['job_description_processed'] = df['job_description_processed'].astype(str).fillna('')

# --- Stopword filtering logic removed from here as per requirements --- #
# Stopword removal has already been handled in the linguistic preprocessing step.
# Avoid double filtering. The following lines are removed:
# nltk.download('stopwords', quiet=True)
# english_stopwords = set(stopwords.words('english'))
# indonesian_stopwords = set([...])
# combined_stopwords = english_stopwords.union(indonesian_stopwords)

# 2. Use only the job_description_processed column as input text
text_data = df['job_description_processed']

# 3. Apply N-Gram extraction using CountVectorizer
#    ngram_range = (1, 3) (Changed from (1, 5) as per requirements)
#    min_df to remove rare terms (e.g., appear in less than 5 documents)
#    max_df to remove overly frequent terms (e.g., appear in more than 85% of documents) (Changed from 80%)
#    Removed stop_words parameter as per requirements (avoid double filtering).
vectorizer = CountVectorizer(ngram_range=(1, 3), min_df=20, max_df=0.85)

print("\nExtracting N-grams...")
X_ngrams = vectorizer.fit_transform(text_data)

# 4. Generate a vocabulary of extracted n-grams
ngram_vocabulary = vectorizer.get_feature_names_out()

# 5. Display: Total number of extracted n-grams
print(f"Total number of extracted N-grams: {len(ngram_vocabulary)}")

# 6. Display: Top 30 most frequent n-grams
# Sum up the counts for each n-gram across all documents
ngram_counts = X_ngrams.sum(axis=0)

# Create a list of (n-gram, count) tuples
n_gram_freq = [(ngram_vocabulary[i], ngram_counts[0, i]) for i in range(len(ngram_vocabulary))]

# Sort by count in descending order
n_gram_freq.sort(key=lambda x: x[1], reverse=True)

print("\nTop 30 most frequent N-grams (after linguistic preprocessing and CountVectorizer filtering):")
for ngram, count in n_gram_freq[:30]:
    print(f"'{ngram}': {int(count)}") # Convert count to int for cleaner display

# 7. Save the resulting n-gram feature dataset for use in the next TF-IDF step.
# Save the sparse matrix (document-term matrix)
npy_filename = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'ngram_document_term_matrix.npz')
scipy.sparse.save_npz(npy_filename, X_ngrams)
print(f"\nN-gram document-term matrix saved as '{npy_filename}'.")

# Save the vocabulary (feature names) as a CSV or text file for easy loading
vocabulary_df = pd.DataFrame(ngram_vocabulary, columns=['ngram'])
vocabulary_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'ngram_vocabulary.csv')
vocabulary_df.to_csv(vocabulary_filepath, index=False)
print(f"N-gram vocabulary saved as '{vocabulary_filepath}'.")

## Skill Matching & Synonym Unification

In [ ]:
import sys
!{sys.executable} -m pip install spacy
print("spaCy installation complete.")

In [ ]:
import spacy

# Download a small English model. Larger models might provide better NER but are slower.
# 'en_core_web_sm' is a good balance for general entity recognition.
print("Downloading spaCy English model 'en_core_web_sm'...")
try:
    nlp = spacy.load('en_core_web_sm')
    print("spaCy model 'en_core_web_sm' loaded.")
except OSError:
    !{sys.executable} -m spacy download en_core_web_sm
    nlp = spacy.load('en_core_web_sm')
    print("spaCy model 'en_core_web_sm' loaded.")

In [ ]:
import pandas as pd
import re
from collections import defaultdict
import nltk
import scipy.sparse # Needed to load the n-gram matrix
import spacy # Import spacy for advanced text processing
import sys # Import sys for sys.exit()
import os # Import os for path manipulation

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Download necessary NLTK data if not already present
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True) # Good practice for spaCy's POS tagging robustness

# Load spaCy model (ensuring it's loaded within this cell for self-containedness)
print("Downloading spaCy English model 'en_core_web_sm'...")
try:
    nlp = spacy.load('en_core_web_sm')
    print("spaCy model 'en_core_web_sm' loaded.")
except OSError:
    # If model not found, download it
    !{sys.executable} -m spacy download en_core_web_sm
    nlp = spacy.load('en_core_web_sm')
    print("spaCy model 'en_core_web_sm' loaded.")

# 1. Load the N-gram vocabulary and document-term matrix
try:
    # Load from the new output folder
    ngram_vocabulary_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'ngram_vocabulary.csv')
    ngram_document_term_matrix_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'ngram_document_term_matrix.npz')

    ngram_vocabulary_df = pd.read_csv(ngram_vocabulary_filepath)
    # Ensure ngrams are lowercase for consistent lookup
    ngram_vocabulary = ngram_vocabulary_df['ngram'].astype(str).str.lower().tolist()

    X_ngrams = scipy.sparse.load_npz(ngram_document_term_matrix_filepath)

    ngram_count_map = {ngram: count for ngram, count in zip(ngram_vocabulary, X_ngrams.sum(axis=0).tolist()[0])}

    extracted_ngrams_initial = set(ngram_vocabulary) # Renamed for clarity for "before filtering" print

    print(f"N-gram vocabulary loaded. Total unique n-grams (initial from CountVectorizer): {len(extracted_ngrams_initial)}")
except FileNotFoundError as e:
    print(f"Error: Make sure '{os.path.basename(ngram_vocabulary_filepath)}' and '{os.path.basename(ngram_document_term_matrix_filepath)}' are in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    sys.exit(1)

# --- Rule-based Filtering (Modified: 'technical_keywords' filter removed) ---
print("\nApplying rule-based filtering to skill candidates (with 'technical_keywords' filter removed)...")

# Define common tech abbreviations to preserve (not directly used as a filter here, but for context)
tech_abbreviations = {'ai', 'ui', 'it', 'go', 'js', 'ml', 'bi', 'qa', 'os', 'sql','html', 'css'}

# Define conjunctions/prepositions (English) to filter out
function_words = {
    'and', 'or', 'with', 'for', 'to', 'in', 'of', 'at', 'like'
}

# Define requirement/time patterns (regex for terms that might indicate non-skill)
requirement_time_patterns = re.compile(
    r'\b(year|month|minimal|least|degree|required|must|should|experience|fresh|graduate|entry|school|hour|fulltime|parttime|stage|probation|period|duration|opportunity|age|religion)\b',
    re.IGNORECASE
)

# Define ROLE keywords to filter out
role_keywords = {'engineer', 'manager', 'officer', 'leader', 'staff', 'intern', 'director', 'developer', 'analyst', 'specialist', 'architect', 'consultant', 'associate', 'programmer', 'admin', 'supervisor'}

# Define BENEFIT/COMPENSATION keywords to filter out
benefit_keywords = {'health', 'dental', 'insurance', 'benefit', 'leave', 'salary', 'compensation', 'package', 'perk', 'bonus'}

# Define abstract terms for filtering
abstract_terms = {
    'efficiency','effectiveness','insight','value','intent',
    'plan','material','ability','quality','improvement',
    'sustainability','performance'
}

# Define generic terms for reverse-order cleanup
generic_reverse_order_terms = {'business','design','analysis','project','management','material', 'analytics'}

# Weak structural terms for cleanup
weak_terms = {
    'support','plan','intent','vision','concept','tool',
    'related','field','skill','mindset'
}

# Action terms for action-pair removal
action_terms = {
    'design','support'
}

# Education/requirement terms for removal
education_requirement_terms = ['related field', 'degree', 'bachelor', 'minimum']

# --- is_valid_skill_candidate function (MODIFIED: 'technical_keywords' filter removed) ---
def is_valid_skill_candidate(term):
    normalized_term = term.strip().lower()

    # Rule: Buang jika ada angka
    if re.search(r'\d', normalized_term):
        return False

    # Process term with spaCy for POS tags and Named Entities
    doc = nlp(normalized_term)
    words = [token.text for token in doc]
    pos_tags = [token.pos_ for token in doc]
    ents = doc.ents

    # Rule: Hanya bigram / trigram (keep phrases of 2 or 3 words)
    if not (2 <= len(words) <= 3):
        return False

    # Rule: Buang jika ada conjunction/preposition
    if any(word in function_words for word in words):
        return False

    # Rule: Buang jika token pertama verb
    if pos_tags and pos_tags[0] == 'VERB':
        return False

    # Rule: Minimal ada 1 NOUN/PROPN (ensure it's a noun-like phrase)
    noun_propn_count = sum(1 for pos in pos_tags if pos in ['NOUN', 'PROPN'])
    if noun_propn_count == 0:
        return False

    # Rule: Buang Named Entity ORG / GPE (remove organizational or geopolitical entities)
    if any(ent.label_ in ['ORG', 'GPE'] for ent in ents):
        return False

    # Rule: Buang requirement/time pattern
    if requirement_time_patterns.search(normalized_term):
        return False

    # Rule: Buang ROLE / BENEFIT keywords
    if any(word in role_keywords for word in words):
        return False
    if any(word in benefit_keywords for word in words):
        return False

    # Rule: Duplicate token removal (e.g., "data data" is removed)
    if len(set(words)) < len(words):
        return False

    # Rule: Indonesian verb phrase (e.g., "memiliki data") -- REMOVED

    # Rule: Abstract-only removal (e.g., "efficiency improvement")
    if all(word in abstract_terms for word in words):
        return False

    # Rule: Reverse generic 2-word cleanup (e.g., "business data" vs "data business")
    if len(words) == 2 and all(word in generic_reverse_order_terms for word in words):
        return False

    # Rule: Weak structural cleanup (e.g., "support system")
    # The previous technical_keywords check was implicitly removed here.
    if len(words) == 2 and any(word in weak_terms for word in words):
        return False

    # Rule: Action-pair removal (e.g., "design support")
    if len(words) == 2 and all(word in action_terms for word in words):
        return False

    # Rule: Education requirement cleanup (e.g., "bachelor degree")
    if any(pat in normalized_term for pat in education_requirement_terms):
        return False

    # --- START OF MODIFICATION: REMOVE technical_keywords FILTER ---
    # User requested to remove filtering based on 'technical_keywords'.
    # The following block of code, related to 'has_technical_keyword' and 'core_technical_keywords',
    # is intentionally REMOVED from this function as per the new requirements.
    # Therefore, no 'technical_keywords' definition or check is present here.
    # --- END OF MODIFICATION ---

    return True

# Provided synonym dictionary (expanded for consistency with previous outputs)
synonym_dict_raw = {
    "ml": "machine learning",
    "machine learning engineer": "machine learning",
    "ai": "artificial intelligence",
    "artificial intelligence engineer": "artificial intelligence",
    "frontend developer": "frontend",
    "backend developer": "backend",
    "fullstack developer": "fullstack",
    "public speaking skill": "public speaking",
    "team work": "teamwork",
    "problem solving skill": "problem solving",
    # Examples from previous outputs for consistency in mapping
    "modeling data": "data modeling",
    "server linux": "linux server",
    "analysis process": "process analysis",
    "data multiple source": "multiple data source",
    "analysis performance": "performance analysis",
    "mobile web": "web mobile", # Assuming 'web mobile' is the canonical form
    "web frontend": "frontend web",
    "support data": "data support",
    "database mysql": "mysql database",
    "network hardware": "hardware network"
}

# --- Helper functions for Skill Standardization & Normalization ---

# 1. Text normalization (lowercase, remove non-alphabetic except spaces, trim spaces)
def normalize_skill_term(term):
    if not isinstance(term, str):
        term = str(term)
    term = term.lower()
    # Remove non-alphabetic characters (excluding spaces and numbers for skill terms)
    # This is in line with the user's explicit request: "hapus simbol non-alfabet kecuali spasi"
    term = re.sub(r'[^a-z0-9\s]', ' ', term)
    term = re.sub(r'\s+', ' ', term).strip()
    return term

# 2. Tokenization: Splits a string by ',', ';', '/', ' dan '
# This function handles a single input string that might contain multiple skill mentions
# separated by the specified delimiters.
def tokenize_raw_skill_string(input_string):
    if not isinstance(input_string, str):
        input_string = str(input_string)
    # Split by comma (with optional spaces), semicolon (with optional spaces), slash (with optional spaces),
    # or the word "dan" surrounded by spaces (using word boundaries \b for "dan")
    tokens = re.split(r',\s*|;\s*|/\s*', input_string) # Removed 'dan' from split pattern
    # Filter out any empty strings that might result from splitting (e.g., from double delimiters)
    return [token.strip() for token in tokens if token.strip()]

# Prepare the canonical synonym maps once
# `canonical_synonym_map`: Direct lookup from normalized key to normalized canonical value.
# `sorted_word_canonical_map`: Lookup for word-order variations (e.g., "modeling data" vs "data modeling").
canonical_synonym_map = {}
sorted_word_canonical_map = {}

for key_raw, value_raw in synonym_dict_raw.items():
    normalized_key = normalize_skill_term(key_raw)
    normalized_value = normalize_skill_term(value_raw)

    canonical_synonym_map[normalized_key] = normalized_value

    # For word-order variations, sort the words and store the mapping
    sorted_key_words = ' '.join(sorted(normalized_key.split()))
    sorted_value_words = ' '.join(sorted(normalized_value.split()))

    # Ensure the canonical form for sorted words is consistently the normalized value
    sorted_word_canonical_map[sorted_key_words] = normalized_value
    sorted_word_canonical_map[sorted_value_words] = normalized_value # Ensure canonical itself is mapped

# Ensure all canonical values (which might not be direct keys) also map to themselves
# This ensures consistency if a canonical form is encountered directly.
for val in set(canonical_synonym_map.values()):
    if val not in canonical_synonym_map:
        canonical_synonym_map[val] = val
    # Also ensure its sorted form maps to itself
    sorted_val_words = ' '.join(sorted(val.split()))
    if sorted_val_words not in sorted_word_canonical_map:
        sorted_word_canonical_map[sorted_val_words] = val

# Helper function to apply synonym unification to a single token
def unify_single_token(token_string, canonical_map, sorted_map):
    # First, normalize the token string for consistent lookup
    normalized_token = normalize_skill_term(token_string)

    # 1. Direct lookup in `canonical_map` (e.g., "ml" -> "machine learning")
    if normalized_token in canonical_map:
        return canonical_map[normalized_token]

    # 2. Sorted word lookup for word-order variations (e.g., "modeling data" -> "data modeling")
    sorted_words = ' '.join(sorted(normalized_token.split()))
    if sorted_words in sorted_map:
        return sorted_map[sorted_words]

    # If no unification applies, the normalized token itself is the canonical form
    return normalized_token

# --- NEW FUNCTION: Clean Structural Prefixes and Transform Verbs to Nouns (EXISTING) ---

# Structural prefixes to identify and remove
structural_prefixes_list = [
    r'strong ability to\s', r'good ability to\s', r'ability to\s',
    r'strong ability\s', r'good ability\s', r'ability\s',
    r'capability to\s', r'capable of\s'
]

# Create a single regex pattern for all prefixes, anchored to the start of the string
structural_prefixes_pattern = re.compile(f'^({"|".join(structural_prefixes_list)})', re.IGNORECASE)

# Mapping for common verb to noun transformations (expanded for new Layer 2 requirements)
verb_to_noun_mapping = {
    'communicate': 'communication',
    'analyze': 'analysis',
    'develop': 'development',
    'collaborate': 'collaboration',
    'adapt': 'adaptability',
    'lead': 'leadership',
    'manage': 'management',
    'solve': 'problem solving', # Maps 'solve' to 'problem solving' directly
    'present': 'presentation',
    'coordinate': 'coordination',
    'organize': 'organization',
    'think': 'thinking',
    'design': 'design',
    'create': 'creativity',
    'drive': 'drive',
    'excel': 'excellence',
    'execute': 'execution',
    'plan': 'planning',
    'evaluate': 'evaluation',
    'improve': 'improvement',
    'innovate': 'innovation',
    'build': 'building',      # For "build" -> "building" (Layer 2 example)
    'perform': 'performance', # For "perform" -> "performance" (Layer 2 example)
    'identify': 'identification', # For "identify" -> "identification" (Layer 2 example)
    'respond': 'responsiveness' # For "respond" -> "responsiveness" (Layer 2 example)
}

def clean_structural_prefix(skill_term_input):
    original_term = skill_term_input.strip() # Assumed to be lowercased from ngram_vocabulary
    cleaned_term = original_term

    # Try to remove structural prefixes
    match = structural_prefixes_pattern.match(original_term)
    if match:
        prefix_matched = match.group(0)
        core_skill_part = original_term[len(prefix_matched):].strip()

        if not core_skill_part: # If only prefix was present, return original as fallback
            return original_term

        # Process the core skill part to find verb and convert to noun
        doc = nlp(core_skill_part)
        first_word_token = doc[0] if doc else None

        if first_word_token and first_word_token.pos_ == 'VERB':
            base_verb = first_word_token.lemma_
            # Attempt direct mapping to noun. If not found, use the original verb text.
            transformed_part = verb_to_noun_mapping.get(base_verb, first_word_token.text)

            # Reconstruct the term: transformed_part + remaining words (if any)
            remaining_words = ' '.join([token.text for token in doc[1:]])
            final_cleaned_term = transformed_part
            if remaining_words:
                final_cleaned_term += ' ' + remaining_words

            # Debug print before returning
            if original_term != final_cleaned_term:
                print(f"    Cleaned prefix & transformed verb (existing): '{original_term}' -> '{final_cleaned_term}'")
            return final_cleaned_term
        else:
            # If no verb is found as the first word or no specific mapping, just return the core skill part
            # after prefix removal. This satisfies 'minimal buang kata "ability" dan simpan verb-nya saja'.
            if original_term != core_skill_part:
                print(f"    Cleaned prefix (no verb transform, existing): '{original_term}' -> '{core_skill_part}'")
            return core_skill_part

    # If no prefix found, return the original term unchanged
    return original_term

# --- NEW LAYER 1: HANDLE "ABLE" PREFIX ---
def handle_able_prefix(skill_term_input):
    original_term = skill_term_input.strip()
    cleaned_term = original_term

    # Prioritize more specific patterns
    patterns = [
        re.compile(r'^\bbe\s+able\s+to\s+', re.IGNORECASE), # "be able to + verb"
        re.compile(r'^\bable\s+to\s+', re.IGNORECASE),     # "able to + verb"
        re.compile(r'^\bable\s+', re.IGNORECASE)           # "able + verb"
    ]

    for pattern in patterns:
        match = pattern.match(cleaned_term)
        if match:
            cleaned_term = cleaned_term[len(match.group(0)):].strip()
            if original_term != cleaned_term:
                print(f"    Handled 'able' prefix (Layer 1): '{original_term}' -> '{cleaned_term}'")
            return cleaned_term # Apply only the first matching pattern

    if original_term != cleaned_term:
        print(f"    Handled 'able' prefix (Layer 1 - no change): '{original_term}'")
    return cleaned_term

# --- NEW LAYER 2: VERB TO NOUN NORMALIZATION ---
def normalize_verb_to_noun(skill_term_input):
    original_term = skill_term_input.strip()
    doc = nlp(original_term)

    # Rule 1: Single-word skill (if verb, try to convert to noun, else drop)
    if len(doc) == 1:
        token = doc[0]
        if token.pos_ == 'VERB':
            base_verb = token.lemma_
            noun_form = verb_to_noun_mapping.get(base_verb, None)
            if noun_form:
                if original_term != noun_form:
                    print(f"    Normalized single-verb skill (Layer 2): '{original_term}' -> '{noun_form}'")
                return noun_form
            else:
                # Drop skill if no valid noun form found for single verb
                print(f"    Dropped single-verb skill (Layer 2, no clear noun form): '{original_term}'")
                return None # Return None to signal that this skill should be dropped
        # If single word but not a verb, keep as is (e.g., 'python', 'data', 'communication')
        return original_term

    # Rule 2: Multi-word phrase starting with a verb, specifically VERB + NOUN -> NOUN + NOUN_FORM_OF_VERB
    if doc[0].pos_ == 'VERB':
        verb_token = doc[0]
        # Check if the next token is a noun or proper noun, indicating a potential object
        if len(doc) > 1 and doc[1].pos_ in ['NOUN', 'PROPN']:
            base_verb = verb_token.lemma_
            noun_form_of_verb = verb_to_noun_mapping.get(base_verb, None)

            if noun_form_of_verb:
                # Reconstruct as "object phrase + noun_form_of_verb"
                # Take all words after the verb as the object phrase for this transformation
                object_phrase = ' '.join([t.text for t in doc[1:]])
                transformed_phrase = f'{object_phrase} {noun_form_of_verb}'
                if original_term != transformed_phrase:
                    print(f"    Normalized multi-word verb phrase (Layer 2, reordered VERB+NOUN): '{original_term}' -> '{transformed_phrase}'")
                return transformed_phrase
            else:
                # Multi-word verb+noun phrase, but the verb has no noun form in mapping.
                # Return original term to avoid aggressive dropping if it's still a potentially valid skill phrase.
                print(f"    Multi-word verb+noun phrase (Layer 2), first verb has no noun form in mapping. Keeping original: '{original_term}'")
                return original_term
        else:
            # Multi-word phrase starts with verb but not followed by a noun (e.g., 'work collaboratively').
            # This pattern is not covered by VERB + NOUN reordering. Keep original.
            print(f"    Multi-word verb phrase (Layer 2) not VERB+NOUN. Keeping original: '{original_term}'")
            return original_term

    # If the skill term does not start with a verb, keep it as is (e.g., 'data analysis', 'critical thinking').
    return original_term

# --- Main Processing Loop for N-gram Vocabulary (with filtering and structural cleaning) ---
final_unique_skills = set() # To store all unique unified skills
skill_unification_mapping = {} # To store original_ngram -> its chosen canonical form (for the CSV output)

print("\n--- Skill Standardization & Normalization Debug Log (with filtering) ---")
processed_ngrams_count = 0

# Apply filtering first to the raw n-gram vocabulary
filtered_ngram_candidates = [ngram for ngram in ngram_vocabulary if is_valid_skill_candidate(ngram)]
print(f"Total skill candidates after rule-based filtering (excluding technical_keywords filter): {len(filtered_ngram_candidates)}")

for original_ngram_raw in filtered_ngram_candidates: # Iterate over the FILTERED n-grams
    processed_ngrams_count += 1
    # Only print debug info for the first 20 items or every 100th item for brevity
    if processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0):
        print(f"\nOriginal N-gram ({processed_ngrams_count}): '{original_ngram_raw}'")

    # Apply existing structural prefix cleaning (e.g., 'strong ability to communicate')
    cleaned_and_transformed_ngram = clean_structural_prefix(original_ngram_raw)
    if (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)) and (original_ngram_raw != cleaned_and_transformed_ngram):
        print(f"  After Existing Structural Prefix Cleaning: '{original_ngram_raw}' -> '{cleaned_and_transformed_ngram}'")
    elif (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)):
        print(f"  After Existing Structural Prefix Cleaning: (No change) '{cleaned_and_transformed_ngram}'")

    # --- Apply NEW LAYER 1: Handle 'Able' Prefix ---
    able_cleaned_ngram = handle_able_prefix(cleaned_and_transformed_ngram)
    if (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)) and (cleaned_and_transformed_ngram != able_cleaned_ngram):
        print(f"  After NEW Layer 1 (Able Prefix Handling): '{cleaned_and_transformed_ngram}' -> '{able_cleaned_ngram}'")
    elif (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)):
        print(f"  After NEW Layer 1 (Able Prefix Handling): (No change) '{able_cleaned_ngram}'")

    # --- Apply NEW LAYER 2: Verb To Noun Normalization ---
    verb_normalized_ngram = normalize_verb_to_noun(able_cleaned_ngram)
    # If Layer 2 decides to drop a skill, verb_normalized_ngram will be None
    if verb_normalized_ngram is None:
        if (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)):
            print(f"  After NEW Layer 2 (Verb To Noun Normalization): Skill dropped.")
        continue # Skip to the next N-gram if dropped
    elif (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)) and (able_cleaned_ngram != verb_normalized_ngram):
        print(f"  After NEW Layer 2 (Verb To Noun Normalization): '{able_cleaned_ngram}' -> '{verb_normalized_ngram}'")
    elif (processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0)):
        print(f"  After NEW Layer 2 (Verb To Noun Normalization): (No change) '{verb_normalized_ngram}'")

    # Step 1: Tokenization (if the N-gram itself contains multiple skill mentions via delimiters)
    # Apply tokenization to the verb_normalized_ngram
    raw_sub_tokens_from_ngram = tokenize_raw_skill_string(verb_normalized_ngram)
    if processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0):
        print(f"  After Primary Tokenization: {raw_sub_tokens_from_ngram}")

    # Determine the canonical form for the *entire original N-gram* for `skill_unification_mapping.csv`
    # The key will be the original N-gram, but the value will reflect the cleaned and unified form.
    # The unification here should be based on the verb_normalized_ngram, not the raw original.
    canonical_for_mapping = unify_single_token(verb_normalized_ngram, canonical_synonym_map, sorted_word_canonical_map)
    skill_unification_mapping[original_ngram_raw] = canonical_for_mapping # Mapping original to its final canonical form

    # Process each sub-token (from verb_normalized_ngram) for the `final_unique_skills` set
    unified_forms_for_this_ngram = [] # For debug output only
    for sub_token_raw in raw_sub_tokens_from_ngram:
        # Step 2: Normalization & Step 3: Synonym Unification for each individual sub-token
        # unify_single_token itself will apply normalize_skill_term
        unified_form = unify_single_token(sub_token_raw, canonical_synonym_map, sorted_word_canonical_map)
        final_unique_skills.add(unified_form) # Add to set for uniqueness
        unified_forms_for_this_ngram.append(unified_form)

    if processed_ngrams_count <= 20 or (processed_ngrams_count % 100 == 0):
        print(f"  After Normalization & Unification (for sub-tokens): {unified_forms_for_this_ngram}")


# Convert final_unique_skills set to list for iterative filtering by new layers
skills_for_further_processing = list(final_unique_skills)
initial_count_before_new_filters = len(skills_for_further_processing)
print(f"\n--- Applying Additional Filtering Layers ---")
print(f"Initial unique skills before additional filters: {initial_count_before_new_filters}")


# --- NEW LAYER 3: ADVERB REMOVAL RULE ---
def apply_adverb_removal(skill_term):
    doc = nlp(skill_term)
    cleaned_tokens = []
    # Track if any adverb was removed to know if the skill changed
    adverb_removed = False
    for token in doc:
        if token.pos_ == 'ADV':
            adverb_removed = True
        else:
            cleaned_tokens.append(token.text)

    cleaned_skill = ' '.join(cleaned_tokens).strip()
    if not cleaned_skill:
        # If skill becomes empty after adverb removal, drop it
        return None

    # Only print if adverb was actually removed (for sample logging)
    if adverb_removed and cleaned_skill != skill_term:
        # Debug print for adverb removal
        # print(f"    Adverb Removal (Layer 3): '{skill_term}' -> '{cleaned_skill}'")
        pass # Remove internal debug print, samples collected externally
    return cleaned_skill

adverb_removed_skills = []
dropped_by_adverb_removal = []
# For sample collection, store up to 10 dropped items
for skill in skills_for_further_processing:
    cleaned_skill = apply_adverb_removal(skill)
    if cleaned_skill is not None:
        adverb_removed_skills.append(cleaned_skill)
    elif len(dropped_by_adverb_removal) < 10: # Only collect samples if needed
        dropped_by_adverb_removal.append(skill)
print(f"Skills dropped by Adverb Removal Rule: {initial_count_before_new_filters - len(adverb_removed_skills)}")
if dropped_by_adverb_removal:
    print(f"  Example dropped: {dropped_by_adverb_removal[:5]}")
print(f"Skills after Adverb Removal Rule: {len(adverb_removed_skills)}")


# --- NEW LAYER 4: STRUCTURAL SKILL VALIDATION ---
def is_structurally_valid(skill_term):
    if not skill_term:
        return False

    doc = nlp(skill_term)
    words = [token.text for token in doc]
    pos_tags = [token.pos_ for token in doc]

    # DROP if it still contains an active verb (not part of noun phrase like 'problem solving')
    # Here we check for any remaining verb, as normalize_verb_to_noun should have handled leading ones.
    # Exception for 'solving' as part of 'problem solving'
    if any(token.pos_ == 'VERB' and token.lemma_ != 'solve' for token in doc):
        return False

    # DROP if it contains any adverb (should be mostly handled by previous layer, but a safeguard)
    if any(token.pos_ == 'ADV' for token in doc):
        return False

    # --- POSITIVE STRUCTURAL PATTERNS ---
    # 1. Single meaningful noun (management, leadership, programming)
    if len(doc) == 1 and pos_tags[0] in ['NOUN', 'PROPN']:
        return True

    # 2. Noun + noun umum sebagai skill (data analysis, project management)
    if len(doc) == 2 and pos_tags[0] in ['NOUN', 'PROPN'] and pos_tags[1] in ['NOUN', 'PROPN']:
        return True

    # 3. Adjective + noun valid (strategic planning, technical skill)
    if len(doc) == 2 and pos_tags[0] == 'ADJ' and pos_tags[1] in ['NOUN', 'PROPN']:
        return True

    # 4. Nama software/tools/teknologi (mysql, python, aws)
    # This is partially covered by PROPN in single/double noun rules.
    # Add a more specific check for known tech-related terms if needed, or rely on NER.
    # For now, if it's a Proper Noun combination, it's likely a tech name.
    if len(doc) >= 2 and all(token.pos_ == 'PROPN' for token in doc): # E.g., 'Google Cloud'
        return True

    # Noun Noun Noun (e.g., 'user experience design')
    if len(doc) == 3 and pos_tags[0] in ['NOUN', 'PROPN'] and pos_tags[1] in ['NOUN', 'PROPN'] and pos_tags[2] in ['NOUN', 'PROPN']:
        return True

    # Adj Noun Noun (e.g., 'effective team management')
    if len(doc) == 3 and pos_tags[0] == 'ADJ' and pos_tags[1] in ['NOUN', 'PROPN'] and pos_tags[2] in ['NOUN', 'PROPN']:
        return True

    # If none of the positive patterns match AND it contains verbs/adverbs, then it's a candidate for dropping.
    # Since we already checked for verbs/adverbs at the beginning, if it reaches here and no positive pattern matched,
    # it implies it's a non-standard structure.

    # Debug print for dropped structural skills
    # print(f"    Structural Validation (Layer 4): Dropped '{skill_term}' (no valid structural pattern matched)")
    return False

structurally_validated_skills = []
dropped_by_structural_validation = []
# For sample collection, store up to 10 dropped items
for skill in adverb_removed_skills:
    if is_structurally_valid(skill):
        structurally_validated_skills.append(skill)
    elif len(dropped_by_structural_validation) < 10: # Only collect samples if needed
        dropped_by_structural_validation.append(skill)
print(f"Skills dropped by Structural Skill Validation: {len(adverb_removed_skills) - len(structurally_validated_skills)}")
if dropped_by_structural_validation:
    print(f"  Example dropped: {dropped_by_structural_validation[:5]}")
print(f"Skills after Structural Skill Validation: {len(structurally_validated_skills)}")


# --- NEW LAYER 5: NOISE BLACKLIST ---
NOISE_BLACKLIST = set([
    'company', 'manufacturer', 'arrangement', 'founded', 'today',
    'encourage', 'requirement', 'according', 'based on',
    'academic performance', 'access requirement',
    'disability', 'soon', 'department', 'faculty', 'university'
])

def is_not_blacklisted(skill_term):
    # Check if the entire skill term is in the blacklist
    if skill_term in NOISE_BLACKLIST:
        return False

    # Check if any word in the skill term is in the blacklist
    # Split by space, then check each word
    for word in skill_term.split():
        if word in NOISE_BLACKLIST:
            return False
    return True

final_filtered_skills_list = []
dropped_by_noise_blacklist = []
# For sample collection, store up to 10 dropped items
for skill in structurally_validated_skills:
    if is_not_blacklisted(skill):
        final_filtered_skills_list.append(skill)
    elif len(dropped_by_noise_blacklist) < 10: # Only collect samples if needed
        dropped_by_noise_blacklist.append(skill)
print(f"Skills dropped by Noise Blacklist: {len(structurally_validated_skills) - len(final_filtered_skills_list)}")
if dropped_by_noise_blacklist:
    print(f"  Example dropped: {dropped_by_noise_blacklist[:5]}")
print(f"Final skills after Noise Blacklist: {len(final_filtered_skills_list)}")


# --- NEW LAYER 6: STRICT POS PATTERN FILTERING ---

def apply_strict_pos_pattern_filtering(skill_term, dropped_sample_list=None, retained_sample_list=None, debug_mode=False):
    original_skill_term = skill_term # Keep for debug messages and sample collection
    if not skill_term:
        if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
            dropped_sample_list.append(f"Empty skill term (was '{original_skill_term}')")
        return None

    doc = nlp(skill_term)
    words = [token.text for token in doc]
    pos_tags = [token.pos_ for token in doc]
    num_tokens = len(doc)

    # Pre-check for active VERB or ADV, consistent with structural validation
    if any(token.pos_ == 'VERB' and token.lemma_ != 'solve' for token in doc):
        if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
            dropped_sample_list.append(f"'{original_skill_term}' (contains active VERB)")
        return None
    if any(token.pos_ == 'ADV' for token in doc):
        if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
            dropped_sample_list.append(f"'{original_skill_term}' (contains ADV)")
        return None

    # Rule: Handle reordering for NOUN ADJ -> ADJ NOUN if possible
    # Example: "communication technical" -> "technical communication"
    if num_tokens == 2 and pos_tags[0] == 'NOUN' and pos_tags[1] == 'ADJ':
        reversed_phrase = f"{words[1]} {words[0]}"
        reversed_doc = nlp(reversed_phrase)
        reversed_pos_tags = [token.pos_ for token in reversed_doc]

        if len(reversed_doc) == 2 and reversed_pos_tags[0] == 'ADJ' and reversed_pos_tags[1] in ['NOUN', 'PROPN']:
            skill_term = reversed_phrase # Use reordered term
            doc = reversed_doc # Update doc for subsequent checks
            words = [token.text for token in doc]
            pos_tags = [token.pos_ for token in doc]
            # No debug print here, will be printed if retained
        else:
            # If reversal doesn't yield ADJ NOUN, drop the original NOUN ADJ
            if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
                dropped_sample_list.append(f"'{original_skill_term}' (NOUN ADJ pattern, reorder failed)")
            return None

    # --- Positive Patterns for Retention ---
    # 1) PROPN+ (any length, if all tokens are proper nouns)
    if all(token.pos_ == 'PROPN' for token in doc):
        if debug_mode and retained_sample_list is not None and len(retained_sample_list) < 10:
            retained_sample_list.append(f"'{skill_term}' (PROPN+ pattern)")
        return skill_term

    # Specific exception for 'acceptance test uat'
    if skill_term == 'acceptance test uat':
        if debug_mode and retained_sample_list is not None and len(retained_sample_list) < 10:
            retained_sample_list.append(f"'{skill_term}' (Specific exception)")
        return skill_term

    # Now apply length-specific rules for non-PROPN patterns
    # 2) NOUN (1 token)
    if num_tokens == 1 and pos_tags[0] == 'NOUN':
        if debug_mode and retained_sample_list is not None and len(retained_sample_list) < 10:
            retained_sample_list.append(f"'{skill_term}' (NOUN pattern)")
        return skill_term
    # 3) NOUN NOUN (2 tokens)
    if num_tokens == 2 and pos_tags[0] == 'NOUN' and pos_tags[1] == 'NOUN':
        if debug_mode and retained_sample_list is not None and len(retained_sample_list) < 10:
            retained_sample_list.append(f"'{skill_term}' (NOUN NOUN pattern)")
        return skill_term
    # 4) ADJ NOUN (2 tokens)
    if num_tokens == 2 and pos_tags[0] == 'ADJ' and pos_tags[1] == 'NOUN':
        if debug_mode and retained_sample_list is not None and len(retained_sample_list) < 10:
            retained_sample_list.append(f"'{skill_term}' (ADJ NOUN pattern)")
        return skill_term

    # --- Negative Patterns (DROP if any matches after positive patterns have been checked) ---
    # Rule: DROP if ADJ ADJ (2 tokens)
    if num_tokens == 2 and pos_tags[0] == 'ADJ' and pos_tags[1] == 'ADJ':
        if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
            dropped_sample_list.append(f"'{original_skill_term}' (ADJ ADJ pattern)")
        return None

    # Rule: Pola lebih dari 2 token (kecuali multi-word PROPN dan 'acceptance test uat' yang sudah ditangani)
    if num_tokens > 2: # and not already returned by PROPN+ or exception
        if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
            dropped_sample_list.append(f"'{original_skill_term}' (more than 2 tokens and not valid PROPN+ or exception)")
        return None

    # Drop anything else that doesn't fit the positive patterns
    if debug_mode and dropped_sample_list is not None and len(dropped_sample_list) < 10:
        dropped_sample_list.append(f"'{original_skill_term}' (did not match any positive pattern)")
    return None

# Convert final_filtered_skills_list (output of Layer 5) to a new list for further processing
skills_for_strict_pos_filtering = list(final_filtered_skills_list)
initial_count_before_strict_pos_filter = len(skills_for_strict_pos_filtering)
print(f"\n--- Applying NEW LAYER 6: STRICT POS PATTERN FILTERING ---")
print(f"Initial unique skills before strict POS filtering: {initial_count_before_strict_pos_filter}")

strictly_pos_filtered_skills = []
dropped_by_strict_pos_filter = []
retained_by_strict_pos_filter = [] # For samples

for skill in skills_for_strict_pos_filtering:
    filtered_skill = apply_strict_pos_pattern_filtering(
        skill,
        dropped_by_strict_pos_filter,
        retained_by_strict_pos_filter,
        debug_mode=True # Enable debug prints to capture samples
    )
    if filtered_skill is not None:
        strictly_pos_filtered_skills.append(filtered_skill)

# Ensure uniqueness again and sort for final output
final_skills = sorted(list(set(strictly_pos_filtered_skills)))

print(f"Total skills after strict POS filtering: {len(final_skills)}")
print(f"\n10 Sample Skills Retained by Strict POS Filter:")
for i, skill in enumerate(retained_by_strict_pos_filter[:10]):
    print(f"{i+1}. {skill}")

print(f"\n10 Sample Skills Dropped by Strict POS Filter:")
for i, skill in enumerate(dropped_by_strict_pos_filter[:10]):
    print(f"{i+1}. {skill}")


print("\n--- Summary (After All Filtering Layers) ---")
print(f"Total unique N-grams considered for detailed processing (after initial filtering): {len(filtered_ngram_candidates)}")
print(f"Total unique skills before additional filters: {initial_count_before_new_filters}")
print(f"Total identified skills after all standardization, normalization and additional filtering: {len(final_skills)}")

# Display 20 Sample of Final Unified Skills
print("\n20 Sample of Final Unified Skills:")
for i, skill in enumerate(final_skills[:20]):
    print(f"{i+1}. {skill}")

# Display 10 Sample of Skill Unification Mapping
print("\n10 Contoh Mapping Nyata (Original N-gram -> Unified Canonical Form):")
sample_mapping_shown = 0
for original, canonical in skill_unification_mapping.items():
    # Only show mappings where the original term's normalized form is different from the canonical form
    # Note: `canonical` is already normalized, `original` needs normalization for comparison
    if normalize_skill_term(original) != canonical:
        print(f"'{original}' -> '{canonical}'")
        sample_mapping_shown += 1
        if sample_mapping_shown >= 10:
            break
if sample_mapping_shown == 0:
    print("Tidak ada contoh mapping yang berhasil digabung/dinormalisasi dari original N-gram.")

# Save the final unique skills list
final_skills_df = pd.DataFrame(final_skills, columns=['unified_skill'])
final_skills_df_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'final_unified_skill_list.csv')
final_skills_df.to_csv(final_skills_df_filepath, index=False)
print(f"\nFinal unified skill list saved to '{final_skills_df_filepath}'.")

# Save the skill unification mapping table
skill_mapping_df = pd.DataFrame(list(skill_unification_mapping.items()), columns=['original_skill', 'canonical_skill'])
skill_mapping_df_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'skill_unification_mapping.csv')
skill_mapping_df.to_csv(skill_mapping_df_filepath, index=False)
print(f"Skill unification mapping saved to '{skill_mapping_df_filepath}'.")

In [ ]:
import spacy
import re
import pandas as pd
import os # Import os for path manipulation

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# 2. Load the final_unified_skill_list.csv file into a pandas DataFrame named final_skills_df.
# 3. Print a success message if the file is loaded correctly, or an error message if the file is not found.
try:
    # Correctly specify the full path using os.path.join
    input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'final_unified_skill_list.csv')
    final_skills_df = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: Make sure 'final_unified_skill_list.csv' is in '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    exit()

nlp = spacy.load("en_core_web_sm")

# Blacklist khusus noise template HR / deskriptif (Now English-only)
blacklist_words = [
    "information", "additional",
    "activity", "global", "office", "company",
    "requirement", "responsibility",
    "job description", "job requirement",
    "minimum requirement", "working experience"
]

def is_meaningful_skill(skill):

    skill_lower = skill.lower().strip()

    # 1️⃣ Remove empty / too short
    if len(skill_lower) < 3:
        return False

    # 2️⃣ Remove phrase containing blacklist words
    if any(word in skill_lower for word in blacklist_words):
        return False

    # 3️⃣ Remove numeric-heavy phrase
    if re.search(r'\d{3,}', skill_lower):
        return False

    # 4️⃣ POS-based filtering (flexible)
    doc = nlp(skill_lower)
    pos_tags = [token.pos_ for token in doc]

    # Remove if mostly verbs (not noun-based phrase)
    noun_count = sum(1 for p in pos_tags if p in ["NOUN", "PROPN"])
    verb_count = sum(1 for p in pos_tags if p == "VERB")

    if noun_count == 0:
        return False

    # Allow soft skill patterns like:
    # NOUN
    # NOUN NOUN
    # ADJ NOUN
    # NOUN VERB (problem solving sometimes tagged like this)
    # VERB NOUN (decision making sometimes)
    # NOUN NOUN NOUN

    if len(pos_tags) > 4:
        return False

    return True


# Apply filtering
before_count = len(final_skills_df)

filtered_skills_df = final_skills_df[
    final_skills_df['unified_skill'].apply(is_meaningful_skill)
].copy()

after_count = len(filtered_skills_df)
filtered_skills_df_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'unigram_final_skill_list.csv')
filtered_skills_df.to_csv(filtered_skills_df_filepath, index=False)
print(f"\nFinal skill list domain cleaned saved to '{filtered_skills_df_filepath}'.")


print("\n--- DOMAIN CLEANING (SOFT SKILL SAFE) ---")
print(f"Before cleaning: {before_count}")
print(f"After cleaning: {after_count}")
print(f"Removed: {before_count - after_count}")
print(f"Retention: {(after_count/before_count*100):.2f}%")

print("\n20 Sample Cleaned Skills:")
for i, skill in enumerate(filtered_skills_df['unified_skill'].head(20)):
    print(f"{i+1}. {skill}")

## CELL 1: Load Bigram Data

In [ ]:
import pandas as pd
import scipy.sparse
import os

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

print("\n--- Loading Bigram Data ---")

try:
    # Load the bigram vocabulary (feature names)
    bigram_vocabulary_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_vocabulary.csv')
    df_bigram_vocab = pd.read_csv(bigram_vocabulary_filepath)

    # Load the bigram document-term matrix to get frequencies
    bigram_dtm_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_document_term_matrix.npz')
    X_bigrams = scipy.sparse.load_npz(bigram_dtm_filepath)

    # Calculate frequencies by summing columns of the DTM
    bigram_frequencies = X_bigrams.sum(axis=0).tolist()[0]

    # Add frequencies to the DataFrame
    df_bigram_vocab['frequency'] = bigram_frequencies
    df_bigram_vocab = df_bigram_vocab.rename(columns={'bigram': 'original_skill'})

    print(f"File '{os.path.basename(bigram_vocabulary_filepath)}' dan '{os.path.basename(bigram_dtm_filepath)}' berhasil dimuat.")
    print(f"Total {len(df_bigram_vocab)} bi-gram dengan frekuensi ditemukan.")

except FileNotFoundError as e:
    print(f"Error: Pastikan file 'bigram_vocabulary.csv' dan 'bigram_document_term_matrix.npz' ada di '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    # Create an empty DataFrame to prevent errors in subsequent cells
    df_bigram_vocab = pd.DataFrame(columns=['original_skill', 'frequency'])

# Tampilkan 5 data teratas
print("\n5 data teratas dari bi-gram vocabulary dengan frekuensi:")
display(df_bigram_vocab.head())

# Validasi kolom
print("\nKolom dalam DataFrame bi-gram:")
print(df_bigram_vocab.columns.tolist())

## CELL 2: Skill Matching + Synonym Unification (Bigram)

In [ ]:
import pandas as pd
import re
from collections import defaultdict
import os
import scipy.sparse # Import scipy.sparse for loading .npz files

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

print("\n--- Performing Skill Matching and Synonym Unification for Bigrams ---")

# --- NEW: Load bigram data directly into this cell ---
print("\n--- Loading Bigram Data ---")
try:
    # Load the bigram vocabulary (feature names)
    bigram_vocabulary_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_vocabulary.csv')
    df_bigram_vocab = pd.read_csv(bigram_vocabulary_filepath)

    # Load the bigram document-term matrix to get frequencies
    bigram_dtm_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_document_term_matrix.npz')
    X_bigrams = scipy.sparse.load_npz(bigram_dtm_filepath)

    # Calculate frequencies by summing columns of the DTM
    bigram_frequencies = X_bigrams.sum(axis=0).tolist()[0]

    # Add frequencies to the DataFrame
    df_bigram_vocab['frequency'] = bigram_frequencies
    df_bigram_vocab = df_bigram_vocab.rename(columns={'bigram': 'original_skill'})

    print(f"File '{os.path.basename(bigram_vocabulary_filepath)}' dan '{os.path.basename(bigram_dtm_filepath)}' berhasil dimuat.")
    print(f"Total {len(df_bigram_vocab)} bi-gram dengan frekuensi ditemukan.")

except FileNotFoundError as e:
    print(f"Error: Pastikan file 'bigram_vocabulary.csv' dan 'bigram_document_term_matrix.npz' ada di '{OUTPUT_FOLDER_INDUSTRY}'. {e}")
    # Create an empty DataFrame to prevent errors in subsequent cells
    df_bigram_vocab = pd.DataFrame(columns=['original_skill', 'frequency'])

# Provided synonym dictionary (as used in main N-gram skill extraction for consistency, but now internal)
synonym_dict_raw = {
    "ml": "machine learning",
    "machine learning engineer": "machine learning",
    "ai": "artificial intelligence",
    "artificial intelligence engineer": "artificial intelligence",
    "frontend developer": "frontend",
    "backend developer": "backend",
    "fullstack developer": "fullstack",
    "public speaking skill": "public speaking",
    "team work": "teamwork",
    "team": "teamwork", # Added 'team' to 'teamwork'
    "problem solving skill": "problem solving",
    "modeling data": "data modeling",
    "server linux": "linux server",
    "analysis process": "process analysis",
    "data multiple source": "multiple data source",
    "analysis performance": "performance performance",
    "mobile web": "web mobile", # Assuming 'web mobile' is the canonical form
    "web frontend": "frontend web",
    "support data": "data support",
    "database mysql": "mysql database",
    "network hardware": "hardware network",

    # NEW: Soft Skill Unifications
    "good communication": "communication",
    "strong communication": "communication",
    "excellent communication": "communication",
    "effective communication": "communication",
    "verbal communication": "communication",
    "written communication": "communication",
    "good problem solving": "problem solving",
    "strong problem solving": "problem solving",
    "effective problem solving": "problem solving",
    "good teamwork": "teamwork",
    "strong teamwork": "teamwork",
    "effective teamwork": "teamwork",
    "strong leadership": "leadership",
    "effective leadership": "leadership",
    "strong analytical": "analytical",
    "analytical skill": "analytical",
    "analytical ability": "analytical",
    "strong critical thinking": "critical thinking",
    "effective critical thinking": "critical thinking",
    "quick adaptability": "adaptability",
    "good adaptability": "adaptability"
}

# --- Helper functions for Skill Standardization & Normalization ---

# 1. Text normalization (lowercase, remove non-alphabetic except spaces, trim spaces)
def normalize_skill_term(term):
    if not isinstance(term, str):
        term = str(term)
    term = term.lower()
    term = re.sub(r'[^a-z0-9\s]', ' ', term) # Remove non-alphanumeric except spaces
    term = re.sub(r'\s+', ' ', term).strip()
    return term

# Prepare the canonical synonym maps once
canonical_synonym_map = {}
sorted_word_canonical_map = {}

for key_raw, value_raw in synonym_dict_raw.items():
    normalized_key = normalize_skill_term(key_raw)
    normalized_value = normalize_skill_term(value_raw)

    canonical_synonym_map[normalized_key] = normalized_value

    # For word-order variations, sort the words and store the mapping
    sorted_key_words = ' '.join(sorted(normalized_key.split()))
    sorted_value_words = ' '.join(sorted(normalized_value.split()))

    sorted_word_canonical_map[sorted_key_words] = normalized_value
    sorted_word_canonical_map[sorted_value_words] = normalized_value # Ensure canonical itself is mapped

# Ensure all canonical values (which might not be direct keys) also map to themselves
# This ensures consistency if a canonical form is encountered directly.
for val in set(canonical_synonym_map.values()):
    if val not in canonical_synonym_map:
        canonical_synonym_map[val] = val
    # Also ensure its sorted form maps to itself
    sorted_val_words = ' '.join(sorted(val.split()))
    if sorted_val_words not in sorted_word_canonical_map:
        sorted_word_canonical_map[sorted_val_words] = val

# Helper function to apply synonym unification to a single token
def unify_single_token(token_string, canonical_map, sorted_map):
    normalized_token = normalize_skill_term(token_string)

    # 1. Direct lookup in `canonical_map` (e.g., "ml" -> "machine learning")
    if normalized_token in canonical_map:
        return canonical_map[normalized_token]

    # 2. Sorted word lookup for word-order variations (e.g., "modeling data" -> "data modeling")
    sorted_words = ' '.join(sorted(normalized_token.split()))
    if sorted_words in sorted_map:
        return sorted_map[sorted_words]

    # If no unification applies, the normalized token itself is the canonical form
    return normalized_token

# --- Process Bigram Vocabulary ---
if not df_bigram_vocab.empty:
    processed_bigrams = []
    for index, row in df_bigram_vocab.iterrows():
        original_bigram = row['original_skill']
        frequency = row['frequency']

        normalized_bigram = normalize_skill_term(original_bigram)

        # Apply synonym unification using the internal maps
        unified_skill = unify_single_token(normalized_bigram, canonical_synonym_map, sorted_word_canonical_map)

        processed_bigrams.append({'unified_skill': unified_skill, 'frequency': frequency})

    df_processed_bigrams = pd.DataFrame(processed_bigrams)

    if not df_processed_bigrams.empty:
        # Gabungkan frekuensi skill yang sama
        df_bigram_final_skill_list = df_processed_bigrams.groupby('unified_skill')['frequency'].sum().reset_index()
        df_bigram_final_skill_list = df_bigram_final_skill_list.sort_values(by='frequency', ascending=False).reset_index(drop=True)

        # Simpan hasil ke "bigram_final_skill_list.csv"
        output_filepath_bigram_skills = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_final_skill_list.csv')
        df_bigram_final_skill_list.to_csv(output_filepath_bigram_skills, index=False)
        print(f"\nFinal skill list bi-gram berhasil disimpan ke '{output_filepath_bigram_skills}'.")

        # Tampilkan 10 skill teratas
        print("\n10 skill bi-gram teratas setelah matching dan unifikasi:")
        display(df_bigram_final_skill_list.head(10))
    else:
        print("Tidak ada bi-gram yang tersisa setelah unifikasi.") # Adjusted message
else:
    print("DataFrame bi-gram kosong, tidak ada pemrosesan yang dilakukan.")

In [ ]:
import pandas as pd
import re
import os
import nltk # Import NLTK for POS tagging
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import sys # Import sys for graceful exit

# Download necessary NLTK data (for POS tagging)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True) # Added to fix LookupError

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Helper function for text normalization (copied for self-containment)
def normalize_skill_term(term):
    if not isinstance(term, str):
        term = str(term)
    term = term.lower()
    term = re.sub(r'[^a-z0-9\\s]', ' ', term) # Remove non-alphanumeric except spaces
    term = re.sub(r'\\s+', ' ', term).strip()
    return term

# ========================
# A. LOAD DATA
# ========================

df_skills = pd.DataFrame() # Initialize df_skills to an empty DataFrame

try:
    input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_final_skill_list.csv')
    df_skills = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully.")
except FileNotFoundError:
    # Fallback to the name specified by the user if the standard one is not found
    input_filepath = os.path.join(OUTPUT_FOLDER_CURRICULUM, 'bigram_final_skill_list_cleaned.csv') # Assuming this is the 'unmapped' list
    try:
        df_skills = pd.read_csv(input_filepath)
        print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully.")
    except FileNotFoundError:
        print(f"Error: Neither 'bigram_final_skill_list.csv' nor 'bigram_final_skill_list_cleaned.csv' found in '{OUTPUT_FOLDER_CURRICULUM}'. Please ensure the file exists.")
        df_skills = pd.DataFrame(columns=['unified_skill', 'frequency']) # Initialize with empty DataFrame on error
        sys.exit(1) # Use sys.exit() to ensure termination

# Ensure 'unified_skill' column is string type and 'frequency' is numeric
df_skills['unified_skill'] = df_skills['unified_skill'].astype(str).fillna('')
df_skills['frequency'] = pd.to_numeric(df_skills['frequency'], errors='coerce').fillna(0).astype(int)


# --- 1. Initial cleaning (verb noise/blacklist) ---
# Get the number of skills before filtering
before_filtering_count_initial = len(df_skills)

# Define verb noise and phrase blacklist (existing logic)
verbs_noise = ["make", "ensure", "assist", "help", "support", "manage", "handle", "create", "perform", "do"]
phrase_blacklist = [
    "make sure", "working hour", "company ask",
    "process please", "added value", "interview asked", "working day", "make payment"
]

# Function to check for verb noise in a skill
def contains_verb_noise(skill):
    words = re.split(r'\\s|-', skill.lower())
    return any(word in verbs_noise for word in words)

# Apply filters
df_filtered = df_skills[~df_skills['unified_skill'].apply(contains_verb_noise)].copy()
df_filtered = df_filtered[~df_filtered['unified_skill'].str.lower().isin(phrase_blacklist)].copy()

after_filtering_count_initial = len(df_filtered)
print(f"Initial cleaning (verb noise/blacklist) removed {before_filtering_count_initial - after_filtering_count_initial} skills.")
print(f"Skills remaining after initial cleaning: {after_filtering_count_initial}")


# --- 2. FIX BROKEN TECHNICAL TERMS (NEW STEP) ---
print("\n--- Fixing Broken Technical Terms ---")
before_fix_broken = len(df_filtered)

broken_tech_terms_mapping = {
    "post gre sql": "postgresql",
    "gre sql": "postgresql",
    "post gre": "postgres",
    "micro service": "microservice",
    "digital ation": "digitalization",
    "data base": "database",
    "web site": "website",
    "machine learning": "machine learning", # Ensure canonical is not broken
    "artificial intelligence": "artificial intelligence",
    "cloud computing": "cloud computing",
    "dev ops": "devops",
    "it": "it"
}

def fix_broken_terms(skill_phrase):
    # Sort keys by length in descending order to match longer phrases first
    for broken_phrase in sorted(broken_tech_terms_mapping.keys(), key=len, reverse=True):
        if broken_phrase in skill_phrase:
            skill_phrase = skill_phrase.replace(broken_phrase, broken_tech_terms_mapping[broken_phrase])
    return skill_phrase

df_filtered['unified_skill'] = df_filtered['unified_skill'].apply(fix_broken_terms)

after_fix_broken = len(df_filtered)
print(f"Skills after fixing broken technical terms: {after_fix_broken}")
#display(df_filtered[df_filtered['unified_skill'].str.contains('postgresql|microservice|database')].head(10)) # Removed display because it may cause issues for the agent.


# --- 3. IMPROVE SOFT SKILL MAPPING (TOKEN-BASED) ---
print("\n--- Starting Soft Skill Mapping and Auto-Detection (Improved) ---")
before_soft_mapping_count = len(df_filtered['unified_skill'].unique())

initial_soft_skill_keywords_raw = {
    "communication": "communication",
    "team": "teamwork",
    "teamwork": "teamwork",
    "leadership": "leadership",
    "management": "management",
    "problem": "problem solving",
    "problem solving": "problem solving",
    "collaboration": "teamwork",
    "critical thinking": "critical thinking",
    "adaptability": "adaptability",
    "decision": "decision making",
    "decision making": "decision making"
}

# Normalize keys and values for consistent matching
soft_skill_keywords = {}
for k, v in initial_soft_skill_keywords_raw.items():
    soft_skill_keywords[normalize_skill_term(k)] = normalize_skill_term(v)

# Additional common soft skill patterns for token-based matching
soft_skill_patterns_for_token_match = {
    "join team": "teamwork",
    "team collaboration": "teamwork",
    "solve problem": "problem solving",
    "critical thinking skill": "critical thinking",
    "good communication": "communication",
    "strong communication": "communication",
    "effective communication": "communication",
    "verbal communication": "communication",
    "written communication": "communication",
    "strong problem": "problem solving",
    "effective problem": "problem solving"
}
for k, v in soft_skill_patterns_for_token_match.items():
    soft_skill_keywords[normalize_skill_term(k)] = normalize_skill_term(v)

# Helper function to apply synonym unification (now handles token-based for soft skills)
def apply_soft_skill_mapping(original_skill_name):
    norm_skill = normalize_skill_term(original_skill_name)

    # 1. Exact match for full normalized skill
    if norm_skill in soft_skill_keywords:
        return soft_skill_keywords[norm_skill]

    # 2. Token-based matching for soft skills (new improvement)
    # Check if any token or sequence of tokens forms a known soft skill keyword
    tokens = norm_skill.split()
    # Iterate through potential multi-word soft skill patterns first (e.g., 'problem solving')
    for pattern in sorted(soft_skill_keywords.keys(), key=len, reverse=True):
        if ' ' in pattern and pattern in norm_skill: # Only for multi-word patterns
            return soft_skill_keywords[pattern]

    # Then check individual tokens
    for token in tokens:
        if token in soft_skill_keywords:
            return soft_skill_keywords[token]

    return original_skill_name # If no soft skill match, return original

df_filtered['unified_skill'] = df_filtered['unified_skill'].apply(apply_soft_skill_mapping)


# --- 4. MERGE FREQUENCY ---
print("\nMerging frequencies for mapped skills...")
df_mapped_skills = df_filtered.groupby('unified_skill')['frequency'].sum().reset_index()
df_mapped_skills = df_mapped_skills.sort_values(by='frequency', ascending=False).reset_index(drop=True)

after_soft_mapping_count = len(df_mapped_skills['unified_skill'].unique())

print("\n--- Soft Skill Mapping Summary ---")
print(f"Jumlah skill unik sebelum mapping (setelah initial cleaning): {before_soft_mapping_count}")
print(f"Jumlah skill unik sesudah mapping: {after_soft_mapping_count}")
print(f"Jumlah skill yang digabungkan/disederhanakan: {before_soft_mapping_count - after_soft_mapping_count}")
print("\n--- 10 Skill Teratas Setelah Soft Skill Mapping ---")
#display(df_mapped_skills.head(10)) # Removed display because it may cause issues for the agent.


# --- START ADDITIONAL CLEANING STEPS (ORDERED AS PER REQUEST) ---
print("\n--- Applying Additional Cleaning Steps ---")
before_additional_cleaning_count = len(df_mapped_skills)

# 5. A. FREQUENCY FILTER
# Hapus skill dengan frequency < 3
df_cleaned = df_mapped_skills[df_mapped_skills['frequency'] >= 3].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills due to frequency < 3.")
before_additional_cleaning_count = len(df_cleaned)

# 6. B. REMOVE VERB-BASED PHRASE
# Definisikan daftar kata kerja umum
verbs_to_remove = ["analyze", "manage", "perform", "make", "ensure", "create", "handle"]

def starts_with_verb(skill_phrase):
    tokens = word_tokenize(skill_phrase.lower())
    if tokens and tokens[0] in verbs_to_remove:
        return True
    return False

df_cleaned = df_cleaned[~df_cleaned['unified_skill'].apply(starts_with_verb)].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills starting with common verbs.")
before_additional_cleaning_count = len(df_cleaned)

# 7. C. REMOVE NOISE WORD
# Definisikan daftar kata yang bukan skill
noise_words = [
    "process", "output", "content", "task",
    "activity", "document", "report", "please"
]

def contains_noise_word(skill_phrase):
    skill_phrase_lower = skill_phrase.lower()
    for word in noise_words:
        if word in skill_phrase_lower.split(): # Check if the word exists as a whole word
            return True
    return False

df_cleaned = df_cleaned[~df_cleaned['unified_skill'].apply(contains_noise_word)].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills containing noise words.")
before_additional_cleaning_count = len(df_cleaned)


# 8. C. CONTEXT-BASED FILTERING (NEW)
print("\n--- Applying Context-Based Filtering ---")

context_noise_words = [
"company", "office", "ltd", "inc", "corp", "group",
"today", "day", "time", "year",
"school", "student", "university",
"com", "www", "http",
"thing", "something", "everyone",
"head office", "third party", "high school", "cause", "pratama", "freedom"
]

def contains_context_noise(skill_phrase):
    skill_phrase_lower = skill_phrase.lower()
    for word in context_noise_words:
        # Use regex to match whole word boundaries
        if re.search(r'\\b' + re.escape(word) + r'\\b', skill_phrase_lower):
            return True
    return False

df_cleaned = df_cleaned[~df_cleaned['unified_skill'].apply(contains_context_noise)].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills due to context noise words.")
before_additional_cleaning_count = len(df_cleaned)


# 9. D. FILTER BAHASA NON-ENGLISH / NOISE (NEW)
print("\n--- Applying Non-English/Noise Word Filtering ---")

indo_noise = ["pada", "saat", "yang", "dan", "atau", "melaku kan"]

def contains_indo_noise(skill_phrase):
    skill_phrase_lower = skill_phrase.lower()
    for word in indo_noise:
        if re.search(r'\\b' + re.escape(word) + r'\\b', skill_phrase_lower):
            return True
    return False

df_cleaned = df_cleaned[~df_cleaned['unified_skill'].apply(contains_indo_noise)].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills due to non-English/noise words.")
before_additional_cleaning_count = len(df_cleaned)


# 10. D. POS FILTERING (IMPORTANT) (existing logic)
# Gunakan NLTK POS tagging
# Ambil hanya skill dengan pola: NN NN atau JJ NN

def is_valid_pos_pattern(skill_phrase):
    tokens = word_tokenize(skill_phrase)
    # Only consider bigrams for these patterns
    if len(tokens) != 2:
        return False

    tagged_tokens = pos_tag(tokens)
    pos_tags = [tag for word, tag in tagged_tokens]

    # Pola NN NN (noun + noun)
    if (pos_tags[0].startswith('N') and pos_tags[1].startswith('N')):
        return True
    # Pola JJ NN (adjective + noun)
    if (pos_tags[0].startswith('J') and pos_tags[1].startswith('N')):
        return True

    return False

df_cleaned = df_cleaned[df_cleaned['unified_skill'].apply(is_valid_pos_pattern)].copy()
print(f"Removed {before_additional_cleaning_count - len(df_cleaned)} skills not matching valid POS patterns (NN NN, JJ NN).")

# F. OUTPUT TAMBAHAN
print("\n--- Final Cleaning Summary ---")
print(f"Total skills before all additional cleaning steps: {len(df_mapped_skills)}")
print(f"Total skills after all additional cleaning steps: {len(df_cleaned)}")
print(f"Total skills removed in all additional cleaning steps: {len(df_mapped_skills) - len(df_cleaned)}")

print("\n10 Top Skills After Final Cleaning:")
#display(df_cleaned.head(10)) # Removed display because it may cause issues for the agent.

# F. SAVE RESULT
# Simpan hasil ke: "bigram_skill_final_cleaned_v2.csv"
output_filepath_final_cleaned = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_skill_final_cleaned_v2.csv')
df_cleaned.to_csv(output_filepath_final_cleaned, index=False)
print(f"\nFinal cleaned bi-gram skill list saved to '{output_filepath_final_cleaned}'.")

## 1. Load Data, Identify Skill Column, and Initial Cleaning

In [ ]:
import pandas as pd
import os
import re

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# --- 1. Load Dataset ---
input_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_skill_final_cleaned_v2.csv')
try:
    df_industry_skills = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully with {len(df_industry_skills)} records.")
except FileNotFoundError:
    print(f"Error: '{os.path.basename(input_filepath)}' not found in '{OUTPUT_FOLDER_INDUSTRY}'. Please ensure the file exists.")
    df_industry_skills = pd.DataFrame() # Create empty DataFrame to avoid errors

# Exit if DataFrame is empty
if df_industry_skills.empty:
    print("Exiting further processing as the DataFrame is empty.")
    raise SystemExit("DataFrame is empty, cannot proceed.")

# --- 2. Identify Skill Column and Rename ---
# Based on previous processing, the skill column is 'unified_skill'
if 'unified_skill' in df_industry_skills.columns:
    df_industry_skills = df_industry_skills.rename(columns={'unified_skill': 'original_skill'})
    print("Skill column 'unified_skill' renamed to 'original_skill'.")
else:
    print("Error: 'unified_skill' column not found. Please verify the input CSV structure.")
    # Attempt to use the first column if 'unified_skill' is not found, as a fallback
    if not df_industry_skills.columns.empty:
        first_col = df_industry_skills.columns[0]
        df_industry_skills = df_industry_skills.rename(columns={first_col: 'original_skill'})
        print(f"Fallback: Renamed first column '{first_col}' to 'original_skill'.")
    else:
        print("No columns found in DataFrame. Cannot identify skill column.")
        raise SystemExit("No columns to process.")

# --- 3. Clean Industry Skills (Light Preprocessing) ---
def clean_text_light(text):
    if pd.isna(text):
        return ''
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    return text

df_industry_skills['original_skill'] = df_industry_skills['original_skill'].apply(clean_text_light)
print("Light preprocessing applied to 'original_skill' column.")

# --- 4. Remove Invalid Skills ---
before_removal_count = len(df_industry_skills)

# Remove null values
df_industry_skills = df_industry_skills.dropna(subset=['original_skill'])

# Remove empty strings
df_industry_skills = df_industry_skills[df_industry_skills['original_skill'].str.strip() != '']

# Remove duplicated skills
df_industry_skills = df_industry_skills.drop_duplicates(subset=['original_skill'])

after_removal_count = len(df_industry_skills)
print(f"Removed {before_removal_count - after_removal_count} invalid or duplicate skills.")

print(f"Final df_industry_skills shape: {df_industry_skills.shape}")
print("First 5 rows of df_industry_skills:")
display(df_industry_skills.head())

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
import json
import re
import os
import time

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Ensure the output directory exists
os.makedirs(OUTPUT_FOLDER_INDUSTRY, exist_ok=True)

# --- Configuration for Translation ---
TRANSLATION_CACHE_FILE = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'translation_cache.json')
UNTRANSLATED_SKILLS_FILE = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'untranslated_or_failed_skills.csv')

# Load existing translation cache or initialize an empty one
translation_cache = {}
if os.path.exists(TRANSLATION_CACHE_FILE):
    with open(TRANSLATION_CACHE_FILE, 'r') as f:
        translation_cache = json.load(f)
    print(f"Loaded {len(translation_cache)} entries from translation cache.")

# Initialize GoogleTranslator
translator = GoogleTranslator(source='en', target='id')

# Technical IT terms to preserve in English (expanded based on common IT terms)
TECH_IT_TERMS = {
    "python", "java", "javascript", "c++", "c#", "php", "ruby", "go", "swift", "kotlin",
    "html", "css", "react", "angular", "vue", "node.js", "spring", "django", "flask", "laravel",
    "sql", "mysql", "postgresql", "mongodb", "redis", "docker", "kubernetes", "aws", "azure", "gcp",
    "cloud", "devops", "agile", "scrum", "linux", "unix", "windows server", "git", "github", "gitlab",
    "api", "rest", "graphql", "microservices", "data science", "machine learning", "artificial intelligence",
    "ai", "ml", "big data", "blockchain", "cybersecurity", "network", "server", "database", "ui", "ux",
    "qa", "testing", "automation", "scripting", "web development", "mobile development", "front end",
    "back end", "full stack", "etl", "sre", "nlp", "computer vision", "data warehousing", "hadoop",
    "spark", "tableau", "power bi", "excel", "jira", "confluence", "salesforce", "sap", "erp", "crm",
    "sso", "oauth", "jwt", "smtp", "dns", "tcp/ip", "vpn", "firewall", "router", "switch",
    "android", "ios", "typescript", "bash", "shell", "r", "matlab", "tensorflow", "pytorch",
    "keras", "pandas", "numpy", "scipy", "scikit-learn", "hadoop", "spark", "kafka", "flink",
    "grafana", "prometheus", "splunk", "elastic search", "logstash", "kibana", "aws s3", "aws ec2",
    "azure vm", "gcp compute engine", "gcp bigquery", "aws redshift", "snowflake", "data lake",
    "data governance", "data quality", "data integration", "data visualization", "business intelligence",
    "software development life cycle", "sdlc", "version control", "ci/cd", "continuous integration",
    "continuous delivery", "containerization", "virtualization", "system design", "architecture", "data modeling"
}

# --- Step 5: Translate Unique Skills with Hybrid Strategy and Caching ---
print("\n--- Translating Unique Skills ---")
unique_skills_to_translate = df_industry_skills['original_skill'].unique()
print(f"Total unique skills to process: {len(unique_skills_to_translate)}")

translated_skills_map = {}
technical_skills_identified = set()
untranslated_or_failed_skills = []
translation_api_calls = 0
cache_hits = 0

for skill in unique_skills_to_translate:
    # Check cache first
    if skill in translation_cache:
        translated_skills_map[skill] = translation_cache[skill]
        cache_hits += 1
        continue

    # Check if it's a technical IT term (case-insensitive and partial match)
    is_technical_it_skill = False
    for term in TECH_IT_TERMS:
        if re.search(r'\b' + re.escape(term) + r'\b', skill, re.IGNORECASE):
            is_technical_it_skill = True
            technical_skills_identified.add(skill)
            break

    if is_technical_it_skill:
        translated_skills_map[skill] = skill # Preserve English
        translation_cache[skill] = skill
    else:
        try:
            # Add a small delay to avoid hitting API rate limits
            time.sleep(0.05) # 50 ms delay
            translated_text = translator.translate(skill)
            if translated_text:
                translated_skills_map[skill] = translated_text
                translation_cache[skill] = translated_text
                translation_api_calls += 1
            else:
                # Fallback to English on translation failure
                translated_skills_map[skill] = skill
                translation_cache[skill] = skill
                untranslated_or_failed_skills.append({'original_skill': skill, 'reason': 'Translation API returned empty string'})
                print(f"  Warning: Empty translation for '{skill}'. Keeping English.")
        except Exception as e:
            # Fallback to English on translation API error
            translated_skills_map[skill] = skill
            translation_cache[skill] = skill
            untranslated_or_failed_skills.append({'original_skill': skill, 'reason': str(e)})
            print(f"  Error translating '{skill}': {e}. Keeping English.")

print(f"Translation complete. Made {translation_api_calls} API calls. {cache_hits} cache hits.")
print(f"Identified {len(technical_skills_identified)} technical IT skills.")

# --- Save Translation Cache ---
with open(TRANSLATION_CACHE_FILE, 'w') as f:
    json.dump(translation_cache, f, indent=4)
print(f"Translation cache saved to '{TRANSLATION_CACHE_FILE}'. Total entries: {len(translation_cache)}")

# --- Save Untranslated/Failed Skills ---
if untranslated_or_failed_skills:
    df_untranslated = pd.DataFrame(untranslated_or_failed_skills)
    df_untranslated.to_csv(UNTRANSLATED_SKILLS_FILE, index=False)
    print(f"Untranslated/failed skills saved to '{UNTRANSLATED_SKILLS_FILE}'. Total: {len(df_untranslated)}")
else:
    print("No untranslated or failed skills to report.")

# --- Step 6: Create Bilingual Representations ---
print("\n--- Creating Bilingual Skill Representations ---")

df_industry_skills['translated_skill'] = df_industry_skills['original_skill'].map(translated_skills_map)

def create_bilingual_representation(row):
    original = row['original_skill']
    translated = row['translated_skill']

    # Determine if the skill was preserved (i.e., it's a technical IT skill or translation failed to ID)
    # Check against the actual output of the translation process, which keeps English if technical or failed
    if original == translated or original in technical_skills_identified:
        return original  # Keep English for technical terms or if translation failed/was not needed
    else:
        # For general competencies, combine English and Indonesian
        return f"{original} ({translated})"

def is_technical_skill_flag(skill):
    return skill in technical_skills_identified

# Apply the function to create the bilingual representation
df_industry_skills['bilingual_skill_representation'] = df_industry_skills.apply(create_bilingual_representation, axis=1)

# Flag if the skill is considered a technical IT skill (based on the TECH_IT_TERMS check)
df_industry_skills['is_technical_it_skill'] = df_industry_skills['original_skill'].apply(is_technical_skill_flag)

# --- Step 7: Clean Bilingual Representation ---
# Apply light preprocessing to the bilingual representation
def clean_text_light(text):
    if pd.isna(text):
        return ''
    text = str(text).lower() # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip() # Normalize whitespace
    return text

df_industry_skills['bilingual_skill_representation_cleaned'] = df_industry_skills['bilingual_skill_representation'].apply(clean_text_light)

# --- Step 8: Final DataFrame Structure and Output ---
print("\n--- Structuring Final Output ---")
df_industry_skills_final = df_industry_skills[[
    'original_skill',
    'translated_skill',
    'bilingual_skill_representation',
    'bilingual_skill_representation_cleaned',
    'is_technical_it_skill'
]].copy()

# Add 'translation_status' column
df_industry_skills_final['translation_status'] = df_industry_skills_final.apply(
    lambda row: 'Preserved (Technical IT)' if row['is_technical_it_skill'] else \
                ('Translated to ID' if row['original_skill'] != row['translated_skill'] else 'Preserved (Failed/Unnecessary)'),
    axis=1
)

print(f"Final df_industry_skills_final shape: {df_industry_skills_final.shape}")
print("First 5 rows of df_industry_skills_final:")
display(df_industry_skills_final.head())

# Save the final DataFrame
output_filepath_final_bilingual = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bilingual_industry_skills.csv')
df_industry_skills_final.to_csv(output_filepath_final_bilingual, index=False)
print(f"Final bilingual skills dataset saved to '{output_filepath_final_bilingual}'.")

# --- Step 9: Print Summary Statistics ---
print("\n--- Summary Statistics for Bilingual Skills ---")

total_skills_processed = len(df_industry_skills)
unique_original_skills = len(unique_skills_to_translate)
unique_translated_successfully = df_industry_skills_final[
    (df_industry_skills_final['translation_status'] == 'Translated to ID')
]['original_skill'].nunique()
unique_technical_it_skills = df_industry_skills_final[
    (df_industry_skills_final['translation_status'] == 'Preserved (Technical IT)')
]['original_skill'].nunique()
unique_preserved_other = df_industry_skills_final[
    (df_industry_skills_final['translation_status'] == 'Preserved (Failed/Unnecessary)')
]['original_skill'].nunique()

print(f"Total skills in dataset: {total_skills_processed}")
print(f"Total unique original skills: {unique_original_skills}")
print(f"Unique skills translated to Indonesian: {unique_translated_successfully}")
print(f"Unique technical IT skills (English preserved): {unique_technical_it_skills}")
print(f"Unique skills preserved (translation failed or unnecessary): {unique_preserved_other}")
print(f"Translation API calls made: {translation_api_calls}")
print(f"Translation cache reused (cache hits): {cache_hits}")


## CELL 4: Comparison (Unigram vs Bigram)

In [ ]:
import pandas as pd
import os

print("\n--- Comparing Unigram and Bigram Skill Lists ---")

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Load the processed skill lists (ensuring they exist from previous cells)
try:
    # Load Bigram Final Skill List
    bigram_skills_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'bigram_final_skill_list.csv')
    df_bigram_skills = pd.read_csv(bigram_skills_filepath)
    print(f"Loaded bigram skills from '{os.path.basename(bigram_skills_filepath)}'.")
except FileNotFoundError:
    print(f"Error: bigram_final_skill_list.csv not found in '{OUTPUT_FOLDER_INDUSTRY}'. Creating empty DataFrame.")
    df_bigram_skills = pd.DataFrame(columns=['unified_skill', 'frequency'])

try:
    # Load Unigram Final Skill List (original 'final_unified_skill_list.csv' is effectively the unigram list)
    unigram_skills_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, 'unigram_final_skill_list.csv')
    df_unigram_skills = pd.read_csv(unigram_skills_filepath)
    # For comparison, we need frequency for unigram as well. If not available, we assume frequency is 1 per occurrence in the vocabulary.
    # This assumes 'unigram_final_skill_list.csv' only contains 'unified_skill' column.
    # For accurate comparison, unigram frequencies would ideally come from its own DTM sum.
    # For now, let's just count unique skills and assume implicit frequency from the source vocabulary.
    # If we need actual frequencies for unigrams, that would require reloading the n-gram DTM and extracting 1-gram frequencies.
    # For the purpose of this comparison, we'll just use the number of unique skills and top items.
    print(f"Loaded unigram skills from '{os.path.basename(unigram_skills_filepath)}'.")
except FileNotFoundError:
    print(f"Error: unigram_final_skill_list.csv not found in '{OUTPUT_FOLDER_INDUSTRY}'. Creating empty DataFrame.")
    df_unigram_skills = pd.DataFrame(columns=['unified_skill'])

# --- Comparison Metrics ---

# 1. Jumlah skill unik
unique_bigram_skills = len(df_bigram_skills['unified_skill'].unique()) if not df_bigram_skills.empty else 0
unique_unigram_skills = len(df_unigram_skills['unified_skill'].unique()) if not df_unigram_skills.empty else 0

# 2. Total frekuensi (untuk bigram, langsung dari kolom 'frequency'; untuk unigram, kita asumsikan 1 per entri jika frekuensi tidak ada)
total_bigram_frequency = df_bigram_skills['frequency'].sum() if not df_bigram_skills.empty else 0
# For unigram, if we don't have frequency, we can't sum it like bigram. We'll use count of items.
# If unigram_final_skill_list.csv had frequencies, we would sum those.
# Given 'unigram_final_skill_list.csv' only has 'unified_skill', let's just report count of unique skills.

# 3. Top 10 skill masing-masing
top_10_bigram = df_bigram_skills.head(10) if not df_bigram_skills.empty else pd.DataFrame(columns=['unified_skill', 'frequency'])
# Unigram top skills are just the first 10 if sorted (which they usually are by frequency if available).
# Since df_unigram_final_skills likely only has skill names, we can't get 'top 10 by frequency' directly here.
# We will just show the first 10 skills from its list.
top_10_unigram = df_unigram_skills.head(10) if not df_unigram_skills.empty else pd.DataFrame(columns=['unified_skill'])

# --- Print Summary ---
print("\n--- Ringkasan Perbandingan Skill (Unigram vs. Bigram) ---")

summary_data = {
    'Metode': ['Unigram', 'Bigram'],
    'Jumlah Skill Unik': [unique_unigram_skills, unique_bigram_skills],
    'Total Frekuensi Teridentifikasi': [len(df_unigram_skills) if not df_unigram_skills.empty else 0, total_bigram_frequency] # For unigram, this is simply the count of skills, not sum of frequencies
}
df_summary = pd.DataFrame(summary_data)
display(df_summary)

print("\n--- Top 10 Bigram Skills ---")
display(top_10_bigram)

print("\n--- Top 10 Unigram Skills ---")
display(top_10_unigram)

## CELL 5: Interpretation

In [ ]:
print("\n--- Interpretasi Perbandingan Unigram vs. Bigram untuk Ekstraksi Skill ---")

print("Berdasarkan analisis perbandingan antara unigram dan bigram:")
print("\n1. **Spesifisitas Skill**:")
print("   - **Bigram** cenderung lebih spesifik dalam menangkap konsep skill. Misalnya, 'machine learning' adalah bigram yang langsung mengidentifikasi sebuah bidang keahlian. 'Data analysis' atau 'project management' juga merupakan contoh yang baik dari bigram yang menyampaikan makna skill secara utuh.")
print("   - **Unigram** lebih umum. Kata seperti 'data', 'management', 'system', atau 'software' adalah unigram yang berdiri sendiri. Meskipun penting, mereka seringkali membutuhkan konteks tambahan untuk sepenuhnya merepresentasikan sebuah skill. 'Data' bisa merujuk pada 'data analysis', 'data science', 'big data', dll.")

print("\n2. **Kesesuaian untuk Ekstraksi Skill**:")
print("   - Untuk **identifikasi skill yang tepat dan actionable**, **bigram (dan n-gram yang lebih tinggi seperti trigram)** umumnya lebih representatif. Mereka membantu membedakan nuansa antara berbagai jenis keahlian yang mungkin menggunakan kata dasar yang sama. Misalnya, memisahkan 'network security' dari hanya 'security'.")
print("   - **Unigram** masih memiliki peran dalam mengidentifikasi domain atau area fokus yang luas, tetapi kurang efektif jika tujuan utamanya adalah untuk mengidentifikasi daftar skill granular yang dapat dicocokkan dengan kualifikasi atau kebutuhan pekerjaan spesifik.")

print("\n3. **Analisis Terkombinasi**:")
print("   - Pendekatan terbaik seringkali melibatkan kombinasi keduanya. Unigram dapat memberikan gambaran umum tentang area dominan, sementara bigram dan trigram mengisi detail skill yang lebih spesifik di dalam area tersebut.")
print("   - Dalam konteks ekstraksi skill dari deskripsi pekerjaan atau kurikulum, **bigram dan trigram adalah pilihan yang lebih kuat** karena kemampuan mereka untuk menangkap frasa multibahasa yang secara intrinsik mendefinisikan suatu keahlian ('critical thinking', 'problem solving', 'software development').")

print("\nKesimpulan:")
print("**Bigram (dan n-gram > 1)** lebih disarankan sebagai representasi skill yang lebih akurat dan relevan untuk analisis demand-supply atau perbandingan kurikulum karena sifatnya yang lebih spesifik dan mampu menangkap konsep keahlian yang utuh.")

## min_df Eksperiment

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load the file industry_demand_processed.csv
try:
    df = pd.read_csv('industry_demand_processed.csv')
    print("Dataset 'industry_demand_processed.csv' loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: Make sure 'industry_demand_processed.csv' is uploaded to the Colab working directory. {e}")
    exit()

# Ensure job_description_processed is string type and handle NaNs safely
df['job_description_processed'] = df['job_description_processed'].astype(str).fillna('')

# Use only the job_description_processed column as input text
text_data = df['job_description_processed']

# Define min_df values to test
min_df_values = [3, 5, 7, 10, 15, 20]
summary_results = []

print("\n--- Experimenting with different min_df values ---")
for min_df_val in min_df_values:
    print(f"\nTesting min_df = {min_df_val}...")
    vectorizer = CountVectorizer(
        ngram_range=(1, 3),
        min_df=min_df_val,
        max_df=0.85
    )
    X_ngrams = vectorizer.fit_transform(text_data)
    ngram_vocabulary = vectorizer.get_feature_names_out()

    total_features = len(ngram_vocabulary)
    summary_results.append({'min_df': min_df_val, 'total_features': total_features})

    print(f"  Total N-grams (vocabulary size): {total_features}")
    print("  10 Sample N-grams:")
    for i, ngram in enumerate(ngram_vocabulary[:10]):
        print(f"    {i+1}. '{ngram}'")

# Display summary table
print("\n--- Summary Comparison of min_df values ---")
summary_df = pd.DataFrame(summary_results)
display(summary_df)

# Analysis
print("\n--- Analysis of min_df Experiment ---")
print("Berdasarkan tabel di atas:")

# Calculate percentage drop for analysis
summary_df['percentage_drop'] = (summary_df['total_features'].diff() / summary_df['total_features'].shift(1) * 100).abs()

first_features = summary_df.iloc[0]['total_features']
last_features = summary_df.iloc[-1]['total_features']
total_reduction_percent = ((first_features - last_features) / first_features) * 100

print(f"- Jumlah fitur awal (min_df=3): {first_features}")
print(f"- Jumlah fitur akhir (min_df=20): {last_features}")
print(f"- Total pengurangan fitur dari min_df=3 ke min_df=20 adalah sekitar {total_reduction_percent:.2f}%")

# Identify stabilization point
stable_range_start = None
for i in range(1, len(summary_df)):
    # Define 'stable' as a drop of less than 10% from the previous step.
    # This threshold can be adjusted if needed.
    if summary_df.loc[i, 'percentage_drop'] < 10:
        if stable_range_start is None:
            stable_range_start = summary_df.loc[i-1, 'min_df']

if stable_range_start:
    print(f"- Penurunan jumlah fitur mulai melambat (menjadi lebih stabil) di sekitar min_df >= {stable_range_start}.")
else:
    print("- Penurunan jumlah fitur masih cukup signifikan di seluruh rentang yang diuji, belum mencapai stabilitas.")

print("\nRekomendasi min_df yang paling rasional:")
print("   Untuk dataset \u00b17000 dokumen, nilai `min_df` yang lebih tinggi akan membantu mengurangi noise dari n-gram yang sangat jarang muncul, yang seringkali tidak relevan sebagai skill. ")
print("   Melihat hasil di atas, `min_df=10` atau `min_df=15` tampaknya merupakan titik yang baik.")
print("   `min_df=10` sudah mengurangi fitur secara signifikan sambil mempertahankan keragaman.")
print("   Jika prioritas adalah 'kebersihan' yang lebih tinggi dan toleransi kehilangan beberapa skill yang sangat jarang, `min_df=15` juga bisa menjadi pilihan yang kuat karena stabilitas mulai terlihat.")
print(f"   Saya merekomendasikan **`min_df=10`** sebagai titik awal yang seimbang, karena sudah mengurangi fitur sekitar {(first_features - summary_df[summary_df['min_df'] == 10]['total_features'].iloc[0]) / first_features * 100:.2f}% dari `min_df=3` (dari `{first_features}` menjadi `{summary_df[summary_df['min_df'] == 10]['total_features'].iloc[0]}`) sambil tetap mencakup n-gram yang muncul setidaknya 10 kali di seluruh korpus, yang cukup untuk dianggap signifikan.")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Assuming 'nlp', and all processing functions and global variables
# (is_valid_skill_candidate, clean_structural_prefix, handle_able_prefix,
# normalize_verb_to_noun, normalize_skill_term, tokenize_raw_skill_string,
# unify_single_token, canonical_synonym_map, sorted_word_canonical_map,
# apply_adverb_removal, is_structurally_valid, is_not_blacklisted,
# apply_strict_pos_pattern_filtering, NOISE_BLACKLIST, verb_to_noun_mapping,
# structural_prefixes_pattern, function_words, requirement_time_patterns,
# role_keywords, benefit_keywords, abstract_terms, generic_reverse_order_terms,
# indonesian_verbs, weak_terms, action_terms, education_requirement_terms)
# are already defined in the global scope from previous cell execution (e.g., cell 50e2f670).

# 1. Load the file industry_demand_processed.csv
try:
    df = pd.read_csv('industry_demand_processed.csv')
    print("Dataset 'industry_demand_processed.csv' loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: Make sure 'industry_demand_processed.csv' is uploaded to the Colab working directory. {e}")
    exit()

# Ensure job_description_processed is string type and handle NaNs safely
df['job_description_processed'] = df['job_description_processed'].astype(str).fillna('')

# Use only the job_description_processed column as input text
text_data = df['job_description_processed']

# Define min_df values to test
min_df_values = [7, 10, 15]
summary_results = []

print("\n--- Experimenting with different min_df values for FINAL skill count ---")
for min_df_val in min_df_values:
    print(f"\nProcessing with min_df = {min_df_val}...")
    vectorizer = CountVectorizer(
        ngram_range=(1, 3),
        min_df=min_df_val,
        max_df=0.85
    )
    X_ngrams = vectorizer.fit_transform(text_data)
    ngram_vocabulary = vectorizer.get_feature_names_out().tolist() # Get as list for iteration

    initial_ngrams_count = len(ngram_vocabulary)

    # --- Replicate the full skill extraction pipeline ---
    current_final_unique_skills_set = set()
    # skill_unification_mapping is not strictly needed for final count, but kept to mirror pipeline
    current_skill_unification_mapping = {}

    # Apply initial rule-based filtering (is_valid_skill_candidate)
    # This filter already removes non-skill candidates before structural cleaning
    filtered_ngram_candidates = [ngram for ngram in ngram_vocabulary if is_valid_skill_candidate(ngram)]

    # Process each candidate through structural cleaning, normalization, and unification
    for original_ngram_raw in filtered_ngram_candidates:
        # Layer 0: Existing Structural Prefix Cleaning (e.g., 'strong ability to communicate')
        cleaned_and_transformed_ngram = clean_structural_prefix(original_ngram_raw)

        # Layer 1: Handle 'Able' Prefix
        able_cleaned_ngram = handle_able_prefix(cleaned_and_transformed_ngram)

        # Layer 2: Verb To Noun Normalization
        verb_normalized_ngram = normalize_verb_to_noun(able_cleaned_ngram)
        if verb_normalized_ngram is None: # Skill dropped by Layer 2
            continue

        # Step 1: Tokenization (if the N-gram itself contains multiple skill mentions via delimiters)
        raw_sub_tokens_from_ngram = tokenize_raw_skill_string(verb_normalized_ngram)

        # Apply Normalization & Synonym Unification for each individual sub-token
        canonical_for_mapping = unify_single_token(verb_normalized_ngram, canonical_synonym_map, sorted_word_canonical_map)
        current_skill_unification_mapping[original_ngram_raw] = canonical_for_mapping

        for sub_token_raw in raw_sub_tokens_from_ngram:
            unified_form = unify_single_token(sub_token_raw, canonical_synonym_map, sorted_word_canonical_map)
            current_final_unique_skills_set.add(unified_form)

    after_matching_count = len(current_final_unique_skills_set) # Count after Layer 0, 1, 2 and unification

    # Convert to list for subsequent iterative filtering layers
    skills_for_further_processing = list(current_final_unique_skills_set)

    # Layer 3: Adverb Removal Rule
    adverb_removed_skills = []
    for skill in skills_for_further_processing:
        cleaned_skill = apply_adverb_removal(skill)
        if cleaned_skill is not None:
            adverb_removed_skills.append(cleaned_skill)

    # Layer 4: Structural Skill Validation
    structurally_validated_skills = []
    for skill in adverb_removed_skills:
        if is_structurally_valid(skill):
            structurally_validated_skills.append(skill)

    # Layer 5: Noise Blacklist
    final_filtered_skills_list = []
    for skill in structurally_validated_skills:
        if is_not_blacklisted(skill):
            final_filtered_skills_list.append(skill)

    # Layer 6: Strict POS Pattern Filtering
    strictly_pos_filtered_skills = []
    for skill in final_filtered_skills_list:
        filtered_skill = apply_strict_pos_pattern_filtering(skill, debug_mode=False) # No samples needed for debug in experiment
        if filtered_skill is not None:
            strictly_pos_filtered_skills.append(filtered_skill)

    final_skill_count = len(set(strictly_pos_filtered_skills)) # Ensure final uniqueness

    summary_results.append({
        'min_df': min_df_val,
        'initial_ngrams': initial_ngrams_count,
        'after_matching': after_matching_count,
        'final_skill_count': final_skill_count
    })

    print(f"  Final Skills count: {final_skill_count}")
    print("  10 Sample Final Skills:")
    # Get 10 samples from the final list of skills
    sorted_final_skills_sample = sorted(list(set(strictly_pos_filtered_skills)))[:10]
    for i, skill in enumerate(sorted_final_skills_sample):
        print(f"    {i+1}. '{skill}'")

# Display summary table
print("\n--- Summary Comparison of min_df values (based on FINAL skill count) ---")
summary_df = pd.DataFrame(summary_results)
display(summary_df)

# Analysis
print("\n--- Analysis of min_df Experiment (based on FINAL skill count) ---")
print("Berdasarkan tabel di atas:")

# Calculate percentage change for analysis
summary_df['percentage_change_final'] = (summary_df['final_skill_count'].diff() / summary_df['final_skill_count'].shift(1) * 100).abs()

first_final_count = summary_df.iloc[0]['final_skill_count']
last_final_count = summary_df.iloc[-1]['final_skill_count']

print(f"- Jumlah final skill awal (min_df={summary_df.iloc[0]['min_df']}): {first_final_count}")
print(f"- Jumlah final skill akhir (min_df={summary_df.iloc[-1]['min_df']}): {last_final_count}")

# Identify stabilization point for final skill count
stable_range_start_final = None
for i in range(1, len(summary_df)):
    if summary_df.loc[i, 'percentage_change_final'] < 5: # Threshold of 5% change for stability
        if stable_range_start_final is None:
            stable_range_start_final = summary_df.loc[i-1, 'min_df']

if stable_range_start_final:
    print(f"- Penurunan/perubahan jumlah final skill mulai melambat (menjadi lebih stabil) di sekitar min_df >= {stable_range_start_final}.")
else:
    print("- Penurunan/perubahan jumlah final skill masih cukup signifikan di seluruh rentang yang diuji, belum mencapai stabilitas.")

print("\nRekomendasi min_df yang paling rasional:")
print("   Tujuan dari `min_df` adalah untuk menghilangkan n-gram yang sangat jarang muncul, yang seringkali merupakan noise atau tidak merepresentasikan skill secara umum. Dengan menerapkan seluruh pipeline filtering:")
print(f"   - Dari `min_df={summary_df.iloc[0]['min_df']}` ke `min_df={summary_df.iloc[1]['min_df']}`, terjadi pengurangan final skill sebesar {summary_df['percentage_change_final'].iloc[1]:.2f}%")
print(f"   - Dari `min_df={summary_df.iloc[1]['min_df']}` ke `min_df={summary_df.iloc[2]['min_df']}`, terjadi pengurangan final skill sebesar {summary_df['percentage_change_final'].iloc[2]:.2f}%")
print("   Meskipun jumlah n-gram awal (`initial_ngrams`) berkurang drastis dengan meningkatnya `min_df`, jumlah skill akhir (`final_skill_count`) menunjukkan penurunan yang lebih moderat, yang mengindikasikan bahwa sebagian besar n-gram 'noise' awal memang sudah terfilter oleh pipeline.")
print("   ")
print(f"   Saya merekomendasikan **`min_df=10`** sebagai titik awal yang baik. Nilai ini memberikan keseimbangan antara mengurangi n-gram yang sangat jarang (yang kemungkinan besar adalah noise) dan mempertahankan skill yang cukup beragam. Perubahan jumlah final skill dari `min_df={summary_df.iloc[0]['min_df']}` ke `min_df={summary_df.iloc[1]['min_df']}` menunjukkan bahwa `min_df=10` sudah cukup efektif dalam memadatkan fitur tanpa kehilangan terlalu banyak informasi berharga setelah semua proses filtering lainnya diterapkan. Jika diinginkan daftar skill yang lebih ringkas, `min_df=15` juga dapat dipertimbangkan, namun mungkin berisiko menghilangkan beberapa skill yang jarang namun relevan.")

## Save Processed Industry Data to Google Drive

In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_INDUSTRY = os.path.join(OUTPUT_BASE_DIR, 'industry_data')

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_INDUSTRY = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/INDUSTRI/PREPROCESSED/'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_INDUSTRY}")

print(f"\nCopying files from '{OUTPUT_FOLDER_INDUSTRY}' to '{GOOGLE_DRIVE_TARGET_PATH_INDUSTRY}'...")

# Iterate through all files in the source directory and copy them
for filename in os.listdir(OUTPUT_FOLDER_INDUSTRY):
    source_filepath = os.path.join(OUTPUT_FOLDER_INDUSTRY, filename)
    destination_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_INDUSTRY, filename)

    try:
        shutil.copy2(source_filepath, destination_filepath)
        print(f"  Copied '{filename}'")
    except Exception as e:
        print(f"  Error copying '{filename}': {e}")

print("\nAll processed industry data saved to Google Drive.")

# CURRICULLUM




## 1. Persiapan: Instalasi Library dan Akses Google Drive

Pertama, kita perlu menginstal library yang diperlukan untuk membaca file PDF, DOCX, dan XLXS. Kemudian, kita akan mengaitkan (mount) Google Drive agar script dapat mengakses folder Anda.

In [ ]:
import sys

# Instal library yang dibutuhkan
!{sys.executable} -m pip install pdfplumber python-docx openpyxl pandas

print("Instalasi library selesai.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
Instalasi library selesai.


## 2. Ekstraksi dan Pembersihan Teks dari File Kurikulum

In [ ]:
import os

OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')

os.makedirs(OUTPUT_FOLDER_CURRICULUM, exist_ok=True)
print(f"Output directory for curriculum data created: {OUTPUT_FOLDER_CURRICULUM}")

Output directory for curriculum data created: /content/processed_data/curriculum_data


Bagian kode ini akan:
1.  Mendefinisikan fungsi untuk mengekstrak teks dari PDF, DOCX, dan XLSX.
2.  Mendefinisikan fungsi untuk membersihkan teks dasar (menghapus _newline_ berlebih, spasi ganda).
3.  Melakukan iterasi pada setiap file di folder Google Drive yang ditentukan.
4.  Memproses file sesuai tipenya dan mengekstrak teks.
5.  Menyimpan hasil ke dalam DataFrame.
6.  Mengekspor DataFrame ke file CSV.

In [ ]:
import os
import pdfplumber
import docx
import pandas as pd
import re

# --- Konfigurasi ---
# Path ke folder Google Drive Anda yang berisi kurikulum
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/2020"
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')
OUTPUT_CSV_NAME = os.path.join(OUTPUT_FOLDER_CURRICULUM, 'curriculum_combined_raw.csv')

# --- Fungsi Ekstraksi Teks ---
def extract_text_from_pdf(filepath):
    text = ""
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                text += page.extract_text(x_tolerance=1, y_tolerance=1) + "\n"
    except Exception as e:
        print(f"  Error extracting text from PDF {os.path.basename(filepath)}: {e}")
        return None
    return text

def extract_text_from_docx(filepath):
    text = ""
    try:
        document = docx.Document(filepath)
        for para in document.paragraphs:
            text += para.text + "\n"
    except Exception as e:
        print(f"  Error extracting text from DOCX {os.path.basename(filepath)}: {e}")
        return None
    return text

def extract_text_from_xlsx(filepath):
    text = ""
    try:
        # pandas read_excel akan membaca semua sheet secara default
        df_excel = pd.read_excel(filepath, sheet_name=None) # sheet_name=None membaca semua sheet ke dalam dict
        for sheet_name, sheet_df in df_excel.items():
            text += f"--- Sheet: {sheet_name} ---\n"
            # Gabungkan semua sel menjadi satu string per sheet
            for col in sheet_df.columns:
                text += sheet_df[col].astype(str).str.cat(sep=' ') + "\n"
    except Exception as e:
        print(f"  Error extracting text from XLSX {os.path.basename(filepath)}: {e}")
        return None
    return text

# --- Fungsi Pembantu untuk Mendapatkan Nama Semester ---
def get_semester_from_path(file_path, base_path):
    # Hapus base_path dari file_path
    relative_path = os.path.relpath(file_path, base_path)

    # Ambil direktori induk dari file
    parent_dir = os.path.dirname(relative_path)

    # Jika file langsung di base_path, semester bisa diisi 'Root_Folder' atau sesuai kebutuhan
    if not parent_dir or parent_dir == '.':
        return "Root_Folder"

    # Jika struktur folder adalah BASE_PATH/Semester_Ganjil_2020_2021/file.pdf
    # maka 'Semester_Ganjil_2020_2021' adalah nama semesternya.
    # Kita hanya perlu nama folder pertama setelah base_path
    semester_name = parent_dir.split(os.sep)[0]
    return semester_name

# --- Proses Utama ---
al_data_kurikulum = []

print(f"Memulai pemrosesan rekursif dari folder: {DRIVE_FOLDER_PATH}")

# Pastikan folder ada
if not os.path.exists(DRIVE_FOLDER_PATH):
    print(f"Error: Folder '{DRIVE_FOLDER_PATH}' tidak ditemukan. Pastikan path sudah benar dan Google Drive sudah di-mount.")
else:
    for root, dirs, files in os.walk(DRIVE_FOLDER_PATH):
        # Ambil nama semester dari path relatif terhadap DRIVE_FOLDER_PATH
        # Contoh: root = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/2020/Semester_1'
        # relative_root = 'Semester_1'
        # current_semester akan menjadi 'Semester_1'

        current_semester = get_semester_from_path(root, DRIVE_FOLDER_PATH)

        for filename in files:
            filepath = os.path.join(root, filename)

            text_content = None
            print(f"\nMemproses file: {filename} (Semester: {current_semester})")

            if filename.lower().endswith('.pdf'):
                text_content = extract_text_from_pdf(filepath)
            elif filename.lower().endswith('.docx'):
                text_content = extract_text_from_docx(filepath)
            elif filename.lower().endswith('.xlsx'):
                text_content = extract_text_from_xlsx(filepath)
            else:
                print(f"  Melewatkan file tidak didukung: {filename}")
                continue

            if text_content is not None:
                al_data_kurikulum.append({
                    'semester': current_semester,
                    'nama_file': filename,
                    'path_file': filepath,
                    'text_content': text_content
                })
            else:
                print(f"  Gagal mengekstrak teks dari {filename}. Melewatkan file ini.")

# Buat DataFrame
df_kurikulum = pd.DataFrame(al_data_kurikulum)

# Tampilkan informasi dan 5 baris pertama DataFrame
print("\n--- Ringkasan Data Kurikulum yang Diekstrak ---")
print(f"Total file yang berhasil diekstrak: {len(df_kurikulum)}")
print("Kolom DataFrame: ", df_kurikulum.columns.tolist())
display(df_kurikulum.head())

# --- Ekspor ke CSV ---
df_kurikulum.to_csv(OUTPUT_CSV_NAME, index=False)

print(f"\nData kurikulum berhasil disimpan ke '{OUTPUT_CSV_NAME}'.")

Memulai pemrosesan rekursif dari folder: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/2020

Memproses file: 093-2020 adendum SK Rektor ttg kurikulum TIF 2019 SK Rektor ttg Kurikulum Operasional TIF 2020-2021.pdf (Semester: Root_Folder)

Memproses file: Copy of FTK121 Aljabar Linear.pdf (Semester: Root_Folder)

Memproses file: Copy of TIF101 Sirkuit Elektronik.pdf (Semester: Root_Folder)

Memproses file: Copy of FTK111 Kalkulus 1.pdf (Semester: Root_Folder)

Memproses file: Copy of TIF107 Pengantar Teknologi Informasi.pdf (Semester: Root_Folder)

Memproses file: Copy of UNI104 English for Academic Purposes 1.pdf (Semester: Root_Folder)

Memproses file: Copy of FTK101 Algoritma & Pemrograman.pdf (Semester: Root_Folder)

Memproses file: Copy of TIF106 Konsep Sistem Informasi.pdf (Semester: Root_Folder)

Memproses file: Copy of UNI204 English AP2.pdf (Semester: Root_Folder)

Memproses file: Copy of FTK211 Kalkulus 2.pdf (Semester: Root_Folder)

Memproses file: Copy of FTK10

,semester,nama_file,path_file,text_content
0,Root_Folder,093-2020 adendum SK Rektor ttg kurikulum TIF 2...,/content/drive/MyDrive/TA_Jennifer Felicia/DAT...,KEPUTUSAN REKTOR UNIVERSITAS BAKRIE\nNOMOR: 09...
1,Root_Folder,Copy of FTK121 Aljabar Linear.pdf,/content/drive/MyDrive/TA_Jennifer Felicia/DAT...,Syllabus\nFTK121\nPage 1/3\n(Rencana Pembelaja...
2,Root_Folder,Copy of TIF101 Sirkuit Elektronik.pdf,/content/drive/MyDrive/TA_Jennifer Felicia/DAT...,SYLLABUs\nTIF101\nHal. 1/6\n- 1 -\nKode Mata K...
3,Root_Folder,Copy of FTK111 Kalkulus 1.pdf,/content/drive/MyDrive/TA_Jennifer Felicia/DAT...,FORM RENCANA PEMBELAJARAN SEMESTER (RPS) FTK11...
4,Root_Folder,Copy of TIF107 Pengantar Teknologi Informasi.pdf,/content/drive/MyDrive/TA_Jennifer Felicia/DAT...,SYLLABUs\nHal. 1/3\n- 1 -\nKode Mata Kuliah (C...



Data kurikulum berhasil disimpan ke '/content/processed_data/curriculum_data/curriculum_combined_raw.csv'.


In [ ]:
import pandas as pd
import re
import sys
import os

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')

# 1. Load dataset
try:
    input_filepath = os.path.join(OUTPUT_FOLDER_CURRICULUM, 'curriculum_combined_raw.csv')
    df_kurikulum = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully from '{OUTPUT_FOLDER_CURRICULUM}'.")
except FileNotFoundError as e:
    print(f"Error: The required file '{os.path.basename(input_filepath)}' was not found in '{OUTPUT_FOLDER_CURRICULUM}'.")
    print("Please ensure that the preceding cells in the 'Preparing Dataset and Preprocessing (Curriculum Data)' section have been executed successfully to generate this file.")
    df_kurikulum = pd.DataFrame() # Create an empty DataFrame to avoid further errors if execution continues
    # Optionally re-raise the exception for clearer failure in some contexts:
    # raise

# Exit if df_kurikulum is empty due to FileNotFoundError
if df_kurikulum.empty:
    print("Exiting further processing because df_kurikulum is empty. Please check the file path and execution of previous cells.")
    sys.exit(1)

# Ensure text_content is string type and handle NaNs safely
df_kurikulum['text_content'] = df_kurikulum['text_content'].astype(str).fillna('')

# --- NEW: Function to detect major sections (Rule 1) ---
def is_major_section(line):
    # Mendeteksi pola seperti: "III. TUJUAN ..." atau "A. PENDAHULUAN"
    return bool(re.match(r'^([IVXLCDM]+\.|[A-Z]\.)\s+[A-Z]', line.strip()))

# --- NEW: Function to normalize text structure (Rule 2) ---
def normalize_text_structure(text):
    lines = text.split('\n')
    processed_lines = []
    buffer = []

    # Patterns for list items (1., a., i.) and smaller sections (A., B., C. for sub-sections)
    list_start_pattern = re.compile(r'^(\d+\.|[a-z]\.|[ivxlcdm]+\.)')
    section_start_pattern = re.compile(r'^([A-Z]\.|[IVXLCDM]+\.)')

    def flush_buffer():
        if buffer:
            processed_lines.append(' '.join(buffer))
            buffer.clear()

    for line in lines:
        stripped_line = line.strip()
        if not stripped_line:
            flush_buffer()
            continue

        # Check for major section, smaller section, or list item start (Rule 2.a, b, c)
        is_new_major_section_start = is_major_section(stripped_line)
        is_new_list_start = list_start_pattern.match(stripped_line)
        is_new_section_start = section_start_pattern.match(stripped_line)

        if is_new_major_section_start or is_new_list_start or is_new_section_start:
            flush_buffer()
            buffer.append(stripped_line)
        else:
            if not buffer: # If buffer is empty, start a new one
                buffer.append(stripped_line)
            else: # Otherwise, append to existing buffer (line continuation)
                buffer.append(stripped_line)
    flush_buffer()
    return '\n'.join(processed_lines)

# --- Perbaiki fungsi force_split_sections (Rule 3) ---
def force_split_sections(text):
    # Updated pattern: (?=\s[A-Z]\.\s)|(?=\s[IVXLCDM]+\.\s+[A-Z])
    split_pattern = r'(?=\s[A-Z]\.\s)|(?=\s[IVXLCDM]+\.\s+[A-Z])'
    sections = re.split(split_pattern, text)
    return '\n'.join([s.strip() for s in sections if s.strip()])

# --- Fungsi untuk mendeteksi false positive visi_misi (Rule 4) ---
def is_false_positive_visi_misi(text):
    # Hapus kata terlalu umum seperti "pendidikan" dan "kurikulum"
    # Gunakan hanya yang lebih spesifik.
    keywords = [
        'tujuan pendidikan',
        'sasaran utama',
        'strategi pengembangan'
    ]
    return not any(keyword in text.lower() for keyword in keywords)

# --- NEW: Fungsi untuk mendeteksi baris visi/misi ---
def is_visi_misi_line(line):
    return bool(re.match(r'^(visi|misi)\s*\d*\.?\s*', line.lower().strip()))

# --- NEW: Fungsi untuk mendeteksi baris list misi ---
def is_mission_list(line):
    return bool(re.match(r'^\d+\.', line.strip()))


# 3. Buat fungsi untuk extract section (Existing function from previous interaction)
def extract_sections(text_content, nama_file):
    sections = []

    # Define keywords and their categories, using regex for patterns
    section_keywords_patterns = {
        # VISI MISI
        r'\bvisi\b|\bmisi\b': 'visi_misi',

        # PROFIL LULUSAN
        r'\bprofil lulusan\b|\bpl\b|\bpl\s*\d+\b': 'profil_lulusan',

        # CPL
        r'\bcapaian pembelajaran\b|\bcpl\b': 'cpl'
        # Removed metode_pembelajaran from here as per instructions
    }

    # Collect all potential boundary markers (keywords and major sections)
    # Each marker is (index, type, original_text_of_marker, category_suggestion)
    all_boundary_markers = []

    # 1. Add keyword matches
    for pattern, category_key in section_keywords_patterns.items():
        for match in re.finditer(pattern, text_content.lower()):
            # For keyword matches, capture the text around the keyword to give context to the boundary
            start_idx = match.start()
            end_idx = match.end()
            # Capture a small window of text for context, or just the matched keyword
            marker_text = text_content[start_idx:end_idx]
            all_boundary_markers.append((start_idx, 'keyword', marker_text, category_key))

    # 2. Add major section headers as boundaries
    current_line_start_idx = 0
    for line in text_content.split('\n'):
        if is_major_section(line):
            # Ensure the major section line is captured as a boundary
            all_boundary_markers.append((current_line_start_idx, 'major_section', line, None))
        current_line_start_idx += len(line) + 1 # +1 for the newline character

    # Sort all markers by their start index
    all_boundary_markers.sort(key=lambda x: x[0])

    current_pos = 0
    current_active_category = 'others'
    for start_idx, marker_type, marker_text_original, category_suggestion in all_boundary_markers:
        # Extract text BEFORE the current marker
        if start_idx > current_pos:
            prior_text = text_content[current_pos:start_idx].strip()
            if prior_text:
                sections.append({'kategori': current_active_category, 'text': prior_text})

        # Process the marker itself
        if marker_type == 'major_section':
            sections.append({'kategori': 'others', 'text': marker_text_original.strip()})
            current_active_category = 'others' # Reset category to others after a major section
            current_pos = start_idx + len(marker_text_original) # Move current_pos past the marker

        elif marker_type == 'keyword':
            # Find the next boundary after this keyword to define its section extent
            next_boundary_idx = len(text_content)
            for next_marker_idx, next_marker_type, _, _ in all_boundary_markers:
                if next_marker_idx > start_idx:
                    next_boundary_idx = next_marker_idx
                    break

            section_text = text_content[start_idx:next_boundary_idx].strip()

            # Apply Rule 5.e: Filter 'visi_misi' false positives
            final_category = category_suggestion

            # PRIORITAS PALING ATAS: Jika baris adalah visi/misi, langsung kategorikan
            if is_visi_misi_line(section_text.split('\n')[0]): # Check only the first line of the section_text
                final_category = 'visi_misi'
            elif category_suggestion == 'visi_misi' and is_false_positive_visi_misi(section_text): # Apply false positive check only if not already confirmed as visi_misi line
                final_category = 'others' # Reassign to 'others' if false positive

            if section_text:
                # Rule: Tambahkan rule untuk mempertahankan list misi
                if final_category == 'visi_misi':
                    current_visi_misi_lines = []
                    for line_in_section in section_text.split('\n'):
                        if is_visi_misi_line(line_in_section) or is_mission_list(line_in_section) or not line_in_section.strip():
                            current_visi_misi_lines.append(line_in_section)
                        else:
                            # If a line breaks the pattern, it's a new section
                            if current_visi_misi_lines:
                                sections.append({'kategori': final_category, 'text': '\n'.join(current_visi_misi_lines).strip()})
                                current_visi_misi_lines = []
                            # Treat the non-matching line as 'others' and restart tracking
                            sections.append({'kategori': 'others', 'text': line_in_section.strip()})
                            final_category = 'others' # Reset category
                    if current_visi_misi_lines:
                        sections.append({'kategori': final_category, 'text': '\n'.join(current_visi_misi_lines).strip()})
                else:
                    sections.append({'kategori': final_category, 'text': section_text})
                current_active_category = final_category # Update active category for subsequent non-marked text
            current_pos = next_boundary_idx

    # Handle any remaining text after the last detected section
    if current_pos < len(text_content):
        remaining_text = text_content[current_pos:].strip()
        if remaining_text:
            sections.append({'kategori': current_active_category, 'text': remaining_text})

    # If no sections were extracted at all, categorize the entire content as 'others'
    if not sections and text_content.strip():
        sections.append({'kategori': 'others', 'text': text_content.strip()})

    return sections

print("\nApplying text structure normalization...")
df_kurikulum['text_content'] = df_kurikulum['text_content'].apply(normalize_text_structure)

print("Applying force section splitting...")
df_kurikulum['text_content'] = df_kurikulum['text_content'].apply(force_split_sections)

# --- Fungsi untuk mendeteksi false positive visi_misi (Rule 4) ---
def is_false_positive_visi_misi(text):
    keywords = [
        'tujuan pendidikan',
        'sasaran utama',
        'strategi pengembangan'
    ]
    # Return True if NONE of the specific keywords are found (meaning it's a false positive visi_misi)
    return not any(keyword in text.lower() for keyword in keywords)


# Helper function to check if a file is an 'SK' file
def is_sk_file(nama_file):
    return 'sk' in nama_file.lower()


print("\nApplying section extraction...")
# Apply the function to each row and flatten the list of lists
all_extracted_sections = []

# Now loop through all files in df_kurikulum and apply the filtering logic AFTER extraction
for index, row in df_kurikulum.iterrows():
    extracted = extract_sections(row['text_content'], row['nama_file'])
    is_current_file_sk = is_sk_file(row['nama_file']) # Determine once per file

    for sec in extracted:
        kategori = sec['kategori']
        text_content_to_add = sec['text']

        if kategori in ['visi_misi', 'profil_lulusan', 'cpl']:
            if is_current_file_sk:
                all_extracted_sections.append({'kategori': kategori, 'text': text_content_to_add})
        elif kategori == 'others': # Handle 'others' category specifically
            if not is_current_file_sk: # Only add if it's a non-SK file
                all_extracted_sections.append({'kategori': 'kompetensi', 'text': text_content_to_add})

# 5. Output DataFrame baru dengan struktur: kategori, text (Intermediate structured data)
df_structured_curriculum = pd.DataFrame(all_extracted_sections)
df_structured_curriculum = df_structured_curriculum[df_structured_curriculum['text'].str.strip() != ''].reset_index(drop=True)

# --- Display Summary for Initial Structured Data ---
print("\n--- Summary of Extracted Sections (Initial Section-Based) ---")
print("Jumlah data per kategori:")
display(df_structured_curriculum['kategori'].value_counts())

print("\nContoh isi teks dari setiap kategori (Initial Section-Based):")
for category in df_structured_curriculum['kategori'].unique():
    print(f"\nKategori: {category}")
    sample_text = df_structured_curriculum[df_structured_curriculum['kategori'] == category]['text'].iloc[0]
    print(f"{sample_text[:500]}...\n")

# --- Implementing Learning Methods Extraction (NEW, IMPROVED) ---
print("\n--- Implementing Learning Methods Extraction from Flattened Text (Improved) ---")

def extract_learning_method_section(text, file_name=None):

    section_groups = {
        'methods_of_instruction': [
            'methods of instructions',
            'methods of instruction',
            'metode pembelajaran',
            'methods of instructions/metode pembelajaran',
            'methods of instruction/metode pembelajaran'
        ],

        'course_outline': [
            'course outline',
            'garis besar materi',
            'pokok bahasan'
        ]
    }

    fallback_keywords = [
        'session targeted competencies',
        'session target competencies',
        'targeted session'
    ]

    stop_keywords = [
        'uas',
        'final exam',
        'final examination',
        'final semester test',
        'attendance requirement'
    ]

    all_keywords = []

    for keywords in section_groups.values():
        all_keywords.extend(keywords)

    all_keywords.extend(fallback_keywords)
    all_keywords.extend(stop_keywords)

    all_section_keywords_regex = '|'.join(
        re.escape(x)
        for x in all_keywords
    )

    extracted_sections = []

    # =====================================================
    # METHODS OF INSTRUCTION + COURSE OUTLINE
    # =====================================================

    for section_name, keywords in section_groups.items():

        section_found = False

        for keyword in keywords:

            pattern = (
                rf'{re.escape(keyword)}'
                rf'\s*[:\-]?\s*'
                rf'(.*?)'
                rf'(?='
                rf'\n\s*(?:{all_section_keywords_regex})\s*(?:\n|:|-)'
                rf'|$)'
            )

            match = re.search(
                pattern,
                text,
                re.IGNORECASE | re.DOTALL
            )

            if match:

                content = match.group(1).strip()

                for unwanted in stop_keywords:

                    content = re.split(
                        rf'\b{re.escape(unwanted)}\b',
                        content,
                        flags=re.IGNORECASE
                    )[0]

                content = content.strip()

                if content:

                    extracted_sections.append(content)

                    section_found = True
                    break

        # =================================================
        # FALLBACK hanya untuk COURSE OUTLINE
        # =================================================

        if (
            section_name == 'course_outline'
            and not section_found
        ):

            for keyword in fallback_keywords:

                pattern = (
                    rf'{re.escape(keyword)}'
                    rf'\s*[:\-]?\s*'
                    rf'(.*?)'
                    rf'(?='
                    rf'\n\s*(?:{all_section_keywords_regex})\s*(?:\n|:|-)'
                    rf'|$)'
                )

                match = re.search(
                    pattern,
                    text,
                    re.IGNORECASE | re.DOTALL
                )

                if match:

                    content = match.group(1).strip()

                    if content:

                        extracted_sections.append(content)
                        break

    # =====================================================
    # RETURN GABUNGAN SEMUA SECTION
    # =====================================================

    return "\n\n".join(extracted_sections)

# Apply the new extraction function to the original df_kurikulum['text_content']
all_learning_methods = []

# Ensure methods only from NON-SK
# Pisahkan dataset berdasarkan jenis file (re-creating df_non_sk for learning methods extraction)
df_non_sk = df_kurikulum[
    ~df_kurikulum['nama_file'].str.contains(
        r'\bsk\b',
        case=False,
        na=False,
        regex=True
    )
]

for _, row in df_non_sk.iterrows():

    extracted_text = extract_learning_method_section(
        row['text_content']
    )

    is_magang = 'magang' in row['nama_file'].lower()

    if extracted_text or is_magang:

        all_learning_methods.append({
            'nama_file': row['nama_file'],
            'kategori': 'metode_pembelajaran',
            'text': extracted_text
        })

df_methods = pd.DataFrame(all_learning_methods)

# \U0001f525 GABUNGKAN DENGAN DATA SEBELUMNYA:
df_final = pd.concat([df_structured_curriculum, df_methods], ignore_index=True)
df_final = df_final.drop_duplicates().reset_index(drop=True)

# 7. Tampilkan summary for FINAL data
print("\n--- Summary of Extracted Sections (Final Combined) ---")
print("Jumlah data per kategori:")
display(df_final['kategori'].value_counts())

print("\nContoh isi teks dari setiap kategori (Final Combined):")
for category in df_final['kategori'].unique():
    print(f"\nKategori: {category}")
    # Display the first non-empty text entry for each category
    sample_text = df_final[df_final['kategori'] == category]['text'].iloc[0]
    print(f"{sample_text[:500]}...\n")

# \U0001f4bd SIMPAN ke: '/content/processed_data/curriculum_data/curriculum_structured_final.csv'
output_filepath_final = os.path.join(OUTPUT_FOLDER_CURRICULUM, 'curriculum_structured_final.csv')
df_final.to_csv(output_filepath_final, index=False)
print(f"Final structured curriculum data saved to '{output_filepath_final}'.")

Dataset 'curriculum_combined_raw.csv' loaded successfully from '/content/processed_data/curriculum_data'.

Applying text structure normalization...
Applying force section splitting...

Applying section extraction...

--- Summary of Extracted Sections (Initial Section-Based) ---
Jumlah data per kategori:


,count
kategori,
kompetensi,548
cpl,20
visi_misi,14
profil_lulusan,14



Contoh isi teks dari setiap kategori (Initial Section-Based):

Kategori: visi_misi
VISI DAN...


Kategori: cpl
Capaian Pembelajaran Lulusan (...


Kategori: profil_lulusan
Profil Lulusan dan jenjang KKNI/SKKNI....


Kategori: kompetensi
Syllabus FTK121 Page 1/3 (Rencana Pembelajaran Semester) Course Code: Course Name: FTK121 Linear Algebra Study Program: Faculty: Informatics Engineering and Computer Science Course Pre-requisite: Credit: 3 – Revision Date: Lecture: Tutorial: Practicum: March 7, 2021 3 0 0 Revision Status: Semester: Genap/Even
2.0 Academic Year: 2020/2021 Schedule: Wednesday, 18.35–21.05 WIB Lecturer’s Name: Berkah...


--- Implementing Learning Methods Extraction from Flattened Text (Improved) ---

--- Summary of Extracted Sections (Final Combined) ---
Jumlah data per kategori:


,count
kategori,
kompetensi,330
metode_pembelajaran,54
cpl,20
profil_lulusan,14
visi_misi,12



Contoh isi teks dari setiap kategori (Final Combined):

Kategori: visi_misi
VISI DAN...


Kategori: cpl
Capaian Pembelajaran Lulusan (...


Kategori: profil_lulusan
Profil Lulusan dan jenjang KKNI/SKKNI....


Kategori: kompetensi
Syllabus FTK121 Page 1/3 (Rencana Pembelajaran Semester) Course Code: Course Name: FTK121 Linear Algebra Study Program: Faculty: Informatics Engineering and Computer Science Course Pre-requisite: Credit: 3 – Revision Date: Lecture: Tutorial: Practicum: March 7, 2021 3 0 0 Revision Status: Semester: Genap/Even
2.0 Academic Year: 2020/2021 Schedule: Wednesday, 18.35–21.05 WIB Lecturer’s Name: Berkah...


Kategori: metode_pembelajaran
Virtual classroom instruction consists of lectures and practical problem solving, supplemented by visual aids designed to assist the student to successfully meet the course’s learning objectives and the interactive discussions in virtual classroom through academic portal Bakrie Information Gateway v2.0. It is imperative that studen

In [ ]:
import os
import pandas as pd

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')

# Load df_final (assuming it's saved as 'curriculum_structured_final.csv' from the previous step)
try:
    df_final = pd.read_csv(os.path.join(OUTPUT_FOLDER_CURRICULUM, 'curriculum_structured_final.csv'))
    print(f"Loaded df_final from '{os.path.join(OUTPUT_FOLDER_CURRICULUM, 'curriculum_structured_final.csv')}'")
except FileNotFoundError:
    print("Error: 'curriculum_structured_final.csv' not found. Please ensure the previous cell was executed.")
    # Initialize an empty DataFrame to prevent errors if the file is not found
    df_final = pd.DataFrame(columns=['kategori', 'text'])

print("\n--- Creating category-specific folders and saving datasets ---")

# Get unique categories from the DataFrame
categories = df_final['kategori'].unique()

for kategori in categories:
    # 1. Buat folder per kategori jika belum ada
    # Contoh: /content/processed_data/curriculum_data/visi_misi/
    category_output_dir = os.path.join(OUTPUT_FOLDER_CURRICULUM, kategori)
    os.makedirs(category_output_dir, exist_ok=True)
    print(f"Created directory: {category_output_dir}")

    # 2. Filter dataframe sesuai kategori
    df_subset = df_final[df_final['kategori'] == kategori].copy() # Use .copy() to avoid SettingWithCopyWarning

    # 3. Bersihkan duplikat dalam subset sebelum disimpan
    # Pastikan tidak ada data kosong yang ikut tersimpan
    df_subset = df_subset.drop_duplicates().reset_index(drop=True)
    # Filter out empty 'text' content, though this should largely be handled upstream
    df_subset = df_subset[df_subset['text'].astype(str).str.strip() != '']


    # 4. Simpan ke folder masing-masing dengan penamaan _raw.csv
    # Contoh: /content/processed_data/curriculum_data/visi_misi/visi_misi_raw.csv
    filename = f"{kategori}_raw.csv"
    filepath = os.path.join(category_output_dir, filename)

    df_subset.to_csv(filepath, index=False)
    print(f"Saved: {filename} ({len(df_subset)} rows) to {category_output_dir}")

print("\nDataset splitting and saving by category complete.")

Loaded df_final from '/content/processed_data/curriculum_data/curriculum_structured_final.csv'

--- Creating category-specific folders and saving datasets ---
Created directory: /content/processed_data/curriculum_data/visi_misi
Saved: visi_misi_raw.csv (12 rows) to /content/processed_data/curriculum_data/visi_misi
Created directory: /content/processed_data/curriculum_data/cpl
Saved: cpl_raw.csv (20 rows) to /content/processed_data/curriculum_data/cpl
Created directory: /content/processed_data/curriculum_data/profil_lulusan
Saved: profil_lulusan_raw.csv (14 rows) to /content/processed_data/curriculum_data/profil_lulusan
Created directory: /content/processed_data/curriculum_data/kompetensi
Saved: kompetensi_raw.csv (330 rows) to /content/processed_data/curriculum_data/kompetensi
Created directory: /content/processed_data/curriculum_data/metode_pembelajaran
Saved: metode_pembelajaran_raw.csv (54 rows) to /content/processed_data/curriculum_data/metode_pembelajaran

Dataset splitting and sa

## Curriculum Data - Category-Specific Cleaning

In [ ]:
import pandas as pd
import re
import os
import numpy as np # For handling NaN values
from collections import Counter

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')

# --- General Text Normalization Function (reused for consistency) ---
def normalize_text_general(text):
    if pd.isna(text) or not isinstance(text, str):
        return ''
    # Convert to lowercase
    text = text.lower()
    # Remove page references like 'halaman x dari y' or 'page x/y'
    text = re.sub(r'\b(halaman|page)\s*\d+\s*(dari|of|\/)\s*\d+', '', text, flags=re.IGNORECASE)
    # Replace multiple spaces with a single space and strip leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Helper function to detect general section headers for stopping conditions
def is_generic_section_header(line):
    line = line.strip()

    # ALL CAPS (more flexible, up to 10 words)
    is_caps = line.isupper() and len(line.split()) <= 10

    # Title Case (results from PDF sometimes like this, 1-6 words)
    is_title = bool(re.match(r'^(?:[A-Z][a-z]*\s){0,5}[A-Z][a-z]*$', line))

    # Contains common section words (e.g., 'learning', 'instruction', 'overview')
    has_section_word = any(word in line.lower() for word in [
        'learning', 'instruction', 'overview', 'summary', 'evaluation', 'goals', 'objectives', 'assessment', 'methodology', 'introduction', 'conclusion', 'references', 'lecturers', 'schedule', 'profil', 'cpl', 'metode' # Added profil, cpl, metode as stop words
    ])

    # Combining conditions
    if (is_caps or is_title or has_section_word) and len(line) > 2:
        return True
    return False

# Function to detect major sections (copied from 138214d2 to ensure it's available and consistent)
def is_major_section_for_stopping(line):
    # Mendeteksi pola seperti: "III. TUJUAN ..." atau "A. PENDAHULUAN"
    return bool(re.match(r'^([IVXLCDM]+\.|[A-Z]\.)\\s+[A-Z]', line.strip()))

# Combined stopping condition for visi misi
def is_stopping_header(line):
    return is_major_section_for_stopping(line) or is_generic_section_header(line)

# --- Cleaning Functions per Category ---
def clean_visi_misi(df):
    print("  Applying Visi Misi cleaning (correct slicing)...")

    df = df.copy()
    df['text'] = df['text'].astype(str).fillna('')

    # 1. Gabungkan semua text
    full_text = ' '.join(df['text'].tolist())
    full_text = re.sub(r'\s+', ' ', full_text)

    results = []

    # =========================
    # STEP 1: CARI POSISI HEADER
    # =========================
    match = re.search(r'(visi\s*program\s*studi|misi\s*program\s*studi)', full_text, re.IGNORECASE)

    if not match:
        print("  Warning: header tidak ditemukan")
        return pd.DataFrame(columns=['kategori', 'text_clean'])

    start_idx = match.end()  # \U0001f525 ambil SETELAH header

    # =========================
    # STEP 2: AMBIL ISI SETELAHNYA
    # =========================
    content = full_text[start_idx:]

    # =========================
    # STEP 3: SPLIT VISI & MISI
    # =========================
    content_lower = content.lower()

    # Cari posisi
    visi_idx = content_lower.find('visi')
    misi_idx = content_lower.find('misi')

    results = []

    # =========================
    # VISI
    # =========================
    if visi_idx != -1:
        if misi_idx != -1 and misi_idx > visi_idx:
            visi_text = content[visi_idx:misi_idx]
        else:
            visi_text = content[visi_idx:]

        visi_text = re.sub(r'visi\s*(program studi)?\s*:?', '', visi_text, flags=re.IGNORECASE)
        # Removed: visi_text = re.sub(r'\b(\d+\.)\s*', '', visi_text)
        visi_text = normalize_text_general(visi_text)

        if visi_text:
            results.append({
                'kategori': 'visi',
                'text_clean': visi_text
            })

    # =========================
    # MISI
    # =========================
    if misi_idx != -1:
        misi_text = content[misi_idx:]

        misi_text = re.sub(r'misi\s*(program studi)?\s*:?', '', misi_text, flags=re.IGNORECASE)
        # Removed: misi_text = re.sub(r'\\b(\\d+\\.)\\s*', '', misi_text)

        if misi_text:
            normalized_misi_text = normalize_text_general(misi_text)

            # Split misi_text by number
            numbered_misi_items = re.findall(r'(\d+\.\s*.*?)(?:\n|\Z)', normalized_misi_text, re.DOTALL)

            if numbered_misi_items:
                for item_content in numbered_misi_items:
                    item_clean = item_content.strip()
                    if item_clean:
                        results.append({
                            'kategori': 'misi',
                            'text_clean': item_clean
                        })
            else:
                # If no numbered items found, add the entire normalized_misi_text as one item
                if normalized_misi_text:
                    results.append({
                        'kategori': 'misi',
                        'text_clean': normalized_misi_text
                    })

    df_result = pd.DataFrame(results)

    # Filter out empty texts after cleaning
    df_result = df_result[df_result['text_clean'] != ''].reset_index(drop=True)

    # Ensure empty DataFrame has expected columns if no data was processed after filtering
    if df_result.empty:
        return pd.DataFrame(columns=['kategori', 'text_clean'])

    print(f"  Visi Misi cleaning complete. {len(df_result)} records remaining.")

    return df_result[['kategori', 'text_clean']]

import pandas as pd
import re

# =========================================================
# FUNCTION CLEAN PROFIL LULUSAN
# =========================================================

def clean_profil_lulusan(df):

    print("Applying Profil Lulusan cleaning...")

    # -----------------------------------------------------
    # Copy dataframe agar data asli tidak berubah
    # -----------------------------------------------------

    df_cleaned = df.copy()

    # -----------------------------------------------------
    # Pastikan semua data berupa string
    # -----------------------------------------------------

    df_cleaned['text'] = df_cleaned['text'].astype(str).fillna('')

    # -----------------------------------------------------
    # Gabungkan seluruh text menjadi 1 string
    # -----------------------------------------------------

    full_text = ' '.join(df['text'].tolist())

    # Rapikan spasi berlebih
    full_text = re.sub(r'\s+', ' ', full_text)

    # =====================================================
    # STEP 1 — CARI HEADER "dokumen KO:"
    # =====================================================

    match = re.search(
        r'dokumen\s*ko\s*:',
        full_text,
        re.IGNORECASE
    )

    # Jika header tidak ditemukan
    if not match:
        print("Header 'dokumen KO:' tidak ditemukan")
        return pd.DataFrame(columns=['kategori', 'text_clean'])

    # -----------------------------------------------------
    # Ambil isi setelah header
    # -----------------------------------------------------

    start_idx = match.end()

    content = full_text[start_idx:]

    # =====================================================
    # STEP 2 — HENTIKAN SAAT MASUK "PEMETAAN PL"
    # =====================================================

    pemetaan_match = re.search(
        r'pemetaan\s*pl',
        content,
        re.IGNORECASE
    )

    # Jika ditemukan "Pemetaan PL"
    # ambil hanya isi sebelum bagian tersebut

    if pemetaan_match:
        content = content[:pemetaan_match.start()]

    # =====================================================
    # STEP 3 — CARI SEMUA PL
    # =====================================================

    pl_matches = list(
        re.finditer(
            r'PL\d+\s*:',
            content,
            re.IGNORECASE
        )
    )

    results = []

    # Jika tidak ada PL ditemukan
    if not pl_matches:
        print("Tidak ditemukan profil lulusan")
        return pd.DataFrame(columns=['kategori', 'text_clean'])

    # =====================================================
    # STEP 4 — LOOP SETIAP PL
    # =====================================================

    for i in range(len(pl_matches)):

        # Posisi awal PL
        start = pl_matches[i].start()

        # Posisi akhir PL
        if i < len(pl_matches) - 1:
            end = pl_matches[i + 1].start()
        else:
            end = len(content)

        # Ambil text tiap PL
        pl_text = content[start:end].strip()

        # ============================================
        # Ambil kode PL (PL1, PL2, PL3, ...)
        # ============================================

        kode_match = re.search(r'(PL\d+)\s*:', pl_text, re.IGNORECASE)

        if kode_match:
            kode_pl = kode_match.group(1).upper()
        else:
            kode_pl = None

        # ============================================
        # Hapus label PL dari isi teks
        # ============================================

        pl_text = re.sub(
            r'PL\d+\s*:',
            '',
            pl_text,
            flags=re.IGNORECASE
        )

        # Rapikan spasi
        pl_text = re.sub(
            r'\s+',
            ' ',
            pl_text
        ).strip()

        # ============================================
        # Simpan hasil
        # ============================================

        if pl_text:
            results.append({
                'kode_profil_lulusan': kode_pl,
                'kategori': 'profil_lulusan',
                'text_clean': pl_text
            })
    # =====================================================
    # UBAH KE DATAFRAME
    # =====================================================

    df_result = pd.DataFrame(results)

    print(
        f"Cleaning selesai. "
        f"Total profil lulusan: {len(df_result)}"
    )

    df_result = df_result[df_result['text_clean'] != ''].reset_index(drop=True)

    # Safeguard against empty df_result after filtering
    if df_result.empty:
        return pd.DataFrame(columns=['kategori', 'text_clean'])

    print(f"  Profil Lulusan cleaning complete. {len(df_result)} records remaining.")
    return df_result[['kode_profil_lulusan', 'kategori', 'text_clean']]

def process_profil_lulusan_text(text):
    # Hilangkan karakter selain huruf, angka, dan spasi
    text = re.sub(r'[^a-z0-9\s]', '', text, flags=re.IGNORECASE)

    return normalize_text_general(text)

def clean_cpl(df):
    """Applies specific cleaning rules for 'cpl' category with new format."""
    print("  Applying CPL cleaning (new format)...")
    df_cleaned = df.copy()
    df_cleaned['text'] = df_cleaned['text'].astype(str).fillna('')

    all_processed_entries = []

    # Regex patterns
    kode_cpl_pattern = r'(CPL-[A-Z]+-\d{2})'
    kkni_pattern = r'KKNI\s*Level\s*(\d+)\s*No\.?\s*(\d+)'

    for idx, row in df_cleaned.iterrows():
        original_text = row['text']

        kode_cpl = None
        kkni_level = None
        kkni_no = None
        text_description = original_text # Start with full text, then remove parts

        # 2.1 Ekstrak Kode CPL
        match_kode_cpl = re.search(kode_cpl_pattern, original_text)
        if match_kode_cpl:
            kode_cpl = match_kode_cpl.group(1).upper() # Keep case as found
            text_description = text_description.replace(match_kode_cpl.group(0), '', 1).strip() # Remove the matched part, 1st occurrence only

        # Filtering WAJIB: Jika tidak ada kode_cpl -> DROP
        if kode_cpl is None:
            # print(f"  Dropped CPL entry due to missing kode_cpl: {original_text[:50]}...")
            continue # Skip this row

        # 2.2 Ekstrak KKNI Level dan Nomor
        match_kkni = re.search(kkni_pattern, original_text)
        if match_kkni:
            kkni_level = int(match_kkni.group(1))
            kkni_no = int(match_kkni.group(2))
            text_description = text_description.replace(match_kkni.group(0), '', 1).strip() # Remove the matched part, 1st occurrence only

        # 2.3 Ambil Isi CPL (after removing kode_cpl and KKNI parts)
        # Further clean the remaining description text
        clean_text_description = re.sub(r'[^a-z0-9\s]', '', text_description, flags=re.IGNORECASE) # Remove symbols, keep alphanumeric
        clean_text_description = normalize_text_general(clean_text_description) # Normalize whitespace

        if clean_text_description: # Only add if there's meaningful description left
            all_processed_entries.append({
                'kategori': row['kategori'],
                'kode_cpl': kode_cpl,
                'kkni_level': kkni_level, # Can be None
                'kkni_no': kkni_no,       # Can be None
                'text_clean': clean_text_description
            })

    df_result = pd.DataFrame(all_processed_entries)

    df_result = df_result[df_result['text_clean'] != ''].reset_index(drop=True)

    # Safeguard: If df_result is empty *after* filtering, ensure it has the expected columns
    if df_result.empty:
        return pd.DataFrame(columns=['kategori', 'kode_cpl', 'kkni_level', 'kkni_no', 'text_clean'])

    print(f"  CPL cleaning complete. {len(df_result)} records remaining.")
    # 2.5 Output CPL Columns: kategori, kode_cpl, kkni_level, kkni_no, text_clean
    return df_result[['kategori', 'kode_cpl', 'kkni_level', 'kkni_no', 'text_clean']]

def clean_metode_pembelajaran(df):
    """Extract learning methods from course outline text."""

    print("  Applying Metode Pembelajaran cleaning...")

    df_cleaned = df.copy()

    df_cleaned['text'] = (
        df_cleaned['text']
        .astype(str)
        .fillna('')
    )

    # ==========================================================
    # LEARNING METHOD KEYWORDS
    # ==========================================================

    learning_method_keywords = [
        'lecture', 'lectures', 'discussion', 'hands-on lab', 'exercise', 'diskusi','seminar', 'project', 'case study', 'assignment',
        'tugas', 'studi kasus','presentation', 'role play', 'praktikum', 'review', 'cooperative learning','problem based learning',
        'model contextual learning', 'quantum teaching','tutorials', 'discussions', 'case studies', 'lab activities', 'quiz',
        'project exposure', 'project review', 'material exposure', 'class-based','practical-based', 'practical problem solving',
        'problem solving', 'practical','exercises', 'work together', 'group project', 'workshop', 'ceramah','presentasi',
        'latihan soal', 'student centered learning', 'lab activities','lab activity', 'project-based', 'lab', 'field visit', 'magang',
        'praktik','simulasi', 'materi', 'tanya jawab', 'aktif debat', 'pair work', 'study case', 'presentations', 'lecturing',
        'classroom course material', 'problem-based learning', 'field trips'
    ]

    # ==========================================================
    # EXTRACT METHODS
    # ==========================================================

    def extract_learning_methods(text):

        text = str(text).lower()

        counter = Counter()

        for method in learning_method_keywords:

            matches = re.findall(
                r'\b' + re.escape(method.lower()) + r'\b',
                text
            )

            counter[method] += len(matches)

        # Filter out methods with frequency of 0
        filtered_counter = {method: count for method, count in counter.items() if count > 0}

        return filtered_counter

    # ==========================================================
    # APPLY EXTRACTION
    # ==========================================================

    df_cleaned['text_clean'] = (
        df_cleaned['text']
        .apply(extract_learning_methods)
    )

    # fallback khusus mata kuliah magang
    mask_magang = (
        df_cleaned['nama_file']
        .str.contains('magang', case=False, na=False)
    )

    df_cleaned.loc[
        mask_magang &
        (df_cleaned['text_clean'] == ''),
        'text_clean'
    ] = 'magang'

    print(
        f"  Metode Pembelajaran cleaning complete. "
        f"{len(df_cleaned)} records remaining."
    )

    return df_cleaned[
        ['nama_file', 'kategori', 'text_clean']
    ]

def clean_kompetensi(df):
    """
    Applies detailed cleaning rules for 'kompetensi' category,
    reconstructs fragmented RPS files,
    extracts important RPS sections,
    and splits sessions individually.

    Output format:
    source_file | course_name | section | extracted_text
    """

    print("  Applying detailed RPS reconstruction, section extraction, and cleaning for 'kompetensi' category...")

    results = []

    # ==========================================================
    # SKIP FIRST ROW
    # ==========================================================
    # First row is not an RPS document
    # ==========================================================

    df = df.iloc[1:].reset_index(drop=True)

    # ==========================================================
    # TARGET SECTIONS ONLY
    # ==========================================================

    target_sections = {

        'course_name': [
            'course name',
            'nama mata kuliah',
            'mata kuliah'
        ],

        'course_description': [
            'course description/deskripsi mata kuliah',
            'course description',
            'deskripsi mata kuliah'
        ],

        'course_objectives': [
            'course objectives',
            'tujuan pembelajaran',
            'course objectives/tujuan pembelajaran'
        ],

        'methods_instruction': [
            'methods of instructions',
            'methods of instruction',
            'metode pembelajaran',
            'methods of instructions/metode pembelajaran',
            'methods of instruction/metode pembelajaran'
        ],

        'course_outline': [
            'course outline',
            'garis besar materi',
            'pokok bahasan'
        ]
    }

    course_outline_fallback_keywords = [
        'session targeted competencies',
        'session target competencies',
        'targeted session'
    ]

    # ==========================================================
    # UNWANTED SECTIONS
    # ==========================================================

    unwanted_sections = [
        'attendance requirement',
        'assesment',
        'material references and required supplies',
        'learning outcome/capaian pembelajaran',
        'subject learning outcome/capaian pembelajaran mk'
    ]

    # ==========================================================
    # BUILD SECTION REGEX
    # ==========================================================

    all_section_keywords = []

    for section_list in target_sections.values():
        all_section_keywords.extend(section_list)

    all_section_keywords.extend(unwanted_sections)

    all_section_keywords_regex = '|'.join(
        re.escape(x)
        for x in all_section_keywords
    )

    # ==========================================================
    # CLEAN
    # ==========================================================

    def clean_section_content(content):

        content = re.sub(r'\s+', ' ', content)

        content = re.sub(
            r'\b\d+\.\s*',
            '',
            content
        )

        content = re.sub(
            r'\b\d+\s*sks\b',
            '',
            content,
            flags=re.IGNORECASE
        )

        content = re.sub(
            r'\s*[:\-]\s*',
            ' ',
            content
        )

        return content.strip()

    # ==========================================================
    # PROCESS EACH RPS FILE
    # ==========================================================

    for idx, row in df.iterrows():

        raw_text = str(row['text_content'])

        original_file_name = row['nama_file']

        # Hilangkan ekstensi file
        current_course_name = (
            original_file_name
            .replace('.pdf', '')
            .replace('.docx', '')
            .replace('.xlsx', '')
        )

        # ===============================
        # CLEANING NAMA FILE
        # ===============================

        # Contoh regex untuk membersihkan nama file
        current_course_name = re.sub(
            r'(?i)^copy[\s_\-]*of[\s_\-]*',
            '',
            current_course_name
        ).strip()

        current_course_name = re.sub(
            r'F-PPK-\d+.*?RPS',
            '',
            current_course_name
        ).strip()

        current_course_name = re.sub(
            r'[_]+',
            ' ',
            current_course_name
        ).strip()

        current_course_name = re.sub(
            r'\.(doc|docx|pdf)$',
            ' ',
            current_course_name
        ).strip()

        current_course_name = re.sub(
            r'\s+\d{4}$',
            ' ',
            current_course_name
        ).strip()

        current_course_name = re.sub(
            r'\s+',
            ' ',
            current_course_name
        ).strip()

        # ===============================
        # Split kode MK dan nama MK
        # ===============================
        match = re.match(
            r'^([A-Za-z]{2,}\d{3,})[\s\-_]*(.*)$',
            current_course_name
        )

        if match:
            course_code = match.group(1).strip()
            course_name = match.group(2).strip()
        else:
            course_code = None
            course_name = current_course_name

        # Track extracted sections
        detected_sections = set()


        # ======================================================
        # PREPROCESSING
        # ======================================================

        cleaned_text_lines = []

        for line in raw_text.split('\n'):

            line = re.sub(r'\b\d+ \/\d+\b', '', line)

            line = re.sub(
                r'\b\d+\s*of\s*\d+\b',
                '',
                line,
                flags=re.IGNORECASE
            )

            line = re.sub(r'\s*\u2022\s*', '- ', line)

            line = re.sub(r'\s*-\s*\d+\s*-\s*', '', line)

            line = re.sub(r'\s*(\d+\.\s*){2,}', '', line)

            line = normalize_text_general(line)

            if line:
                cleaned_text_lines.append(line)

        processed_text = '\n'.join(cleaned_text_lines)

        processed_text = re.sub(r'\n+', '\n', processed_text)

        normalized_search_text = re.sub(
            r'\s+',
            ' ',
            processed_text
        ).lower()
        # ======================================================
        # EXTRACT COURSE NAME (THIS PART IS NOW SKIPPED AS PER USER REQUEST)
        # The `current_course_name` is already derived from `original_file_name`.
        # ======================================================

        # ======================================================
        # EXTRACT ONLY TARGET SECTIONS
        # ======================================================

        for section_label, keywords in target_sections.items():

            if section_label == 'course_name':
                continue

            # Skip if section already extracted
            if section_label in detected_sections:
                continue

            section_found = False

            for keyword in keywords:

                pattern = (
                    rf'{re.escape(keyword)}'
                    rf'\s*[:\-]?\s*'
                    rf'(.*?)'
                    rf'(?='
                    rf'\n\s*(?:{all_section_keywords_regex})\s*[:\-]?'
                    rf'|$)'
                )

                matches = re.finditer(
                    pattern,
                    processed_text,
                    re.IGNORECASE | re.DOTALL
                )

                for match in matches:

                    content = match.group(1).strip()

                    # ==========================================
                    # REMOVE UNWANTED SUBSECTIONS
                    # ==========================================

                    for unwanted in unwanted_sections:

                        content = re.split(
                            rf'\b{re.escape(unwanted.upper())}\b',
                            content,
                            flags=re.IGNORECASE
                        )[0]

                    content = clean_section_content(content)

                    # ==========================================
                    # SKIP EMPTY
                    # ==========================================

                    if not content:
                        continue

                    # ==========================================
                    # SAVE RESULT
                    # ==========================================

                    results.append({
                        'source_file': original_file_name,
                        'course_code': course_code,
                        'course_name': course_name,
                        'section': section_label,
                        'extracted_text': content
                    })

                    # Mark section as extracted
                    detected_sections.add(section_label)

                    section_found = True

                    # Stop checking other keywords
                    break

                if section_found:
                  break

            # ==================================================
            # FALLBACK FOR COURSE OUTLINE
            # ==================================================

            if (
                section_label == 'course_outline'
                and not section_found
            ):

                print(f"Fallback activated: {course_name}")

                match = re.search(
                    r'targeted\s+session',
                    processed_text,
                    re.IGNORECASE
                )

                if match:

                    print("FOUND TARGETED SESSION")

                    start_idx = match.start()

                    content = processed_text[start_idx:]

                    content = clean_section_content(content)

                    results.append({
                        'source_file': original_file_name,
                        'course_code': course_code,
                        'course_name': course_name,
                        'section': section_label,
                        'extracted_text': content
                    })

                    detected_sections.add(section_label)
                    section_found = True

                    break

    # ==========================================================
    # FINAL DATAFRAME
    # ==========================================================

    df_result = pd.DataFrame(results)

    if not df_result.empty:

        df_result = df_result[
            df_result['extracted_text']
            .astype(str)
            .str.strip() != ''
        ].reset_index(drop=True)

    # ==========================================================
    # SAFEGUARD
    # ==========================================================

    if df_result.empty:

        return pd.DataFrame(
            columns=[
                'source_file',
                'course_code',
                'course_name',
                'section',
                'extracted_text'
            ]
        )

    print(
        f"  Detailed RPS section extraction complete. "
        f"{len(df_result)} records generated."
    )
    # Jumlah mata kuliah yang berhasil diekstrak
    total_courses = df_result['course_name'].nunique()

    print(f"Jumlah mata kuliah berhasil diekstrak: {total_courses}")

    return df_result[
        [
            'source_file',
            'course_code',
            'course_name',
            'section',
            'extracted_text'
        ]
    ]
# --- Main Processing Loop ---

cleaning_functions = {
    'visi_misi': clean_visi_misi,
    'profil_lulusan': clean_profil_lulusan,
    'cpl': clean_cpl,
    'metode_pembelajaran': clean_metode_pembelajaran, # Using 'metode_pembelajaran' as per previous cell output
    'kompetensi': clean_kompetensi # 'others' category is now 'kompetensi'
}

print("\n--- Starting category-specific cleaning for curriculum data ---")

for category, clean_func in cleaning_functions.items():
    print(f"\nProcessing category: {category}")
    category_output_dir = os.path.join(OUTPUT_FOLDER_CURRICULUM, category)
    if category == "kompetensi":

        input_filepath = os.path.join(
            OUTPUT_FOLDER_CURRICULUM,
            'curriculum_combined_raw.csv'
        )

    else:

        input_filepath = os.path.join(
            category_output_dir,
            f'{category}_raw.csv'
        )
    output_filepath = os.path.join(category_output_dir, f'{category}_cleaned.csv')

    # Add a debug print for the path
    print(f"  Attempting to load from: {input_filepath}")
    # Add an explicit file existence check
    if not os.path.exists(input_filepath):
        print(f"  DEBUG: File DOES NOT EXIST at: {input_filepath}")
        print(f"  DEBUG: Listing contents of directory '{category_output_dir}':")
        try:
            for item in os.listdir(category_output_dir):
                print(f"    - {item}")
        except FileNotFoundError:
            print(f"    Directory '{category_output_dir}' itself not found.")
        print(f"Error: Input file not found for {category}: {input_filepath}. Skipping this category.")
        continue

    try:
        df_raw = pd.read_csv(input_filepath)
        print(f"  Loaded {len(df_raw)} raw records for {category}.")

        df_cleaned_category = clean_func(df_raw)

        if not df_cleaned_category.empty:
            df_cleaned_category.to_csv(output_filepath, index=False)
            print(f"  Cleaned data saved to '{output_filepath}' ({len(df_cleaned_category)} records).")
        else:
            # Save an empty CSV with headers if the dataframe is empty
            df_cleaned_category.to_csv(output_filepath, index=False)
            print(f"  No data remaining for {category} after cleaning. Saved an empty file with headers to '{output_filepath}'.")

    except Exception as e:
        print(f"  An error occurred during cleaning for {category}: {e}")

print("\n--- All category-specific cleaning complete ---")


--- Starting category-specific cleaning for curriculum data ---

Processing category: visi_misi
  Attempting to load from: /content/processed_data/curriculum_data/visi_misi/visi_misi_raw.csv
  Loaded 12 raw records for visi_misi.
  Applying Visi Misi cleaning (correct slicing)...
  Visi Misi cleaning complete. 2 records remaining.
  Cleaned data saved to '/content/processed_data/curriculum_data/visi_misi/visi_misi_cleaned.csv' (2 records).

Processing category: profil_lulusan
  Attempting to load from: /content/processed_data/curriculum_data/profil_lulusan/profil_lulusan_raw.csv
  Loaded 14 raw records for profil_lulusan.
Applying Profil Lulusan cleaning...
Cleaning selesai. Total profil lulusan: 5
  Profil Lulusan cleaning complete. 5 records remaining.
  Cleaned data saved to '/content/processed_data/curriculum_data/profil_lulusan/profil_lulusan_cleaned.csv' (5 records).

Processing category: cpl
  Attempting to load from: /content/processed_data/curriculum_data/cpl/cpl_raw.csv
  Lo

## Save Cleaned Curriculum Data to Google Drive

In [ ]:
import pandas as pd
import os
from google.colab import drive

# Mount Google Drive if not already mounted
# drive.mount('/content/drive', force_remount=True) # Uncomment if drive is not mounted

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')

# Define the target Google Drive path
# This path will mirror the internal structure of OUTPUT_FOLDER_CURRICULUM
GOOGLE_DRIVE_TARGET_BASE_DIR = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED'

# List of categories that were cleaned
categories = ['visi_misi', 'profil_lulusan', 'cpl', 'metode_pembelajaran', 'kompetensi']

print(f"Saving cleaned curriculum data to: {GOOGLE_DRIVE_TARGET_BASE_DIR}")

for category in categories:
    # Source file path (from processed_data folder)
    source_filepath = os.path.join(OUTPUT_FOLDER_CURRICULUM, category, f'{category}_cleaned.csv')

    # Destination folder path in Google Drive
    destination_category_dir = os.path.join(GOOGLE_DRIVE_TARGET_BASE_DIR, category)

    # Ensure destination directory exists in Google Drive
    os.makedirs(destination_category_dir, exist_ok=True)

    # Destination file path in Google Drive
    destination_filepath = os.path.join(destination_category_dir, f'{category}_cleaned.csv')

    try:
        # Load the cleaned CSV from local Colab storage
        df_cleaned = pd.read_csv(source_filepath)

        # Save the DataFrame to Google Drive
        df_cleaned.to_csv(destination_filepath, index=False)
        print(f"Successfully saved '{os.path.basename(source_filepath)}' to '{destination_filepath}' ({len(df_cleaned)} rows).")
    except FileNotFoundError:
        print(f"Error: Source file not found for category '{category}' at '{source_filepath}'. Skipping.")
    except Exception as e:
        print(f"An error occurred while saving '{category}_cleaned.csv': {e}")

print("\nAll specified cleaned curriculum data saved to Google Drive.")

Saving cleaned curriculum data to: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED
Successfully saved 'visi_misi_cleaned.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/visi_misi/visi_misi_cleaned.csv' (2 rows).
Successfully saved 'profil_lulusan_cleaned.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/profil_lulusan/profil_lulusan_cleaned.csv' (5 rows).
Successfully saved 'cpl_cleaned.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/cpl/cpl_cleaned.csv' (11 rows).
Successfully saved 'metode_pembelajaran_cleaned.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/metode_pembelajaran/metode_pembelajaran_cleaned.csv' (54 rows).
Successfully saved 'kompetensi_cleaned.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/kompetensi/kompetensi_cleaned.csv' (203 rows).

All specified cleaned curriculum data saved to Google 

## KO

In [ ]:
# ============================================================
# MEMBANGUN MASTER MATA KULIAH
# (Semi Manual dari df_kompetensi + Tugas Akhir)
# ============================================================

import pandas as pd
import re

df_kompetensi = pd.read_csv(
    "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/kompetensi/kompetensi_cleaned.csv"
)

display(df_kompetensi.head())

print("="*80)
print("MEMBANGUN MASTER MATA KULIAH")
print("="*80)

# ------------------------------------------------------------------
# 1. Ambil daftar mata kuliah unik
# ------------------------------------------------------------------

df_master_courses = (
    df_kompetensi[
        ["course_code", "course_name"]
    ]
    .drop_duplicates()
    .copy()
)

# ------------------------------------------------------------------
# 2. Bersihkan text
# ------------------------------------------------------------------

df_master_courses["course_code"] = (
    df_master_courses["course_code"]
    .astype(str)
    .str.strip()
)

df_master_courses["course_name"] = (
    df_master_courses["course_name"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ------------------------------------------------------------------
# 3. Ubah nama kolom agar konsisten
# ------------------------------------------------------------------

df_master_courses = df_master_courses.rename(
    columns={
        "course_code": "kode_mk",
        "course_name": "nama_mata_kuliah"
    }
)

# -------------------------------------------------------------
# Tambahkan mata kuliah yang tidak memiliki RPS
# -------------------------------------------------------------

additional_courses = pd.DataFrame({
    "kode_mk": [
        "TIF403",   # sesuaikan jika berbeda
        "TIF353",   # sesuaikan dengan KO
        "TIF451",   # sesuaikan dengan KO
        "TIF453"    # sesuaikan dengan KO
    ],
    "nama_mata_kuliah": [
        "Tugas Akhir",
        "Mobile System Programming",
        "Analisis Perancangan Sistem Informasi",
        "Advanced Ethical Hacking"
    ]
})

df_master_courses = pd.concat(
    [df_master_courses, additional_courses],
    ignore_index=True
).drop_duplicates(subset="kode_mk")

# -------------------------------------------------------------
# Tambahkan kategori
# -------------------------------------------------------------

df_master_courses["kategori"] = "Wajib Prodi"

# Mata kuliah Universitas
mk_universitas = [
    "Agama",
    "English for Academic Purposes 1",
    "English AP2",
    "Pendidikan Pancasila dan Kewarganegaraan",
    "Bahasa Indonesia"
]

df_master_courses.loc[
    df_master_courses["nama_mata_kuliah"].isin(mk_universitas),
    "kategori"
] = "Wajib Universitas"

# Mata kuliah Peminatan
mk_peminatan = [
    "Mobile System Programming",
    "Data Warehouse dan Mining",
    "Pengolahan Citra",
    "Sistem Informasi Manajemen",
    "Ethical Hacking",
    "Manajemen Jaringan",
    "Rekayasa Ilmu",
    "Analisis Perancangan Sistem Informasi",
    "Advanced Ethical Hacking",
    "Mobil Computing",
    "Computer Vision",
    "Advanced Broadband Networking",
    "Tata Kelola Sistem Informasi"
]

df_master_courses.loc[
    df_master_courses["nama_mata_kuliah"].isin(mk_peminatan),
    "kategori"
] = "Peminatan"

# Tugas Akhir
df_master_courses.loc[
    df_master_courses["nama_mata_kuliah"] == "Tugas Akhir",
    "kategori"
] = "Tugas Akhir"

# ------------------------------------------------------------------
# 8. Hapus duplikasi & urutkan
# ------------------------------------------------------------------

df_master_courses = (
    df_master_courses
    .drop_duplicates(subset="kode_mk")
    .sort_values("kode_mk")
    .reset_index(drop=True)
)

# ------------------------------------------------------------------
# 9. Validasi
# ------------------------------------------------------------------

print(f"Jumlah Mata Kuliah : {len(df_master_courses)}")
print(f"Jumlah Kode Unik   : {df_master_courses['kode_mk'].nunique()}")

display(df_master_courses)

,source_file,course_code,course_name,section,extracted_text
0,Copy of FTK121 Aljabar Linear.pdf,FTK121,Aljabar Linear,course_description,this course provides an understanding of apply...
1,Copy of FTK121 Aljabar Linear.pdf,FTK121,Aljabar Linear,course_objectives,"upon completion of this course, the student sh..."
2,Copy of FTK121 Aljabar Linear.pdf,FTK121,Aljabar Linear,methods_instruction,virtual classroom instruction consists of lect...
3,Copy of FTK121 Aljabar Linear.pdf,FTK121,Aljabar Linear,course_outline,session topics & sub topics methods references...
4,Copy of TIF101 Sirkuit Elektronik.pdf,TIF101,Sirkuit Elektronik,course_description,this course provides a complete and straightfo...


MEMBANGUN MASTER MATA KULIAH
Jumlah Mata Kuliah : 58
Jumlah Kode Unik   : 58


,kode_mk,nama_mata_kuliah,kategori
0,FTK101,Algoritma & Pemrograman,Wajib Prodi
1,FTK103,Pemrograman Visual,Wajib Prodi
2,FTK111,Kalkulus 1,Wajib Prodi
3,FTK121,Aljabar Linear,Wajib Prodi
4,FTK151,Metodologi Penelitian dan Ilmiah,Wajib Prodi
5,FTK161,Statistika,Wajib Prodi
6,FTK203,Sistem Basis Data,Wajib Prodi
7,FTK205,LAN and Wireless,Wajib Prodi
8,FTK206,Pemrograman Berorientasi Objek,Wajib Prodi
9,FTK211,Kalkulus 2,Wajib Prodi


In [ ]:
# ============================================================
# MASTER RELASI PRASYARAT MATA KULIAH
# (Diinput manual berdasarkan Kurikulum Operasional)
# ============================================================

import pandas as pd

df_prerequisite = pd.DataFrame([
    ["UNI204", "Bahasa Inggris 2", "UNI104", "Bahasa Inggris 1"],
    ["FTK211", "Kalkulus 2", "FTK111", "Kalkulus 1"],
    ["FTK103", "Pemrograman Visual", "TIF101", "Sirkuit Elektronik"],
    ["TIF206", "Sistem Operasi", "TIF109", "Struktur Data"],
    ["TIF203", "Sistem Dijital", "TIF101", "Sirkuit Elektronik"],
    ["FTK203", "Sistem Basis Data", "TIF109", "Struktur Data"],
    ["TIF204", "Sistem Basis Data Lanjut", "FTK203", "Sistem Basis Data"],
    ["FTK205", "LAN and Wireless", "TIF207", "Komunikasi Data"],
    ["TIF209", "Arsitektur dan Organisasi Komputer", "TIF203", "Sistem Dijital"],
    ["FTK206", "Pemrograman Berorientasi Objek", "FTK101", "Algoritma dan Pemrograman"],
    ["TIF301", "Routing Protocols and Concepts", "FTK205", "LAN and Wireless"],
    ["TIF302", "Pengantar Intelejensi Artifisial", "FTK101", "Algoritma dan Pemrograman"],
    ["FTK301", "E-Business dan Pemrograman Berbasis Web", "FTK101", "Algoritma dan Pemrograman"],
    ["TIF311", "Sistem Multimedia", "TIF208", "Proses Sinyal Digital"],
    ["TIF305", "Pemodelan dan Simulasi Jaringan", "FTK205", "LAN and Wireless"],
    ["TIF313", "Wide Area Network (WAN)", "TIF301", "Routing Protocols and Concepts"],
    ["TIF307", "Komunikasi Nirkabel", "TIF207", "Data Communication"],
    ["TIF309", "Sistem Terdistribusi", "FTK203", "Sistem Basis Data"],
    ["TIF401", "IT Security", "FTK205", "LAN and Wireless"],
    ["TIF402", "Interaksi Manusia dan Komputer", "TIF206", "Sistem Operasi"],
    ["TIF403", "Tugas Akhir", "TIF301", "Metodologi Penelitian dan Penulisan Ilmiah Research Methods"],
    ["TIF356", "Pengolahan Citra", "TIF208", "Proses Sinyal Dijital"],
    ["TIF356", "Pengolahan Citra", "TIF311", "Sistem Multimedia"],
    ["TIF352", "Tata Kelola Sistem Informasi", "TIF106", "Konsep Sistem Informasi"],
    ["TIF456", "Computer Vision", "TIF356", "Pengolahan Citra"]
], columns=[
    "kode_mk",
    "nama_mata_kuliah",
    "prerequisite_code",
    "prerequisite_name"
])

print("="*80)
print("MASTER RELASI PRASYARAT")
print("="*80)

display(df_prerequisite)

print(f"\nJumlah relasi prasyarat : {len(df_prerequisite)}")
print(f"Jumlah mata kuliah yang memiliki prasyarat : {df_prerequisite['kode_mk'].nunique()}")

MASTER RELASI PRASYARAT


,kode_mk,nama_mata_kuliah,prerequisite_code,prerequisite_name
0,UNI204,Bahasa Inggris 2,UNI104,Bahasa Inggris 1
1,FTK211,Kalkulus 2,FTK111,Kalkulus 1
2,FTK103,Pemrograman Visual,TIF101,Sirkuit Elektronik
3,TIF206,Sistem Operasi,TIF109,Struktur Data
4,TIF203,Sistem Dijital,TIF101,Sirkuit Elektronik
5,FTK203,Sistem Basis Data,TIF109,Struktur Data
6,TIF204,Sistem Basis Data Lanjut,FTK203,Sistem Basis Data
7,FTK205,LAN and Wireless,TIF207,Komunikasi Data
8,TIF209,Arsitektur dan Organisasi Komputer,TIF203,Sistem Dijital
9,FTK206,Pemrograman Berorientasi Objek,FTK101,Algoritma dan Pemrograman



Jumlah relasi prasyarat : 25
Jumlah mata kuliah yang memiliki prasyarat : 24


In [ ]:
import os
from google.colab import drive

# Pastikan Google Drive ter-mount
drive.mount('/content/drive', force_remount=True)

# Definisikan path folder output di Google Drive
OUTPUT_PATH_DRIVE = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED'

# Pastikan direktori tujuan ada
os.makedirs(OUTPUT_PATH_DRIVE, exist_ok=True)

# Simpan df_master_courses
master_courses_filepath = os.path.join(OUTPUT_PATH_DRIVE, 'master_courses.csv')
df_master_courses.to_csv(master_courses_filepath, index=False)
print(f"'master_courses.csv' berhasil disimpan ke: {master_courses_filepath}")

# Simpan df_prerequisite
prerequisite_filepath = os.path.join(OUTPUT_PATH_DRIVE, 'prerequisite.csv')
df_prerequisite.to_csv(prerequisite_filepath, index=False)
print(f"'prerequisite.csv' berhasil disimpan ke: {prerequisite_filepath}")

Mounted at /content/drive
'master_courses.csv' berhasil disimpan ke: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/master_courses.csv
'prerequisite.csv' berhasil disimpan ke: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/prerequisite.csv


# TRACER STUDY

### Step 1: Mount Google Drive and Define Folder Path

First, we'll mount Google Drive to access your datasets. Please ensure your Excel files are in the specified `DRIVE_FOLDER_PATH`.

In [ ]:
from google.colab import drive

# --- Configuration --- #
# IMPORTANT: Update this path to your specific Google Drive folder containing the tracer study Excel files.
DRIVE_FOLDER_PATH = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/TRACER STUDY/2024-2025/"

print(f"Google Drive mounted. Data folder set to: {DRIVE_FOLDER_PATH}")

Google Drive mounted. Data folder set to: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/TRACER STUDY/2024-2025/


### Step 2: Load All Excel Files

This step automatically finds all Excel files ('.xlsx' and '.xls') within the specified `DRIVE_FOLDER_PATH`. Each file is then loaded into a pandas DataFrame and added to a list. The original filename is also stored to facilitate extracting the source year later.

In [ ]:
import pandas as pd
import glob
import os

# List to store individual DataFrames and their original filenames/paths
list_of_dfs = []
list_of_filepaths = []

print(f"Searching for Excel files in: {DRIVE_FOLDER_PATH}")

# Use glob to find all .xlsx and .xls files in the specified directory
excel_files = glob.glob(os.path.join(DRIVE_FOLDER_PATH, '*.xlsx'))
excel_files.extend(glob.glob(os.path.join(DRIVE_FOLDER_PATH, '*.xls')))

if not excel_files:
    print(f"No Excel files found in '{DRIVE_FOLDER_PATH}'. Please check the path and file types.")
else:
    print(f"Found {len(excel_files)} Excel files.")
    for filepath in excel_files:
        try:
            df_temp = pd.read_excel(filepath)
            list_of_dfs.append(df_temp)
            list_of_filepaths.append(filepath)
            print(f"  Successfully loaded: {os.path.basename(filepath)} with {len(df_temp)} rows.")
        except Exception as e:
            print(f"  Error loading {os.path.basename(filepath)}: {e}")

print(f"Total DataFrames loaded: {len(list_of_dfs)}")
print(f"Total file paths stored: {len(list_of_filepaths)}")

# Display the first few rows of the first loaded DataFrame as an example
if list_of_dfs:
    print("\nFirst 5 rows of the first loaded DataFrame:")
    display(list_of_dfs[0].head())
else:
    print("No DataFrames were loaded.")

Searching for Excel files in: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/TRACER STUDY/2024-2025/
Found 2 Excel files.
  Successfully loaded: TS 2025.xlsx with 14 rows.
  Successfully loaded: TS 2024.xlsx with 12 rows.
Total DataFrames loaded: 2
Total file paths stored: 2

First 5 rows of the first loaded DataFrame:


,No.,NIM,Nama,Kode Prodi,Program Studi,Tahun Lulus,Jelaskan status Anda saat ini?,Dalam berapa bulan Anda mendapatkan pekerjaan pertama ?,Berapa lama masa tunggu Anda dalam mendapatkan pekerjaan pertama Anda?,Berapa rata-rata pendapatan Anda per bulan? (take home pay),...,Berapa bulan sebelum lulus ?,Berapa bulan sesudah lulus ?,Bagaimana Anda mencari pekerjaan tersebut? Jawaban bisa lebih dari satu\n,Berapa perusahaan/instansi/institusi yang sudah Anda lamar (lewat surat atau e-mail) sebelum Anda memperoleh pekerjaan pertama?,Berapa banyak perusahaan/instansi/institusi yang merespons lamaran Anda?\n,Berapa banyak perusahaan/instansi/institusi yang mengundang Anda untuk wawancara?\n,f1001,Apakah Anda aktif mencari pekerjaan dalam 4 minggu terakhir? Pilihlah satu jawaban\n,"Jika menurut Anda pekerjaan Anda saat ini tidak sesuai dengan : pendidikan Anda, mengapa Anda mengambilnya? Jawaban bisa lebih dari satu\n",Tuliskan saran dan masukan Anda bagi Universitas dan Prodi Anda
0,1,1172001006,Rizal Fahlepi,55201,Teknik Informatika S1,2024,Bekerja (full time / part time),0.0,< 6 Bulan,9000000.0,...,16.0,NaN,Mencari lewat internet/iklan online/milis\nMem...,8,4,2,"Bekerja, namun tetap mencari pekerjaan dengan ...","Bekerja, namun tetap mencari pekerjaan dengan ...",Di pekerjaan ini saya memeroleh prospek karir ...,NaN
1,2,1202001007,Sheila Riva Rezqian,55201,Teknik Informatika S1,2024,Bekerja (full time / part time),2.0,< 6 Bulan,5600000.0,...,NaN,2.0,Mencari lewat internet/iklan online/milis,20,10,8,2,"Tidak, tapi saya sedang menunggu hasil lamaran...",Pertanyaan tidak sesuai; pekerjaan saya sekara...,NaN
2,3,1172001032,Ananda Anggita Pratama,55201,Teknik Informatika S1,2024,Bekerja (full time / part time),0.0,< 6 Bulan,12000000.0,...,6.0,NaN,"Melalui relasi (misalnya dosen, orang tua, sau...",0,0,0,4,"Ya, tapi saya belum pasti akan bekerja dalam 2...",Pertanyaan tidak sesuai; pekerjaan saya sekara...,"Karena perusahaan ini, saya bisa bertemu denga..."
3,4,1202001025,Mumtaz Aaliyah Fasya,55201,Teknik Informatika S1,2024,Bekerja (full time / part time),8.0,6 sampai 18 Bulan,10000000.0,...,2.0,NaN,Mencari lewat internet/iklan online/milis,60,40,9,4,"Ya, tapi saya belum pasti akan bekerja dalam 2...",Pertanyaan tidak sesuai; pekerjaan saya sekara...,UI/UX is fun
4,5,1192001003,Muhammad Alvin Alyundra,55201,Teknik Informatika S1,2024,Bekerja (full time / part time),2.0,< 6 Bulan,7400000.0,...,NaN,2.0,"Melalui relasi (misalnya dosen, orang tua, sau...",4,2,1,1,Tidak,Pertanyaan tidak sesuai; pekerjaan saya sekara...,Saya berharap Universitas dapat terus meningka...


### Step 3: Clean Column Names

For each DataFrame, this step will:
- convert column names to lowercase
- strip whitespace
- replace spaces with underscores
- remove special characters

This standardization is crucial for consistent data processing across different years and sources.

In [ ]:
import re

def clean_column_name(col_name):
    # Convert to lowercase
    cleaned_name = str(col_name).lower()
    # Strip leading/trailing whitespace
    cleaned_name = cleaned_name.strip()
    # Replace spaces with underscores
    cleaned_name = cleaned_name.replace(' ', '_')
    # Remove special characters (keep alphanumeric and underscores)
    cleaned_name = re.sub(r'[^a-z0-9_]', '', cleaned_name)
    # Remove multiple underscores
    cleaned_name = re.sub(r'_+', '_', cleaned_name)
    return cleaned_name

# Apply cleaning to all loaded DataFrames
cleaned_list_of_dfs = []

if not list_of_dfs:
    print("No DataFrames to process. Please ensure Excel files are loaded in Step 2.")
else:
    print(f"Cleaning column names for {len(list_of_dfs)} DataFrames...")
    for i, df_temp in enumerate(list_of_dfs):
        original_columns = df_temp.columns.tolist()
        df_temp.columns = [clean_column_name(col) for col in original_columns]
        cleaned_list_of_dfs.append(df_temp)
        print(f"\nDataFrame {i+1} (from {os.path.basename(list_of_filepaths[i])}) cleaned. Original vs Cleaned Columns:")
        for orig_col, clean_col in zip(original_columns, df_temp.columns.tolist()):
            print(f"  '{orig_col}' -> '{clean_col}'")

    # Replace the original list_of_dfs with the cleaned one for subsequent steps
    list_of_dfs = cleaned_list_of_dfs
    print("\nColumn cleaning complete. Updated list_of_dfs with cleaned DataFrames.")


Cleaning column names for 2 DataFrames...

DataFrame 1 (from TS 2025.xlsx) cleaned. Original vs Cleaned Columns:
  'No.' -> 'no'
  'NIM' -> 'nim'
  'Nama' -> 'nama'
  'Kode Prodi' -> 'kode_prodi'
  'Program Studi' -> 'program_studi'
  'Tahun Lulus' -> 'tahun_lulus'
  'Jelaskan status Anda saat ini?' -> 'jelaskan_status_anda_saat_ini'
  'Dalam berapa bulan Anda mendapatkan pekerjaan pertama ?' -> 'dalam_berapa_bulan_anda_mendapatkan_pekerjaan_pertama_'
  'Berapa lama masa tunggu Anda dalam mendapatkan pekerjaan pertama Anda?' -> 'berapa_lama_masa_tunggu_anda_dalam_mendapatkan_pekerjaan_pertama_anda'
  'Berapa rata-rata pendapatan Anda per bulan? (take home pay)' -> 'berapa_ratarata_pendapatan_anda_per_bulan_take_home_pay'
  'Apa jenis perusahaan/intansi/institusi tempat Anda bekerja sekarang?' -> 'apa_jenis_perusahaanintansiinstitusi_tempat_anda_bekerja_sekarang'
  'Apakah Anda saat ini bekerja di perusahaan pada Kelompok Usaha Bakrie (KUB) ?' -> 'apakah_anda_saat_ini_bekerja_di_perusah

In [ ]:
import re

# --- STEP 4: CREATE FLEXIBLE COLUMN DETECTION ---

# Define keyword patterns for each target standardized column
# The order of keywords matters for some patterns (e.g., 'tahun_lulus' should match 'tahun' and 'lulus')
column_patterns = {
    'nim': ['nim', 'nomor_induk_mahasiswa'],
    'nama': ['nama', 'nama_lengkap'],
    'tahun_lulus': ['tahun', 'lulus'], # This needs to be handled carefully, check for both keywords
    'status_pekerjaan': ['status_pekerjaan', 'status_anda'],
    'nama_perusahaan': ['perusahaan', 'kantor_tempat'], # Match either 'perusahaan' or 'kantor_tempat'
    'posisi_pekerjaan': ['posisi', 'jabatan'],
    'masa_tunggu_bulan': ['dalam_berapa_bulan', 'dalam_bulan'], # Combine multiple keywords for better detection
    'gaji': ['gaji', 'pendapatan', 'take_home_pay'],
    'saran': ['saran', 'masukan_anda'] # Added new pattern for 'saran'
}

# This will store the detected column names for each DataFrame
# Key: original DataFrame index, Value: dictionary of {'standard_col_name': 'actual_cleaned_col_name'}
alumni_data_column_mappings = []

print("\n--- Detecting Flexible Columns ---")

for i, df_temp in enumerate(list_of_dfs):
    current_df_mapping = {}
    print(f"Processing DataFrame from {os.path.basename(list_of_filepaths[i])}")

    # Iterate through each target standardized column
    for standard_col, patterns in column_patterns.items():
        found_col = None
        # Iterate through the cleaned columns of the current DataFrame
        for cleaned_df_col in df_temp.columns:
            # For 'tahun_lulus', check if both keywords are present
            if standard_col == 'tahun_lulus':
                if all(keyword in cleaned_df_col for keyword in patterns):
                    found_col = cleaned_df_col
                    break
            # For other columns, check if any of the keywords are present
            else:
                if any(keyword in cleaned_df_col for keyword in patterns):
                    found_col = cleaned_df_col
                    break

        if found_col:
            current_df_mapping[standard_col] = found_col
            print(f"  Detected '{standard_col}' as '{found_col}'")
        else:
            # Optional: Log if a column was not found, depending on strictness requirements
            print(f"  Warning: '{standard_col}' not found in this DataFrame.")
    alumni_data_column_mappings.append(current_df_mapping)

print("\nFlexible column detection complete. Mappings stored in 'alumni_data_column_mappings'.")
# Display the detected mappings for the first DataFrame as an example
if alumni_data_column_mappings:
    print("\nExample mapping for the first DataFrame:")
    print(alumni_data_column_mappings[0])


--- Detecting Flexible Columns ---
Processing DataFrame from TS 2025.xlsx
  Detected 'nim' as 'nim'
  Detected 'nama' as 'nama'
  Detected 'tahun_lulus' as 'tahun_lulus'
  Detected 'status_pekerjaan' as 'jelaskan_status_anda_saat_ini'
  Detected 'nama_perusahaan' as 'apa_jenis_perusahaanintansiinstitusi_tempat_anda_bekerja_sekarang'
  Detected 'posisi_pekerjaan' as 'apa_jabatan_anda_saat_ini'
  Detected 'masa_tunggu_bulan' as 'dalam_berapa_bulan_anda_mendapatkan_pekerjaan_pertama_'
  Detected 'gaji' as 'berapa_ratarata_pendapatan_anda_per_bulan_take_home_pay'
  Detected 'saran' as 'tuliskan_saran_dan_masukan_anda_bagi_universitas_dan_prodi_anda'
Processing DataFrame from TS 2024.xlsx
  Detected 'nim' as 'harap_tuliskan_nim_anda_jika_tidak_ingat_mohon_tuliskan_angka_0'
  Detected 'nama' as 'tuliskan_nama_lengkap_anda_sesuai_yang_tertera_di_ijazah_ubakrie'
  Detected 'tahun_lulus' as 'pilihlah_tahun_kelulusanyudisium_anda'
  Detected 'status_pekerjaan' as 'f8_pilihlah_status_pekerjaan_a

In [ ]:
import re

# --- STEP 5: DETECT COMPETENCY COLUMNS ---

# Define keyword patterns for each target standardized column
competency_keywords = {
    'ethics': ['etika', 'etika_f1761', 'etika_f1762'],
    'communication': ['komunikasi', 'komunikasi_f1769', 'komunikasi_f1770'],
    'it_skill': [ 'teknologi', 'penggunaan_teknologi_informasi', 'penggunaan_teknologi_informasi_f1767', 'penggunaan_teknologi_informasi_f1768'],
    'teamwork': ['kerjasama', 'kerjasama_tim', 'kerja_sama_timf1771', 'kerja_sama_tim_f1772','kerja_sama_tim'],
    'english_language': ['bahasa_inggris', 'bahasa_inggris_f1765', 'bahasa_inggris_f1766'],
    'self_development': ['pengembangan_diri', 'pengembangan_diri_f1773', 'pengembangan_diri_f1774', 'pengembangan'],
    'domain_knowledge': ['keahlian', 'keahlian_berdasarkan_bidang_ilmu', 'keahlian_berdasarkan_bidang_ilmuf1763', 'keahlian_berdasarkan_bidang_ilmuf1764' ], # Added 'memimpin'
}

# Define patterns for graduation and work context
graduation_patterns = ['kuasai', 'menguasai']
work_patterns = ['dibutuhkan', 'diperlukan', 'pekerjaan', 'dalam_pekerjaan', 'kompetensi_dalam_pekerjaan', 'pekerjaan_anda', 'dunia_kerja'] # Added 'dunia_kerja'

# This will store the detected competency column mappings for each DataFrame
# Key: original DataFrame index, Value: dictionary of {'standard_competency_col_name': 'actual_cleaned_col_name'}
alumni_competency_column_mappings = []

# --- NEW: DETECT LEARNING METHOD COLUMNS ---
# Define learning method keywords and their standardized names
learning_method_keywords = {
    'perkuliahan': 'lecture',
    'simulasirole_play': 'demonstration',
    'partisipasi_dalam_proyek_riset': 'research_project_participation',
    'magang': 'internship',
    'praktikum': 'lab_practicum',
    'kerja_lapangan': 'field_work',
    'kerjalapangan': 'field_work', # Added mapping for 'kerjalapangan'
    'partisipasidalamproyekriset': 'research_project_participation', # Added mapping for 'partisipasidalamproyekriset'
    'diskusi': 'discussion'
}

# This will store the detected learning method column mappings for each DataFrame
alumni_learning_method_column_mappings = []

print("\n--- Detecting Competency and Learning Method Columns ---")

for i, df_temp in enumerate(list_of_dfs):
    current_df_competency_mapping = {}
    current_df_learning_method_mapping = {} # NEW: For learning methods

    print(f"Processing DataFrame from {os.path.basename(list_of_filepaths[i])}")

    # --- COMPETENCY COLUMN DETECTION ---
    for cleaned_df_col in df_temp.columns:
        lower_cleaned_df_col = cleaned_df_col.lower()

        # Skip if this column is likely a learning method column
        if 'metode_pembelajaran' in lower_cleaned_df_col:
            continue

        is_graduation = any(p in lower_cleaned_df_col for p in graduation_patterns)
        is_work = any(p in lower_cleaned_df_col for p in work_patterns)

        for comp_standard_name, comp_keywords in competency_keywords.items():
            if any(k in lower_cleaned_df_col for k in comp_keywords):
                if is_graduation and not is_work:
                    standard_comp_col = f"{comp_standard_name}_graduation"
                    current_df_competency_mapping[standard_comp_col] = cleaned_df_col
                    print(f"  Detected '{standard_comp_col}' as '{cleaned_df_col}'")
                elif is_work and not is_graduation:
                    standard_comp_col = f"{comp_standard_name}_work"
                    current_df_competency_mapping[standard_comp_col] = cleaned_df_col
                    print(f"  Detected '{standard_comp_col}' as '{cleaned_df_col}'")
                elif is_graduation and is_work:
                    print(f"  Ambiguous competency column: '{cleaned_df_col}'. Contains both graduation and work keywords. Skipping for now.")

    alumni_competency_column_mappings.append(current_df_competency_mapping)

    # --- NEW: LEARNING METHOD COLUMN DETECTION ---
    for cleaned_df_col in df_temp.columns:
        lower_cleaned_df_col = cleaned_df_col.lower()

        if 'metode_pembelajaran' in lower_cleaned_df_col: # Primary identifier for learning methods
            for method_keyword, standard_method_name in learning_method_keywords.items():
                if method_keyword in lower_cleaned_df_col:
                    standard_lm_col = f"{standard_method_name}_implementation"
                    current_df_learning_method_mapping[standard_lm_col] = cleaned_df_col
                    print(f"  Detected '{standard_lm_col}' as '{cleaned_df_col}'")
                    break # Move to next df_temp column once a method is found

    alumni_learning_method_column_mappings.append(current_df_learning_method_mapping)

print("\nCompetency and learning method column detection complete. Mappings stored.")
# Display the detected mappings for the first DataFrame as an example
if alumni_competency_column_mappings:
    print("\nExample competency mapping for the first DataFrame:")
    print(alumni_competency_column_mappings[0])
if alumni_learning_method_column_mappings:
    print("\nExample learning method mapping for the first DataFrame:")
    print(alumni_learning_method_column_mappings[0])



--- Detecting Competency and Learning Method Columns ---
Processing DataFrame from TS 2025.xlsx
Processing DataFrame from TS 2024.xlsx
  Detected 'ethics_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_etika'
  Detected 'domain_knowledge_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_keahlian_berdasarkan_bidang_ilmu'
  Detected 'english_language_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_bahasa_inggris'
  Detected 'it_skill_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_penggunaan_teknologi_informasi'
  Detected 'communication_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_komunikasi'
  Detected 'teamwork_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompetensi_di_bawah_ini_anda_kuasai_kerjasama_tim'
  Detected 'self_development_graduation' as 'pada_saat_lulus_pada_tingkat_mana_kompeten

In [ ]:
import re
import numpy as np # For np.nan

# --- NEW: Function to parse semi-structured 'kompetensi' column ---
def parse_competency_text(text):
    if pd.isna(text):
        return {}

    text = str(text)

    competency_dict = {}

    # Pattern: Nama Kompetensi | A | B
    # Updated pattern to be more flexible, matching words and spaces for skill name
    pattern = r'([A-Za-z0-9\s_\-]+)\s*\|\s*(\d)\s*\|\s*(\d)'
    matches = re.findall(pattern, text)

    for match in matches:
        skill_name = match[0].strip().lower()

        # --- CLEANING NOISE (INI YANG PENTING BANGET) ---
        skill_name = re.sub(r'[\n\r]', ' ', skill_name)         # hapus newline
        skill_name = re.sub(r'[^a-z\s]', '', skill_name)        # hapus karakter aneh
        skill_name = re.sub(r'\s+', ' ', skill_name).strip()    # rapihin spasi

        # --- HAPUS PREFIX KAYAK "b " ---
        skill_name = re.sub(r'^(a|b)\s+', '', skill_name)

        val_a = int(match[1])
        val_b = int(match[2])

        # Standardize skill naming
        skill_name = skill_name.replace(' ', '_')

        mapping = {
            'etika': 'ethics',
            'keahlian_berdasarkan_bidang_ilmu': 'domain_knowledge',
            'bahasa_inggris': 'english_language',
            'penggunaan_teknologi_informasi': 'it_skill',
            'komunikasi': 'communication',
            'kerja_sama_tim': 'teamwork',
            'pengembangan': 'self_development',
            'pengembangan_diri': 'self_development' # Added for completeness
        }

        skill_name = mapping.get(skill_name, skill_name)

        # baru ubah ke underscore
        skill_name = skill_name.replace(' ', '_')

        competency_dict[f"{skill_name}_graduation"] = val_a
        competency_dict[f"{skill_name}_work"] = val_b

    return competency_dict

# --- NEW: Function to parse semi-structured 'learning method' column ---
def parse_learning_method_text(text):
    if pd.isna(text):
        return {}

    text = str(text)
    lm_dict = {}

    # Values for qualitative assessment to 1-5 scale
    groups = {
        5: ['sangat tinggi', 'sangat besar'],
        4: ['tinggi', 'besar'],
        3: ['sedang', 'cukup besar','cukup'],
        2: ['kurang besar', 'rendah'],
        1: ['sangat kecil', 'sangat rendah']
    }

    value_mapping = {
        text: score
        for score, texts in groups.items()
        for text in texts
    }

    # Lines are in format "Method Name | Value"
    # Skip the header line "Item | Jawaban"
    lines = text.split('\n')[1:]

    for line in lines:
        if not line.strip():
            continue
        parts = line.split('|')
        if len(parts) == 2:
            method_name_raw = parts[0].strip().lower()
            method_value_raw = parts[1].strip().lower()

            # Clean method name
            method_name_cleaned = re.sub(r'\s+\(.*?\)', '', method_name_raw) # Remove text in parenthesis
            method_name_cleaned = re.sub(r'[^a-z0-9_]', '', method_name_cleaned) # Remove non-alphanumeric
            method_name_cleaned = re.sub(r'\s+', '_', method_name_cleaned) # Replace spaces with underscore

            # Map to standardized names (from learning_method_keywords in Step 5)
            mapping = {
                'perkuliahan': 'lecture',
                'simulasi/role_play': 'demonstration',
                'demonstrasi': 'demonstration', # Added for 'Demonstrasi (Peragaan)'
                'partisipasi_dalam_proyek_riset': 'research_project_participation',
                'magang': 'internship',
                'praktikum': 'lab_practicum',
                'kerja_lapangan': 'field_work',
                'kerjalapangan': 'field_work', # Ensure 'kerjalapangan' maps to 'field_work'
                'partisipasidalamproyekriset': 'research_project_participation', # Ensure 'partisipasidalamproyekriset' maps to 'research_project_participation'
                'diskusi': 'discussion'
            }
            standardized_method_name = mapping.get(method_name_cleaned, method_name_cleaned)
            standardized_method_name = f"{standardized_method_name}_implementation"

            # Convert value to 1-5 scale
            score = value_mapping.get(method_value_raw, None)
            if score is not None:
                lm_dict[standardized_method_name] = score

    return lm_dict


# --- STEP 6: EXTRACT DATA USING DETECTED COLUMNS ---

standardized_dfs = []
all_standardized_competency_cols = set()
# NEW: Define a set to collect all standardized learning method columns
all_standardized_learning_method_cols = set()

print("\n--- Extracting and Standardizing Data ---")

for i, df_temp in enumerate(list_of_dfs):
    current_df_filepath = list_of_filepaths[i]
    # FIX: Updated regex to handle spaces or underscores between 'TS' and the year
    current_year = re.search(r'TS[ _]?(\d{4})', os.path.basename(current_df_filepath), re.IGNORECASE)
    if current_year: # Extract year from filename, e.g., 'TS 2025.xlsx' -> '2025'
        source_year = int(current_year.group(1))
    else:
        source_year = None # Or some default value if year cannot be extracted

    # --- DEBUGGING: Removed Debugging Raw Competency Columns --- #

    # NEW DEBUG: Inspect semi-structured learning method column if it exists for TS 2025
    if os.path.basename(current_df_filepath) == 'TS 2025.xlsx': # Check for specific file
        semi_structured_lm_col_pattern = 'menurut_anda_seberapa_besar_penekanan_pada_metode_pembelajaran_dibawah_ini_dilaksanakan_di_program_studi_anda'
        if semi_structured_lm_col_pattern in df_temp.columns:
            print(f"\n[DEBUG] Sample raw content of semi-structured learning method column '{semi_structured_lm_col_pattern}' for TS 2025.xlsx:")
            unique_content_lm = df_temp[semi_structured_lm_col_pattern].dropna().unique()
            for val in unique_content_lm[:5]: # Print first 5 unique values
                print(f"    - {val}")
            if len(unique_content_lm) > 5: print("    ...")

    # Get mappings for current DataFrame
    data_mapping = alumni_data_column_mappings[i]
    competency_mapping = alumni_competency_column_mappings[i]
    learning_method_mapping = alumni_learning_method_column_mappings[i] # NEW

    # Combine all mappings for the current DataFrame
    full_mapping = {**data_mapping, **competency_mapping, **learning_method_mapping}

    # Invert the mapping: {'actual_cleaned_col_name': 'standard_col_name'} for renaming
    inverse_full_mapping = {v: k for k, v in full_mapping.items()}

    # Identify the semi-structured competency column for parsing if it exists
    # This targets the column that was previously marked as "ambiguous" in Step 5 for 2025 data.
    semi_structured_comp_col_name_in_df = None
    for col in df_temp.columns:
        lower_col = col.lower()
        # Look for columns containing 'kompetensi' AND ('kuasai' OR 'diperlukan')
        # AND that are NOT already explicitly mapped to a standard competency_graduation/work column.
        if 'kompetensi' in lower_col and ('kuasai' in lower_col or 'diperlukan' in lower_col):
            is_already_mapped = False
            for mapped_original_col in inverse_full_mapping.keys():
                if mapped_original_col == col:
                    is_already_mapped = True
                    break
            if not is_already_mapped:
                semi_structured_comp_col_name_in_df = col
                print(f"  Identified semi-structured competency column for parsing: '{col}'")
                break # Found it, no need to check other columns

    # Identify the semi-structured learning method column for parsing if it exists for TS 2025
    semi_structured_lm_col_name_in_df = None
    if os.path.basename(current_df_filepath) == 'TS 2025.xlsx':
        semi_structured_lm_col_pattern = 'menurut_anda_seberapa_besar_penekanan_pada_metode_pembelajaran_dibawah_ini_dilaksanakan_di_program_studi_anda'
        if semi_structured_lm_col_pattern in df_temp.columns:
            semi_structured_lm_col_name_in_df = semi_structured_lm_col_pattern
            print(f"  Identified semi-structured learning method column for parsing: '{semi_structured_lm_col_name_in_df}'")


    # Prepare list of columns to select from df_temp
    initial_cols_to_keep = [col for col in inverse_full_mapping.keys() if col in df_temp.columns]
    if semi_structured_comp_col_name_in_df and semi_structured_comp_col_name_in_df not in initial_cols_to_keep:
        initial_cols_to_keep.append(semi_structured_comp_col_name_in_df)
    # Add semi-structured LM column if found and not already in list
    if semi_structured_lm_col_name_in_df and semi_structured_lm_col_name_in_df not in initial_cols_to_keep:
        initial_cols_to_keep.append(semi_structured_lm_col_name_in_df)

    df_standardized = df_temp[initial_cols_to_keep].rename(columns=inverse_full_mapping)

    # Add a 'source_year' column
    df_standardized['source_year'] = source_year

    # --- HANDLE SEMI-STRUCTURED COMPETENCY FORMAT ---
    if semi_structured_comp_col_name_in_df and semi_structured_comp_col_name_in_df in df_standardized.columns:
        print(f"  Attempting to extract structured competency data from semi-structured column: '{semi_structured_comp_col_name_in_df}'...")

        # DEBUG: Print actual content of the semi-structured column
        print(f"  Sample raw content of '{semi_structured_comp_col_name_in_df}' (first 3 unique values):")
        unique_content = df_standardized[semi_structured_comp_col_name_in_df].dropna().unique()
        for val in unique_content[:3]:
            print(f"    - {val}")
        if len(unique_content) > 3: print("    ...")

        # Apply the new parsing function
        parsed_comp = df_standardized[semi_structured_comp_col_name_in_df].apply(parse_competency_text)

        # DEBUG: Check if parsed_comp actually contains data
        parsed_non_empty = parsed_comp[parsed_comp.apply(lambda x: bool(x))]
        if not parsed_non_empty.empty: # Only proceed if some data was actually parsed
            df_comp = pd.json_normalize(parsed_comp)
            # Concatenate the extracted competency DataFrame with the main standardized DataFrame
            df_standardized = pd.concat([df_standardized, df_comp], axis=1)

            print(f"  Extracted {len(df_comp.columns)} competency columns from text. Columns: {df_comp.columns.tolist()}")
            # Drop the original semi-structured text column as its data is now in new columns
            df_standardized = df_standardized.drop(columns=[semi_structured_comp_col_name_in_df], errors='ignore')
        else:
            print("  No data successfully parsed from '{semi_structured_comp_col_name_in_df}' using the defined pattern.")

    # --- NEW: HANDLE SEMI-STRUCTURED LEARNING METHOD FORMAT ---
    if semi_structured_lm_col_name_in_df and semi_structured_lm_col_name_in_df in df_standardized.columns:
        print(f"  Attempting to extract structured learning method data from semi-structured column: '{semi_structured_lm_col_name_in_df}'...")

        # Apply the new parsing function
        parsed_lm = df_standardized[semi_structured_lm_col_name_in_df].apply(parse_learning_method_text)

        parsed_lm_non_empty = parsed_lm[parsed_lm.apply(lambda x: bool(x))]
        if not parsed_lm_non_empty.empty:
            df_lm = pd.json_normalize(parsed_lm)
            df_standardized = pd.concat([df_standardized, df_lm], axis=1)

            print(f"  Extracted {len(df_lm.columns)} learning method columns from text. Columns: {df_lm.columns.tolist()}")
            df_standardized = df_standardized.drop(columns=[semi_structured_lm_col_name_in_df], errors='ignore')
        else:
            print("  No data successfully parsed from '{semi_structured_lm_col_name_in_df}' using the defined pattern.")


    # Track all unique competency column names encountered across all DFs
    # Get competency columns from the df_standardized after all extractions
    # FIX: Use .endswith() for precise matching of _graduation and _work suffixes.
    current_competency_cols = [col for col in df_standardized.columns if col.endswith('_graduation') or col.endswith('_work')]
    for std_comp_col_name in current_competency_cols:
        all_standardized_competency_cols.add(std_comp_col_name)

    # NEW: Track all unique learning method columns
    current_learning_method_cols = [col for col in df_standardized.columns if col.endswith('_implementation')]
    for std_lm_col_name in current_learning_method_cols:
        all_standardized_learning_method_cols.add(std_lm_col_name)

    # --- NEW DEBUG: Removed Debugging Competency columns after extraction --- #

    # --- NEW DEBUG: Removed Debugging Learning method columns after extraction --- #

    standardized_dfs.append(df_standardized)
    print(f"  Processed {os.path.basename(current_df_filepath)}: {df_standardized.shape[0]} rows, {df_standardized.shape[1]} columns.")

# Standardized columns for the final merged DataFrame
# FIX: Explicitly include 'source_year' in the standard_columns list
standard_columns = [
    'nim',
    'nama',
    'tahun_lulus',
    'status_pekerjaan',
    'nama_perusahaan',
    'posisi_pekerjaan',
    'masa_tunggu_bulan',
    'gaji',
    'source_year',
    'saran' # Added 'saran' to the standard columns
] + sorted(list(all_standardized_competency_cols)) + sorted(list(all_standardized_learning_method_cols)) # NEW: Add learning method columns

# Combine all standardized DataFrames
# We will use pd.concat later in Step 10, but for now, let's prepare the individual DFs.
# The final merge will ensure consistent columns, so for now, we just ensure each df has its detected cols.

print("\nIndividual DataFrames standardized. They will be merged in a later step.")
print(f"Total {len(standardized_dfs)} standardized DataFrames created.")
print(f"All unique standardized competency columns found: {sorted(list(all_standardized_competency_cols))}")
print(f"All unique standardized learning method columns found: {sorted(list(all_standardized_learning_method_cols))}") # NEW

# Display the first few rows and columns of the first standardized DataFrame as an example
if standardized_dfs:
    print("\nFirst 5 rows of the first standardized DataFrame (example):")
    display(standardized_dfs[0])
    print("Columns of the first standardized DataFrame:")
    print(standardized_dfs[0].columns.tolist())


--- Extracting and Standardizing Data ---

[DEBUG] Sample raw content of semi-structured learning method column 'menurut_anda_seberapa_besar_penekanan_pada_metode_pembelajaran_dibawah_ini_dilaksanakan_di_program_studi_anda' for TS 2025.xlsx:
    - Item | Jawaban
Perkuliahan | Sangat Besar
Demonstrasi (Peragaan) | Besar
Partisipasi dalam proyek riset | Cukup Besar
Magang | Besar
Praktikum | Besar
Kerja Lapangan | Besar
Diskusi | Besar
    - Item | Jawaban
Perkuliahan | Sangat Besar
Demonstrasi (Peragaan) | Sangat Besar
Partisipasi dalam proyek riset | Sangat Besar
Magang | Cukup Besar
Praktikum | Sangat Besar
Kerja Lapangan | Sangat Besar
Diskusi | Sangat Besar
    - Item | Jawaban
Perkuliahan | Sangat Besar
Demonstrasi (Peragaan) | Sangat Besar
Partisipasi dalam proyek riset | Sangat Besar
Magang | Sangat Besar
Praktikum | Sangat Besar
Kerja Lapangan | Sangat Besar
Diskusi | Sangat Besar
    - Item | Jawaban
Perkuliahan | Cukup Besar
Demonstrasi (Peragaan) | Cukup Besar
Partisipasi da

,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,saran,source_year,...,teamwork_work,self_development_graduation,self_development_work,lecture_implementation,demonstration_implementation,research_project_participation_implementation,internship_implementation,lab_practicum_implementation,field_work_implementation,discussion_implementation
0,1172001006,Rizal Fahlepi,2024,Bekerja (full time / part time),Perusahaan swasta,"IT Governance, Risk & Compliance Lead",0.0,9000000.0,NaN,2025,...,5,4,5,5,4,3,4,4,4,4
1,1202001007,Sheila Riva Rezqian,2024,Bekerja (full time / part time),Perusahaan swasta,IT Quality Assurance,2.0,5600000.0,NaN,2025,...,5,5,5,5,5,5,3,5,5,5
2,1172001032,Ananda Anggita Pratama,2024,Bekerja (full time / part time),Perusahaan swasta,"Content Manager, Content Creator, Semi-Team Cr...",0.0,12000000.0,"Karena perusahaan ini, saya bisa bertemu denga...",2025,...,5,5,5,5,5,5,5,5,5,5
3,1202001025,Mumtaz Aaliyah Fasya,2024,Bekerja (full time / part time),Perusahaan luar negeri,Junior UI/UX Designer,8.0,10000000.0,UI/UX is fun,2025,...,5,4,4,3,3,3,5,2,4,5
4,1192001003,Muhammad Alvin Alyundra,2024,Bekerja (full time / part time),Wiraswasta/perusahaan sendiri,Staff,2.0,7400000.0,Saya berharap Universitas dapat terus meningka...,2025,...,2,5,5,4,3,5,5,5,5,5
5,1172001019,RITLY TANIA SASIANG,2024,Tidak kerja tetapi sedang mencari kerja,NaN,NaN,NaN,NaN,Saya berterima kasih kepada Universitas Bakrie...,2025,...,5,4,5,5,3,3,4,4,3,4
6,1202921001,Donny Ostrianto Sundawa,2024,Bekerja (full time / part time),Perusahaan swasta,Project Manager,1.0,30000000.0,Tugas-tugas berbasis project lebih diperbanyak...,2025,...,5,5,4,4,4,5,4,4,4,5
7,1202001005,Chika Humaira Abidatillah,2024,Bekerja (full time / part time),Pendidikan,Guru Informatika,2.0,4000000.0,Semoga universitas bisa memberikan lebih banya...,2025,...,5,5,5,5,4,4,5,5,3,4
8,1182001006,Al Syahga,2024,Bekerja (full time / part time),Wiraswasta/perusahaan sendiri,CTO,0.0,12000000.0,Untuk Universitas Bakrie : sudah bagus dalam m...,2025,...,5,5,5,5,5,5,5,5,5,5
9,1192001001,Iqtarra Rizqiva Syachturi,2024,Bekerja (full time / part time),Perusahaan swasta,Magang KOL Specialis,2.0,3000000.0,Semoga kedepannya untuk jurusan teknik informa...,2025,...,5,5,5,4,4,4,3,4,3,4


Columns of the first standardized DataFrame:
['nim', 'nama', 'tahun_lulus', 'status_pekerjaan', 'nama_perusahaan', 'posisi_pekerjaan', 'masa_tunggu_bulan', 'gaji', 'saran', 'source_year', 'ethics_graduation', 'ethics_work', 'domain_knowledge_graduation', 'domain_knowledge_work', 'english_language_graduation', 'english_language_work', 'it_skill_graduation', 'it_skill_work', 'communication_graduation', 'communication_work', 'teamwork_graduation', 'teamwork_work', 'self_development_graduation', 'self_development_work', 'lecture_implementation', 'demonstration_implementation', 'research_project_participation_implementation', 'internship_implementation', 'lab_practicum_implementation', 'field_work_implementation', 'discussion_implementation']


In [ ]:
# --- STEP 7: FIX DATA TYPES ---FIX DATA TYPES, SALARY CORRECTION, AND COMPETENCY NORMALIZATION ---

print("\n--- Fixing Data Types, Applying Salary Correction & Competency Normalization ---")

# --- Debugging: Track total rows after Step 7 ---
total_rows_step7 = 0
all_dfs_step7 = [] # To store DataFrames at end of step 7 for debugging

for i, df_standardized in enumerate(standardized_dfs):
    current_df_filepath = list_of_filepaths[i]
    print(f"Processing DataFrame from {os.path.basename(current_df_filepath)}")

    # --- NEW: STANDARDIZE 'tahun_lulus' Column EARLY ---
    if 'tahun_lulus' in df_standardized.columns:
        # Ensure 'tahun_lulus' is string type for regex application
        df_standardized['tahun_lulus'] = df_standardized['tahun_lulus'].astype(str)

        # Regex to find a four-digit number, typically representing a year
        def extract_year(value):
            if pd.isna(value):
                return None

            value = str(value)

            # Cari angka tahun (4 digit)
            match = re.search(r'(20\d{2})', value)

            if match:
                return int(match.group(1))

            return None

        df_standardized['tahun_lulus'] = df_standardized['tahun_lulus'].apply(extract_year)
        # Convert 'tahun_lulus' to nullable integer (Int64) for consistency
        df_standardized['tahun_lulus'] = pd.to_numeric(df_standardized['tahun_lulus'], errors='coerce').astype('Int64')
        print("  'tahun_lulus' column standardized (year extracted and converted to Int64).")
        print(f"  Unique graduation years after standardization: {sorted(df_standardized['tahun_lulus'].dropna().unique())}")

    # Convert NIM to string
    if 'nim' in df_standardized.columns:
        df_standardized['nim'] = df_standardized['nim'].astype(str)
        print("  'nim' column converted to string.")

    # Convert masa_tunggu_bulan to numeric
    if 'masa_tunggu_bulan' in df_standardized.columns:
        # Ensure string type for regex processing
        df_standardized['masa_tunggu_bulan'] = df_standardized['masa_tunggu_bulan'].astype(str)

        # Function to extract waiting time and convert to months
        def extract_masa_tunggu(value):
            if pd.isna(value):
                return None

            value = str(value).lower()

            # Cari angka (bisa negatif & desimal)
            match = re.search(r'(-?\d+\.?\d*)', value)

            if not match:
                return None

            num = float(match.group(1))

            # Konversi ke bulan
            if 'tahun' in value:
                return int(num * 12)
            elif 'bulan' in value:
                return int(num)
            elif 'minggu' in value:
                return int(num / 4)
            elif re.fullmatch(r'-?\d+\.?\d*', value):
                return int(num)  # asumsi = bulan

            return None

        # Apply extraction
        df_standardized['masa_tunggu_bulan'] = df_standardized['masa_tunggu_bulan'].apply(extract_masa_tunggu)

        # Convert ke numeric (nullable integer)
        df_standardized['masa_tunggu_bulan'] = pd.to_numeric(
            df_standardized['masa_tunggu_bulan'],
            errors='coerce'
        ).astype('Int64')
        print("  'masa_tunggu_bulan' column standardized (extracted and converted to months).")
        print("\nUnique values after standardization:")
        print(sorted(df_standardized['masa_tunggu_bulan'].dropna().unique()))

    # Convert gaji to numeric
    if 'gaji' in df_standardized.columns:
        # First, ensure it's string to handle potential scientific notation or commas
        df_standardized['gaji'] = df_standardized['gaji'].astype(str).str.replace(r'[^\d]', '', regex=True)
        df_standardized['gaji'] = pd.to_numeric(df_standardized['gaji'], errors='coerce')
        print("  'gaji' column converted to numeric.")

        # --- IMPROVEMENT 1: FIX SALARY VALUES ---
        print("  Applying salary sanity checks and automatic correction...")
        original_high_salary_count = (df_standardized['gaji'] > 100_000_000).sum()

        def correct_salary(salary):
            if pd.isna(salary):
                return salary
            salary_val = float(salary)
            if salary_val > 100_000_000: # Check if salary is abnormally high
                original_salary = salary_val
                # Attempt automatic correction by dividing by powers of 10
                while salary_val > 50_000_000 and salary_val > 1000: # Max salary 50M, min meaningful value 1000
                    salary_val /= 10
                # If after division it's still too high or became too low (below 1M threshold), revert to original
                if salary_val < 1_000_000 or salary_val > 50_000_000:
                    print(f"    Salary {original_salary:.0f} was corrected to {salary_val:.0f}, but still outside 1M-50M range. Reverting to original or setting NaN.")
                    return original_salary # Revert to original if couldn't normalize well enough
                print(f"    Corrected salary from {original_salary:.0f} to {salary_val:.0f}")
                return salary_val
            return salary_val

        df_standardized['gaji'] = df_standardized['gaji'].apply(correct_salary)
        # Convert salary to integer after correction, allowing for NaN
        # FIX: Explicitly round the values before casting to Int64
        df_standardized['gaji'] = pd.to_numeric(df_standardized['gaji'], errors='coerce').round().astype('Int64')
        print(f"  {original_high_salary_count} high salary values were identified and attempts made to correct them.")

    # Iterate through all detected competency columns and convert to numeric
    for comp_col in all_standardized_competency_cols:
        if comp_col in df_standardized.columns:

            print(f"\nProcessing column: {comp_col}")

            # --- STEP 1: EXTRACT NUMERIC VALUE FROM TEXT ---
            def extract_score(value):
                if pd.isna(value):
                    return None

                value_str = str(value).lower()

                # PRIORITY 1: ambil angka di dalam []
                match_bracket = re.search(r'\[(\d+)\]', value_str)
                if match_bracket:
                    return float(match_bracket.group(1))

                # PRIORITY 2: ambil angka biasa
                match_number = re.search(r'(\d+\.?\d*)', value_str)
                if match_number:
                    return float(match_number.group(1))

                # PRIORITY 3: mapping dari teks (fallback)
                if 'sangat tinggi' in value_str or 'sangat besar' in value_str:
                    return 5
                elif 'tinggi' in value_str or 'besar' in value_str:
                    return 4
                elif 'cukup' in value_str or 'sedang' in value_str:
                    return 3
                elif 'rendah' in value_str or 'kecil' in value_str:
                    return 2
                elif 'sangat rendah' in value_str or 'sangat kecil' in value_str:
                    return 1

                return None

            df_standardized[comp_col] = df_standardized[comp_col].apply(extract_score)

            # --- STEP 2: NORMALIZE TO 1\u20135 SCALE ---
            print(f"  Normalizing '{comp_col}' to 1\u20135 scale...")

            def normalize_competency_score(score):
                if pd.isna(score):
                    return None

                score = float(score)

                # Case: skala 0\u2013100
                if score > 5:
                    if score > 100:
                        score = 100
                    return round((score / 100) * 4) + 1

                # Case: nilai 0
                elif score < 1:
                    return 1

                return round(score)

            df_standardized[comp_col] = df_standardized[comp_col].apply(normalize_competency_score).astype('Int64')

            # Debug output
            print(f"  Unique values after cleaning: {sorted(df_standardized[comp_col].dropna().unique())}")

    # NEW: Iterate through all detected learning method columns and convert to numeric
    for lm_col in all_standardized_learning_method_cols:
        if lm_col in df_standardized.columns:
            print(f"\nProcessing learning method column: {lm_col}")
            # Apply the same score extraction and normalization as for competency columns
            df_standardized[lm_col] = df_standardized[lm_col].apply(extract_score)
            df_standardized[lm_col] = df_standardized[lm_col].apply(normalize_competency_score).astype('Int64')
            print(f"  Unique values after cleaning: {sorted(df_standardized[lm_col].dropna().unique())}")

    # Convert 'source_year' to numeric (Int64)
    if 'source_year' in df_standardized.columns:
        df_standardized['source_year'] = pd.to_numeric(df_standardized['source_year'], errors='coerce').astype('Int64')
        print("  'source_year' column converted to numeric (Int64).")

    # Update the DataFrame in the list
    standardized_dfs[i] = df_standardized

    # --- Debugging: Add to list of all DFs at this step for later comparison
    all_dfs_step7.append(df_standardized)
    total_rows_step7 += len(df_standardized)

print("\nData type fixing, salary correction, and competency normalization complete for all DataFrames.")

# Display data types of the first standardized DataFrame as an example
if standardized_dfs:
    print("\nData types of the first standardized DataFrame (example):")
    display(standardized_dfs[0].dtypes)


--- Fixing Data Types, Applying Salary Correction & Competency Normalization ---
Processing DataFrame from TS 2025.xlsx
  'tahun_lulus' column standardized (year extracted and converted to Int64).
  Unique graduation years after standardization: [np.int64(2024)]
  'nim' column converted to string.
  'masa_tunggu_bulan' column standardized (extracted and converted to months).

Unique values after standardization:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(8)]
  'gaji' column converted to numeric.
  Applying salary sanity checks and automatic correction...
    Corrected salary from 120000000 to 12000000
    Corrected salary from 300000000 to 30000000
    Corrected salary from 120000000 to 12000000
  3 high salary values were identified and attempts made to correct them.

Processing column: it_skill_graduation
  Normalizing 'it_skill_graduation' to 1–5 scale...
  Unique values after cleaning: [np.int64(4), np.int64(5)]

Processing column: teamwork_work
  Normalizing 't

,0
nim,object
nama,object
tahun_lulus,Int64
status_pekerjaan,object
nama_perusahaan,object
posisi_pekerjaan,object
masa_tunggu_bulan,Int64
gaji,Int64
saran,object
source_year,Int64


In [ ]:
# --- STEP 8: REMOVE INVALID ROWS (WITH DEBUGGING) ---

print("\n--- Removing Invalid Rows (Step 8 Debugging) ---")

# Snapshot before Step 8
initial_total_rows_step8 = sum(len(df) for df in standardized_dfs)
print(f"Initial total rows across all DataFrames before Step 8: {initial_total_rows_step8}")

filtered_dfs = []
removed_rows_count_step8 = 0
all_removed_rows_step8_df = pd.DataFrame() # To store all removed rows for inspection (Step G)

for i, df_standardized in enumerate(standardized_dfs):
    current_df_filepath = list_of_filepaths[i]
    df_name = os.path.basename(current_df_filepath)
    current_df_initial_rows_before_filter = len(df_standardized)

    print(f"\nProcessing DataFrame: {df_name} (Initial rows: {current_df_initial_rows_before_filter})")

    # --- STEP B: CHECK MISSING VALUES ---
    print("  Missing values in key columns (nim, nama, tahun_lulus) before filtering:")
    display(df_standardized[['nim', 'nama', 'tahun_lulus']].isnull().sum())

    print("  Rows with missing 'nim':")
    display(df_standardized[df_standardized['nim'].isnull()])

    print("  Rows with missing 'nama':")
    display(df_standardized[df_standardized['nama'].isnull()])

    print("  Rows with missing 'tahun_lulus':")
    display(df_standardized[df_standardized['tahun_lulus'].isnull()])

    # Identify rows to be removed (for Step G)
    rows_to_drop_by_key = df_standardized[
        df_standardized['nim'].isnull() |
        df_standardized['nama'].isnull() |
        df_standardized['tahun_lulus'].isnull() |
        (df_standardized['nim'].astype(str).str.strip() == '') |
        (df_standardized['nama'].astype(str).str.strip() == '')
    ].copy() # Use copy to avoid SettingWithCopyWarning

    # --- STEP C: TRACK ROW REMOVAL (CONTROLLED FILTERING) ---
    # Drop rows where 'nim', 'nama', or 'tahun_lulus' are missing (NaN or None) or empty strings
    df_filtered = df_standardized.dropna(subset=['nim', 'nama', 'tahun_lulus']).copy() # Start with dropna

    # Additionally, filter out empty strings if they exist after dropna (astype(str) might create 'nan' strings)
    if 'nim' in df_filtered.columns: # Check again if column exists after dropna
        df_filtered = df_filtered[df_filtered['nim'].astype(str).str.strip() != ''].copy()
    if 'nama' in df_filtered.columns: # Check again if column exists after dropna
        df_filtered = df_filtered[df_filtered['nama'].astype(str).str.strip() != ''].copy()

    current_df_removed_rows = current_df_initial_rows_before_filter - len(df_filtered)
    removed_rows_count_step8 += current_df_removed_rows

    print(f"  Rows removed due to missing key data (nim, nama, tahun_lulus): {current_df_removed_rows}")
    filtered_dfs.append(df_filtered)

    # --- STEP G: EXPORT REMOVED ROWS (Individual DF Level) ---
    if not rows_to_drop_by_key.empty:
        rows_to_drop_by_key['removed_reason'] = 'Missing_Key_Data'
        all_removed_rows_step8_df = pd.concat([all_removed_rows_step8_df, rows_to_drop_by_key], ignore_index=True)

standardized_dfs = filtered_dfs # Update the list of DataFrames for subsequent steps
final_total_rows_step8 = sum(len(df) for df in standardized_dfs)

print(f"\nTotal rows removed in Step 8: {removed_rows_count_step8}")
print(f"Final total rows across all DataFrames after Step 8: {final_total_rows_step8}")

# Save all removed rows from Step 8 to CSV (Step G)
if not all_removed_rows_step8_df.empty:
    all_removed_rows_step8_df.to_csv("removed_tracer_rows_step8_debug.csv", index=False)
    print("Removed rows from Step 8 saved to 'removed_tracer_rows_step8_debug.csv' for inspection.")
else:
    print("No rows were removed in Step 8, so no debug file generated.")

# Display shape of the first filtered DataFrame as an example
if standardized_dfs:
    print("\nShape of the first filtered DataFrame (example):")
    print(standardized_dfs[0].shape)


--- Removing Invalid Rows (Step 8 Debugging) ---
Initial total rows across all DataFrames before Step 8: 26

Processing DataFrame: TS 2025.xlsx (Initial rows: 14)
  Missing values in key columns (nim, nama, tahun_lulus) before filtering:


,0
nim,0
nama,0
tahun_lulus,0


  Rows with missing 'nim':


,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,saran,source_year,...,teamwork_work,self_development_graduation,self_development_work,lecture_implementation,demonstration_implementation,research_project_participation_implementation,internship_implementation,lab_practicum_implementation,field_work_implementation,discussion_implementation


  Rows with missing 'nama':


...

  Rows with missing 'nama':


,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,saran,ethics_graduation,...,teamwork_work,self_development_work,lecture_implementation,demonstration_implementation,research_project_participation_implementation,internship_implementation,lab_practicum_implementation,field_work_implementation,discussion_implementation,source_year


  Rows with missing 'tahun_lulus':


,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,saran,ethics_graduation,...,teamwork_work,self_development_work,lecture_implementation,demonstration_implementation,research_project_participation_implementation,internship_implementation,lab_practicum_implementation,field_work_implementation,discussion_implementation,source_year


  Rows removed due to missing key data (nim, nama, tahun_lulus): 0

Total rows removed in Step 8: 0
Final total rows across all DataFrames after Step 8: 26
No rows were removed in Step 8, so no debug file generated.

Shape of the first filtered DataFrame (example):
(14, 31)


In [ ]:
# --- STEP 10: MERGE ALL DATASETS ---

print("\n--- Merging All DataFrames ---")

# Define the target data types for standardization
dtype_map = {
    'nim': str,  # Keep NIM as string
    'nama': str,
    'tahun_lulus': 'Int64', # Nullable integer
    'status_pekerjaan': str,
    'nama_perusahaan': str,
    'posisi_pekerjaan': str,
    'masa_tunggu_bulan': 'Int64', # Nullable integer
    'gaji': 'Int64', # Nullable integer
    'source_year': 'Int64', # Nullable integer
    'saran': str
}

# Add competency columns to the dtype_map, assuming they should be nullable integers
for col in standard_columns:
    if '_graduation' in col or '_work' in col or '_implementation' in col: # NEW: Include learning method columns
        dtype_map[col] = 'Int64'

final_dfs_for_concat = []
for df in standardized_dfs:
    # Create an empty DataFrame with the desired columns and dtypes
    df_reindexed = pd.DataFrame(columns=standard_columns)

    # Apply dtypes to the empty DataFrame before filling, to prevent type coercion to object
    for col, dtype in dtype_map.items():
        if col in df_reindexed.columns: # Only apply to columns that are actually in standard_columns
            df_reindexed[col] = df_reindexed[col].astype(dtype)

    for col in df_reindexed.columns:
        if col in df.columns:
            # Assign data, ensuring it conforms to the target dtype, coercing errors to NaN
            if col in dtype_map and dtype_map[col] != str: # For numeric columns, coerce errors
                df_reindexed[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype_map[col])
            else:
                df_reindexed[col] = df[col].astype(str).replace('<NA>', pd.NA) # Ensure string columns handle NA correctly
        else:
            df_reindexed[col] = pd.NA # Explicitly set missing columns to pd.NA

    final_dfs_for_concat.append(df_reindexed)

# Concatenate all DataFrames into a single one
tracer_study_df = pd.concat(final_dfs_for_concat, ignore_index=True)

# Reset index as requested
tracer_study_df = tracer_study_df.reset_index(drop=True)

print(f"Merged DataFrame created with {len(tracer_study_df)} rows and {len(tracer_study_df.columns)} columns.")

# Display the head of the merged DataFrame
print("\nFirst 5 rows of the merged tracer_study_df:")
display(tracer_study_df.head())

# Display data types of the merged DataFrame for verification
print("\nData types of the merged tracer_study_df:")
display(tracer_study_df.dtypes)


--- Merging All DataFrames ---
Merged DataFrame created with 26 rows and 31 columns.

First 5 rows of the merged tracer_study_df:


,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,source_year,saran,...,self_development_work,teamwork_graduation,teamwork_work,demonstration_implementation,discussion_implementation,field_work_implementation,internship_implementation,lab_practicum_implementation,lecture_implementation,research_project_participation_implementation
0,1172001006,Rizal Fahlepi,2024,Bekerja (full time / part time),Perusahaan swasta,"IT Governance, Risk & Compliance Lead",0,90000000,2025,nan,...,5,4,5,4,4,4,4,4,5,3
1,1202001007,Sheila Riva Rezqian,2024,Bekerja (full time / part time),Perusahaan swasta,IT Quality Assurance,2,56000000,2025,nan,...,5,5,5,5,5,5,3,5,5,5
2,1172001032,Ananda Anggita Pratama,2024,Bekerja (full time / part time),Perusahaan swasta,"Content Manager, Content Creator, Semi-Team Cr...",0,12000000,2025,"Karena perusahaan ini, saya bisa bertemu denga...",...,5,5,5,5,5,5,5,5,5,5
3,1202001025,Mumtaz Aaliyah Fasya,2024,Bekerja (full time / part time),Perusahaan luar negeri,Junior UI/UX Designer,8,100000000,2025,UI/UX is fun,...,4,5,5,3,5,4,5,2,3,3
4,1192001003,Muhammad Alvin Alyundra,2024,Bekerja (full time / part time),Wiraswasta/perusahaan sendiri,Staff,2,74000000,2025,Saya berharap Universitas dapat terus meningka...,...,5,5,2,3,5,5,5,5,4,5



Data types of the merged tracer_study_df:


,0
nim,object
nama,object
tahun_lulus,Int64
status_pekerjaan,object
nama_perusahaan,object
posisi_pekerjaan,object
masa_tunggu_bulan,Int64
gaji,Int64
source_year,Int64
saran,object


In [ ]:
import pandas as pd
import numpy as np

# --- IMPROVEMENT 4: ENSURE COMPETENCY PAIRS EXIST ---

print("\n--- Ensuring Competency Pairs Exist ---")

# Create a set of all unique competency roots (e.g., 'ethics', 'communication')
competency_roots = set()
for comp_col in all_standardized_competency_cols:
    # Split 'ethics_graduation' -> 'ethics'
    root = comp_col.replace('_graduation', '').replace('_work', '')
    competency_roots.add(root)

# Prepare a list of all desired full competency column names (e.g., 'ethics_graduation', 'ethics_work')
all_expected_competency_cols = set()
for root in competency_roots:
    all_expected_competency_cols.add(f"{root}_graduation")
    all_expected_competency_cols.add(f"{root}_work")

all_expected_competency_cols = sorted(list(all_expected_competency_cols))

# NEW: Prepare a list of all desired learning method column names
all_expected_learning_method_cols = sorted(list(all_standardized_learning_method_cols))

# Iterate through each DataFrame and add missing competency columns
for i, df in enumerate(standardized_dfs):
    current_df_filepath = list_of_filepaths[i]
    print(f"Processing DataFrame from {os.path.basename(current_df_filepath)}")

    for expected_col in all_expected_competency_cols:
        if expected_col not in df.columns:
            df[expected_col] = pd.NA # Add missing column with NaN
            print(f"  Added missing competency column: '{expected_col}'")

    # NEW: Add missing learning method columns
    for expected_col in all_expected_learning_method_cols:
        if expected_col not in df.columns:
            df[expected_col] = pd.NA # Add missing column with NaN
            print(f"  Added missing learning method column: '{expected_col}'")

    standardized_dfs[i] = df # Update the DataFrame in the list

# Update standard_columns to include all expected competency and learning method columns
# This is crucial for the final concat operation to have a consistent schema.
# It's important that this `standard_columns` list is defined *before* Step 10.
standard_columns_updated = [
    'nim',
    'nama',
    'tahun_lulus',
    'status_pekerjaan',
    'nama_perusahaan',
    'posisi_pekerjaan',
    'masa_tunggu_bulan',
    'gaji',
    'source_year',
    'saran' # Added 'saran' here to ensure it's in the updated standard columns
] + all_expected_competency_cols + all_expected_learning_method_cols # NEW: Add all learning method columns here

# Overwrite the global standard_columns with the updated list for consistency in later steps
standard_columns = standard_columns_updated

print("\nEnsuring competency pairs and learning methods complete.")
print(f"Updated list of standard columns now includes: {standard_columns}")


--- Ensuring Competency Pairs Exist ---
Processing DataFrame from TS 2025.xlsx
Processing DataFrame from TS 2024.xlsx

Ensuring competency pairs and learning methods complete.
Updated list of standard columns now includes: ['nim', 'nama', 'tahun_lulus', 'status_pekerjaan', 'nama_perusahaan', 'posisi_pekerjaan', 'masa_tunggu_bulan', 'gaji', 'source_year', 'saran', 'communication_graduation', 'communication_work', 'domain_knowledge_graduation', 'domain_knowledge_work', 'english_language_graduation', 'english_language_work', 'ethics_graduation', 'ethics_work', 'it_skill_graduation', 'it_skill_work', 'self_development_graduation', 'self_development_work', 'teamwork_graduation', 'teamwork_work', 'demonstration_implementation', 'discussion_implementation', 'field_work_implementation', 'internship_implementation', 'lab_practicum_implementation', 'lecture_implementation', 'research_project_participation_implementation']


In [ ]:
# --- STEP 11: REMOVE DUPLICATES ---

print("\n--- Removing Duplicate Alumni Entries ---")

initial_rows_before_dedup = len(tracer_study_df)
print(f"Initial rows in merged DataFrame: {initial_rows_before_dedup}")

# Sort by 'nim' and then by a proxy for completeness (e.g., number of non-null values)
# This helps in keeping the 'most complete' record when duplicates are dropped.
# First, count non-nulls for each row
tracer_study_df['non_null_count'] = tracer_study_df.count(axis=1)

# Sort by nim and then by non_null_count in descending order
tracer_study_df_sorted = tracer_study_df.sort_values(by=['nim', 'non_null_count'], ascending=[True, False])

# Drop duplicates based on 'nim', keeping the first (most complete due to sorting)
final_tracer_study_df = tracer_study_df_sorted.drop_duplicates(subset=['nim'], keep='first').reset_index(drop=True)

# Remove the temporary 'non_null_count' column
final_tracer_study_df = final_tracer_study_df.drop(columns=['non_null_count'])

final_rows_after_dedup = len(final_tracer_study_df)
removed_duplicates_count = initial_rows_before_dedup - final_rows_after_dedup

print(f"Removed {removed_duplicates_count} duplicate alumni entries.")
print(f"Final rows after deduplication: {final_rows_after_dedup}")

# Display summary of duplicates if any were found
if removed_duplicates_count > 0:
    print("\nSample of records before and after deduplication (if duplicates existed):")
    # Display some original rows that contained duplicates (if any)
    duplicated_nims = tracer_study_df_sorted[tracer_study_df_sorted.duplicated(subset=['nim'], keep=False)]['nim'].unique()
    if len(duplicated_nims) > 0:
        print("Original records involved in deduplication (first 5 duplicated NIMs):")
        display(tracer_study_df_sorted[tracer_study_df_sorted['nim'].isin(duplicated_nims[:5])].head(10))

print("\nFirst 5 rows of the final DataFrame after deduplication:")
display(final_tracer_study_df.head())


--- Removing Duplicate Alumni Entries ---
Initial rows in merged DataFrame: 26
Removed 0 duplicate alumni entries.
Final rows after deduplication: 26

First 5 rows of the final DataFrame after deduplication:


,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,source_year,saran,...,self_development_work,teamwork_graduation,teamwork_work,demonstration_implementation,discussion_implementation,field_work_implementation,internship_implementation,lab_practicum_implementation,lecture_implementation,research_project_participation_implementation
0,0,Muhammad Alif Farhan Dewa,2023,[1] Bekerja (full time / part time),PT Xapiens Teknologi Indonesia,Business Analyst,-3,12900000,2024,nan,...,3,3,3,3,4,4,4,4,3,3
1,1162001002,PILIPUS DELEVIA VEGAS,2023,[1] Bekerja (full time / part time),"Globalindo Group, (PT. Vista Mandiri Gemilang ...",IT Programmer,<NA>,<NA>,2024,"pendapat saya untuk prodi, karna ilmu bukan ha...",...,4,4,4,4,3,4,4,3,3,4
2,1162001004,Refa Nurleana,2023,[1] Bekerja (full time / part time),PT. Asuransi Jiwa Generali Indonesia,Staff Reporting,-18,5000000,2024,-,...,4,4,4,4,4,4,4,4,4,4
3,1162001008,Binya Amary,2023,[1] Bekerja (full time / part time),PT Asuransi Jiwa Generali Indonesia,Data Reporting,-18,5000000,2024,Memperbanyak proyek riset yang kemungkinan aka...,...,5,4,4,2,4,2,3,4,4,2
4,1162001011,Muhammad Fiqih Husain,2023,[1] Bekerja (full time / part time),-,-,-1,7000000,2024,"Untuk Teknik Informatika, lebih diperbanyak la...",...,5,5,5,2,2,2,3,2,4,2


In [ ]:
import pandas as pd
import numpy as np
import sys # Import sys for graceful exit

print("\n--- Filtering Tracer Study Data for 2020 Cohort ---")

# Check if final_tracer_study_df exists. If not, inform the user and exit.
if 'final_tracer_study_df' not in locals() and 'final_tracer_study_df' not in globals():
    print("Error: 'final_tracer_study_df' not found. Please ensure that the preceding cell (f1d333af) has been executed successfully.")
    sys.exit(1)

# 1. Get initial row count before filtering
initial_rows_before_filtering = len(final_tracer_study_df)
print(f"Total rows before 2020 cohort filtering: {initial_rows_before_filtering}")

# 2. Ensure 'nim' column is string type and handle NaN values
# Fill NaN with empty string to prevent errors during string operations
final_tracer_study_df['nim'] = final_tracer_study_df['nim'].astype(str).fillna('')

# 3. Remove data with invalid NIMs (length < 3)
# Create a mask for valid NIMs based on length
valid_nim_mask = final_tracer_study_df['nim'].str.len() >= 3
# Store rows that will be removed for potential debugging (optional, but good practice)
removed_invalid_nim_df = final_tracer_study_df[~valid_nim_mask].copy()
final_tracer_study_df = final_tracer_study_df[valid_nim_mask].copy()

print(f"Removed {len(removed_invalid_nim_df)} rows with invalid NIMs (length < 3 or NaN).")

# 4. Extract the 2nd and 3rd digits from NIM (index 1 to 3, exclusive of 3)
# and store it in a new column 'tahun_masuk_nim'
# Convert to numeric, coercing errors to NaN, then to nullable integer (Int64)
final_tracer_study_df['tahun_masuk_nim'] = pd.to_numeric(
    final_tracer_study_df['nim'].str[1:3],
    errors='coerce'
).astype('Int64')

# --- MODIFIED: Convert tahun_masuk_nim from '20' to '2020' ---
final_tracer_study_df['tahun_masuk_nim'] = final_tracer_study_df['tahun_masuk_nim'].apply(lambda x: 2000 + x if pd.notna(x) else x)

# 5. Filter for students with 'tahun_masuk_nim' equal to 2020 (representing 2020)
cohort_2020_mask = (final_tracer_study_df['tahun_masuk_nim'] == 2020)

# Store rows that will be removed (not 2020 cohort)
removed_non_2020_cohort_df = final_tracer_study_df[~cohort_2020_mask].copy()

# Overwrite final_tracer_study_df with only the filtered data
final_tracer_study_df = final_tracer_study_df[cohort_2020_mask].copy()

# 6. Display results
final_rows_after_filtering = len(final_tracer_study_df)
rows_removed_by_cohort_filter = initial_rows_before_filtering - final_rows_after_filtering

print(f"Removed {len(removed_non_2020_cohort_df)} rows not belonging to the 2020 cohort.")
print(f"Total rows after 2020 cohort filtering: {final_rows_after_filtering}")
print(f"Total rows removed during 2020 cohort filtering step: {rows_removed_by_cohort_filter}")

print("\nFirst 5 rows of the filtered 2020 cohort data:")
display(final_tracer_study_df[['nim', 'tahun_masuk_nim', 'nama', 'tahun_lulus']].head())


--- Filtering Tracer Study Data for 2020 Cohort ---
Total rows before 2020 cohort filtering: 26
Removed 1 rows with invalid NIMs (length < 3 or NaN).
Removed 18 rows not belonging to the 2020 cohort.
Total rows after 2020 cohort filtering: 7
Total rows removed during 2020 cohort filtering step: 19

First 5 rows of the filtered 2020 cohort data:


,nim,tahun_masuk_nim,nama,tahun_lulus
19,1202001004,2020,Faatihah Rahmatillah,2024
20,1202001005,2020,Chika Humaira Abidatillah,2024
21,1202001007,2020,Sheila Riva Rezqian,2024
22,1202001010,2020,ramadhani asri,2024
23,1202001025,2020,Mumtaz Aaliyah Fasya,2024


In [ ]:
# --- PROBLEM 5: FINAL VALIDATION ---

print("\n--- Final Validation Summary ---")

# Total rows
print(f"Total rows in final dataset: {len(final_tracer_study_df)}")

# Rows per source_year
if 'source_year' in final_tracer_study_df.columns:
    print("\nRows per source year:")
    display(final_tracer_study_df['source_year'].value_counts().sort_index())
else:
    print("\n'source_year' column not found for breakdown.")

# Salary statistics
print("\nSalary Statistics (gaji):")
if 'gaji' in final_tracer_study_df.columns:
    display(final_tracer_study_df['gaji'].describe())
else:
    print("'gaji' column not found.")

# Competency column list
print("\nCompetency Columns in Final Dataset:")
# Filter for columns that are part of the competency structure (e.g., ending with _graduation or _work)
competency_cols_in_df = [col for col in final_tracer_study_df.columns if col.endswith('_graduation') or col.endswith('_work')]
if competency_cols_in_df:
    for col in sorted(competency_cols_in_df):
        print(f"- {col}")
else:
    print("No specific competency columns identified.")

# NEW: Learning Method column list
print("\nLearning Method Columns in Final Dataset:")
lm_cols_in_df = [col for col in final_tracer_study_df.columns if col.endswith('_implementation')]
if lm_cols_in_df:
    for col in sorted(lm_cols_in_df):
        print(f"- {col}")
else:
    print("No specific learning method columns identified.")

# Missing value percentage
print("\nMissing Value Percentage per Column:")
missing_percentage = final_tracer_study_df.isnull().sum() * 100 / len(final_tracer_study_df)
missing_percentage = missing_percentage[missing_percentage > 0].sort_values(ascending=False)
display(missing_percentage)

print("\nFinal validation complete. Dataset is ready for further analysis.")


--- Final Validation Summary ---
Total rows in final dataset: 7

Rows per source year:


,count
source_year,
2024,1
2025,6



Salary Statistics (gaji):


,gaji
count,7.0
mean,51857142.857143
std,30986172.031131
min,7000000.0
25%,35000000.0
50%,50000000.0
75%,68000000.0
max,100000000.0



Competency Columns in Final Dataset:
- communication_graduation
- communication_work
- domain_knowledge_graduation
- domain_knowledge_work
- english_language_graduation
- english_language_work
- ethics_graduation
- ethics_work
- it_skill_graduation
- it_skill_work
- self_development_graduation
- self_development_work
- teamwork_graduation
- teamwork_work

Learning Method Columns in Final Dataset:
- demonstration_implementation
- discussion_implementation
- field_work_implementation
- internship_implementation
- lab_practicum_implementation
- lecture_implementation
- research_project_participation_implementation

Missing Value Percentage per Column:


,0



Final validation complete. Dataset is ready for further analysis.


In [ ]:
# --- STEP 12: SAVE FINAL DATASET ---

OUTPUT_FILENAME = 'tracer_study_2020.csv'
final_tracer_study_df.to_csv(OUTPUT_FILENAME, index=False)

print(f"\nFinal standardized tracer study dataset saved to '{OUTPUT_FILENAME}'.")


Final standardized tracer study dataset saved to 'tracer_study_2020.csv'.


In [ ]:
# --- STEP 13: PRINT SUMMARY ---

print("\n--- Final Tracer Study Data Summary ---")

# Total rows
total_rows = len(final_tracer_study_df)
print(f"Total rows in final dataset: {total_rows}")

# Rows per source_year
if 'source_year' in final_tracer_study_df.columns:
    print("\nRows per source year:")
    display(final_tracer_study_df['source_year'].value_counts().sort_index())
else:
    print("\n'source_year' column not found to provide rows per year summary.")

# Detected competency columns (from all_standardized_competency_cols)
print("\nDetected competency columns:")
if all_standardized_competency_cols:
    for comp_col in sorted(list(all_standardized_competency_cols)):
        print(f"- {comp_col}")
else:
    print("No specific competency columns were detected.")

# NEW: Detected learning method columns
print("\nDetected learning method columns:")
if all_standardized_learning_method_cols:
    for lm_col in sorted(list(all_standardized_learning_method_cols)):
        print(f"- {lm_col}")
else:
    print("No specific learning method columns were detected.")

# Final column list
print("\nFinal column list:")
print(final_tracer_study_df.columns.tolist())

# 5 example records
print("\n5 example records from the final dataset:")
display(final_tracer_study_df.head(5))


--- Final Tracer Study Data Summary ---
Total rows in final dataset: 7

Rows per source year:


,count
source_year,
2024,1
2025,6



Detected competency columns:
- communication_graduation
- communication_work
- domain_knowledge_graduation
- domain_knowledge_work
- english_language_graduation
- english_language_work
- ethics_graduation
- ethics_work
- it_skill_graduation
- it_skill_work
- self_development_graduation
- self_development_work
- teamwork_graduation
- teamwork_work

Detected learning method columns:
- demonstration_implementation
- discussion_implementation
- field_work_implementation
- internship_implementation
- lab_practicum_implementation
- lecture_implementation
- research_project_participation_implementation

Final column list:
['nim', 'nama', 'tahun_lulus', 'status_pekerjaan', 'nama_perusahaan', 'posisi_pekerjaan', 'masa_tunggu_bulan', 'gaji', 'source_year', 'saran', 'communication_graduation', 'communication_work', 'domain_knowledge_graduation', 'domain_knowledge_work', 'english_language_graduation', 'english_language_work', 'ethics_graduation', 'ethics_work', 'it_skill_graduation', 'it_skill_wo

,nim,nama,tahun_lulus,status_pekerjaan,nama_perusahaan,posisi_pekerjaan,masa_tunggu_bulan,gaji,source_year,saran,...,teamwork_graduation,teamwork_work,demonstration_implementation,discussion_implementation,field_work_implementation,internship_implementation,lab_practicum_implementation,lecture_implementation,research_project_participation_implementation,tahun_masuk_nim
19,1202001004,Faatihah Rahmatillah,2024,Bekerja (full time / part time),Instansi pemerintah,Staff Administrasi Honorer,1,50000000,2025,Untuk prodi Informatika mungkin dapat menjalin...,...,4,5,5,4,5,5,4,5,5,2020
20,1202001005,Chika Humaira Abidatillah,2024,Bekerja (full time / part time),Pendidikan,Guru Informatika,2,40000000,2025,Semoga universitas bisa memberikan lebih banya...,...,4,5,4,4,3,5,5,5,4,2020
21,1202001007,Sheila Riva Rezqian,2024,Bekerja (full time / part time),Perusahaan swasta,IT Quality Assurance,2,56000000,2025,nan,...,5,5,5,5,5,3,5,5,5,2020
22,1202001010,ramadhani asri,2024,Bekerja (full time / part time),Wiraswasta/perusahaan sendiri,pemilik,1,80000000,2025,sudah cukup baik,...,4,4,3,3,5,4,4,3,3,2020
23,1202001025,Mumtaz Aaliyah Fasya,2024,Bekerja (full time / part time),Perusahaan luar negeri,Junior UI/UX Designer,8,100000000,2025,UI/UX is fun,...,5,5,3,5,4,5,2,3,3,2020


In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
# drive.mount('/content/drive', force_remount=True)

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/TRACER STUDY/PROCESSED'
OUTPUT_FILENAME = 'tracer_study_2020.csv' # Use the same filename as local save

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH, exist_ok=True)

destination_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH, OUTPUT_FILENAME)

try:
    # Save the DataFrame to Google Drive
    final_tracer_study_df.to_csv(destination_filepath, index=False)
    print(f"Final standardized tracer study dataset successfully saved to Google Drive: '{destination_filepath}'.")
except Exception as e:
    print(f"An error occurred while saving the tracer study data to Google Drive: {e}")

Final standardized tracer study dataset successfully saved to Google Drive: '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/TRACER STUDY/PROCESSED/tracer_study_2020.csv'.


# KKNI

### 1. Define Paths and Mount Google Drive

We'll define the Google Drive folder path for your KKNI PDF files and ensure Google Drive is mounted to access them.

In [ ]:
import sys

# Instal library yang dibutuhkan
!{sys.executable} -m pip install pdfplumber python-docx openpyxl pandas

In [ ]:
import os
import pandas as pd
import re

# --- Konfigurasi --- #
# Path ke folder Google Drive Anda yang berisi file KKNI PDF
DRIVE_FOLDER_PATH_KKNI = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/KKNI/" # Assuming this path from previous context
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_KKNI = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'KKNI')
OUTPUT_CSV_NAME_KKNI = os.path.join(OUTPUT_FOLDER_KKNI, 'kkni_combined_raw.csv')

# Create local directories if they don't exist
os.makedirs(OUTPUT_FOLDER_KKNI, exist_ok=True)
print(f"Created local directory: {OUTPUT_FOLDER_KKNI}")

print(f"KKNI data will be extracted from: {DRIVE_FOLDER_PATH_KKNI}")
print(f"Output CSV for KKNI will be saved to: {OUTPUT_CSV_NAME_KKNI}")

In [ ]:
import os
import pandas as pd
import re
import pdfplumber
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# --- Konfigurasi --- #
DRIVE_FOLDER_PATH_KKNI = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/KKNI"
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_KKNI = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'KKNI')
OUTPUT_CSV_NAME_KKNI = os.path.join(OUTPUT_FOLDER_KKNI, 'kkni_combined_raw.csv')

# --- Fungsi Ekstraksi Teks PDF (dari `curriculum_combined_raw.csv` logic) ---
def extract_text_from_pdf(filepath):
    text = ""
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                # Menghapus duplikat baris kosong dan multiple spaces
                page_text = page.extract_text(x_tolerance=1, y_tolerance=1)
                if page_text:
                    cleaned_page_text = re.sub(r'\n\s*\n', '\n', page_text) # Remove multiple blank lines
                    cleaned_page_text = re.sub(r'\s+', ' ', cleaned_page_text).strip() # Normalize spaces
                    text += cleaned_page_text + "\n"
    except Exception as e:
        print(f"  Error extracting text from PDF {os.path.basename(filepath)}: {e}")
        return None
    return text

# --- Proses Utama ---
al_data_kkni = []

print(f"Memulai pemrosesan file dari folder: {DRIVE_FOLDER_PATH_KKNI}")

# Pastikan folder ada
if not os.path.exists(DRIVE_FOLDER_PATH_KKNI):
    print(f"Error: Folder '{DRIVE_FOLDER_PATH_KKNI}' tidak ditemukan. Pastikan path sudah benar dan Google Drive sudah di-mount.")
else:
    for filename in os.listdir(DRIVE_FOLDER_PATH_KKNI):
        filepath = os.path.join(DRIVE_FOLDER_PATH_KKNI, filename)

        text_content = None
        print(f"\nMemproses file: {filename}")

        if filename.lower().endswith('.pdf'):
            text_content = extract_text_from_pdf(filepath)
        else:
            print(f"  Melewatkan file tidak didukung (bukan PDF): {filename}")
            continue

        if text_content is not None:
            al_data_kkni.append({
                'nama_file': filename,
                'path_file': filepath,
                'text_content': text_content
            })
        else:
            print(f"  Gagal mengekstrak teks dari {filename}. Melewatkan file ini.")

# Buat DataFrame
df_kkni = pd.DataFrame(al_data_kkni)

# Tampilkan informasi dan 5 baris pertama DataFrame
print("\n--- Ringkasan Data KKNI yang Diekstrak ---")
print(f"Total file yang berhasil diekstrak: {len(df_kkni)}")
print("Kolom DataFrame: ", df_kkni.columns.tolist())
display(df_kkni.head())

# --- Ekspor ke CSV ---
df_kkni.to_csv(OUTPUT_CSV_NAME_KKNI, index=False)

print(f"\nData KKNI berhasil disimpan ke '{OUTPUT_CSV_NAME_KKNI}'.")


### 3. Ekstraksi Konten Lampiran KKNI

Langkah ini bertujuan untuk memfilter `text_content` yang sudah diekstrak agar hanya menyertakan bagian yang relevan dengan Lampiran KKNI. Ini akan mempermudah proses parsing selanjutnya untuk mendapatkan data 'jenjang' dan 'deskripsi' secara spesifik.

In [ ]:
import os
import pandas as pd
import re

# Define output path for consistency
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_KKNI = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'KKNI')

# Load the raw KKNI data
try:
    input_filepath = os.path.join(OUTPUT_FOLDER_KKNI, 'kkni_combined_raw.csv')
    df_kkni = pd.read_csv(input_filepath)
    print(f"Dataset '{os.path.basename(input_filepath)}' loaded successfully.")
except FileNotFoundError:
    print(f"Error: '{os.path.basename(input_filepath)}' not found. Please ensure the previous step was executed.")
    df_kkni = pd.DataFrame(columns=['nama_file', 'path_file', 'text_content'])

# Ensure text_content is string type and handle NaNs safely
df_kkni['text_content'] = df_kkni['text_content'].astype(str).fillna('')

def extract_lampiran_content(text):
    # Regex to find 'Lampiran' at the beginning of a line, optionally followed by Roman numerals, letters, or numbers.
    # Using re.IGNORECASE to catch 'LAMPIRAN', 'lampiran', etc.
    # Using re.DOTALL to allow '.' to match newlines, ensuring the match goes until the end of the string
    match = re.search(r'\b(LAMPIRAN\s*(?:[IVXLCDM]+|[A-Z]|[0-9])?)\b.*', text, re.DOTALL)
    if match:
        # Return everything from the start of 'Lampiran' keyword onwards
        return match.group(0).strip()
    return "" # Return empty string if 'Lampiran' is not found

print("\nExtracting 'Lampiran' content from KKNI text...")
df_kkni['lampiran_content'] = df_kkni['text_content'].apply(extract_lampiran_content)

# Filter out entries where no lampiran content was found
df_kkni_lampiran = df_kkni[df_kkni['lampiran_content'] != ''].reset_index(drop=True)

# Display summary
print("\n--- Ringkasan Data Lampiran KKNI yang Diekstrak ---")
print(f"Total file dengan Lampiran yang berhasil diekstrak: {len(df_kkni_lampiran)}")
display(df_kkni_lampiran[['nama_file', 'lampiran_content']].head())

# Save the extracted lampiran content to a new CSV
OUTPUT_CSV_NAME_KKNI_LAMPIRAN = os.path.join(OUTPUT_FOLDER_KKNI, 'kkni_lampiran_raw.csv')
df_kkni_lampiran.to_csv(OUTPUT_CSV_NAME_KKNI_LAMPIRAN, index=False)
print(f"\nData Lampiran KKNI berhasil disimpan ke '{OUTPUT_CSV_NAME_KKNI_LAMPIRAN}'.")

destination_filepath = os.path.join(DRIVE_FOLDER_PATH_KKNI, 'kkni_lampiran_raw.csv')
# Save the DataFrame to Google Drive
df_kkni_lampiran.to_csv(destination_filepath, index=False, encoding="utf-8-sig")
print(f"'kkni_lampiran_raw.csv' successfully saved to Google Drive: '{destination_filepath}'.")


## 4. Ekstraksi KKNI Level 6

In [ ]:
import pandas as pd
import re
import os

# Path file hasil ekstraksi lampiran
input_path = "/content/processed_data/dictionary_data/KKNI/kkni_lampiran_raw.csv"

# Load CSV
df = pd.read_csv(input_path)

# Gabungkan seluruh isi dataframe menjadi satu teks panjang
text = " ".join(df.astype(str).fillna("").values.flatten())

# Rapikan whitespace
text = re.sub(r'\s+', ' ', text)

# Marker awal dan akhir KKNI Level 6
start_marker = "Mampu mengaplikasikan bidang keahliannya"
end_marker = "Mampu merencanakan dan mengelola sumberdaya"

# Cari posisi marker
start_idx = text.find(start_marker)
end_idx = text.find(end_marker)

# Validasi marker
if start_idx != -1 and end_idx != -1:

    # Ambil isi KKNI level 6
    kkni_level_6 = text[start_idx:end_idx].strip()

    # Rapikan per kalimat
    sentences = re.split(r'(?<=\.)\s+', kkni_level_6)

    # Define cleaning function for a single line
    # Define cleaning function for a single line

    def clean_single_line(line):

        # Remove duplicated OCR fragment like "Menguasai ... --"
        line = re.sub(
            r'^Menguasai\s*(\.{3}|…)\s*-*\s*',
            '',
            line,
            flags=re.IGNORECASE
        )

        # Remove PDF artifacts
        line = re.sub(r'\.{3}|…', ' ', line)
        line = re.sub(r'--+', ' ', line)
        line = re.sub(r'-\s*\d+\s*-', ' ', line)


        # Remove common OCR/PDF header noise
        line = re.sub(r'PRESIDEN REPUBLIK INDONESIA', ' ', line, flags=re.IGNORECASE)

        # Remove numbers
        line = re.sub(r'\d+', '', line)

        # Remove words that are all capital letters
        line = ' '.join(
            word for word in line.split(' ')
            if not word.isupper()
        )

        # Clean extra spaces
        line = re.sub(r'\s+', ' ', line).strip()

        return line

    # Apply cleaning to each sentence, filter out empty lines, and store as individual entries
    processed_points = []
    for i, sentence in enumerate(sentences):
        cleaned_line = clean_single_line(sentence)
        if cleaned_line: # Only add non-empty cleaned lines
            processed_points.append({
                "level": 6,
                "point_id": i + 1, # Start numbering from 1
                "description": cleaned_line
            })

    # Buat dataframe dari list of dictionaries
    df_kkni_level_6 = pd.DataFrame(processed_points)

    # Tentukan path output
    output_path = "/content/processed_data/dictionary_data/KKNI/kkni_level_6.csv"

    # Save dataframe
    df_kkni_level_6.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Content of '{os.path.basename(output_path)}':")
    display(df_kkni_level_6.head())
    print(f"\nFile berhasil disimpan di:\n{output_path}")

else:
    print("Marker awal atau akhir tidak ditemukan.")

destination_filepath = os.path.join(DRIVE_FOLDER_PATH_KKNI, 'kkni_level_6.csv')
# Save the DataFrame to Google Drive
df_kkni_level_6.to_csv(destination_filepath, index=False, encoding="utf-8-sig")
print(f"'kkni_level_6.csv' successfully saved to Google Drive: '{destination_filepath}'.")


# ESCO


## 1. Load File Zip

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/ESCO/ESCOdataset-v1.2.1-classification--rdf.zip"

# Check if the file exists before attempting to open
if not os.path.exists(zip_path):
    print(f"Error: ZIP file not found at '{zip_path}'. Please verify the path and file name.")
else:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.printdir()

In [ ]:
import os

directory_to_list = "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/ESCO/"

print(f"Listing contents of: {directory_to_list}")

if os.path.exists(directory_to_list):
    for item in os.listdir(directory_to_list):
        print(item)
else:
    print(f"Error: Directory not found at '{directory_to_list}'.")

In [ ]:
import os

extract_path = "/content/processed_data/dictionary_data/ESCO/"

# Create the directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"ESCO ZIP file extracted to: {extract_path}")


In [ ]:
import os

extracted_files = []
for root, dirs, files in os.walk(extract_path):
    for file in files:
        file_path = os.path.join(root, file)
        extracted_files.append(file_path)
        print(file_path)

# Identify the RDF file - assuming it's the one we saw in printdir()
esco_rdf_file = ""
for f in extracted_files:
    if f.endswith('.rdf'): # or .ttl depending on the actual file
        esco_rdf_file = f
        break

if esco_rdf_file:
    print(f"\nIdentified ESCO RDF/TTL file: {esco_rdf_file}")
else:
    print("\nNo RDF/TTL file found in the extracted directory.")

## Load File RDF

In [ ]:
import sys
!{sys.executable} -m pip install rdflib
print("rdflib installation complete.")

In [ ]:
import os
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace

# Identify the RDF file - assuming it's the one we saw in printdir()
esco_rdf_file = ""
for root, dirs, files in os.walk(extract_path):
    for file in files:
        if file.endswith('.rdf'): # or .ttl depending on the actual file
            esco_rdf_file = os.path.join(root, file)
            break
    if esco_rdf_file: break

g = Graph()

if esco_rdf_file:
    print(f"Identified ESCO RDF/TTL file: {esco_rdf_file}")
    if esco_rdf_file.endswith('.rdf'):
        g.parse(esco_rdf_file, format="xml")
        print(f"Loaded RDF file '{esco_rdf_file}' with format 'xml'")
    elif esco_rdf_file.endswith('.ttl'):
        g.parse(esco_rdf_file, format="ttl")
        print(f"Loaded RDF file '{esco_rdf_file}' with format 'ttl'")
    else:
        print(f"Unsupported RDF file format for: {esco_rdf_file}")
else:
    print("Error: No RDF/TTL file found in the extracted directory.")

print(f"Graph g has {len(g)} triples")

In [ ]:
import pandas as pd
import re
import json
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace
import os

# Define namespaces
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")
ESCO_MODEL = Namespace("http://data.europa.eu/esco/model#")
DCTERMS = Namespace("http://purl.org/dc/terms/")

# Define the ESCO concept scheme for skills (from previous cells)
SKILLS_SCHEME = URIRef("http://data.europa.eu/esco/concept-scheme/skills")

# Define output paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_ESCO = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'ESCO')

os.makedirs(OUTPUT_FOLDER_ESCO, exist_ok=True)
print(f"Ensured output directory exists: {OUTPUT_FOLDER_ESCO}")

In [ ]:
# Helper function to normalize text (lowercase, remove extra whitespace)
def normalize_text_light(text):
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Helper function to get all English labels (pref and alt) for a given URI
def get_all_english_labels(concept_uri):
    pref_label = None
    alt_labels = []

    # Preferred Label
    for xl_label_uri in g.objects(concept_uri, SKOSXL.prefLabel):
        for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
            if isinstance(literal_form, Literal) and literal_form.language == 'en':
                pref_label = str(literal_form)
                break
        if pref_label: # Break outer loop if pref_label found
            break

    # Alternative Labels
    for xl_label_uri in g.objects(concept_uri, SKOSXL.altLabel):
        for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
            if isinstance(literal_form, Literal) and literal_form.language == 'en':
                alt_labels.append(str(literal_form))

    return pref_label, list(set(alt_labels)) # Return unique alt labels

# Helper function to get English definition/description
def get_english_definition(concept_uri):
    for definition_literal in g.objects(concept_uri, SKOS.definition):
        if isinstance(definition_literal, Literal) and definition_literal.language == 'en':
            return str(definition_literal)
    return None

# Helper function to get related concepts (URIs)
def get_related_concept_uris(concept_uri, predicate):
    return [str(o) for o in g.objects(concept_uri, predicate)]

# Helper function to get labels for a list of concept URIs
def get_labels_for_uris(uri_list):
    labels = []
    for uri in uri_list:
        pref, _ = get_all_english_labels(URIRef(uri))
        if pref:
            labels.append(pref)
    return labels

In [ ]:
import pandas as pd
import re
import json
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace
import os

# BEGIN FIX FOR NameError: name 'g' is not defined (Reinforced)
# Ensure g and SKOSXL are defined at the very top of this cell for robustness
# This assumes the ESCO RDF file is located at the standard extracted path
extract_path = "/content/processed_data/dictionary_data/ESCO/"
esco_rdf_file = os.path.join(extract_path, 'esco-v1.2.1.rdf')

g = Graph()
if os.path.exists(esco_rdf_file):
    print(f"Loading ESCO RDF file: {esco_rdf_file}")
    g.parse(esco_rdf_file, format="xml")
    print(f"Loaded RDF file '{esco_rdf_file}' with format 'xml'")
else:
    print(f"Error: ESCO RDF file not found at {esco_rdf_file}. Please ensure ZIP extraction was successful and path is correct.")

# Define namespaces (re-added for self-containment)
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")
ESCO_MODEL = Namespace("http://data.europa.eu/esco/model#")
DCTERMS = Namespace("http://purl.org/dc/terms/")

# Define the ESCO concept scheme for skills (re-added for self-containment)
SKILLS_SCHEME = URIRef("http://data.europa.eu/esco/concept-scheme/skills")
# END FIX FOR NameError


# Define output paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_ESCO = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'ESCO')

os.makedirs(OUTPUT_FOLDER_ESCO, exist_ok=True)
print(f"Ensured output directory exists: {OUTPUT_FOLDER_ESCO}")

# Helper function to normalize text (lowercase, remove extra whitespace)
def normalize_text_light(text):
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Helper function to get all English labels (pref and alt) for a given URI
def get_all_english_labels(concept_uri):
    pref_label = None
    alt_labels = []

    # Preferred Label
    for xl_label_uri in g.objects(concept_uri, SKOSXL.prefLabel):
        for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
            if isinstance(literal_form, Literal) and literal_form.language == 'en':
                pref_label = str(literal_form)
                break
        if pref_label: # Break outer loop if pref_label found
            break

    # Alternative Labels
    for xl_label_uri in g.objects(concept_uri, SKOSXL.altLabel):
        for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
            if isinstance(literal_form, Literal) and literal_form.language == 'en':
                alt_labels.append(str(literal_form))

    return pref_label, list(set(alt_labels)) # Return unique alt labels

# Helper function to get English definition/description
def get_english_definition(concept_uri):
    for definition_literal in g.objects(concept_uri, SKOS.definition):
        if isinstance(definition_literal, Literal) and definition_literal.language == 'en':
            return str(definition_literal)
    return None

# Helper function to get related concepts (URIs)
def get_related_concept_uris(concept_uri, predicate):
    uris = [str(o) for o in g.objects(concept_uri, predicate)]
    # Debug print for empty narrower/related skills for a few samples
    if (predicate == SKOS.narrower or predicate == SKOS.related) and not uris and len(esco_skill_data) < 10: # Only for first 10 skills
        print(f"  DEBUG: No URIs found for {predicate} for skill {concept_uri}")
    return uris

# Helper function to get labels for a list of concept URIs
def get_labels_for_uris(uri_list):
    labels = []
    for uri in uri_list:
        pref, _ = get_all_english_labels(URIRef(uri))
        if pref:
            labels.append(pref)
    return labels


print("\nExtracting ESCO skill concepts...")

esco_skill_data = []

# Removed: Keywords for ICT domain filtering (for is_ict_related)
# Removed: ict_keywords = [
# Removed:     'software', 'programming', 'ict', 'information and communication technology',
# Removed:     'networking', 'database', 'software engineering', 'machine learning',
# Removed:     'artificial intelligence', 'data science', 'cybersecurity', 'cloud computing',
# Removed:     'information systems', 'computer science', 'developer', 'engineer', 'system administration',
# Removed:     'web development', 'mobile development', 'devops', 'it infrastructure'
# Removed: ]
# Removed: ict_keywords_pattern = re.compile(r'\b(' + '|'.join(ict_keywords) + r')\b', re.IGNORECASE)

# Find all subjects that are SKOS.Concept and belong to the SKILLS_SCHEME
skill_concepts = set()
for s in g.subjects(RDF.type, SKOS.Concept):
    if (s, SKOS.inScheme, SKILLS_SCHEME) in g:
        skill_concepts.add(s)

print(f"Found {len(skill_concepts)} skill concepts in SKILLS_SCHEME.")

for skill_uri_ref in skill_concepts:
    skill_uri = str(skill_uri_ref)

    pref_label, alt_labels = get_all_english_labels(skill_uri_ref)
    description = get_english_definition(skill_uri_ref)

    # Broader, Narrower, Related skills (get URIs first)
    broader_uris = get_related_concept_uris(skill_uri_ref, SKOS.broader)
    narrower_uris = get_related_concept_uris(skill_uri_ref, SKOS.narrower)
    related_uris = get_related_concept_uris(skill_uri_ref, SKOS.related)

    # Get labels for broader, narrower, related URIs
    broader_skill_labels = get_labels_for_uris(broader_uris)
    narrower_skill_labels = get_labels_for_uris(narrower_uris)
    related_skill_labels = get_labels_for_uris(related_uris)

    # Taxonomy Path: For simplicity and to avoid flattening, we'll use immediate broader skill labels.
    # For a full path, a recursive traversal would be needed. Storing immediate broader is a good balance.
    taxonomy_path = ' > '.join(broader_skill_labels) if broader_skill_labels else None

    # Cleaned text columns (light preprocessing)
    preferred_label_clean = normalize_text_light(pref_label)
    description_clean = normalize_text_light(description)

    # Removed: ICT-related filtering (using keywords)
    # Removed: is_ict_related = False
    # Removed: if pref_label and ict_keywords_pattern.search(pref_label):
    # Removed:     is_ict_related = True
    # Removed: elif description and ict_keywords_pattern.search(description):
    # Removed:     is_ict_related = True

    # ESCO Semantic Representation
    semantic_parts = []
    if preferred_label_clean: semantic_parts.append(preferred_label_clean)
    if alt_labels: semantic_parts.append(' '.join([normalize_text_light(lbl) for lbl in alt_labels]))
    if description_clean: semantic_parts.append(description_clean)
    if broader_skill_labels: semantic_parts.append(' '.join([normalize_text_light(lbl) for lbl in broader_skill_labels]))
    esco_semantic_representation = ' '.join(semantic_parts).strip()

    esco_skill_data.append({
        'skill_uri': skill_uri,
        'preferred_label': pref_label,
        'preferred_label_clean': preferred_label_clean,
        'alternative_labels': alt_labels,
        'description': description,
        'description_clean': description_clean,
        'broader_skill_uris': broader_uris,
        'broader_skill_labels': broader_skill_labels,
        'narrower_skill_uris': narrower_uris,
        'narrower_skill_labels': narrower_skill_labels,
        'related_skill_uris': related_uris,
        'related_skill_labels': related_skill_labels,
        'taxonomy_path': taxonomy_path,
        'esco_semantic_representation': esco_semantic_representation
    })

df_esco_skills = pd.DataFrame(esco_skill_data)

### Summary of Extracted ESCO Skills

In [ ]:
# Print summary statistics
total_skills = len(df_esco_skills)
total_broader_concepts = df_esco_skills['broader_skill_uris'].apply(len).sum()
total_narrower_concepts = df_esco_skills['narrower_skill_uris'].apply(len).sum()
total_related_skill_links = df_esco_skills['related_skill_uris'].apply(len).sum()

print(f"Total ESCO skills extracted: {total_skills}")
print(f"Total broader concept links: {total_broader_concepts}")
print(f"Total narrower concept links: {total_narrower_concepts}")
print(f"Total related skill links: {total_related_skill_links}")

print("\nSample rows of df_esco_skills (first 5):")
display(df_esco_skills.head())

### Saving Extracted ESCO Skills Data

In [ ]:
import os

# Define output paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_ESCO = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'ESCO')

os.makedirs(OUTPUT_FOLDER_ESCO, exist_ok=True)
print(f"Ensured output directory exists: {OUTPUT_FOLDER_ESCO}")

# Save to CSV
csv_filepath = os.path.join(OUTPUT_FOLDER_ESCO, 'df_esco_skills.csv')
df_esco_skills.to_csv(csv_filepath, index=False)
print(f"DataFrame saved to CSV: {csv_filepath}")

# Save to JSON
json_filepath = os.path.join(OUTPUT_FOLDER_ESCO, 'df_esco_skills.json')
df_esco_skills.to_json(json_filepath, orient='records', indent=4)
print(f"DataFrame saved to JSON: {json_filepath}")

print("\nESCO skills data extraction and saving complete. The DataFrame `df_esco_skills` is ready for embedding generation, ontology-assisted category representation, semantic category enrichment, and curriculum-industry alignment analysis.")

## AMBIL SKILL LABEL (ENGLISH)

In [ ]:
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace

# Define SKOSXL namespace explicitly
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")

esco_skills = {}

# Define the ESCO namespace for specific properties if needed
SKILLS_SCHEME = URIRef("http://data.europa.eu/esco/concept-scheme/skills")

# Step 1: Find all subjects that are SKOS.Concept
concept_subjects = set(g.subjects(RDF.type, SKOS.Concept))
print(f"DEBUG: Total subjects with rdf:type SKOS.Concept: {len(concept_subjects)}")

# Step 2: Filter those that are specifically in the SKILLS_SCHEME
skills_in_scheme_subjects = set()
for s in concept_subjects:
    if (s, SKOS.inScheme, SKILLS_SCHEME) in g:
        skills_in_scheme_subjects.add(s)
print(f"DEBUG: SKOS.Concept subjects found in SKILLS_SCHEME: {len(skills_in_scheme_subjects)}")

if not skills_in_scheme_subjects:
    print("DEBUG: No skill concepts found in the specified scheme. Re-check scheme URI or RDF structure.")
else:
    # Step 3: Extract English prefLabels using skos-xl:literalForm
    for s in list(skills_in_scheme_subjects): # Iterate all for actual data extraction
        for xl_label_uri in g.objects(s, SKOSXL.prefLabel):
            for _, _, o in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
                if isinstance(o, Literal) and o.language == 'en':
                    esco_skills[str(s)] = str(o).lower()
                    break # Found the English literal form, move to next skill

print(f"Found {len(esco_skills)} English skill labels.")
print("Contoh 5 skill teratas:")
for i, (uri, skill_label) in enumerate(esco_skills.items()):
    if i >= 5: break
    print(f"- {skill_label}")

In [ ]:
print("\n--- Inspecting Sample Skill Concepts for Label Predicate ---")

sample_skills = list(skills_in_scheme_subjects)[:5] # Take 5 samples

if not sample_skills:
    print("No skill concepts available for inspection.")
else:
    for i, s_uri in enumerate(sample_skills):
        print(f"\nSample Skill {i+1} URI: {s_uri}")
        for p, o in g.predicate_objects(s_uri):
            if isinstance(o, Literal):
                print(f"  Predicate: {p}, Object: '{o}' (Lang: {o.language if hasattr(o, 'language') else 'N/A'})")
            else:
                print(f"  Predicate: {p}, Object URI: {o}")

print("--- Sample Skill Inspection Complete ---")

## Buat Skill List

In [ ]:
skill_list = list(set(esco_skills.values())) # Use set to ensure unique skill labels

print(f"Created a list of {len(skill_list)} unique ESCO skill labels.")

## AMBIL DATA ESCO OCCUPATION DAN RELASI SKILL

In [ ]:
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace

ESCO_NAMESPACE = Namespace("http://data.europa.eu/esco/model#")
SKOSXL = Namespace("http://www.w3.org/2008/05/skos-xl#")

esco_occupations_with_skills = {}

# Get all occupation URIs
all_occupation_uris = list(g.subjects(RDF.type, ESCO_NAMESPACE.Occupation))
print(f"Total ESCO occupations found: {len(all_occupation_uris)}")

if not all_occupation_uris:
    print("No ESCO occupations found in the graph. Please check ESCO_NAMESPACE.Occupation.")
else:
    print("Extracting skills for all occupations...")

    # Helper function to get all English labels (pref and alt) for a skill URI
    def get_all_english_skill_labels(skill_uri):
        labels = set()
        # Get prefLabel
        for xl_label_uri in g.objects(skill_uri, SKOSXL.prefLabel):
            for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
                if isinstance(literal_form, Literal) and literal_form.language == 'en':
                    labels.add(str(literal_form).lower())
        # Get altLabel
        for xl_label_uri in g.objects(skill_uri, SKOSXL.altLabel):
            for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
                if isinstance(literal_form, Literal) and literal_form.language == 'en':
                    labels.add(str(literal_form).lower())
        return list(labels) # Return as list for appending

    for occ_uri in all_occupation_uris:
        occ_label = None
        # Get preferred English label for the occupation
        for xl_label_uri in g.objects(occ_uri, SKOSXL.prefLabel):
            for _, _, literal_form in g.triples((xl_label_uri, SKOSXL.literalForm, None)):
                if isinstance(literal_form, Literal) and literal_form.language == 'en':
                    occ_label = str(literal_form).lower()
                    break
            if occ_label:
                break

        if occ_label:
            associated_skills = set()
            # Collect skills linked by relatedEssentialSkill
            for s_occ, p_skill, o_skill_uri in g.triples((occ_uri, ESCO_NAMESPACE.relatedEssentialSkill, None)):
                labels_for_this_skill = get_all_english_skill_labels(o_skill_uri)
                associated_skills.update(labels_for_this_skill)

            # Collect skills linked by relatedOptionalSkill
            for s_occ, p_skill, o_skill_uri in g.triples((occ_uri, ESCO_NAMESPACE.relatedOptionalSkill, None)):
                labels_for_this_skill = get_all_english_skill_labels(o_skill_uri)
                associated_skills.update(labels_for_this_skill)

            if associated_skills:
                # Convert set to list for storage
                esco_occupations_with_skills[occ_label] = list(associated_skills)

print(f"Found {len(esco_occupations_with_skills)} English occupations with associated skills (including altLabels).")
print("Contoh 3 occupations teratas dengan skill:")
for i, (occ, skills) in enumerate(esco_occupations_with_skills.items()):
    if i >= 3: break
    print(f"- {occ}: {skills[:5]}...") # Display first 5 skills for brevity


In [ ]:
import json
import os

OUTPUT_FOLDER_ESCO = os.path.join('/content/processed_data/dictionary_data', 'ESCO')

# Load esco_occupations_with_skills from the saved JSON file
# This assumes the file structure and content are as expected from the previous steps
esco_occupations_with_skills_filepath = os.path.join(OUTPUT_FOLDER_ESCO, 'filtered_esco_it_occupations.json')

try:
    with open(esco_occupations_with_skills_filepath, 'r') as f:
        # The filtered_esco_it_occupations.json actually contains the filtered list already
        # So we can load it directly into filtered_esco_occupations_with_skills
        filtered_esco_occupations_with_skills = json.load(f)
    print(f"Loaded {len(filtered_esco_occupations_with_skills)} filtered ESCO IT occupations from '{esco_occupations_with_skills_filepath}'.")
except FileNotFoundError:
    print(f"Error: '{esco_occupations_with_skills_filepath}' not found. Please ensure previous steps were executed correctly.")
    filtered_esco_occupations_with_skills = {} # Initialize as empty to prevent errors

# Also define esco_occupations_with_skills for consistency, though it's implicitly part of the filtered data
esco_occupations_with_skills = filtered_esco_occupations_with_skills

print(f"Found {len(esco_occupations_with_skills)} English occupations with associated skills (including altLabels).")
print("Contoh 3 occupations teratas dengan skill:")
for i, (occ, skills) in enumerate(esco_occupations_with_skills.items()):
    if i >= 3: break
    print(f"- {occ}: {skills[:5]}...") # Display first 5 skills for brevity


In [ ]:
print(f"Total IT-related ESCO occupations under ISCO Group 25: {len(filtered_esco_occupations_with_skills)}")
print("Contoh 5 IT-related occupations yang difilter:")
for i, occ_label in enumerate(list(filtered_esco_occupations_with_skills.keys())[:5]):
    print(f"- {occ_label}")


## Filter ESCO Occupations to ISCO Group 25 (IT-related Roles)


In [ ]:
from rdflib import Graph, Literal, RDF, URIRef
from rdflib.namespace import SKOS, Namespace
import json

# Define ISCO namespace
ISCO_NAMESPACE = Namespace("http://data.europa.eu/esco/isco/")

# ISCO Group 25 URI for 'Information and Communications Technology Professionals'
# We need to find the exact URI for ISCO 25. Let's try to query it.

isco_25_uri = None

# Search for the ISCO concept with notation '25'
# The notation property for ISCO is `skos:notation`
for s, p, o in g.triples((None, SKOS.notation, Literal("25"))):
    # Check if this concept is also part of the ISCO concept scheme
    if (s, SKOS.inScheme, URIRef("http://data.europa.eu/esco/concept-scheme/isco")) in g:
        isco_25_uri = s
        break

print(f"Identified ISCO Group 25 URI: {isco_25_uri}")

if not isco_25_uri:
    print("Error: Could not find ISCO Group 25 URI. Please check the RDF structure and notation.")
else:
    # Now, filter occupations that are narrower than or related to ISCO_25_URI
    # ESCO occupations use `skos:broader` or `skos:broaderTransitive` to link to ISCO.
    # Let's collect all occupations that are explicitly linked to ISCO 25.

    filtered_esco_occupations_with_skills = {}
    all_it_isco_occupations = set() # To store the lowercased labels of filtered occupations

    print("Filtering ESCO occupations by ISCO Group 25...")

    for occ_label, skills in esco_occupations_with_skills.items():
        # Get the URI for the current occupation label from the graph
        # This is a bit indirect, but we previously built esco_occupations_with_skills based on URIs and their prefLabels
        # We need to find the original URI for `occ_label` to check its ISCO relation
        occ_uri = None
        for s_occ_uri, p, o_label_uri in g.triples((None, SKOSXL.prefLabel, None)): # Find the XL label URI
            for _, _, literal_form in g.triples((o_label_uri, SKOSXL.literalForm, None)):
                if isinstance(literal_form, Literal) and str(literal_form).lower() == occ_label:
                    # Check if s_occ_uri is an actual ESCO Occupation
                    if (s_occ_uri, RDF.type, Namespace("http://data.europa.eu/esco/model#").Occupation) in g:
                        occ_uri = s_occ_uri
                        break
            if occ_uri: break
        if not occ_uri: continue # Skip if URI not found

        # Check if the occupation is 'narrower' (more specific) than ISCO 25
        # `skos:broader` and `skos:broaderTransitive` are used here. We need to check if ISCO 25 is a broader concept.
        if (occ_uri, SKOS.broader, isco_25_uri) in g or \
           (occ_uri, SKOS.broaderTransitive, isco_25_uri) in g:
            filtered_esco_occupations_with_skills[occ_label] = skills
            all_it_isco_occupations.add(occ_label)

    print(f"Found {len(filtered_esco_occupations_with_skills)} IT-related ESCO occupations under ISCO Group 25.")
    print("Contoh 5 IT-related occupations:")
    for i, occ_label in enumerate(list(filtered_esco_occupations_with_skills.keys())[:5]):
        print(f"- {occ_label}")

# Save the filtered ESCO occupations with skills to a JSON file
output_json_path = "filtered_esco_it_occupations.json"
with open(output_json_path, 'w') as f:
    json.dump(filtered_esco_occupations_with_skills, f, indent=4)

print(f"Filtered ESCO IT occupations saved to '{output_json_path}'.")


In [ ]:
print(f"Menampilkan 3 contoh IT-related occupations dengan semua skill terasosiasi (termasuk altLabels):")
count = 0
for occ_label, skills in filtered_esco_occupations_with_skills.items():
    if count >= 3: break
    print(f"\n- Occupation: {occ_label}")
    print(f"  Jumlah skill: {len(skills)}")
    print(f"  Contoh 5 skill teratas: {skills[:5]}...") # Display first 5 skills for brevity
    count += 1

In [ ]:
print(f"Total IT-related ESCO occupations under ISCO Group 25: {len(filtered_esco_occupations_with_skills)}")
print("Contoh 5 IT-related occupations yang difilter:")
for i, occ_label in enumerate(list(filtered_esco_occupations_with_skills.keys())[:5]):
    print(f"- {occ_label}")

## Save ESCO Skills and Occupations

In [ ]:
import os
import pandas as pd
import json

# Define output paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_ESCO = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'ESCO')

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_ESCO = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/ESCO'

# Create local directories if they don't exist
os.makedirs(OUTPUT_FOLDER_ESCO, exist_ok=True)
print(f"Created local directory: {OUTPUT_FOLDER_ESCO}")

# Create Google Drive directories if they don't exist
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_ESCO, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_ESCO}")

### Save ESCO Skill List (English)

In [ ]:
df_esco_skill_list = pd.DataFrame(skill_list, columns=['skill_label'])

# Save to local processed data
local_skill_list_filepath = os.path.join(OUTPUT_FOLDER_ESCO, 'esco_skill_list.csv')
df_esco_skill_list.to_csv(local_skill_list_filepath, index=False)
print(f"ESCO skill list saved locally to: '{local_skill_list_filepath}'")

# Save to Google Drive
drive_skill_list_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_ESCO, 'esco_skill_list.csv')
df_esco_skill_list.to_csv(drive_skill_list_filepath, index=False)
print(f"ESCO skill list saved to Google Drive: '{drive_skill_list_filepath}'")

### Save Filtered ESCO IT Occupations with Skills

In [ ]:
# Save to local processed data
local_filtered_occupations_filepath = os.path.join(OUTPUT_FOLDER_ESCO, 'filtered_esco_it_occupations.json')
with open(local_filtered_occupations_filepath, 'w') as f:
    json.dump(filtered_esco_occupations_with_skills, f, indent=4)
print(f"Filtered ESCO IT occupations saved locally to: '{local_filtered_occupations_filepath}'")

# Save to Google Drive
drive_filtered_occupations_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_ESCO, 'filtered_esco_it_occupations.json')
with open(drive_filtered_occupations_filepath, 'w') as f:
    json.dump(filtered_esco_occupations_with_skills, f, indent=4)
print(f"Filtered ESCO IT occupations saved to Google Drive: '{drive_filtered_occupations_filepath}'")

In [ ]:
# Save to Google Drive
drive_esco_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_ESCO, 'esco-v121.rdf')
with open(drive_esco_filepath, 'w') as f:
    json.dump(filtered_esco_occupations_with_skills, f, indent=4)
print(f"Filtered ESCO IT occupations saved to Google Drive: '{drive_esco_filepath}'")

# PETA OKUPASI

## LOAD DATA

In [ ]:
pip install PyMuPDF

In [ ]:
import fitz # PyMuPDF

PON_TIK_DICTIONARY_PATH = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK/'

# Define PDF path
pdf_path = PON_TIK_DICTIONARY_PATH + "Buku Publikasi Peta Okupasi Bidang TIK - PON TIK.pdf"

# Check if the PDF file exists
if not os.path.exists(pdf_path):
    print(f"Error: PDF file not found at {pdf_path}")
    full_pdf_text = ""
else:
    print(f"Loading PDF from: {pdf_path}")
    text_content = []
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                text_content.append(page.get_text())
        full_pdf_text = "\n".join(text_content)
        # Clean up multiple newlines and leading/trailing whitespace for better readability
        global cleaned_text
        cleaned_text = '\n'.join([line.strip() for line in full_pdf_text.split('\n')]).strip()
        print("PDF text extracted and cleaned successfully.")
    except Exception as e:
        print(f"Error extracting text from PDF: {e}")
        full_pdf_text = ""
        cleaned_text = ""

Loading PDF from: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK/Buku Publikasi Peta Okupasi Bidang TIK - PON TIK.pdf
PDF text extracted and cleaned successfully.


In [ ]:
print("--- Sample of Extracted PDF Text (First 1000 characters) ---")
if full_pdf_text:
    # Clean up multiple newlines and leading/trailing whitespace for better readability
    # MODIFICATION: Do not filter out empty lines, just strip them, to preserve spacing and structural context
    global cleaned_text
    cleaned_text = '\n'.join([line.strip() for line in full_pdf_text.split('\n')]).strip()
    print(cleaned_text[:1000])
    print("\n...")
else:
    print("No text extracted or PDF not found.")
    cleaned_text = "" # Ensure cleaned_text is defined even if empty

--- Sample of Extracted PDF Text (First 1000 characters) ---
PETA OKUPASI NASIONAL
BIDANG TEKNOLOGI INFORMASI DAN KOMUNIKASI
Perubahan Kesatu (Okt 2025)


Tim Penyusun:

Pengarah​
​
:  Bonifasius Wahyu Pudjianto, Ph.D (Kepala BPSDM Komdigi)
Penanggung Jawab​
:  Dr. Nusirwan, S.Ag., M.Si., (Kepala Pusbang Ekosistem SDM Komdigi)
Sekretariat​
​
:  Aldhino Anggorosesar, Rieka Mustika, Dewi Hernikawati, Renata
Octaviani Priono, Fitri Widyaningsih, Cut Medika Z., Olivia Nelar, Fitri
Widyaningsih, Irfan Setiawan, Renata Octaviani, Fikri Jodi Pratama,
Sharon Gracia Gabriela, Adinda Natasya, Nandita Ayu, Otto Satya
Hutama, Tasya Apriliana.

Tim Perumus:

Dr.rer.nat. I Made Wiryana, M.Sc., Satriyo Wibowo, MBA, M.H. IPM, CERG, Ardhanti Nurwidya,
S.H., LL.M.

Narasumber dan Kontributor:

Ilafi Firsta Putri, Muharman Lubis, Insan Ardiansyah, Muh. Armil Syam, Chandra Yulistia,
Moh Amir Syarifuddin, Farrell Jake, Muhammad Deckri Algamar, Adhi Prasetya, Humairoh,
Grace Winnee Malia, Sinta Novanana, No

## Locate 'DAFTAR ISI' and Extract TIK Codes

In [ ]:
import re

daftar_isi_occupations = []

# Keywords to define the section for TIK code listing
daftar_isi_marker = "DAFTAR ISI"
matrix_occupasi_marker = "DAFTAR PEMUTAKHIRAN OKUPASI PON TIK"

# Find the start of the DAFTAR ISI section
start_daftar_isi = cleaned_text.find(daftar_isi_marker)

if start_daftar_isi == -1:
    print(f"Error: '{daftar_isi_marker}' not found. Cannot proceed with extraction.")
    daftar_isi_section_raw = ""
else:
    # The actual list of occupations in DAFTAR ISI usually starts after a specific sub-heading or just a few lines after "DAFTAR ISI"
    # Let's search for "DAFTAR DAN DESKRIPSI OKUPASI PON TIK" after "DAFTAR ISI" as a more precise start for the list.
    start_occup_list = cleaned_text.find("DAFTAR DAN DESKRIPSI OKUPASI PON TIK", start_daftar_isi)

    if start_occup_list == -1:
        # If "DAFTAR DAN DESKRIPSI OKUPASI PON TIK" is not found, start immediately after "DAFTAR ISI"
        section_start_index = start_daftar_isi + len(daftar_isi_marker)
        print(f"Warning: 'DAFTAR DAN DESKRIPSI OKUPASI PON TIK' not found after '{daftar_isi_marker}'. Starting list extraction from after '{daftar_isi_marker}'.")
    else:
        # Start after "DAFTAR DAN DESKRIPSI OKUPASI PON TIK" and its page number line
        # Find the next newline after this marker, and start from there.
        section_start_index = cleaned_text.find("\n", start_occup_list + len("DAFTAR DAN DESKRIPSI OKUPASI PON TIK"))
        if section_start_index != -1:
            section_start_index += 1 # Start from the character after the newline
        else:
            section_start_index = start_occup_list + len("DAFTAR DAN DESKRIPSI OKUPASI PON TIK") + 1 # Fallback if no newline after

    # Find the end of the section, which is before "DAFTAR PEMUTAKHIRAN OKUPASI PON TIK"
    section_end_index = cleaned_text.find(matrix_occupasi_marker, section_start_index)

    if section_start_index == -1: # This should not happen if daftar_isi_marker was found
        print("Could not determine start of TIK list. Aborting.")
        daftar_isi_section_raw = ""
    elif section_end_index == -1:
        print(f"Warning: '{matrix_occupasi_marker}' not found after the start of the TIK list. Assuming list continues to end of document for now.")
        daftar_isi_section_raw = cleaned_text[section_start_index:]
    else:
        daftar_isi_section_raw = cleaned_text[section_start_index:section_end_index]

    print(f"DAFTAR ISI section (raw) to process: {len(daftar_isi_section_raw)} characters.")

if daftar_isi_section_raw:
    # Pattern to capture TIK code and its title from a line in DAFTAR ISI
    # Example lines:
    # "TIK.ITG0701 - Ahli Pengembangan Perangkat Lunak ........ 22"
    # "TIK.ITG0702 Asisten Ahli Pengembangan Perangkat Lunak .... 23"
    # "TIK.ITG0901 – PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER)............................................................... 6"

    # Simplified pattern without re.VERBOSE to avoid potential formatting issues,
    # and merged \s* into the optional page number group for robustness.
    tik_code_and_title_pattern = re.compile(
        r"^(TIK\.[A-Z]{3}\d{4})\s*"      # Group 1: TIK code
        r"(?:[–-—]\s*)?"                 # Optional separator
        r"(.*?)"                         # Group 2: Non-greedy title capture
        r"(?:\s*\.{2,}\s*\d+)?"             # Optional page number part
        r"$",                            # End of line
        re.IGNORECASE | re.MULTILINE
    )

    for line in daftar_isi_section_raw.splitlines():
        line = line.strip()
        if not line:
            continue

        match = tik_code_and_title_pattern.search(line)
        if match:
            code = match.group(1).strip()
            title = match.group(2).strip()
            if title:
                # Remove common list-like prefixes/suffixes like numbers and dots or trailing dashes if they erroneously get captured
                title = re.sub(r'^[\d\.]+\s*', '', title) # remove leading numbers/dots
                title = re.sub(r'[\s.,-]+$', '', title).strip()
                if title: # Re-check if title is still not empty after cleaning
                    daftar_isi_occupations.append({
                        'code': code,
                        'title': title
                    })

print(f"Extracted {len(daftar_isi_occupations)} TIK codes and titles from the DAFTAR ISI section.")

if daftar_isi_occupations:
    print("\nSample of extracted occupations (first 10):")
    for i, occ in enumerate(daftar_isi_occupations[:10]):
        print(f"- Code: {occ['code']}, Title: {occ['title']}")
    print("\nSample of extracted occupations (last 10):")
    for i, occ in enumerate(daftar_isi_occupations[-10:]):
        print(f"- Code: {occ['code']}, Title: {occ['title']}")
else:
    print("No TIK codes and titles were extracted.")

DAFTAR ISI section (raw) to process: 34312 characters.
Extracted 269 TIK codes and titles from the DAFTAR ISI section.

Sample of extracted occupations (first 10):
- Code: TIK.ITG0901, Title: PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER)
- Code: TIK.ITG0902, Title: PEMIMPIN KEPATUHAN (CHIEF COMPLIANCE OFFICER (CCO))
- Code: TIK.ITG0801, Title: KEPALA ARSITEKTUR PERUSAHAAN (LEAD ENTERPRISE ARCHITECTURE)
- Code: TIK.ITG0802, Title: KEPALA PENASIHAT TI (LEAD IT ADVISOR)
- Code: TIK.ITG0803, Title: KEPALA MANAJEMEN PROGRAM (LEAD PROGRAM MANAGEMENT)
- Code: TIK.ITG0804, Title: KEPALA PORTOFOLIO PROYEK TI (LEAD IT PROJECT PORTFOLIO)
- Code: TIK.ITG0805, Title: SPESIALIS PENASIHAT TI (SPECIALIST IT ADVISOR)
- Code: TIK.ITG0806, Title: KEPALA KONSULTAN TI (LEAD IT CONSULTANT)
- Code: TIK.ITG0807, Title: KEPALA MANAJEMEN SISTEM (LEAD SYSTEMS MANAGEMENT)
- Code: TIK.ITG0808, Title: PEJABAT PENGAWAS PDP (DATA PROTECTION OFFICER)

Sample of extracted occupations (last 10):
- Code: TIK.SRV0504, Ti

## EXTRACT OCCUPATION BLOCKS

In [ ]:
import re

def extract_occupation_blocks(text, daftar_isi_occupations_list):
    blocks = []
    processed_codes = set() # To keep track of codes already processed

    # =========================================================
    # 1. NORMALIZE DAFTAR ISI CODES
    # =========================================================
    daftar_isi_codes_set = {
        entry['code'].strip().encode('ascii', 'ignore').decode('ascii')
        for entry in daftar_isi_occupations_list
    }

    print(f"INFO: Total daftar isi occupation codes: {len(daftar_isi_codes_set)}")

    # =========================================================
    # 2. REGEX UNTUK TIK CODE
    # =========================================================
    occupation_start_pattern = re.compile(
        r'(TIK\.[A-Z]{3}\d{4}).{0,150}?DEFINISI',
        flags=re.IGNORECASE | re.DOTALL
    )

    # =========================================================
    # 3. CARI SEMUA TIK CODE YANG VALID
    #    (HARUS PUNYA DEFINISI DI DEKATNYA DAN HANYA KEMUNCULAN PERTAMA)
    # =========================================================
    temp_tik_code_locations = []

    for match in occupation_start_pattern.finditer(text):

        found_code_original = match.group(1).strip()

        found_code_normalized = (
            found_code_original
            .encode('ascii', 'ignore')
            .decode('ascii')
        )

        if (
            found_code_normalized in daftar_isi_codes_set
            and found_code_normalized not in processed_codes
        ):

            temp_tik_code_locations.append(
                (
                    match.start(),
                    found_code_original
                )
            )

            processed_codes.add(
                found_code_normalized
            )

    # =========================================================
    # 4. SORT BERDASARKAN POSISI
    # =========================================================
    temp_tik_code_locations.sort(key=lambda x: x[0])

    print(f"INFO: Valid occupation blocks found: {len(temp_tik_code_locations)}")

    if not temp_tik_code_locations:
        print("WARNING: No valid occupation blocks found.")
        return []

    # =========================================================
    # 5. EXTRACT BLOCKS
    # =========================================================
    for i, (current_start, current_code) in enumerate(temp_tik_code_locations):

        # default end = akhir text
        block_end = len(text)

        # jika bukan block terakhir
        if i + 1 < len(temp_tik_code_locations):
            block_end = temp_tik_code_locations[i + 1][0]

        raw_content = text[current_start:block_end].strip()

        blocks.append({
            "code": current_code,
            "content": raw_content
        })

    return blocks


# =============================================================
# MAIN EXTRACTION LOGIC
# =============================================================

start_detailed_descriptions_marker = "DAFTAR DAN DESKRIPSI OKUPASI PON TIK"
end_detailed_descriptions_marker = "DAFTAR PEMUTAKHIRAN OKUPASI PON TIK"

text_for_block_extraction = ""

# =============================================================
# 1. CARI MARKER AWAL
# =============================================================
first_start_idx = cleaned_text.find(
    start_detailed_descriptions_marker
)

if first_start_idx == -1:
    print(f"ERROR: '{start_detailed_descriptions_marker}' not found.")
else:

    # =========================================================
    # 2. AMBIL KEMUNCULAN KEDUA
    #    (karena pertama biasanya daftar isi)
    # =========================================================
    second_start_idx = cleaned_text.find(
        start_detailed_descriptions_marker,
        first_start_idx + len(start_detailed_descriptions_marker)
    )

    if second_start_idx != -1:
        start_idx = second_start_idx
        print(f"INFO: Using second occurrence at index {start_idx}")
    else:
        start_idx = first_start_idx
        print("WARNING: Second occurrence not found. Using first occurrence.")

    # =========================================================
    # 3. AMBIL SEGMENT SETELAH MARKER
    # =========================================================
    segment_after_marker = cleaned_text[
        start_idx + len(start_detailed_descriptions_marker):
    ]

    # =========================================================
    # 4. CARI BLOK PERTAMA YANG PUNYA DEFINISI
    # =========================================================
    real_block_pattern = re.compile(
        r"(TIK\.[A-Z]{3}\d{4}.*?DEFINISI)",
        re.DOTALL
    )

    real_match = real_block_pattern.search(
        segment_after_marker
    )

    if real_match:

        actual_content_start = real_match.start()

        print(
            f"INFO: Real occupation content starts at offset "
            f"{actual_content_start}"
        )

        content_segment = segment_after_marker[
            actual_content_start:
        ]

    else:
        print("WARNING: Could not find block with DEFINISI.")
        content_segment = segment_after_marker

    # =========================================================
    # 5. CARI MARKER AKHIR
    # =========================================================
    end_idx = content_segment.find(
        end_detailed_descriptions_marker
    )

    if end_idx != -1:

        text_for_block_extraction = content_segment[
            :end_idx
        ].strip()

        print(
            f"INFO: Extraction segment length: "
            f"{len(text_for_block_extraction)} chars"
        )

    else:

        text_for_block_extraction = content_segment.strip()

        print(
            "WARNING: End marker not found. "
            "Using until end of document."
        )

# =============================================================
# 6. CLEAN ASCII
# =============================================================
if text_for_block_extraction:

    text_for_block_extraction = (
        text_for_block_extraction
        .encode('ascii', 'ignore')
        .decode('ascii')
    )

    print(
        f"INFO: ASCII-cleaned extraction text length: "
        f"{len(text_for_block_extraction)}"
    )

# =============================================================
# 7. EXTRACT OCCUPATION BLOCKS
# =============================================================
occupation_blocks = []

if text_for_block_extraction:

    if (
        'daftar_isi_occupations' in globals()
        and daftar_isi_occupations
    ):

        occupation_blocks = extract_occupation_blocks(
            text_for_block_extraction,
            daftar_isi_occupations
        )

        print(
            f"SUCCESS: Extracted "
            f"{len(occupation_blocks)} occupation blocks."
        )

    else:

        print(
            "ERROR: daftar_isi_occupations not found or empty."
        )

else:

    print(
        "ERROR: text_for_block_extraction is empty."
    )

# =============================================================
# 8. SAMPLE OUTPUT
# =============================================================
if occupation_blocks:

    print("\n==============================")
    print("FIRST OCCUPATION BLOCK SAMPLE")
    print("==============================")

    first_block = occupation_blocks[0]

    print(f"Code: {first_block['code']}")
    print()

    print(first_block['content'][:2000])

    print("\n==============================")
    print("BLOCK STATISTICS")
    print("==============================")

    for i in range(min(5, len(occupation_blocks))):

        print(
            f"Block {i+1}: "
            f"{occupation_blocks[i]['code']} | "
            f"Length = {len(occupation_blocks[i]['content'])}"
        )

else:

    print("No occupation blocks extracted.")

INFO: Using second occurrence at index 67194
INFO: Real occupation content starts at offset 561
INFO: Extraction segment length: 730058 chars
INFO: ASCII-cleaned extraction text length: 724113
INFO: Total daftar isi occupation codes: 269
INFO: Valid occupation blocks found: 269
SUCCESS: Extracted 269 occupation blocks.

FIRST OCCUPATION BLOCK SAMPLE
Code: TIK.ITG0901

TIK.ITG0901  PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER)
DEFINISI
Orang yang bertanggung jawab untuk memimpin dan mengelola teknologi informasi (TI)
organisasi dengan menggunakan keterampilan teknis dan non-teknis untuk memastikan
bahwa TI dapat mendukung strategi bisnis organisasi.
KUALIFIKASI
Level 9
LINGKUP BIDANG PEKERJAAN
1. Mengembangkan dan mengimplementasikan strategi teknologi informasi
2. Mengelola dan mengoptimalkan aset TI
PROFIL
1. Berintegritas
2. Analitis
3. Mengatasi masalah (problem-solving)
4. Merencanakan dan mengorganisasi pekerjaan
5. Memimpin tim
6. Bertanggung jawab
7. Mampu mengarahkan dan mempu

## PARSE INDIVIDUAL OCCUPATION DETAILS

In [ ]:
import re

# =====================================================
# HELPER FUNCTIONS
# =====================================================

def clean_whitespace(text):
    if not text:
        return text

    text = text.replace('\n', ' ')
    text = text.replace('\u200b', '')
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def extract_list_section(section_content):

    numbered = extract_numbered_items(
        section_content
    )

    # sudah ada numbering
    if numbered:
        return numbered

    lines = [
        line.strip()
        for line in section_content.splitlines()
        if line.strip()
    ]

    if not lines:
        return []

    merged = []
    current = lines[0]

    for line in lines[1:]:

        # line sebelumnya panjang
        # kemungkinan wrap PDF
        if len(current) >= 70:

            current += " " + line

        else:

            merged.append(current)
            current = line

    merged.append(current)

    return [
        clean_whitespace(x)
        for x in merged
        if clean_whitespace(x)
    ]

def extract_numbered_items(section_content):
    """
    Menangkap item bernomor yang bisa terdiri dari banyak baris.

    Contoh:

    1. Pengetahuan yang mendalam tentang hukum,
       peraturan dan kebijakan

    akan menjadi:

    Pengetahuan yang mendalam tentang hukum,
    peraturan dan kebijakan
    """

    matches = re.findall(
        r'\d+\.\s*(.*?)(?=\n\s*\d+\.|\Z)',
        section_content,
        flags=re.DOTALL
    )

    results = []

    for item in matches:

        item = clean_whitespace(item)

        if item and item != '-':
            results.append(item)

    return results


def extract_profile_items(section_content):
    """
    PROFIL kadang:
    1. Analitis
    2. Teliti

    kadang:
    Analitis
    Teliti

    Fungsi ini menangani keduanya.
    """

    numbered = extract_numbered_items(section_content)

    if numbered:
        return numbered

    lines = [
        clean_whitespace(line)
        for line in section_content.split('\n')
        if line.strip()
    ]

    return lines

def extract_task_units(section_content):

    lines = [
        line.strip()
        for line in section_content.splitlines()
        if line.strip()
    ]

    units = []
    current = []

    code_patterns = [

        # BSBMGT608C
        r'^[A-Z]{3,}[A-Z0-9]{2,}$',

        # BSBMGT608C (SKKNI 2019-022)
        r'^[A-Z]{3,}[A-Z0-9]{2,}\s*\(',

        # J.612000.029.01
        r'^[A-Z]\.[A-Z0-9]+(?:\.[A-Z0-9]+)+',

        # TIK.SM02.015.01
        r'\b[A-Z]{2,}\.[A-Z0-9]+(?:\.[A-Z0-9]+)+\b',
    ]

    for line in lines:

        if line.upper() == "KETERSEDIAAN STANDAR":
            continue

        if "(SKKNI, SKKI, SKKK)" in line:
            continue

        if line.strip() == "-":

            competency = " ".join(current)

            competency = re.sub(
                r'\s+',
                ' ',
                competency
            ).strip()

            if competency:
                units.append(competency)

            current = []
            continue

        current.append(line)

        is_code_line = any(
            re.match(pattern, line)
            for pattern in code_patterns
        )

        is_dash_line = line.strip() == "-"

        if is_code_line or is_dash_line:

            competency = " ".join(current)

            competency = re.sub(
                r'\s+',
                ' ',
                competency
            ).strip()

            units.append(competency)

            current = []

    if current:

        competency = " ".join(current)

        competency = re.sub(
            r'\s+',
            ' ',
            competency
        ).strip()

        units.append(competency)

    return units

def clean_task_units(task_list):

    cleaned = []

    for item in task_list:

        # hapus heading
        item = re.sub(
            r'KETERSEDIAAN STANDAR',
            '',
            item,
            flags=re.IGNORECASE
        )

        # hapus seluruh kurung yang mengandung SKKNI/SKKI/SKKK
        item = re.sub(
            r'\([^)]*(?:SKKNI|SKKI|SKKK)[^)]*\)',
            '',
            item,
            flags=re.IGNORECASE
        )

        # hapus (SKKNI xxxx)
        # Hapus seluruh referensi SKKNI
        item = re.sub(
            r'\(\s*SKKNI\s+\d{4}\s*-\s*\d+\s*\)',
            '',
            item,
            flags=re.IGNORECASE
        )

        print("AFTER SKKNI :", item)

        # ICAICT605A
        # ICANWK616A
        # BSBMGT608C
        item = re.sub(
            r'\b[A-Z]{3,}[A-Z0-9]{2,}\b',
            '',
            item
        )

        # J.612000.029.01
        # M.702090.001.01
        # J.62PDP00.001.1
        item = re.sub(
            r'\b[A-Z]\.[A-Z0-9]+(?:\.[A-Z0-9]+)+\b',
            '',
            item
        )

        item = re.sub(
            r'\b[A-Z]\s*\.\s*[A-Z0-9]+(?:\s*\.\s*[A-Z0-9]+)+\b',
            '',
            item
        )

        # hapus TIK.SM02.015.01
        item = re.sub(
            r'\b[A-Z]{2,}\.[A-Z0-9]+(?:\.[A-Z0-9]+)+\b',
            '',
            item
        )

        item = re.sub(
            r'^\d+\s+',
            '',
            item
        )

        item = clean_whitespace(item)

        if len(item) > 5:
            cleaned.append(item)

    return cleaned

In [ ]:
import pandas as pd
import re
import ast

# =====================================================
# PARSER PON-TIK
# =====================================================

def parse_occupation_details(text):

    details = {
        'definisi': None,
        'kualifikasi': None,
        'lingkup_bidang_pekerjaan': [],
        'profil': [],
        'tanggung_jawab': [],
        'wewenang': [],
        'persyaratan': [],
        'tugas_utama': [],
        'tugas_khusus': []
    }

    if pd.isna(text):
        return details

    text = str(text)

    # Hilangkan zero width
    text = text.replace('\u200b', '')

    sections = [
        "DEFINISI",
        "KUALIFIKASI",
        "LINGKUP BIDANG PEKERJAAN",
        "PROFIL",
        "TANGGUNG JAWAB",
        "WEWENANG",
        "PERSYARATAN",
        "JENJANG KARIER",
        "TUGAS UTAMA",
        "TUGAS KHUSUS",
        "SERTIFIKASI",
        "VERIFIKASI"
    ]

    positions = []

    for sec in sections:
        for m in re.finditer(
            rf'(?m)^\s*{re.escape(sec)}\s*$',
            text,
            flags=re.IGNORECASE
        ):
            positions.append(
                (sec, m.start(), m.end())
            )

    positions = sorted(
        positions,
        key=lambda x: x[1]
    )

    extracted = {}

    for i, (section, start, end) in enumerate(positions):

        next_start = (
            positions[i + 1][1]
            if i < len(positions) - 1
            else len(text)
        )

        content = text[end:next_start].strip()

        extracted[section] = content

    # =================================================
    # DEFINISI
    # =================================================

    if "DEFINISI" in extracted:

        details["definisi"] = clean_whitespace(
                extracted["DEFINISI"]
            )

    # =================================================
    # KUALIFIKASI
    # =================================================

    if "KUALIFIKASI" in extracted:

        details["kualifikasi"] = clean_whitespace(
                extracted["KUALIFIKASI"]
            )
    # =================================================
    # LIST SECTION
    # =================================================

    list_mapping = {
        "LINGKUP BIDANG PEKERJAAN":
            "lingkup_bidang_pekerjaan",

        "PROFIL":
            "profil",

        "TANGGUNG JAWAB":
            "tanggung_jawab",

        "WEWENANG":
            "wewenang",

        "PERSYARATAN":
            "persyaratan"
    }

    for source, target in list_mapping.items():

        if source not in extracted:
            continue

        content = extracted[source]

        details[target] = extract_list_section(
            content
        )

    # =================================================
    # TUGAS UTAMA
    # =================================================

    if "TUGAS UTAMA" in extracted:

        details["tugas_utama"] = clean_task_units(
            extract_task_units(
                extracted["TUGAS UTAMA"]
            )
        )

    # =================================================
    # TUGAS KHUSUS
    # =================================================

    if "TUGAS KHUSUS" in extracted:

        details["tugas_khusus"] = clean_task_units(
            extract_task_units(
                extracted["TUGAS KHUSUS"]
            )
        )

    return details

# =====================================================
# JALANKAN PARSER KE DATASET
# =====================================================

# Create df_raw from occupation_blocks before using it
df_raw = pd.DataFrame(occupation_blocks)
df_raw.rename(columns={'content': 'occupation_content'}, inplace=True)

parsed_rows = []

for _, row in df_raw.iterrows():

    parsed = parse_occupation_details(
        row["occupation_content"]
    )

    parsed_rows.append({
        "code": row["code"],
        **parsed
    })

df_pon_tik_occupations = pd.DataFrame(
    parsed_rows
)

# Merge df_daftar_isi into df_pon_tik_occupations to get 'occupation_name'
df_daftar_isi = pd.DataFrame(daftar_isi_occupations)
print(df_daftar_isi.head())

df_pon_tik_occupations = pd.merge(
    df_pon_tik_occupations,
    df_daftar_isi[['code', 'title']],
    on='code',
    how='left'
)

print(
    "Jumlah okupasi:",
    len(df_pon_tik_occupations)
)

display(
    df_pon_tik_occupations.head()
)


# =====================================================
# MEMBUAT occupation_text
# =====================================================

competency_cols = [
    "definisi",
    "lingkup_bidang_pekerjaan",
    "profil",
    "tanggung_jawab",
    "wewenang",
    "persyaratan",
    "tugas_utama",
    "tugas_khusus"
]


def combine_competencies(row):

    texts = []

    for col in competency_cols:

        value = row[col]

        if value is None:
            continue

        if isinstance(value, list):

            texts.extend([
                str(x).strip()
                for x in value
                if str(x).strip()
            ])

        else:

            value = str(value).strip()

            if value:
                texts.append(value)

    return " ".join(texts)


df_pon_tik_occupations[
    "occupation_text"
] = df_pon_tik_occupations.apply(
    combine_competencies,
    axis=1
)

# =====================================================
# SAVE
# =====================================================

# Save the DataFrame with the new 'occupation_text' column
import os
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_PON_TIK = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'PON_TIK')

# Ensure the output directory exists
os.makedirs(OUTPUT_FOLDER_PON_TIK, exist_ok=True)

output_filepath_combined = os.path.join(OUTPUT_FOLDER_PON_TIK, 'pon_tik_occupations.csv')
df_pon_tik_occupations.to_csv(output_filepath_combined, index=False)
print(f"\nUpdated PON TIK occupations data saved to '{output_filepath_combined}'.")

AFTER SKKNI : Manage Innovation and Continuous Improvement (Mengelola Inovasi dan Peningkatan Berkelanjutan) BSBMGT608C 
AFTER SKKNI : Develop a Knowledge Management Strategy (Mengembangkan Strategi Manajemen Pengetahuan) ICADBS602A 
AFTER SKKNI : Implement a Knowledge Management Strategy ICAICT605A
AFTER SKKNI : Lead the Evaluation and Implementation of Current Industry-Specific Technologies ICAICT609A
AFTER SKKNI : Design and Implement a Security System ICANWK601A
AFTER SKKNI : Manage Security, Privacy and Compliance of Cloud Service Deployment ICANWK616A
AFTER SKKNI : Manage and Control IT Project Risks ICAPMG607A
AFTER SKKNI : Conduct Knowledge Audits ICASAD602A
AFTER SKKNI : Manage Assessment and Validation of IT Solutions ICASAD607A
AFTER SKKNI : Implement Change-Management Processes ICASAS601A
AFTER SKKNI : Integrate Sustainability in ICT Planning and Design Projects ICTSUS6233A
AFTER SKKNI : Establish a Business Case for Sustainability and Competitive Advantage in ICT Projects 

,code,definisi,kualifikasi,lingkup_bidang_pekerjaan,profil,tanggung_jawab,wewenang,persyaratan,tugas_utama,tugas_khusus,title
0,TIK.ITG0901,Orang yang bertanggung jawab untuk memimpin da...,Level 9,[Mengembangkan dan mengimplementasikan strateg...,"[Berintegritas, Analitis, Mengatasi masalah (p...",[Mengembangkan dan menerapkan program kepatuha...,[Mengembangkan dan mengimplementasikan kebijak...,[Memiliki sertifikasi kompetensi terkait setar...,[Manage Innovation and Continuous Improvement ...,[],PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER)
1,TIK.ITG0902,Orang yang bertanggung jawab untuk memastikan ...,Level 9,[Mengembangkan dan mengimplementasikan kebijak...,"[Berintegritas, Mampu memimpin tim, Komunikati...","[Memimpin dan mengelola tim TI, Mendefinisikan...","[Mengakses dan menggunakan TI, Melakukan perub...",[Memiliki sertifikasi okupasi terkait setara K...,[Menentukan Metode Pemodelan Arsitektur Bisnis...,[],PEMIMPIN KEPATUHAN (CHIEF COMPLIANCE OFFICER (...
2,TIK.ITG0801,Orang yang bertanggung jawab untuk mengembangk...,Level 8,[Mengelola arsitektur yang mendukung tujuan bi...,"[Bertaqwa kepada Tuhan Yang Maha Esa, Memiliki...",[Mendefinisikan dan mengimplementasikan arsite...,[Menetapkan visi dan strategi arsitektur perus...,[Memiliki sertifikasi okupasi terkait setara K...,[Lead Innovative Thinking and Practice (Memimp...,[],KEPALA ARSITEKTUR PERUSAHAAN (LEAD ENTERPRISE ...
3,TIK.ITG0802,Orang yang memiliki pengalaman dan keahlian ya...,Level 8,"[Perencanaan teknologi informasi (TI), Impleme...","[Kemampuan untuk mengelola proyek dan tim, Kem...",[Memberikan saran dan nasihat teknis kepada ma...,[Mengembangkan dan mengimplementasikan strateg...,[Memiliki sertifikasi okupasi terkait setara K...,"[Direct ICT Services, Synchronise ICT Projects...",[],KEPALA PENASIHAT TI (LEAD IT ADVISOR)
4,TIK.ITG0803,"Orang yang bertanggung jawab atas perencanaan,...",Level 8,[Memimpin dan mengelola tim pengembangan dan i...,"[Komunikatif, Senang dengan teknologi baru, Te...",[Merancang dan mengimplementasikan strategi pr...,[Merancang antarmuka pengguna yang sesuai deng...,[Memiliki sertifikasi okupasi terkait setara K...,[Menentukan Prioritas Proyek dan Menyesuaikan ...,[],KEPALA MANAJEMEN PROGRAM (LEAD PROGRAM MANAGEM...



Updated PON TIK occupations data saved to '/content/processed_data/dictionary_data/PON_TIK/pon_tik_occupations.csv'.


In [ ]:
item = "Mengidentifikasi Peraturan Perundang-undangan Terkait Pelindungan Data Pribadi J.62PDP00.004.1 (SKKNI 2023-103)"

print(item)

item = re.sub(
    r'\(\s*SKKNI\s+\d{4}\s*-\s*\d+\s*\)',
    '',
    item,
    flags=re.IGNORECASE
)

print(item)

Mengidentifikasi Peraturan Perundang-undangan Terkait Pelindungan Data Pribadi J.62PDP00.004.1 (SKKNI 2023-103)
Mengidentifikasi Peraturan Perundang-undangan Terkait Pelindungan Data Pribadi J.62PDP00.004.1 


## COMBINE COMPETENCY-RELATED COLUMNS

In [ ]:
competency_cols = [
    'definisi',
    'lingkup_bidang_pekerjaan',
    'profil',
    'tanggung_jawab',
    'wewenang',
    'persyaratan',
    'tugas_utama',
    'tugas_khusus'
]

# Ensure all columns exist before concatenating
existing_competency_cols = [col for col in competency_cols if col in df_pon_tik_occupations.columns]

# Combine into 'occupation_text'
df_pon_tik_occupations['occupation_text'] = df_pon_tik_occupations[existing_competency_cols] \
    .fillna('') \
    .astype(str) \
    .agg(' '.join, axis=1)

print("df_pon_tik_occupations with new 'occupation_text' column head:")
display(df_pon_tik_occupations[['title', 'occupation_text']].head())

# Save the DataFrame with the new 'occupation_text' column
import os
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_PON_TIK = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'PON_TIK')

# Ensure the output directory exists
os.makedirs(OUTPUT_FOLDER_PON_TIK, exist_ok=True)

output_filepath_combined = os.path.join(OUTPUT_FOLDER_PON_TIK, 'pon_tik_occupations_combined.csv')
df_pon_tik_occupations.to_csv(output_filepath_combined, index=False)
print(f"\nUpdated PON TIK occupations data saved to '{output_filepath_combined}'.")

df_pon_tik_occupations with new 'occupation_text' column head:


,title,occupation_text
0,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Orang yang bertanggung jawab untuk memimpin da...
1,PEMIMPIN KEPATUHAN (CHIEF COMPLIANCE OFFICER (...,Orang yang bertanggung jawab untuk memastikan ...
2,KEPALA ARSITEKTUR PERUSAHAAN (LEAD ENTERPRISE ...,Orang yang bertanggung jawab untuk mengembangk...
3,KEPALA PENASIHAT TI (LEAD IT ADVISOR),Orang yang memiliki pengalaman dan keahlian ya...
4,KEPALA MANAJEMEN PROGRAM (LEAD PROGRAM MANAGEM...,"Orang yang bertanggung jawab atas perencanaan,..."



Updated PON TIK occupations data saved to '/content/processed_data/dictionary_data/PON_TIK/pon_tik_occupations_combined.csv'.


## SPLIT COMPETENCY

In [ ]:
import pandas as pd
import ast
import re

competency_cols = [
    'definisi',
    'lingkup_bidang_pekerjaan',
    'profil',
    'tanggung_jawab',
    'wewenang',
    'persyaratan',
    'tugas_utama',
    'tugas_khusus'
]

rows = []

for _, row in df_pon_tik_occupations.iterrows():

    for col in competency_cols:

        value = row[col]

        # kosong
        if value is None:
            continue

        # NaN
        if isinstance(value, float) and pd.isna(value):
            continue

        # LIST
        if isinstance(value, list):

            for item in value:

                item = str(item).strip()

                if item:

                    rows.append({
                        "occupation_id": row["code"],
                        "occupation_name": row["title"],
                        "competency_unit": item
                    })

        # STRING
        elif isinstance(value, str):

            value = value.strip()

            if value:

                rows.append({
                    "occupation_id": row["code"],
                    "occupation_name": row["title"],
                    "competency_unit": value
                })

df_pon_tik_competencies = pd.DataFrame(rows)

print("Total competency rows:", len(df_pon_tik_competencies))

display(df_pon_tik_competencies.head())

# ==========================================
# FILTERING NOISE
# ==========================================

def is_noise_competency(text):

    if pd.isna(text):
        return True

    text = str(text).strip().lower()

    # kosong
    if text == "":
        return True

    # hanya "-"
    if text == "-":
        return True

    # hanya angka
    if re.fullmatch(r'\d+', text):
        return True

    # ==========================
    # PENDIDIKAN
    # ==========================
    education_patterns = [
        r'\bsarjana\b',
        r'\bmagister\b',
        r'\bdoktor\b',
        r'\bdiploma\b',
        r'\bs1\b',
        r'\bs2\b',
        r'\bs3\b',
        r'\bd1\b',
        r'\bd2\b',
        r'\bd3\b',
        r'\bd4\b',
        r'pendidikan terakhir',
        r'gelar',
        r'lulusan',
        r'jurusan',
        r'pendidikan'
    ]

    # ==========================
    # PENGALAMAN
    # ==========================
    experience_patterns = [
        r'pengalaman',
        r'berpengalaman',
        r'minimal\s+\d+\s+tahun',
        r'sekurang-kurangnya\s+\d+\s+tahun',
        r'lebih\s+dari\s+\d+\s+tahun',
        r'bekerja\s+di\s+bidang',
        r'pengalaman kerja'
    ]

    # ==========================
    # SERTIFIKASI
    # ==========================
    certification_patterns = [
        r'sertifikasi',
        r'sertifikat',
        r'sertifikasi kompetensi',
        r'sertifikasi okupasi',
        r'sertifikasi profesional',
        r'sertifikat profesional',
        r'kkni level',
        r'tik\.[a-z0-9]',
        r'certified information systems security professional',
        r'cissp'
    ]

    patterns = (
        education_patterns
        + experience_patterns
        + certification_patterns
    )

    for pattern in patterns:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            return True

    return False

before_filter = len(df_pon_tik_competencies)

df_pon_tik_competencies["competency_unit"] = (
    df_pon_tik_competencies["competency_unit"]
    .astype(str)
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df_pon_tik_competencies = df_pon_tik_competencies[
    ~df_pon_tik_competencies["competency_unit"]
        .apply(is_noise_competency)
].reset_index(drop=True)

after_filter = len(df_pon_tik_competencies)

print("NOISE FILTERING")
print("="*50)
print("Before :", before_filter)
print("After  :", after_filter)
print("Removed:", before_filter - after_filter)

# ==========================================
# INSPECT
# ==========================================

display(
    df_pon_tik_competencies.sample(
        min(20, len(df_pon_tik_competencies)),
        random_state=42
    )
)

# Save the DataFrame with the new 'occupation_text' column
import os
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_PON_TIK = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'PON_TIK')

# Ensure the output directory exists
os.makedirs(OUTPUT_FOLDER_PON_TIK, exist_ok=True)

output_filepath_combined = os.path.join(OUTPUT_FOLDER_PON_TIK, 'pon_tik_competency.csv')
df_pon_tik_competencies.to_csv(output_filepath_combined, index=False)
print(f"\nUpdated PON TIK occupations data saved to '{output_filepath_combined}'.")

Total competency rows: 7842


,occupation_id,occupation_name,competency_unit
0,TIK.ITG0901,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Orang yang bertanggung jawab untuk memimpin da...
1,TIK.ITG0901,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Mengembangkan dan mengimplementasikan strategi...
2,TIK.ITG0901,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Mengelola dan mengoptimalkan aset TI
3,TIK.ITG0901,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Berintegritas
4,TIK.ITG0901,PEMIMPIN INFORMASI (CHIEF INFORMATION OFFICER),Analitis


NOISE FILTERING
Before : 7842
After  : 7092
Removed: 750


,occupation_id,occupation_name,competency_unit
6938,TIK.SRV0506,DESAINER SITUS WEB MUDA (ASSOCIATE WEBSITE DES...,melaksanakan cutover aplikasi
4950,TIK.INF0902,PEMIMPIN INFRASTRUKTUR (CHIEF INFRASTRUCTURE O...,menetapkan strategi dan rencana infrastruktur ...
696,TIK.ITG0605,KONSULTAN TI (IT CONSULTANT),membantu mengimplementasikan solusi it
733,TIK.ITG0606,AUDITOR TI (IT AUDITOR),menganalisis bukti tindak lanjut audit teknolo...
1862,TIK.DEV0611,SUPERVISOR PEMROGRAM MOBILE (MOBILE PROGRAMMER...,memberikan petunjuk teknis kepada pelanggan
1038,TIK.DEV0801,KEPALA PEMROGRAM (LEAD PROGRAMMER),menganalisis dan memecahkan masalah
2465,TIK.DEV0514,ANALIS PENGAMBILAN KEPUTUSAN ERP MUDA (ASSOCIA...,membantu pengguna memahami dan menggunakan data
2925,TIK.DSC0801,KEPALA MANAJEMEN GUDANG DATA (LEAD DATA WAREHO...,mengakses dan menggunakan data
1430,TIK.DEV0712,MANAJER DUKUNGAN TI (IT SUPPORT MANAGER),melakukan perencanaan dan pengembangan sistem ti
5112,TIK.INF0703,INSINYUR JARINGAN SENIOR (SENIOR NETWORK ENGIN...,mengumpulkan data peralatan jaringan dengan te...



Updated PON TIK occupations data saved to '/content/processed_data/dictionary_data/PON_TIK/pon_tik_competency.csv'.


## SAVE

In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
# drive.mount('/content/drive', force_remount=True)

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_PON_TIK = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK'
OUTPUT_FILENAME_PON_TIK = 'pon_tik_occupations.csv'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, exist_ok=True)

destination_filepath_pon_tik = os.path.join(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, OUTPUT_FILENAME_PON_TIK)

try:
    # Save the DataFrame to Google Drive
    df_pon_tik_occupations.to_csv(destination_filepath_pon_tik, index=False)
    print(f"Final PON TIK occupations dataset successfully saved to Google Drive: '{destination_filepath_pon_tik}'.")
except Exception as e:
    print(f"An error occurred while saving the PON TIK occupations data to Google Drive: {e}")

Final PON TIK occupations dataset successfully saved to Google Drive: '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK/pon_tik_occupations.csv'.


In [ ]:
import os
from google.colab import drive

# Mount Google Drive if not already mounted
# drive.mount('/content/drive', force_remount=True)

# Define the target Google Drive path
GOOGLE_DRIVE_TARGET_PATH_PON_TIK = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK'
OUTPUT_FILENAME_PON_TIK = 'pon_tik_competency.csv'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, exist_ok=True)

destination_filepath_pon_tik = os.path.join(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, OUTPUT_FILENAME_PON_TIK)

try:
    # Save the DataFrame to Google Drive
    df_pon_tik_competencies.to_csv(destination_filepath_pon_tik, index=False)
    print(f"Final PON TIK occupations dataset successfully saved to Google Drive: '{destination_filepath_pon_tik}'.")
except Exception as e:
    print(f"An error occurred while saving the PON TIK occupations data to Google Drive: {e}")

Final PON TIK occupations dataset successfully saved to Google Drive: '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK/pon_tik_competency.csv'.


In [ ]:
import pandas as pd
import os
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Mapping Area Function
area_mapping = {
    "ITG": "IT Governance",
    "DEV": "Digital Product Development",
    "DSC": "Data Science & Artificial Intelligence",
    "SEC": "Information & Cyber Security",
    "INF": "Infrastructure & Technology",
    "SRV": "IT Services"
}

# Mengambil prefix area dari kode
df_pon_tik_occupations["area_code"] = (
    df_pon_tik_occupations["code"]
    .str.extract(r"TIK\.([A-Z]{3})")
)

# Menambahkan nama area fungsi
df_pon_tik_occupations["occupation_group"] = (
    df_pon_tik_occupations["area_code"]
    .map(area_mapping)
)

# Hapus kolom sementara jika tidak diperlukan
df_pon_tik_occupations.drop(columns="area_code", inplace=True)

# Define the target Google Drive path for PON TIK
GOOGLE_DRIVE_TARGET_PATH_PON_TIK = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, exist_ok=True)

# Define the output filename
OUTPUT_FILENAME_PON_TIK_GROUND_TRUTH = 'pon_tik_occupations_ground_truth.csv'

# Construct the full destination path
destination_filepath_pon_tik = os.path.join(GOOGLE_DRIVE_TARGET_PATH_PON_TIK, OUTPUT_FILENAME_PON_TIK_GROUND_TRUTH)

try:
    # Save the DataFrame to Google Drive
    df_pon_tik_occupations.to_csv(destination_filepath_pon_tik, index=False, encoding="utf-8-sig")
    print(f"'pon_tik_occupations_ground_truth.csv' successfully saved to Google Drive: '{destination_filepath_pon_tik}'.")
except Exception as e:
    print(f"An error occurred while saving the PON TIK occupations ground truth data to Google Drive: {e}")

# display(df_pon_tik_occupations.head())

Mounted at /content/drive
'pon_tik_occupations_ground_truth.csv' successfully saved to Google Drive: '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/PON_TIK/pon_tik_occupations_ground_truth.csv'.


# KKO TAXONOMY BLOOM


## Raw KKO Taxonomy Bloom Extraction

In [ ]:
import pandas as pd
import re

# =========================================================
# RAW KKO DATASET
# =========================================================

kko_raw = {
    "C1": [
        "Membaca","Memberi Label","Membuat Daftar","Membuat Tabel",
        "Menamai","Mencatat","Mencocokkan","Mendefinisikan",
        "Mendeklamasikan","Menggambar","Menggandakan",
        "Menggarisbawahi","Menghafal","Mengingat","Mengulang",
        "Mengutip","Menyalin","Menyatakan","Menyebutkan",
        "Mereproduksi","Menjelaskan","Membilang",
        "Mengidentifikasi","Mendaftar","Menunjukkan",
        "Memberi indeks","Memasagkan","Menandai","Meniru",
        "Meninjau","Memilih","Mentabulasi","Memberi kode",
        "Menulis","Menelusuri","Menemukenali",
        "Mengingat kembali","Melafalkan","Melafazkan",
        "Menuliskan","Menyusun daftar","Menjodohkan",
        "Memberi definisi"
    ],

    "C2": [
        "Melaporkan","Memahami","Memberi Contoh",
        "Memparafrasakan","Memperluas","Memprediksi",
        "Mendiskusikan","Menemukan","Menerjemahkan",
        "Mengamati","Mengartikulasikan","Mengasosiasikan",
        "Mengekstrapolasi","Menggeneralisasi",
        "Menginterpolasi","Mengkarakterisasi",
        "Mengklarifikasi","Mengklasifikasikan",
        "Mengubah","Mengulang Kembali",
        "Mengungkapkan","Meninjau","Menjelaskan",
        "Menulis Ulang","Menunjukkan","Merangkum",
        "Merepresentasikan","Meringkas",
        "Memperkirakan","Menceritakan",
        "Mengkatagorikan","Mencirikan","Merinci",
        "Membandingkan","Menghitung",
        "Mengkontraskan","Menjalin",
        "Mencontohkan","Mengemukakan",
        "Mempolakan","Menyimpulkan",
        "Meramalkan","Menjabarkan","Menggali",
        "Mempertahankan","Mengartikan",
        "Menerangkan","Menafsirkan",
        "Membedakan","Menginterpretasikan",
        "Menampilkan","Menguraikan",
        "Menyadur","Menggantikan",
        "Menarik kesimpulan",
        "Mengembangkan","Membuktikan"
    ],

    "C3": [
        "Bertindak","Melengkapi","Melukis",
        "Memanfaatkan","Memanipulasi",
        "Membuat Sketsa","Memecahkan",
        "Memilih","Mempraktikkan",
        "Mencadangkan","Mendemonstrasikan",
        "Mendramatisasi","Menerapkan",
        "Menggunakan","Menghasilkan",
        "Menghitung","Mengilustrasikan",
        "Mengoperasikan","Mengubah",
        "Menjadwalkan","Menyimulasikan",
        "Menunjukkan","Menyesuaikan",
        "Menyiapkan","Mewawancarai",
        "Menugaskan","Mengurutkan",
        "Menentukan","Mengkalkulasi",
        "Memodifikasi","Membangun",
        "Mencegah","Menggambarkan",
        "Menilai","Melatih","Menggali",
        "Mengemukakan","Mengadaptasi",
        "Menyelidikit","Mempersoalkan",
        "Mengkonsepkan","Melaksanakan",
        "Memproduksi","Memproses",
        "Mengaitkan","Menyusun",
        "Melakukan","Mensimulasikan",
        "Mentabulasi","Membiasakan",
        "Mengklasifikasi","Meramalkan",
        "Mengimplementasikan",
        "Mengonsepkan","Memproseskan",
        "Menghubungkan","Membuktikan",
        "Memperagakan"
    ],

    "C4": [
        "Berdebat","Bereksperimen","Membagi",
        "Membagi Kecil","Membandingkan",
        "Membedah","Membedakan",
        "Membuat Diagram",
        "Membuat Garis Besar",
        "Membuat Inventarisasi",
        "Memeriksa","Memisahkan",
        "Mempertanyakan",
        "Memprioritaskan",
        "Mendeteksi","Mendiagnosis",
        "Mendiskriminasi",
        "Menganalisis",
        "Mengelompokkan",
        "Menghitung",
        "Menghubungkan",
        "Mengategorikan",
        "Mengontraskan",
        "Menguji","Menguraikan",
        "Menyurvei","Menunjukkan",
        "Menyelidiki","Menyimpulkan",
        "Mengaudit","Mengatur",
        "Menganimasi","Mengumpulkan",
        "Memecahkan","Menegaskan",
        "Menyeleksi","Merinci",
        "Menominasikan",
        "Mendiagramkan",
        "Mengkorelasikan",
        "Mencerahkan",
        "Membagankan",
        "Menjelajah",
        "Memaksimalkan",
        "Memerintahkan",
        "Mengaitkan",
        "Mentransfer",
        "Melatih","Mengedit",
        "Menemukan",
        "Mengoreksi",
        "Menelaah",
        "Mengukur",
        "Membangunkan",
        "Merasionalkan",
        "Memfokuskan",
        "Memadukan",
        "Mendiferensiasikan",
        "Mengorganisasikan",
        "Mengatribusikan",
        "Memerinci","Memilih",
        "Mempertentangkan",
        "Mendistribusikan",
        "Memilah-milah",
        "Menerima pendapat"
    ],

    "C5": [
        "Berargumen","Melampirkan",
        "Melepaskan","Memberi Nasihat",
        "Memberi Nilai",
        "Mempertimbangkan",
        "Memutuskan","Memverifikasi",
        "Mendamaikan","Menengahi",
        "Menentukan","Mengatur",
        "Mengawasi","Mengevaluasi",
        "Menghargai","Mengkritik",
        "Mengutamakan","Menilai",
        "Menimbang","Menjustifikasi",
        "Menyintesis","Menyelidiki",
        "Menyimpulkan",
        "Menyusun Kembali",
        "Membandingkan",
        "Mengarahkan",
        "Memprediksi",
        "Memperjelas",
        "Menugaskan",
        "Menafsirkan",
        "Mempertahankan",
        "Memerinci",
        "Mengukur","Merangkum",
        "Membuktikan",
        "Memvalidasi",
        "Mengetes","Mendukung",
        "Memilih",
        "Memproyeksikan",
        "Memisahkan",
        "Mengecek",
        "Memperbandingkan",
        "Memberi saran",
        "Memberi argumentasi",
        "Merekomendasi"
    ],

    "C6": [
        "Memainkan Peran","Membuat",
        "Membuat Hipotesis",
        "Memfasilitasi",
        "Memperbaiki",
        "Menciptakan",
        "Mendesain","Mendukung",
        "Menentukan","Mengarang",
        "Mengembangkan",
        "Menghasilkan",
        "Mengintegrasikan",
        "Mengombinasikan",
        "Mengorganisir",
        "Mengusulkan",
        "Menilai",
        "Menjelaskan Alasan",
        "Menyusun","Merakit",
        "Merancang Kembali",
        "Merekonstruksi",
        "Merencanakan",
        "Merevisi","Merumuskan",
        "Mengumpulkan",
        "Mengabstraksi",
        "Mengatur",
        "Menganimasi",
        "Mengkatagorikan",
        "Membangun",
        "Mengkreasikan",
        "Mengoreksi",
        "Memadukan",
        "Mendikte",
        "Membentuk",
        "Meningkatkan",
        "Menanggulangi",
        "Menggeneralisasi",
        "Menggabungkan",
        "Merancang",
        "Membatas",
        "Mereparasi",
        "Menyiapkan",
        "Memproduksi",
        "Memperjelas",
        "Merangkum",
        "Mengkode",
        "Mengkombinasikan",
        "Mengkonstruksi",
        "Menghubungkan",
        "Menampilkan",
        "Mengkategorikan",
        "Merangkaikan",
        "Menyusun kembali",
        "Menyimpulkan",
        "Membuat pola"
    ]
}

# =========================================================
# NORMALIZATION
# =========================================================

TYPO_FIXES = {
    "mengkatagorikan": "mengkategorikan",
    "mensimulasikan": "menyimulasikan",
    "memasagkan": "memasangkan",
    "merangcang": "merancang",
    "menyelidikit": "menyelidiki",
    "mengklasifikasi": "mengklasifikasikan",
    "mengkombinasikan": "mengombinasikan",
    "merekomendasi": "merekomendasikan",
    "membatas": "membatasi"
}

def clean_text(text):
    text = text.lower().strip()

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # remove weird characters
    text = re.sub(r'[^a-zA-Z\s-]', '', text)

    # typo correction
    if text in TYPO_FIXES:
        text = TYPO_FIXES[text]

    return text.strip()

# =========================================================
# BUILD DATASET
# =========================================================

rows = []

for bloom_level, verbs in kko_raw.items():

    cleaned_unique = set()

    for verb in verbs:

        clean_verb = clean_text(verb)

        # skip empty
        if not clean_verb:
            continue

        # remove duplicates inside same level
        if clean_verb in cleaned_unique:
            continue

        cleaned_unique.add(clean_verb)

        rows.append({
            "raw_verb": clean_verb,
            "bloom_level": bloom_level
        })

# =========================================================
# CREATE DATAFRAME
# =========================================================

df_kko = pd.DataFrame(rows)

# sort
df_kko = df_kko.sort_values(
    by=["bloom_level", "raw_verb"]
).reset_index(drop=True)

# =========================================================
# SAVE CSV
# =========================================================

output_path = "kko_bloom_dataset_clean.csv"

df_kko.to_csv(output_path, index=False)

# =========================================================
# SUMMARY
# =========================================================

print("=" * 50)
print("KKO BLOOM DATASET SUMMARY")
print("=" * 50)

for level in sorted(df_kko["bloom_level"].unique()):
    total = df_kko[df_kko["bloom_level"] == level].shape[0]
    print(f"{level}: {total} verbs")

print("-" * 50)
print(f"TOTAL VERBS: {len(df_kko)}")

print("\nSample Data:")
print(df_kko.head(20))

print(f"\nDataset saved to: {output_path}")

In [ ]:
import os
import pandas as pd
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define output paths consistent with the dictionary_data structure
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_DICTIONARY = os.path.join(OUTPUT_BASE_DIR, 'dictionary_data')
OUTPUT_FOLDER_KKO_BLOOM = os.path.join(OUTPUT_FOLDER_DICTIONARY, 'KKO_TAXONOMY_BLOOM')

# Ensure local output directory exists
os.makedirs(OUTPUT_FOLDER_KKO_BLOOM, exist_ok=True)
print(f"Created local directory: {OUTPUT_FOLDER_KKO_BLOOM}")

# Define Google Drive target path
GOOGLE_DRIVE_TARGET_PATH_KKO_BLOOM = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/Dictionary/KKO_TAXONOMY_BLOOM'

# Ensure Google Drive target directory exists
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_KKO_BLOOM, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_KKO_BLOOM}")

# Save to local processed data (ensure df_kko is available from previous cell)
local_cleaned_kko_filepath = os.path.join(OUTPUT_FOLDER_KKO_BLOOM, 'kko_bloom_dataset_clean.csv')
df_kko.to_csv(local_cleaned_kko_filepath, index=False)
print(f"KKO Bloom dataset saved locally to: '{local_cleaned_kko_filepath}'")

# Save to Google Drive
drive_cleaned_kko_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_KKO_BLOOM, 'kko_bloom_dataset_clean.csv')
df_kko.to_csv(drive_cleaned_kko_filepath, index=False)
print(f"KKO Bloom dataset saved to Google Drive: '{drive_cleaned_kko_filepath}'")

# BERITA ACARA

In [ ]:
import sys
!{sys.executable} -m pip install PyMuPDF

print("PyMuPDF installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.3 MB/s eta 0:00:00
PyMuPDF installation complete.


## BAP Data Extraction

In [ ]:
import numpy as np
import pandas as pd
import re
import os
import fitz # PyMuPDF

# --- Configuration and File Paths (re-declaration for robustness) ---
BAP_PDF_PATHS = [
    "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/BERITA ACARA/BAP 20241.pdf",
    "/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/BERITA ACARA/BAP 20242.pdf"
]

OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')
OUTPUT_CSV_FILENAME = 'bap_meeting_detail.csv'
OUTPUT_CSV_PATH = os.path.join(OUTPUT_FOLDER_CURRICULUM, OUTPUT_CSV_FILENAME)

# Ensure output directory exists
os.makedirs(OUTPUT_FOLDER_CURRICULUM, exist_ok=True)
print(f"Output directory ensured: {OUTPUT_FOLDER_CURRICULUM}")

# --- Helper Functions (re-declaration for robustness) ---
def clean_extracted_text(text):
    if text is None:
        return None
    text = re.sub(r'\n+', ' ', text).strip()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_text_from_pdf(pdf_path):
    text_content = []
    try:
        with fitz.open(pdf_path) as doc:
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                text_content.append(page.get_text())
        return "\n".join(text_content)
    except Exception as e:
        print(f"Error extracting text from {os.path.basename(pdf_path)}: {e}")
        return None

def parse_bap_document_state_machine(text_content, academic_year):
    lines = text_content.split('\n')
    parsed_meetings = []

    STATE_IDLE = 0
    STATE_IN_MEETING_CONTENT = 1

    current_state = STATE_IDLE
    current_course_name = None
    current_meeting_number = None
    current_meeting_content = []

    def reset_meeting_data():
        nonlocal current_meeting_number, current_meeting_content

        current_meeting_number = None
        current_meeting_content = []

    def save_current_meeting():
        if current_course_name and current_meeting_number is not None:

            meeting_content = " ".join(current_meeting_content)
            meeting_content = re.sub(r"\s+", " ", meeting_content).strip()

            parsed_meetings.append({
                "academic_year": academic_year,
                "course_name": current_course_name,
                "meeting_number": current_meeting_number,
                "meeting_content": meeting_content
            })

        reset_meeting_data()

    meeting_number_pattern = re.compile(r'^\s*(\d{1,2})\s*$')

    cleaned_lines = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue

        if line.endswith('-') and i + 1 < len(lines):
            merged_line = line[:-1] + lines[i+1].strip()
            cleaned_lines.append(merged_line.strip())
            i += 2
        else:
            cleaned_lines.append(line)
            i += 1

    processed_lines = []

    for line in cleaned_lines:

        # Pisahkan Java7 -> Java\n7
        line = line.replace("Java7", "Java\n7")
        line = line.replace("Software14", "Software\n14")

        processed_lines.extend(line.split("\n"))

    cleaned_lines = processed_lines

    for line_idx, line in enumerate(cleaned_lines):
        line = line.strip()

        if re.fullmatch(r"\d{1,2}", line):

            prev_line = cleaned_lines[line_idx - 1].lower() if line_idx > 0 else ""
            next_line = cleaned_lines[line_idx + 1].strip() if line_idx + 1 < len(cleaned_lines) else ""

            # Kasus:
            # Chapter
            # 1
            # 1. Introduction
            if (
                "chapter" in prev_line
                and re.match(r"^\d+\.", next_line)
            ):
                continue

        # --- FIX: Check for meeting number BEFORE applying general line filters ---
        match_meeting = meeting_number_pattern.match(line)
        if match_meeting:
            num = int(match_meeting.group(1))
            if 1 <= num <= 14:
                if current_meeting_number is not None:
                    save_current_meeting()

                current_meeting_number = num
                current_state = STATE_IN_MEETING_CONTENT

                if current_meeting_number == 1:
                    for j in range(line_idx - 1, -1, -1):
                        prev_line = cleaned_lines[j].strip()
                        if prev_line:
                            if not any(kw in prev_line.lower() for kw in ["topic", "reference", "description", "mata kuliah", "namamk", "pertemuank", "topik", "referensi", "deskripsi", "no.", "academic year", "semester"]):
                                current_course_name = prev_line
                                break

                continue
        # --- END FIX ---

        # Apply general line filters only if it wasn't a meeting number
        if len(line) < 3 or line.lower() in ["topic", "reference", "description", "mata kuliah", "namamk", "pertemuank", "topik", "referensi", "deskripsi"]:
            continue

        if current_course_name and current_meeting_number is not None:
            current_meeting_content.append(line)

    if current_course_name and current_meeting_number is not None:
        save_current_meeting()

    return parsed_meetings

# --- Main Processing Loop (Re-execute) ---
all_extracted_data = []

for pdf_path in BAP_PDF_PATHS:
    academic_year_match = re.search(r'BAP\s*(\d+)\.pdf', os.path.basename(pdf_path), re.IGNORECASE)
    if academic_year_match:
        academic_year = academic_year_match.group(1)
        print(f"Processing {os.path.basename(pdf_path)} for academic year {academic_year}...")
        full_text_content = extract_text_from_pdf(pdf_path)
        if full_text_content:
            data_from_pdf = parse_bap_document_state_machine(full_text_content, academic_year)
            all_extracted_data.extend(data_from_pdf)
        else:
            print(f"No text extracted from {os.path.basename(pdf_path)}. Skipping parsing.")

    else:
        print(f"Could not extract academic year from filename: {os.path.basename(pdf_path)}. Skipping.")

df_bap = pd.DataFrame(all_extracted_data)

# Fill missing values with None explicitly for object columns
for col in ['academic_year', 'course_name', 'topic', 'reference', 'description']:
    if col in df_bap.columns:
        df_bap[col] = df_bap[col].replace('', None)

print("\n--- Raw Extracted BAP Data (First 5 Rows) ---")
display(df_bap.head())
print(f"Total extracted meeting records: {len(df_bap)}")

# --- Data Validation ---
print("\n--- Starting Data Validation ---")

# 1. Check for duplicates: (academic_year, course_name, meeting_number)
# Combine meeting_content with the same meeting_number
grouped_cols = ['academic_year', 'course_name', 'meeting_number']

# Check if there are any duplicates that would require concatenation
if not df_bap.empty and df_bap.duplicated(subset=grouped_cols).any():
    print("Duplicate (academic_year, course_name, meeting_number) combinations found. Combining 'meeting_content'...")

    # Group by the specified columns and concatenate unique 'meeting_content' values
    df_bap = df_bap.groupby(grouped_cols, as_index=False).agg(
        meeting_content=('meeting_content', lambda x: ' | '.join(x.astype(str).unique()))
    )

    print(f"Combined 'meeting_content' for {len(df_bap)} unique meeting entries.")
else:
    print("No duplicate (academic_year, course_name, meeting_number) combinations found that require content merging.")

# 2. meeting_number must be an integer
if 'meeting_number' in df_bap.columns and not df_bap.empty:
    df_bap['meeting_number'] = pd.to_numeric(df_bap['meeting_number'], errors='coerce')
    if df_bap['meeting_number'].isnull().any():
        print("Warning: Some meeting_number values are not numeric. They have been converted to NaN.")
        df_bap.dropna(subset=['meeting_number'], inplace=True)
        print(f"Removed rows with non-numeric meeting_number. Remaining rows: {len(df_bap)}")
    df_bap['meeting_number'] = df_bap['meeting_number'].astype('Int64')
    print("meeting_number column converted to nullable integer.")
else:
    print("Skipping meeting_number numeric conversion: DataFrame is empty or 'meeting_number' column is missing.")

# 3. meeting_number must be within range 1-14
if 'meeting_number' in df_bap.columns and not df_bap.empty:
    invalid_meeting_numbers = df_bap[(df_bap['meeting_number'] < 1) | (df_bap['meeting_number'] > 14)]
    if not invalid_meeting_numbers.empty:
        print(f"Warning: {len(invalid_meeting_numbers)} records found with meeting_number outside 1-14 range.")
        print("Sample of invalid meeting numbers:")
        display(invalid_meeting_numbers.head())
        df_bap.drop(invalid_meeting_numbers.index, inplace=True)
        print(f"Removed invalid meeting numbers. Remaining rows: {len(df_bap)}")
    else:
        print("All meeting_number values are within the 1-14 range.")
else:
    print("Skipping meeting_number range check: DataFrame is empty or 'meeting_number' column is missing.")

# --- Summary Statistics ---
print("\n--- Summary Statistics ---")

if not df_bap.empty and 'course_name' in df_bap.columns and 'academic_year' in df_bap.columns and 'meeting_number' in df_bap.columns:
    total_courses = df_bap['course_name'].nunique()
    print(f"Total unique courses: {total_courses}")

    total_meetings = len(df_bap)
    print(f"Total meetings: {total_meetings}")

    print("\nMeetings per course:")
    meetings_per_course = df_bap.groupby(['academic_year', 'course_name'])['meeting_number'].nunique().sort_values(ascending=False)
    display(meetings_per_course)
else:
    print("Cannot generate full summary statistics: DataFrame is empty or essential columns are missing.")

# --- Final Output ---
print("\n--- Final Processed BAP Data (First 5 Rows) ---")
display(df_bap.head())

# Save to CSV
df_bap.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nFinal BAP meeting details saved to '{OUTPUT_CSV_PATH}'.")

Output directory ensured: /content/processed_data/curriculum_data
Processing BAP 20241.pdf for academic year 20241...
Processing BAP 20242.pdf for academic year 20242...

--- Raw Extracted BAP Data (First 5 Rows) ---


,academic_year,course_name,meeting_number,meeting_content
0,20241,Advanced Ethical Hacking,1,Course Introduction and Indonesia Cyber Securi...
1,20241,Advanced Ethical Hacking,2,Distributed Denial of Service (DDOS) CEH v12 U...
2,20241,Advanced Ethical Hacking,3,Chapter 11: Session Hijacking CEH v12 Capable ...
3,20241,Advanced Ethical Hacking,4,"Module 12 Evading IDS, Firewall, and Honeypot ..."
4,20241,Advanced Ethical Hacking,5,Hacking Web Servers CEH v12 Capable of knowing...


Total extracted meeting records: 686

--- Starting Data Validation ---
Duplicate (academic_year, course_name, meeting_number) combinations found. Combining 'meeting_content'...
Combined 'meeting_content' for 672 unique meeting entries.
meeting_number column converted to nullable integer.
All meeting_number values are within the 1-14 range.

--- Summary Statistics ---
Total unique courses: 47
Total meetings: 672

Meetings per course:


academic_year  course_name                               
20241          Advanced Ethical Hacking                      14
               Algoritma dan Pemrograman                     14
               Aljabar Linear                                14
               Analisis Perancangan Sistem Informasi         14
               Bahasa Inggris 1                              14
               Computer Vision                               14
               E-Business dan Pemrograman Berbasis Web       14
               Interaksi Manusia dan Komputer                14
               Interpersonal Skill                           14
               Kalkulus I                                    14
               Keamanan Teknologi Informasi                  14
               Kewirausahaan Berbasis Teknologi              14
               Komunikasi Data                               14
               Leadership and Management for IT Engineer     14
               Metodologi Penelitian dan Penulisan Ilmiah    14
               Mobile Computing                              14
               Pemodelan dan Simulasi Jaringan               14
               Pengantar Intelejensi Artifisial              14
               Pengantar Teknologi Informasi                 14
               Rekayasa Ilmu                                 14
               Routing Protocols and Concepts                14
               Sirkuit Elektronik                            14
               Sistem Basis Data                             14
               Sistem Dijital                                14
               Sistem Multimedia                             14
               Sistem Operasi                                14
               Statistik                                     14
20242          Arsitektur dan Organisasi Komputer            14
               Bahasa Inggris 2                              14
               Ethical Hacking                               14
               Etika Profesi dan Hak Kekayaan Intelektual    14
               IT Project Management                         14
               Informatika Sosial                            14
               Interpersonal Skill                           14
               Kalkulus II                                   14
               Komunikasi Nirkabel                           14
               Konsep Sistem Informasi                       14
               LAN and Wireless                              14
               Manajemen Jaringan                            14
               Matematika Diskrit                            14
               Pemrograman Berorientasi Objek                14
               Pemrograman Visual                            14
               Proses Sinyal Dijital                         14
               Rekayasa perangkat Lunak                      14
               Sistem Basis Data Lanjut                      14
               Sistem Terdistribusi                          14
               Struktur Data                                 14
               Wide Area Network (WAN)                       14
Name: meeting_number, dtype: int64


--- Final Processed BAP Data (First 5 Rows) ---


,academic_year,course_name,meeting_number,meeting_content
0,20241,Advanced Ethical Hacking,1,Course Introduction and Indonesia Cyber Securi...
1,20241,Advanced Ethical Hacking,2,Distributed Denial of Service (DDOS) CEH v12 U...
2,20241,Advanced Ethical Hacking,3,Chapter 11: Session Hijacking CEH v12 Capable ...
3,20241,Advanced Ethical Hacking,4,"Module 12 Evading IDS, Firewall, and Honeypot ..."
4,20241,Advanced Ethical Hacking,5,Hacking Web Servers CEH v12 Capable of knowing...



Final BAP meeting details saved to '/content/processed_data/curriculum_data/bap_meeting_detail.csv'.


In [ ]:
import os
import shutil
from google.colab import drive

# Mount Google Drive if not already mounted
drive.mount('/content/drive', force_remount=True)

# Define paths (consistent with previous cells)
OUTPUT_BASE_DIR = '/content/processed_data'
OUTPUT_FOLDER_CURRICULUM = os.path.join(OUTPUT_BASE_DIR, 'curriculum_data')
OUTPUT_CSV_FILENAME = 'bap_meeting_detail.csv'

# Define the target Google Drive path for BAP files
GOOGLE_DRIVE_TARGET_PATH_BAP = '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/BERITA ACARA/'

# Ensure the target directory exists in Google Drive
os.makedirs(GOOGLE_DRIVE_TARGET_PATH_BAP, exist_ok=True)
print(f"Created Google Drive directory: {GOOGLE_DRIVE_TARGET_PATH_BAP}")

# Define source and destination file paths
source_filepath = os.path.join(OUTPUT_FOLDER_CURRICULUM, OUTPUT_CSV_FILENAME)
destination_filepath = os.path.join(GOOGLE_DRIVE_TARGET_PATH_BAP, OUTPUT_CSV_FILENAME)

try:
    shutil.copy2(source_filepath, destination_filepath)
    print(f"Successfully copied '{OUTPUT_CSV_FILENAME}' to '{destination_filepath}'.")
except FileNotFoundError:
    print(f"Error: Source file '{OUTPUT_CSV_FILENAME}' not found at '{source_filepath}'.")
except Exception as e:
    print(f"An error occurred while copying '{OUTPUT_CSV_FILENAME}': {e}")

Mounted at /content/drive
Created Google Drive directory: /content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/BERITA ACARA/
Successfully copied 'bap_meeting_detail.csv' to '/content/drive/MyDrive/TA_Jennifer Felicia/DATASET/KURIKULUM/PROCESSED/BERITA ACARA/bap_meeting_detail.csv'.
